In [ ]:
# =============================================================================
# CELL 0: FRAMEWORK METADATA, CONFIGURATION, & SCENARIO SELECTION
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Single source of truth for every experimental parameter. One run is one
#   scenario: a region of interest (active_roi), a real Earth-observation
#   fleet (fleet, planet_swath_case) and an analysis window (window_hours,
#   live from the newest orbital elements, from a common reference_epoch_utc,
#   or retrospective from analysis_epoch). The Starlink proxy (fleet="starlink",
#   N in VALID_N_VALUES) and the Botswana bounding box / Nepal corridor ROIs are
#   kept only to reproduce the earliest runs.
#
# Theoretical basis:
#   - Cardinality-constrained weighted maximum coverage (NP-hard). The k-step
#     greedy attains at least 1 - 1/e (~0.632) of the optimum of a monotone
#     submodular objective (Nemhauser, Wolsey & Fisher 1978, Math. Program. 14,
#     265-294, doi:10.1007/BF01588971). Both selection objectives of Cell 6 are
#     of this kind; Cell 7C measures their exact gaps with an ILP.
#
# Execution workflow:
#   1. Set active_roi, fleet, planet_swath_case (and the window) in the
#      ARFConfig(...) call below, or pass ARF_CONFIG_OVERRIDES from a batch
#      driver (arf_batch_runner.py), and run this cell.
#   2. Run Cells 1 -> 9 in order, all from this release (each cell checks).
#   3. Results accumulate in the scenario_results ledger until the kernel
#      restarts; each scenario also writes its own output folder. From v1.5.3
#      each run's report leads with that run's own statuses; an "overall"
#      status pools every scenario in the ledger and says so.
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

"""
ARF Scenario Configuration Module
==================================

Defines and validates every parameter; later cells only read them.

Regions of interest (official WDPA boundaries, Cell 1):
    - ckgr_okavango: Central Kalahari Game Reserve + Okavango Delta (Botswana)
    - lower_zambezi: Lower Zambezi National Park (Zambia)
    - karoo_nature_reserve: Karoo Nature Reserve (South Africa)
Fleets (CelesTrak GP data, Cell 2; swaths and daylight limits in MISSION_TABLE):
    - earth-resources: Sentinel-2A/B/C and Landsat 8/9 (free and open data)
    - planet: PlanetScope Doves (commercial)
    - earth-resources+planet: both
Legacy: fleet="starlink" (proxy, N in VALID_N_VALUES), botswana_full,
nepal_trishuli_corridor.
"""

import os
import sys
import json
import logging
from dataclasses import dataclass, asdict
from typing import Dict, Any, Set, ClassVar, Tuple, Optional

# =============================================================================
# FRAMEWORK METADATA
# =============================================================================

__version__ = "1.5.3"   # 1.5.3: presentation only, every computed value as in 1.5.2 -- Figure 3 and composite panel (c) zoomed to their data with the legend under the axes, Figure 7 panel (b) title shortened so the map fits two IEEE columns, its cloud settings moved to the footer (Cell 7); Figures 5 and 9 at the H1a level of reachable tiles, Figure 5 in hours and Figure 9 in minutes on linear axes, legends under the panels (Cell 7C); superseded hypotheses labelled as such in the console, the report, the JSON and the pipeline log, and each run's own statuses printed first with ledger-wide statuses labelled as pooled (Cells 7, 7C, 8, 9); 1.5.2: per-cell audit -- sourced acquisition exclusions (Sentinel-2A tandem pause) applied per pass time, and an exclude_satellites option for fleet sensitivity runs (Cells 0, 2, 5); default scenario CKGR + Okavango with the open-data + PlanetScope fleet, and a complete run of another release in the output folder kept rather than overwritten (Cell 0); validated package versions pinned and checked (Cell 1), recorded with JPype and Java (Cell 9); deterministic sample tile (Cell 1); scenario ledger keyed by fleet tag and reset per run (Cells 0, 6); end-cap count after the daylight filter, per-satellite swath-edge print (Cell 5); figures at IEEE column widths, latency figures in hours against the physical floor and the H1a margin instead of the falsified 10-min line, new H1a-gap figure (Cells 7, 7C); coverage-vs-time CSV and undelivered-image-safe ordering (Cell 7B); per-platform archive lags and precision/recall (Cell 7D); H3 overall status by the same rule as H1a, H1b and H2 (Cell 8); documentation (Cells 0-9); 1.5.1: each tile observation assigned to the strip of its closest approach, no end-cap duplicates (Cell 5); fleet-independent reference epoch and optional common window start for live runs (Cells 0, 2); version guard in every cell; exact ILP tolerances (Cell 7C); time-aware weights on delivery time (Cell 6); single-window H1 statistics labelled descriptive; documentation corrected (Cells 0-9); 1.5.0: time-aware selection on a time-weighted coverage objective with the 1-1/e guarantee and its exact ILP gap (Cells 6, 7C); spatially and temporally correlated cloud with optional per-tile climatology (Cell 6); comparative H2 over cloud seeds (Fail-Signal vs open-loop plans with the same images) and H1a/H1b on the earliest-completion cover (Cells 7C, 8); per-cell version check (Cell 9); protected-area outline rasterised in saved maps (Cells 3, 7); config overrides for batch runs (Cell 0); 1.4.0: fleet-aware time-decay scale (Cell 6); H1 as latency non-inferiority (H1a) + image economy (H1b) with evaluability rules, H3 not-evaluable label (Cells 7C, 8); map furniture kept off the ROI (Cells 3, 7); Cell 7 metadata fleet size; NaN-safe delivery order; 1.3.1: journal-standard study-area map (Cell 3) and mosaic map (Cell 7), figures only; 1.3.0: Planet FOV swath, realistic cloud, ECC + frontier, retrospective runs, archive validation; 1.2.0: revised H1; earlier runs report 1.0.0
__framework_name__ = "Algorithmic Reconstruction Framework (ARF)"
__license__ = "MIT"
__authors__ = ["Tatenda Allen Tobaiwa"]
# Cells run in this session, with their release (version guard since v1.5.0, in
# every cell since v1.5.1; Cell 9 refuses a manifest from a mixed notebook).
# Reset whenever Cell 0 runs.
_ARF_CELLS_RUN = {"0": __version__}
__institution__ = "Botswana International University of Science and Technology (BIUST)"

# =============================================================================
# CUSTOM EXCEPTIONS
# =============================================================================

class ConfigurationError(Exception):
    """Raised when an invalid scenario or parameter configuration is detected."""
    pass


def _utc_naive(iso):
    """ISO 8601 time -> naive datetime in UTC. 'Z' and offsets (+02:00) are
    converted to UTC (v1.5.1; an offset previously crashed the future-time check
    or was dropped when formatting)."""
    from datetime import datetime as _dt, timezone as _tz
    d = _dt.fromisoformat(str(iso).strip().replace("Z", "+00:00"))
    return d.astimezone(_tz.utc).replace(tzinfo=None) if d.tzinfo is not None else d


def _utc_now_naive():
    from datetime import datetime as _dt, timezone as _tz
    return _dt.now(_tz.utc).replace(tzinfo=None)

# =============================================================================
# CONFIGURATION DATACLASS
# =============================================================================

@dataclass
class ARFConfig:
    """
    Configuration container for the Algorithmic Reconstruction Framework.
    Enforces runtime type validation and physical parameter sanity checks.
    """

    # --- Allowed Scenarios ---
    VALID_N_VALUES: ClassVar[Set[int]] = {5, 20, 100, 250, 1000}
    VALID_FLEETS: ClassVar[Set[str]] = {"starlink", "earth-resources", "planet", "earth-resources+planet"}
    VALID_ROI_NAMES: ClassVar[Set[str]] = {"botswana_full", "nepal_trishuli_corridor", "ckgr_okavango",
                                           "lower_zambezi", "karoo_nature_reserve"}

    # --- Scenario Parameters ---
    active_n: int = 1000                 # Starlink proxy only (legacy); real fleets use every operational satellite
    active_roi: str = "ckgr_okavango"    # v1.5.2 default (was the legacy Nepal corridor)
    cloud_probability: float = 0.35
    max_selection_steps: int = 30
    random_seed: int = 42

    # Imaging fleet. "starlink" = the Starlink proxy used for all results before
    # Path 1 (uniform swath_half_angle_deg, no daylight limit, N sampled). The
    # others are real Earth-observation fleets from CelesTrak GP data: every
    # operational satellite with a MISSION_TABLE entry, each with its own swath
    # and minimum sun elevation. Satellites without an entry are excluded and
    # listed, never given a guessed swath. v1.5.2 default: the open-data +
    # PlanetScope fleet (was the Starlink proxy).
    fleet: str = "earth-resources+planet"
    # PlanetScope frame orientation is not stated in the sources found, and the
    # generation of each Dove cannot be told from its name, so its swath is run
    # as a bound from Planet's field of view: "conservative" = PS2.SD Dove-R
    # (3.0 deg, ~26-27 km at 500 km) or "optimistic" = PSB.SD SuperDove
    # (4.0 deg, ~35-36 km). See MISSION_TABLE.
    planet_swath_case: str = "conservative"
    # Used only for missions whose sources give no sun-elevation limit
    # (Sentinel-2). An ASSUMPTION, borrowed from the Landsat criterion (5 deg).
    assumed_min_sun_elev_deg: float = 5.0

    # --- Physical & Astrodynamics Parameters ---
    # Simulation window. 48 h is the rapid-response case. For the open-data
    # fleet (fleet="earth-resources") use 384 h (16 days): the longest single-
    # satellite repeat cycle in that fleet, so every satellite flies its whole
    # ground track once. Sources (accessed 2026-10-09): Landsat 9 "Has a 16-day
    # repeat cycle" (https://www.usgs.gov/landsat-missions/landsat-9);
    # Sentinel-2 "The revisit frequency of each single Sentinel-2 satellite is
    # 10 days" (https://sentiwiki.copernicus.eu/web/s2-mission). A window other
    # than 48 h is added to the output folder name (e.g. _384h).
    window_hours: int = 48
    swath_half_angle_deg: float = 9.22
    h3_resolution: int = 7
    downlink_min_elevation_deg: float = 15.0

    # Fallback wait-time bound only. Overwritten dynamically in Cell 2.
    t_max_wait_minutes: float = 95.0

    # Time-aware objective (Cells 6 and 7C): marginal gain weighted by
    # exp(-rate * t_min / tau). Rate 0 == coverage-first.
    # The default 3.0 is where the Nepal decay sweep reached the physical-best
    # latency; it was chosen after seeing that sweep, so the full sweep
    # (decay_sweep, reported in Cell 7C) is the result to cite, not 3.0 alone.
    latency_decay_rate: float = 3.0
    decay_sweep: Tuple[float, ...] = (0.5, 1.0, 2.0, 3.0, 5.0)
    # tau of that weight (v1.4.0). "fleet": the longer of one orbital period
    # and the physical floor of the candidate passes (time for ALL of them to
    # cover h1_reachable_coverage_pct of the reachable tiles, clear-sky
    # geometry), so a sparse fleet that needs a day to cover the ROI is not
    # restricted to its first orbit. "fixed": one orbital period, as in v1.3.1
    # (T_MAX_WAIT_MINUTES). Identical whenever the floor is within one orbit.
    latency_decay_scale: str = "fleet"

    # Onset-robustness analysis (Cell 7C): request time shifted across
    # [0, onset_shift_hours] in onset_shift_step_min steps, each using the
    # following (window_hours - onset_shift_hours) hours of passes.
    onset_robustness_enabled: bool = True
    onset_shift_hours: int = 24
    onset_shift_step_min: int = 20

    # Baselines (Cell 7C): random draws per comparison, and the exact ILP
    # optimum on small sub-problems (clear sky) to measure the greedy's gap.
    random_baseline_draws: int = 10
    ilp_enabled: bool = True
    ilp_instances: int = 5
    ilp_horizon_hours: int = 6
    ilp_budget_k: int = 10
    # Instances are shrunk (shorter window) until they fit ilp_max_pairs; at
    # ~100k pass-tile pairs HiGHS proves optimality well inside the time limit.
    ilp_time_limit_s: float = 120.0
    ilp_max_pairs: int = 100_000

    # Equal-coverage latency (Cell 7C): time at which each policy's delivered
    # mosaic first reaches these coverage levels (% of ROI tiles). Completion
    # time alone is only comparable between policies reaching equal coverage.
    coverage_targets_pct: Tuple[float, ...] = (95.0, 99.0)

    # Revised H1 (pre-specified 2026-10-09, before the fresh-data test; the
    # original "< 10 min" H1 is still evaluated and reported as falsified):
    #   At >= h1_min_onset_pct of onsets, the time-aware selector's delivered
    #   mosaic reaches h1_reachable_coverage_pct of the tiles REACHABLE in that
    #   onset's horizon no later than h1_floor_margin_min after the physical
    #   floor (all passes) reaches the same level, using at most
    #   max_selection_steps passes. Clear sky is the test; cloud is reported.
    # h1_floor_margin_min=None derives the margin as ONE ORBITAL PERIOD at the
    # fleet's median altitude (Kepler's third law): missing one revisit costs
    # about one orbit. h1_min_onset_pct=90 is an ASSUMPTION set by the authors,
    # not derived; change it only before seeing fresh-data results.
    h1_reachable_coverage_pct: float = 95.0
    h1_floor_margin_min: Optional[float] = None
    h1_min_onset_pct: float = 90.0
    # H1 framing from v1.4.0 (Cells 7C and 8). The criterion above is H1a,
    # LATENCY NON-INFERIORITY: first-come-first-served also meets the floor by
    # construction when the budget allows, so H1a shows the selector loses no
    # time, not that it beats the baselines (their pass rates are reported).
    # H1b, IMAGE ECONOMY (added 2026-10-12, after the 15 runs of that day, so
    # confirmatory only on fresh data): at >= h1_min_onset_pct of onsets where
    # first-come-first-served needs two or more images to reach the level, the
    # earliest-completion cover reaches it within the same margin of the floor
    # using FEWER images than first-come-first-served. Mosaic quality at that
    # level (time-aware vs first-come-first-served) is reported, not tested.
    # Evaluability: a share of onsets can only show ">= h1_min_onset_pct" with
    # 95% confidence if enough onsets are evaluated -- with every onset a pass,
    # the exact one-sided 95% lower bound (Clopper & Pearson 1934) is
    # 0.05**(1/n), which reaches 0.90 only from n = 29. Below that the test is
    # NOT EVALUABLE. None derives n from h1_min_onset_pct; an integer overrides.
    h1_min_evaluable_onsets: Optional[int] = None

    # What the selector may know about cloud when it chooses passes.
    #   "realistic" (default): no image's cloud is known before that image
    #     ARRIVES (delivery time). How the remaining budget is planned is set by
    #     cloud_planning below (v1.5.0 default: expected coverage from each
    #     tile's cloud probability, re-planned after every arrival; v1.4.0: clear-
    #     sky geometry, re-planned when a planned observation comes back cloudy,
    #     the Fail-Signal). After the budget, compensation takes each arriving
    #     image that clearly covers a still-missing tile.
    #   "oracle": the selector knows in advance which tiles each pass will see
    #     clear (the behaviour of every run before 2026-10-09). Not achievable
    #     in practice; reported as a clairvoyant reference (greedy, so not a
    #     strict upper bound).
    cloud_knowledge: str = "realistic"

    # Time-aware objective (v1.5.0, Cells 6 and 7C).
    #   "facility" (default): greedy on the time-weighted coverage
    #     f(S) = sum over tiles of max over selected passes e of w_e * u_e,t,
    #     w_e = exp(-rate * t_e / tau). f is a weighted facility-location
    #     function, hence monotone submodular, so the k-step greedy keeps the
    #     1 - 1/e guarantee of Nemhauser, Wolsey & Fisher (1978), Math. Program.
    #     14, 265-294, doi:10.1007/BF01588971; Cell 7C measures its exact gap
    #     with the same ILP as coverage-first, utilities replaced by w * u.
    #   "legacy": v1.4.0 (unweighted marginal gain times w_e). Those scores are
    #     not the marginal gains of any set function, so no guarantee applies.
    time_aware_objective: str = "facility"

    # Cloud model (v1.5.0, Cell 6).
    #   "correlated" (default): a latent Gaussian field over the region,
    #     spatially correlated with e-folding length cloud_corr_length_km and
    #     evolving between pass times as an AR(1) process with e-folding time
    #     cloud_corr_time_h (separable covariance). A tile is cloudy for a pass
    #     when the field there exceeds the threshold that makes its marginal
    #     probability exactly the tile's cloud probability. Passes minutes apart
    #     therefore see nearly the same clouds, and clouds cover neighbouring
    #     tiles together.
    #   "iid": every (pass, tile) pair independent (v1.4.0 and earlier).
    # Why: about half of global cloud cover comes from clouds 200 km or larger
    # and about 15% from clouds under 10 km (Wood & Field 2011, J. Climate 24,
    # 4800-4816, doi:10.1175/2011JCLI4056.1), so one independent draw per
    # ~2.4 km tile is unrealistic. The two scales are ASSUMPTIONS within that
    # range, not measured values: Cell 7C sweeps them (cloud_sweep).
    cloud_model: str = "correlated"
    cloud_corr_length_km: float = 50.0
    cloud_corr_time_h: float = 3.0
    # Each tile's cloud probability.
    #   "constant": cloud_probability everywhere (all runs before v1.5.0).
    #   "earthenv": the 2000-2014 mean cloud frequency of the analysis month
    #     at the tile (EarthEnv Global 1-km Cloud Frequency v1; Wilson & Jetz
    #     2016, PLoS Biol 14(3): e1002415, doi:10.1371/journal.pbio.1002415;
    #     licence CC BY-NC 4.0), read from
    #     CLIMATOLOGY_DIR/MODCF_monthlymean_MM.tif (Cell 6 downloads it there
    #     once if missing). cloud_climatology_month overrides the month.
    cloud_probability_source: str = "constant"
    cloud_climatology_month: Optional[int] = None
    # How the adaptive (Fail-Signal) selector plans the images it has not yet
    # seen (v1.5.0, Cell 6).
    #   "expected" (default): plan the remaining budget to maximise EXPECTED
    #     coverage of the tiles still missing, from each tile's cloud
    #     probability (redundancy where cloud is likely), and re-plan after
    #     every arrival from the coverage actually achieved.
    #   "clear": plan on clear-sky geometry and re-plan only when a planned
    #     observation comes back cloudy (v1.4.0).
    cloud_planning: str = "expected"
    # Cloud realisations for the cloud analyses of Cell 7C (H2, seeds
    # random_seed, random_seed + 1, ...). The maps and the onset analysis use
    # the single realisation random_seed.
    cloud_seeds: int = 20
    # Sensitivity of the cloud results to the two correlation scales (Cell 7C):
    # lengths x times, cloud_sweep_seeds realisations each; empty to skip.
    cloud_sweep_lengths_km: Tuple[float, ...] = (0.0, 10.0, 25.0, 50.0, 100.0, 200.0)
    cloud_sweep_times_h: Tuple[float, ...] = (1.0, 3.0, 6.0, 24.0)
    cloud_sweep_seeds: int = 10

    # Retrospective runs of a REAL fleet (not Starlink): simulate from this past
    # UTC instant (ISO 8601, e.g. "2026-09-15T00:00:00") instead of "now". The
    # fleet's satellites are taken from the current CelesTrak list, and their
    # orbital elements AS OF this instant (newest element set with epoch at or
    # before it, at most 7 days old) from Space-Track gp_history (needs the
    # SPACETRACK_USERNAME / SPACETRACK_PASSWORD Colab secrets). A past window
    # can be checked at once against the real image archive (Cell 7D).
    # None = live run from the newest elements.
    analysis_epoch: Optional[str] = None
    # Live runs (v1.5.1): start the window at this UTC instant (ISO 8601) instead
    # of at the newest element set. Runs that will be COMPARED (one region with
    # different fleets, say) must share one window: give them the same value
    # here, or use analysis_epoch. None = the newest element-set epoch over BOTH
    # CelesTrak groups (resource and planet) whichever fleet is run, so runs
    # fetched together share their start. The 15 runs of 12 October 2026 did
    # not: the CKGR open-data run started 4 h 51 min before the open + PlanetScope
    # run, so their times to coverage are not comparable as they stand.
    reference_epoch_utc: Optional[str] = None
    # Strip assignment (v1.5.1, Cell 5). False (default): each tile observation
    # belongs to the one-minute strip that contains its closest approach.
    # True: the v1.5.0-and-earlier rule, under which every strip also claimed
    # an "end cap" of radius half the swath beyond each end (most Sentinel-2
    # observations appeared in two strips). Only to reproduce earlier runs.
    legacy_strip_end_caps: bool = False
    # Fleet sensitivity (v1.5.2, Cell 2): satellites to leave out, by CelesTrak
    # OBJECT_NAME (case-insensitive, exact), e.g. ("SENTINEL-2A",). Being tracked
    # is not the same as acquiring: Sentinel-2A flies an extended campaign with a
    # reduced acquisition plan (it imaged the CKGR every 20 days in August-October
    # 2026, Earth Search catalogue), so its predicted passes are an upper bound;
    # run with and without it to bracket the open-data results. Added to the
    # output folder name (_without-...).
    exclude_satellites: Tuple[str, ...] = ()

    def __post_init__(self):
        """Validate configuration parameters upon instantiation."""
        self._validate()

    def _validate(self):
        if self.active_roi not in self.VALID_ROI_NAMES:
            raise ConfigurationError(
                f"ACTIVE_ROI={self.active_roi!r} is invalid. Valid ROIs: {sorted(self.VALID_ROI_NAMES)}"
            )
        if self.active_n not in self.VALID_N_VALUES:
            raise ConfigurationError(
                f"ACTIVE_N={self.active_n} is invalid. Valid scenarios: {sorted(self.VALID_N_VALUES)}"
            )
        if not 0.0 <= self.cloud_probability <= 1.0:
            raise ConfigurationError(f"CLOUD_PROBABILITY={self.cloud_probability} must be in [0.0, 1.0].")
        if self.max_selection_steps < 1:
            raise ConfigurationError(f"MAX_SELECTION_STEPS={self.max_selection_steps} must be >= 1.")
        if not isinstance(self.random_seed, int):
            raise ConfigurationError(f"RANDOM_SEED must be an integer, received {type(self.random_seed)}.")
        if self.window_hours < 1:
            raise ConfigurationError(f"WINDOW_HOURS={self.window_hours} must be >= 1.")
        if self.fleet not in self.VALID_FLEETS:
            raise ConfigurationError(f"FLEET={self.fleet!r} is invalid. Valid: {sorted(self.VALID_FLEETS)}")
        if self.planet_swath_case not in ("conservative", "optimistic"):
            raise ConfigurationError(f"PLANET_SWATH_CASE={self.planet_swath_case!r} must be 'conservative' or 'optimistic'.")
        if not -10.0 <= self.assumed_min_sun_elev_deg < 90.0:
            raise ConfigurationError(f"ASSUMED_MIN_SUN_ELEV_DEG={self.assumed_min_sun_elev_deg} must be in [-10, 90).")
        if not 0.0 < self.swath_half_angle_deg < 90.0:
            raise ConfigurationError(f"SWATH_HALF_ANGLE_DEG={self.swath_half_angle_deg} must be strictly in (0, 90).")
        if not 0 <= self.h3_resolution <= 15:
            raise ConfigurationError(f"H3_RESOLUTION={self.h3_resolution} must be an integer in [0, 15].")
        if not 0.0 <= self.downlink_min_elevation_deg < 90.0:
            raise ConfigurationError(f"DOWNLINK_MIN_ELEVATION_DEG={self.downlink_min_elevation_deg} must be in [0, 90).")
        if self.t_max_wait_minutes <= 0:
            raise ConfigurationError(f"T_MAX_WAIT_MINUTES={self.t_max_wait_minutes} must be > 0.")
        if self.latency_decay_rate < 0:
            raise ConfigurationError(f"LATENCY_DECAY_RATE={self.latency_decay_rate} must be >= 0.")
        if not self.decay_sweep or any(r < 0 for r in self.decay_sweep):
            raise ConfigurationError(f"DECAY_SWEEP={self.decay_sweep} must be non-empty with all rates >= 0.")
        if not 0 < self.onset_shift_hours < self.window_hours:
            raise ConfigurationError(f"ONSET_SHIFT_HOURS={self.onset_shift_hours} must be in (0, WINDOW_HOURS).")
        if self.onset_shift_step_min < 1:
            raise ConfigurationError(f"ONSET_SHIFT_STEP_MIN={self.onset_shift_step_min} must be >= 1.")
        if self.random_baseline_draws < 1:
            raise ConfigurationError(f"RANDOM_BASELINE_DRAWS={self.random_baseline_draws} must be >= 1.")
        if self.ilp_instances < 1 or self.ilp_budget_k < 1 or self.ilp_max_pairs < 1:
            raise ConfigurationError("ILP_INSTANCES, ILP_BUDGET_K and ILP_MAX_PAIRS must all be >= 1.")
        if not 0 < self.ilp_horizon_hours <= self.window_hours:
            raise ConfigurationError(f"ILP_HORIZON_HOURS={self.ilp_horizon_hours} must be in (0, WINDOW_HOURS].")
        if self.ilp_time_limit_s <= 0:
            raise ConfigurationError(f"ILP_TIME_LIMIT_S={self.ilp_time_limit_s} must be > 0.")
        if not self.coverage_targets_pct or any(not 0 < t <= 100 for t in self.coverage_targets_pct):
            raise ConfigurationError(f"COVERAGE_TARGETS_PCT={self.coverage_targets_pct} must be non-empty, each in (0, 100].")
        if not 0 < self.h1_reachable_coverage_pct <= 100:
            raise ConfigurationError(f"H1_REACHABLE_COVERAGE_PCT={self.h1_reachable_coverage_pct} must be in (0, 100].")
        if self.h1_floor_margin_min is not None and self.h1_floor_margin_min < 0:
            raise ConfigurationError(f"H1_FLOOR_MARGIN_MIN={self.h1_floor_margin_min} must be >= 0 or None.")
        if not 0 < self.h1_min_onset_pct <= 100:
            raise ConfigurationError(f"H1_MIN_ONSET_PCT={self.h1_min_onset_pct} must be in (0, 100].")
        if self.h1_min_evaluable_onsets is not None and int(self.h1_min_evaluable_onsets) < 1:
            raise ConfigurationError(f"H1_MIN_EVALUABLE_ONSETS={self.h1_min_evaluable_onsets} must be >= 1 or None.")
        if self.latency_decay_scale not in ("fleet", "fixed"):
            raise ConfigurationError(f"LATENCY_DECAY_SCALE={self.latency_decay_scale!r} must be 'fleet' or 'fixed'.")
        if self.cloud_knowledge not in ("realistic", "oracle"):
            raise ConfigurationError(f"CLOUD_KNOWLEDGE={self.cloud_knowledge!r} must be 'realistic' or 'oracle'.")
        if self.time_aware_objective not in ("facility", "legacy"):
            raise ConfigurationError(f"TIME_AWARE_OBJECTIVE={self.time_aware_objective!r} must be 'facility' or 'legacy'.")
        if self.cloud_model not in ("correlated", "iid"):
            raise ConfigurationError(f"CLOUD_MODEL={self.cloud_model!r} must be 'correlated' or 'iid'.")
        if not self.cloud_corr_length_km > 0:
            raise ConfigurationError(f"CLOUD_CORR_LENGTH_KM={self.cloud_corr_length_km} must be > 0 "
                                     "(use cloud_model='iid' for independent tiles).")
        if self.cloud_corr_time_h < 0:
            raise ConfigurationError(f"CLOUD_CORR_TIME_H={self.cloud_corr_time_h} must be >= 0 "
                                     "(0 = independent between pass times).")
        if self.cloud_planning not in ("expected", "clear"):
            raise ConfigurationError(f"CLOUD_PLANNING={self.cloud_planning!r} must be 'expected' or 'clear'.")
        if self.cloud_probability_source not in ("constant", "earthenv"):
            raise ConfigurationError(f"CLOUD_PROBABILITY_SOURCE={self.cloud_probability_source!r} must be "
                                     "'constant' or 'earthenv'.")
        if self.cloud_climatology_month is not None and not 1 <= int(self.cloud_climatology_month) <= 12:
            raise ConfigurationError(f"CLOUD_CLIMATOLOGY_MONTH={self.cloud_climatology_month} must be 1-12 or None.")
        if self.cloud_seeds < 2 or self.cloud_sweep_seeds < 2:
            raise ConfigurationError("CLOUD_SEEDS and CLOUD_SWEEP_SEEDS must be >= 2 (a confidence interval needs "
                                     "at least two realisations).")
        if any(x < 0 for x in self.cloud_sweep_lengths_km) or any(x < 0 for x in self.cloud_sweep_times_h):
            raise ConfigurationError("CLOUD_SWEEP_LENGTHS_KM and CLOUD_SWEEP_TIMES_H must be >= 0 "
                                     "(length 0 = independent tiles).")
        for _name in ("analysis_epoch", "reference_epoch_utc"):
            _v = getattr(self, _name)
            if _v is None:
                continue
            try:
                _ae = _utc_naive(_v)
            except (ValueError, TypeError):
                raise ConfigurationError(f"{_name.upper()}={_v!r} is not an ISO 8601 time.")
            if self.fleet == "starlink":
                raise ConfigurationError(f"{_name.upper()} applies to real fleets only; Starlink historical runs use "
                                         "the ROI's historical_epoch.")
            if _name == "analysis_epoch" and _ae > _utc_now_naive():
                raise ConfigurationError(f"ANALYSIS_EPOCH={_v} is in the future.")
        if self.analysis_epoch is not None and self.reference_epoch_utc is not None:
            raise ConfigurationError("Set ANALYSIS_EPOCH (retrospective, elements as of then) or REFERENCE_EPOCH_UTC "
                                     "(live elements, common window start), not both.")
        if not isinstance(self.legacy_strip_end_caps, bool):
            raise ConfigurationError("LEGACY_STRIP_END_CAPS must be True or False.")
        if (isinstance(self.exclude_satellites, str) or not isinstance(self.exclude_satellites, (tuple, list))
                or any(not isinstance(n, str) or not n.strip() for n in self.exclude_satellites)):
            raise ConfigurationError(f"EXCLUDE_SATELLITES={self.exclude_satellites!r} must be a tuple of satellite "
                                     "names (CelesTrak OBJECT_NAME), e.g. ('SENTINEL-2A',).")
        if self.exclude_satellites and self.fleet == "starlink":
            raise ConfigurationError("EXCLUDE_SATELLITES applies to real fleets only.")

    def to_dict(self) -> Dict[str, Any]:
        return asdict(self)

    def to_json(self) -> str:
        return json.dumps(self.to_dict(), indent=2)

# =============================================================================
# CONFIGURATION INITIALISATION
# =============================================================================

# Primary Configuration Instantiation. v1.5.2 default scenario: the Botswana
# study area (CKGR + Okavango Delta) with the open-data + PlanetScope fleet,
# conservative Dove swath -- the primary design of the confirmatory protocol.
# The Starlink proxy (fleet="starlink", active_n) and the Nepal corridor /
# Botswana bounding-box ROIs reproduce the earliest runs only.
config = ARFConfig(
    active_n=1000,
    active_roi="ckgr_okavango",
    cloud_probability=0.35,
    max_selection_steps=30,
    random_seed=42,
    fleet="earth-resources+planet",
    planet_swath_case="conservative",
    assumed_min_sun_elev_deg=5.0,
    window_hours=48,
    swath_half_angle_deg=9.22,
    h3_resolution=7,
    downlink_min_elevation_deg=15.0,
    t_max_wait_minutes=95.0,
    latency_decay_rate=3.0,
    decay_sweep=(0.5, 1.0, 2.0, 3.0, 5.0),
    latency_decay_scale="fleet",   # "fixed" reproduces v1.3.1 (one orbital period)
    onset_robustness_enabled=True,
    onset_shift_hours=24,
    onset_shift_step_min=20,
    random_baseline_draws=10,
    ilp_enabled=True,
    ilp_instances=5,
    ilp_horizon_hours=6,
    ilp_budget_k=10,
    ilp_time_limit_s=120.0,
    ilp_max_pairs=100_000,
    coverage_targets_pct=(95.0, 99.0),
    h1_reachable_coverage_pct=95.0,
    h1_floor_margin_min=None,      # None = one orbital period at the fleet's median altitude
    h1_min_onset_pct=90.0,
    h1_min_evaluable_onsets=None,  # None = derived (29 for 90%): fewer evaluable onsets -> NOT EVALUABLE
    cloud_knowledge="realistic",   # "oracle" reproduces the pre-2026-10-09 (clairvoyant) cloud runs
    time_aware_objective="facility",  # "legacy" reproduces v1.4.0's time-aware selection
    cloud_model="correlated",      # "iid" reproduces v1.4.0's cloud mask
    cloud_corr_length_km=50.0,     # ASSUMPTION, swept in Cell 7C
    cloud_corr_time_h=3.0,         # ASSUMPTION, swept in Cell 7C
    cloud_probability_source="constant",  # "earthenv": per-tile monthly climatology (Wilson & Jetz 2016)
    cloud_climatology_month=None,  # None = month of the analysis start
    cloud_planning="expected",     # "clear" reproduces v1.4.0's Fail-Signal planning
    cloud_seeds=20,
    cloud_sweep_lengths_km=(0.0, 10.0, 25.0, 50.0, 100.0, 200.0),
    cloud_sweep_times_h=(1.0, 3.0, 6.0, 24.0),
    cloud_sweep_seeds=10,
    analysis_epoch=None,           # e.g. "2026-09-15T00:00:00" for a retrospective real-fleet run
    reference_epoch_utc=None,      # live runs to be compared: one common window start (v1.5.1)
    legacy_strip_end_caps=False,   # True reproduces the end-cap strips of v1.5.0 and earlier (Cell 5)
    exclude_satellites=(),         # e.g. ("SENTINEL-2A",): fleet sensitivity run (v1.5.2, Cell 2)
)

# Batch runs (v1.5.0): a driver may set ARF_CONFIG_OVERRIDES = {field: value}
# in the namespace before running the cells; they replace the values above and
# are validated by the same checks. Recorded in the manifest (Cell 9).
ARF_CONFIG_OVERRIDES_APPLIED = dict(globals().get("ARF_CONFIG_OVERRIDES") or {})
if ARF_CONFIG_OVERRIDES_APPLIED:
    from dataclasses import replace as _dc_replace, fields as _dc_fields
    _unknown = sorted(set(ARF_CONFIG_OVERRIDES_APPLIED) - {f.name for f in _dc_fields(ARFConfig)})
    if _unknown:
        raise ConfigurationError(f"ARF_CONFIG_OVERRIDES has unknown fields: {_unknown}")
    config = _dc_replace(config, **{k: (tuple(v) if isinstance(v, list) else v)
                                    for k, v in ARF_CONFIG_OVERRIDES_APPLIED.items()})
    print(f"⚙️  Config overrides applied: {ARF_CONFIG_OVERRIDES_APPLIED}")

# Expose module-level globals for downstream pipeline cells
ACTIVE_N = config.active_n
ACTIVE_ROI = config.active_roi
CLOUD_PROBABILITY = config.cloud_probability
MAX_SELECTION_STEPS = config.max_selection_steps
RANDOM_SEED = config.random_seed
WINDOW_HOURS = config.window_hours
SWATH_HALF_ANGLE_DEG = config.swath_half_angle_deg
FLEET = config.fleet
PLANET_SWATH_CASE = config.planet_swath_case
ASSUMED_MIN_SUN_ELEV_DEG = config.assumed_min_sun_elev_deg
EXCLUDE_SATELLITES = tuple(str(n).strip().upper() for n in config.exclude_satellites)
FLEET_TAG = f"N{ACTIVE_N}" if FLEET == "starlink" else FLEET.replace("+", "_") + (
    f"-{PLANET_SWATH_CASE}" if "planet" in FLEET else "")
# Non-default windows get their own output folder, so a 16-day run never
# overwrites the 48 h run of the same region and fleet.
if config.window_hours != 48:
    FLEET_TAG += f"_{config.window_hours}h"
if config.analysis_epoch is not None:          # retrospective runs get their own folder too
    FLEET_TAG += "_from" + _utc_naive(config.analysis_epoch).strftime("%Y%m%d")
if config.reference_epoch_utc is not None:     # live runs with a common start (v1.5.1)
    FLEET_TAG += "_at" + _utc_naive(config.reference_epoch_utc).strftime("%Y%m%dT%H%M")
if EXCLUDE_SATELLITES:                          # fleet sensitivity runs (v1.5.2)
    import re as _re0
    FLEET_TAG += "_without-" + "-".join(_re0.sub(r"[^A-Z0-9]+", "", n) for n in sorted(EXCLUDE_SATELLITES))
# Scenario identity in the scenario_results ledger: (ROI, N) for the Starlink
# proxy, as before; (ROI, fleet tag) for real fleets (v1.5.2; was (ROI, fleet),
# so the two Planet swath cases, a 384 h window or a past window of the same
# fleet overwrote each other's entry when run in one kernel).
SCENARIO_N = ACTIVE_N if FLEET == "starlink" else FLEET_TAG

# -----------------------------------------------------------------------------
# MISSION TABLE (real fleets): every value carries its status and source.
#   sourced    -- stated on the cited page (quote recorded)
#   assumption -- no source found; labelled, and to be covered by a sweep
# name_pattern is matched against CelesTrak OBJECT_NAME. The patterns
# themselves are unconfirmed until a live fetch: Cell 2 lists every
# satellite left unmatched, so a missed name shows up rather than vanishing.
# -----------------------------------------------------------------------------
CELESTRAK_GROUPS = {
    # GROUP values of https://celestrak.org/NORAD/elements/gp.php ("resource" =
    # Earth Resources, "planet" = Planet), as listed in
    # https://resources.wolframcloud.com/FunctionRepository/resources/CelesTrakTLE
    "earth-resources": ["resource"],
    "planet": ["planet"],
    "earth-resources+planet": ["resource", "planet"],
}
_ACCESSED = "2026-10-08"
MISSION_TABLE = [
    {"mission": "Sentinel-2", "name_pattern": r"^SENTINEL[- ]?2[A-Z]?$",
     "swath_km": {"value": 290.0, "status": "sourced", "accessed": _ACCESSED,
                  "source_url": "https://sentiwiki.copernicus.eu/web/s2-mission",
                  "quote": "The orbital swath width is 290 km."},
     "min_sun_elev_deg": {"value": ASSUMED_MIN_SUN_ELEV_DEG, "status": "assumption",
                          "note": "Source states MSI collects reflected sunlight but gives no sun-elevation "
                                  "limit; the Landsat 5 deg criterion is borrowed (ASSUMED_MIN_SUN_ELEV_DEG)."}},
    {"mission": "Landsat 8/9", "name_pattern": r"^LANDSAT[- ]?[89]$",
     "swath_km": {"value": 185.0, "status": "sourced", "accessed": _ACCESSED,
                  "source_url": "https://science.nasa.gov/mission/landsat-9",
                  "quote": "Swath width: 185 km (115 mi)"},
     "min_sun_elev_deg": {"value": 5.0, "status": "sourced", "accessed": _ACCESSED,
                          "source_url": "https://www.usgs.gov/landsat-missions/landsat-9",
                          "quote": "including islands at solar elevation angles greater than 5 degrees"}},
    # PlanetScope swath is given by Planet as a FIELD OF VIEW ACROSS THE SWATH
    # (PSScene spec, June 2026, Table 2-A): PS2 and PS2.SD "3.0° (swath)",
    # PSB.SD "4.0° (swath)"; the frame sizes in the same table (24 x 8 km,
    # 24 x 16 km, 32.5 x 19.6 km at 475 km) are swath x scene length, which the
    # FOV confirms (3.0 deg at 475 km = 24.9 km; 1.0 deg = 8.3 km). An earlier
    # version of this table used 8 km -- the along-track scene length -- as the
    # swath; that was wrong. PS2 (Dove Classic) imagery ended April 2022, so the
    # current fleet is PS2.SD (Dove-R, 3.0 deg) or PSB.SD (SuperDove, 4.0 deg).
    # Which Dove carries which instrument cannot be told from its name, so the
    # two cases bracket it. Each satellite's swath is computed from the FOV at
    # its own altitude (spherical Earth), not taken at the 475 km reference.
    {"mission": "PlanetScope", "name_pattern": r"^FLOCK",
     "fov_deg": ({"value": 3.0, "status": "sourced (lower bound: PS2.SD Dove-R)", "accessed": "2026-10-09",
                  "source_url": "https://assets.planet.com/docs/Planet_PSScene_Imagery_Product_Spec_letter_screen.pdf",
                  "quote": "Field of View 3.0° (swath) 1.0° (scene length) 3.0° (swath) 2.0° (scene length) "
                           "4.0° (swath) 2.3° (scene length) [PS2, PS2.SD, PSB.SD]"}
                 if PLANET_SWATH_CASE == "conservative" else
                 {"value": 4.0, "status": "sourced (upper bound: PSB.SD SuperDove)", "accessed": "2026-10-09",
                  "source_url": "https://assets.planet.com/docs/Planet_PSScene_Imagery_Product_Spec_letter_screen.pdf",
                  "quote": "Field of View 3.0° (swath) 1.0° (scene length) 3.0° (swath) 2.0° (scene length) "
                           "4.0° (swath) 2.3° (scene length) [PS2, PS2.SD, PSB.SD]"}),
     "min_sun_elev_deg": {"value": 10.0, "status": "sourced", "accessed": _ACCESSED,
                          "source_url": "https://docs.planet.com/data/imagery/planetscope/",
                          "quote": "All of the Doves have a system constraint to image regions that have a "
                                   "sun elevation angle greater than 10°."}},
]
# -----------------------------------------------------------------------------
# ACQUISITION EXCLUSIONS (v1.5.2): periods in which a satellite of the fleet is
# tracked (it has orbital elements, so Cell 2 keeps it) but its operator states
# that it does not acquire nominal imagery. Cell 5 drops the passes whose time
# falls in [start_utc, end_utc) and counts them. Every entry carries its source.
# Not covered here: routine acquisition plans, e.g. the Sentinel-2A extension
# campaign (from 13 March 2025; "through 31 December 2027", Copernicus Data
# Space Ecosystem news, 9 October 2026), under which Sentinel-2A images a given
# region on a reduced schedule (over the CKGR: every 20 days in August-October
# 2026 in the Earth Search catalogue). Use exclude_satellites (above) to
# bracket such a satellite. Checked and not listed: the Sentinel-2A night-time
# campaign from 23 July 2026 (20 days), whose page states that "The S2A
# expected data volume during that period should not be impacted" (daytime
# imaging continued: an S2A scene of the CKGR on 4 August 2026 is in the archive).
# -----------------------------------------------------------------------------
ACQUISITION_EXCLUSIONS = [
    {"name_pattern": r"^SENTINEL[- ]?2A$", "start_utc": "2026-01-13T00:00:00", "end_utc": "2026-03-09T00:00:00",
     "status": "sourced", "accessed": "2026-10-14",
     "source_url": "https://sentinels.copernicus.eu/-/sentinel-2a-and-sentinel-2b-tandem-acquisitions",
     "quote": "From 13 January to 08 March 2026, Sentinel-2A will temporarily pause its nominal acquisition "
              "activities.",
     "note": "Sentinel-2A/2B tandem calibration (page published 18 December 2025). The end date is inclusive, so "
             "the window ends at 9 March 00:00 UTC. Images taken in vicarious (VIC) mode during the drift were "
             "distributed to expert users only and are not modelled."},
]
H3_RESOLUTION = config.h3_resolution
DOWNLINK_MIN_ELEVATION_DEG = config.downlink_min_elevation_deg
T_MAX_WAIT_MINUTES = config.t_max_wait_minutes
LATENCY_DECAY_RATE = config.latency_decay_rate
LATENCY_DECAY_SCALE = config.latency_decay_scale
DECAY_SWEEP = tuple(config.decay_sweep)
ONSET_ROBUSTNESS_ENABLED = config.onset_robustness_enabled
ONSET_SHIFT_HOURS = config.onset_shift_hours
ONSET_SHIFT_STEP_MIN = config.onset_shift_step_min
RANDOM_BASELINE_DRAWS = config.random_baseline_draws
ILP_ENABLED = config.ilp_enabled
ILP_INSTANCES = config.ilp_instances
ILP_HORIZON_HOURS = config.ilp_horizon_hours
ILP_BUDGET_K = config.ilp_budget_k
ILP_TIME_LIMIT_S = config.ilp_time_limit_s
ILP_MAX_PAIRS = config.ilp_max_pairs
COVERAGE_TARGETS_PCT = tuple(sorted(float(t) for t in config.coverage_targets_pct))
H1_REACHABLE_COVERAGE_PCT = config.h1_reachable_coverage_pct
H1_FLOOR_MARGIN_MIN_CONFIG = config.h1_floor_margin_min
H1_MIN_ONSET_PCT = config.h1_min_onset_pct
H1_MIN_EVALUABLE_ONSETS_CONFIG = config.h1_min_evaluable_onsets
CLOUD_KNOWLEDGE = config.cloud_knowledge
TIME_AWARE_OBJECTIVE = config.time_aware_objective
CLOUD_MODEL = config.cloud_model
CLOUD_CORR_LENGTH_KM = config.cloud_corr_length_km
CLOUD_CORR_TIME_H = config.cloud_corr_time_h
CLOUD_PROBABILITY_SOURCE = config.cloud_probability_source
CLOUD_CLIMATOLOGY_MONTH = config.cloud_climatology_month
CLOUD_PLANNING = config.cloud_planning
CLOUD_SEEDS = config.cloud_seeds
CLOUD_SWEEP_LENGTHS_KM = tuple(config.cloud_sweep_lengths_km)
CLOUD_SWEEP_TIMES_H = tuple(config.cloud_sweep_times_h)
CLOUD_SWEEP_SEEDS = config.cloud_sweep_seeds
ANALYSIS_EPOCH = (None if config.analysis_epoch is None else
                  _utc_naive(config.analysis_epoch).strftime("%Y-%m-%dT%H:%M:%S.%f"))
REFERENCE_EPOCH_UTC = (None if config.reference_epoch_utc is None else
                       _utc_naive(config.reference_epoch_utc).strftime("%Y-%m-%dT%H:%M:%S.%f"))
LEGACY_STRIP_END_CAPS = bool(config.legacy_strip_end_caps)

# =============================================================================
# GOOGLE COLAB DRIVE MOUNT & OUTPUT DIRECTORY
# =============================================================================

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    OUT_DIR = f"/content/drive/MyDrive/ARF_Outputs_{ACTIVE_ROI}_{FLEET_TAG}"
    # v1.5.2: a complete run of ANOTHER release in this folder (its Cell 9
    # manifest names the release) is kept, never overwritten: this run writes
    # to <folder>_v<release digits> instead. Protects, e.g., the 12 October
    # 2026 runs of release 1.5.0, whose arf_refined_events.csv files are the
    # inputs of arf_colab_parity.py. A folder without a manifest (an incomplete
    # run) is reused as before; an unreadable manifest counts as another release.
    _prev_manifest = os.path.join(OUT_DIR, "arf_reproducibility_manifest.json")
    if os.path.exists(_prev_manifest):
        try:
            with open(_prev_manifest) as _f:
                _prev_release = json.load(_f).get("framework_version")
        except Exception:
            _prev_release = None
        if _prev_release != __version__:
            print(f"📁 {OUT_DIR} holds a complete run of release {_prev_release or '(unreadable manifest)'}: "
                  f"kept unchanged; this run writes to its own folder.")
            OUT_DIR += "_v" + __version__.replace(".", "")
    os.makedirs(OUT_DIR, exist_ok=True)
    print(f"📁 Google Drive mounted. Outputs will be saved to: {OUT_DIR}")
else:
    OUT_DIR = os.getcwd()
    print(f"📁 Running locally. Outputs will be saved to: {OUT_DIR}")

# =============================================================================
# RESULTS LEDGER INITIALISATION
# =============================================================================

# Preserve multi-scenario state in memory across cell runs
if 'scenario_results' not in globals():
    scenario_results: Dict[Tuple[str, Any], Dict[str, Any]] = {}
    print("📊 Initialised empty scenario_results ledger (fresh kernel).")
else:
    print(f"📊 Existing scenario_results preserved for (ROI, N or fleet tag) = "
          f"{sorted(scenario_results.keys(), key=str)}.")

# =============================================================================
# SUMMARY & PERSISTENCE
# =============================================================================

print("\n" + "=" * 60)
print(f"  {__framework_name__} v{__version__}")
print("=" * 60)
print(f"  Active Configuration Summary:")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Region (ROI):              {ACTIVE_ROI}")
if FLEET == "starlink":
    print(f"  Starlink proxy, N:         {ACTIVE_N}")
else:
    print(f"  Fleet:                     {FLEET}" + (f" (PlanetScope {PLANET_SWATH_CASE})" if "planet" in FLEET else ""))
    print(f"  Window start:              " + (f"{ANALYSIS_EPOCH} UTC (retrospective)" if ANALYSIS_EPOCH else
                                              f"{REFERENCE_EPOCH_UTC} UTC (set)" if REFERENCE_EPOCH_UTC else
                                              "newest orbital elements (live)"))
    if EXCLUDE_SATELLITES:
        print(f"  Left out (sensitivity):    {', '.join(EXCLUDE_SATELLITES)}")
print(f"  Cloud Obscuration Prob:    {CLOUD_PROBABILITY:.2f}")
print(f"  Selection Budget (k):      {MAX_SELECTION_STEPS}")
print(f"  Random Seed:               {RANDOM_SEED}")
print(f"  Simulation Window:         {WINDOW_HOURS} hours")
if FLEET == "starlink":
    print(f"  Sensor Half-Angle:         {SWATH_HALF_ANGLE_DEG}°")
print(f"  H3 DGGS Resolution:        {H3_RESOLUTION}")
print(f"  Downlink Min Elevation:    {DOWNLINK_MIN_ELEVATION_DEG}°")
print(f"  Working Directory:         {OUT_DIR}")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  License:                   {__license__}")
print(f"  Recorded Scenarios:        {len(scenario_results)}")
print("=" * 60)

config_filepath = os.path.join(OUT_DIR, 'arf_config.json')
try:
    with open(config_filepath, 'w') as f:
        f.write(config.to_json())
    print(f"✅ Configuration exported to {config_filepath}")
except Exception as e:
    print(f"⚠️ Warning: Could not write configuration file: {e}")

print("\n=== CELL 0 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 1: ENVIRONMENT SETUP, OREKIT INITIALISATION, & SPATIAL DISCRETISATION
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Establishes the astrodynamics engine (Orekit) and discretises the continuous
#   Region of Interest (ROI) into a Discrete Global Grid System (DGGS) using
#   the H3 hexagonal index at resolution 7 (average cell 5.161 km2, average edge
#   1.406 km; H3 resolution table, https://h3geo.org/docs/core-library/restable/
#   -- the 1.22 km figure of the H3 v3 table is a different edge definition).
#
# Research Proposal Mapping:
#   - Section 3.2.1: Geodesic Grid Model (The Target Manifold R)
#   - Section 3.2.3: Ground Station Network (Telemetry Anchors)
#   - Gap 2 & Justification 2.4.1: Hexagonal uniform adjacency over Lat/Lon
#
# Dependencies:
#   - Cell 0 must be executed (provides ACTIVE_ROI, H3_RESOLUTION, OUT_DIR)
#
# Outputs (Globals for Downstream Pipeline):
#   - itrf: Orekit ITRF reference frame (IERS 2010 conventions)
#   - earth: Orekit WGS84 ellipsoid model
#   - target_tiles: Set[str] of H3 cell indices covering the ROI
#   - ground_station_points: Dict[str, GeodeticPoint]
#   - GROUND_STATIONS: Dict[str, Dict] of ground station metadata
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

"""
ARF Environment Setup & Spatial Discretisation
===============================================

This module executes three foundational initialisation tasks:

1. **JVM & Astrodynamics Engine Initialisation**: Boots the Java Virtual Machine
   (JVM) and loads high-precision physical models (IERS Earth Orientation Parameters,
   leap seconds, and gravitational harmonics).

2. **Geodetic Reference Frame Setup**: Instantiates the International Terrestrial
   Reference Frame (ITRF 2010) and WGS84 ellipsoid for SGP4 orbital propagation.

3. **Discrete Global Grid System (DGGS) Discretisation**: Projects the continuous
   target manifold into H3 hexagonal cells (near-uniform area; H3 cells are not
   exactly equal-area), enabling O(1) set-theoretic spatial operations. ROIs are
   official WDPA protected-area boundaries (the Botswana, Zambia and South
   Africa study areas), or, for the legacy runs, a bounding box (Botswana) or a
   buffered polyline (Nepal disaster corridor).
"""

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 1 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["1"] = _ARF_CELL_VERSION

import os
import sys
import platform
import logging
import json
from typing import Dict, List, Tuple, Set, Optional, Any
from dataclasses import dataclass

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logger = logging.getLogger("arf.cell1")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# =============================================================================
# CUSTOM EXCEPTIONS
# =============================================================================

class EnvironmentSetupError(Exception):
    """Raised when JVM, Orekit physical data, or dependencies fail to initialise."""
    pass

class SpatialDiscretisationError(Exception):
    """Raised when the H3 polygon discretisation fails or yields empty tile sets."""
    pass

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()

# =============================================================================
# 1. DEPENDENCY INSTALLATION & JVM INITIALISATION
# =============================================================================

logger.info("Verifying and installing astrodynamics & geospatial packages...")

# v1.5.0: a batch driver that runs many configurations in one session sets
# ARF_SKIP_INSTALL = True after the first run (pip, and the orekit-data clone
# from git, would otherwise repeat for every run).
import subprocess
# v1.5.2: the packages whose behaviour the results depend on are pinned to the
# versions the release was validated with (Colab, 12 October 2026, and the
# test environment): orekit-jpype 13.1.9.0 (with jdk4py 25.0.2.1), h3 4.5.0,
# pyshp 3.1.6. numpy, pandas, shapely, scipy and matplotlib are Colab's own
# (the refined events reproduced to 1e-13 deg across numpy 2.1 and 2.5); every
# version actually used is recorded in the manifest (Cell 9). orekit-data
# (Earth orientation and leap seconds) is installed from its repository head on
# purpose -- a pinned copy would lack the Earth orientation data of new windows
# -- and its version is recorded.
ARF_VALIDATED_VERSIONS = {"orekit-jpype": "13.1.9.0", "jdk4py": "25.0.2.1", "h3": "4.5.0", "pyshp": "3.1.6"}
if globals().get("ARF_SKIP_INSTALL"):
    logger.info("ARF_SKIP_INSTALL: package installation skipped (installed earlier in this session).")
else:
    try:
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             f"orekit-jpype[jdk4py]=={ARF_VALIDATED_VERSIONS['orekit-jpype']}",
             f"jdk4py=={ARF_VALIDATED_VERSIONS['jdk4py']}", f"h3=={ARF_VALIDATED_VERSIONS['h3']}",
             f"pyshp=={ARF_VALIDATED_VERSIONS['pyshp']}", "numpy", "pandas", "shapely"],
            check=True
        )
        subprocess.run(
            [sys.executable, "-m", "pip", "install", "-q",
             "git+https://gitlab.orekit.org/orekit/orekit-data.git"],
            check=True
        )
        logger.info("Library installation complete.")
    except subprocess.CalledProcessError as e:
        logger.error(f"Package installation failed: {e}")
        raise EnvironmentSetupError("Failed to install required dependencies.") from e

import jdk4py
import orekit_jpype

# Auto-detect JVM Path
try:
    _jvm_path = os.path.join(jdk4py.JAVA_HOME, "lib", "server", "libjvm.so") if platform.system() == "Linux" else None
    if _jvm_path and os.path.exists(_jvm_path):
        orekit_jpype.initVM(jvmpath=_jvm_path)
    else:
        orekit_jpype.initVM()
    logger.info("Orekit JVM initialised successfully.")
except Exception as e:
    logger.info(f"JVM already running or init handled: {e}")

# =============================================================================
# 2. OREKIT PHYSICAL DATA CONTEXT
# =============================================================================

from orekit_jpype.pyhelpers import setup_orekit_curdir

_orekit_data_loaded = False
try:
    setup_orekit_curdir(from_pip_library=True)
    _orekit_data_loaded = True
    logger.info("Orekit physical data context loaded successfully (pip library).")
except Exception:
    try:
        import orekitdata
        from orekit_jpype.pyhelpers import setup_orekit_data
        setup_orekit_data(filenames=orekitdata.__path__[0], from_pip_library=False)
        _orekit_data_loaded = True
        logger.info("Orekit physical data loaded via fallback package path.")
    except Exception as e:
        logger.error(f"Failed to load physical data context: {e}")

if not _orekit_data_loaded:
    raise EnvironmentSetupError("Could not load Orekit physical data (EOP/Leap Seconds).")

# =============================================================================
# 3. SPATIAL & ASTRODYNAMICS IMPORTS
# =============================================================================

import h3
import numpy as np
import pandas as pd

if not h3.__version__.startswith("4."):
    raise EnvironmentSetupError(f"Detected H3 version {h3.__version__}. ARF requires H3 v4.x.")
logger.info(f"H3 DGGS library version confirmed: {h3.__version__}")
# v1.5.2: report any package that differs from the validated version (a warning,
# not an error: the run is still recorded with the versions it used).
from importlib import metadata as _md1
ARF_PACKAGE_VERSIONS = {}
for _pkg, _want in ARF_VALIDATED_VERSIONS.items():
    try:
        ARF_PACKAGE_VERSIONS[_pkg] = _md1.version(_pkg)
    except _md1.PackageNotFoundError:
        ARF_PACKAGE_VERSIONS[_pkg] = None
    if ARF_PACKAGE_VERSIONS[_pkg] != _want:
        logger.warning(f"{_pkg} {ARF_PACKAGE_VERSIONS[_pkg]} differs from the validated {_want}; results may "
                       f"differ from the release's reference values (arf_colab_parity.py checks them).")

from org.orekit.frames import FramesFactory
from org.orekit.utils import IERSConventions, Constants
from org.orekit.bodies import OneAxisEllipsoid, GeodeticPoint
from org.hipparchus.util import FastMath

# Establish ITRF Reference Frame and WGS84 Oblate Spheroid
itrf = FramesFactory.getITRF(IERSConventions.IERS_2010, True)
earth = OneAxisEllipsoid(
    Constants.WGS84_EARTH_EQUATORIAL_RADIUS,
    Constants.WGS84_EARTH_FLATTENING,
    itrf
)
logger.info("Astrodynamics reference frame (ITRF 2010) & WGS84 ellipsoid initialised.")

# =============================================================================
# 4. REGION OF INTEREST (ROI) GEOMETRY DEFINITION
# =============================================================================

def _buffer_waypoints_to_polygon(waypoints_latlon: List[Tuple[float, float]], buffer_km: float) -> List[Tuple[float, float]]:
    """
    Buffer a lat/lon waypoint polyline by buffer_km and return the resulting polygon.
    Uses a flat-earth degree/km approximation local to the waypoints' average latitude.
    """
    import math
    from shapely.geometry import LineString

    lat_avg = sum(w[0] for w in waypoints_latlon) / len(waypoints_latlon)
    km_per_deg_lat = 111.0
    km_per_deg_lon = 111.0 * math.cos(math.radians(lat_avg))
    buffer_deg = buffer_km / min(km_per_deg_lat, km_per_deg_lon)

    line_lonlat = LineString([(lon, lat) for lat, lon in waypoints_latlon])
    corridor_lonlat = line_lonlat.buffer(buffer_deg)
    return [(lat, lon) for lon, lat in corridor_lonlat.exterior.coords]

ROI_DEFINITIONS = {
    "botswana_full": {
        "short_name": "Botswana (bounding box)",  # label on the maps (Cells 3 and 7)
        "description": "Botswana full national boundary (bounding rectangle)",
        "roi_coords": [
            (-17.6618156877, 19.8954577979), (-17.6618156877, 29.4321883481),
            (-26.8285429827, 29.4321883481), (-26.8285429827, 19.8954577979)
        ],
        "ground_stations": {
            "BIUST": {"lat": -22.59, "lng": 27.14, "alt_m": 994.0},
            "Kasane": {"lat": -17.80, "lng": 25.15, "alt_m": 930.0},
            "Kebonye": {"lat": -24.84, "lng": 25.84, "alt_m": 994.0},
        },
        "tle_source": "celestrak_live",
    },
    "nepal_trishuli_corridor": {
        "short_name": "Trishuli corridor",  # label on the maps (Cells 3 and 7)
        "description": (
            "2026 Nepal Trishuli/Lende Khola debris flow and flash flood corridor. "
            "Buffered 7.5km either side of the waypoint polyline."
        ),
        "waypoints_latlon": [
            (28.27778, 85.37778),      # Rasuwagadhi
            (28.1635, 85.3363),        # Syaphru / Syabrubesi
            (27.9833, 85.1833),        # Betrawati
            (27.8881907, 85.1340051),  # Devighat
        ],
        "buffer_km": 7.5,
        "historical_epoch": "2026-08-26T02:52:00",
        "tle_source": "spacetrack_history",
        "ground_stations": {
            "NAST_Khumaltar": {"lat": 27.65479, "lng": 85.32882, "alt_m": 1335.0},
        },
    },
    "ckgr_okavango": {
        "short_name": "CKGR + Okavango Delta",  # label on the maps (Cells 3 and 7)
        "description": (
            "Central Kalahari Game Reserve and the Okavango Delta World Heritage Site, "
            "mosaicked together from their official WDPA boundaries."
        ),
        # Expected GIS areas are the WDPA's own polygon areas (GIS_AREA), used
        # to flag a revised boundary in a later WDPA release.
        "wdpa_sites": {
            7510: {"name": "Central Kalahari Game Reserve",
                   "expected_gis_area_km2": 52225.35, "designation_contains": "Game Reserve"},
            555577555: {"name": "Okavango Delta (World Heritage Site)",
                        "expected_gis_area_km2": 20551.25, "designation_contains": "World Heritage"},
        },
        "ground_stations": {
            "BIUST": {"lat": -22.59, "lng": 27.14, "alt_m": 994.0},
            "Kasane": {"lat": -17.80, "lng": 25.15, "alt_m": 930.0},
            "Kebonye": {"lat": -24.84, "lng": 25.84, "alt_m": 994.0},
        },
        "tle_source": "celestrak_live",
    },
    # Site IDs, names, countries and reported areas below are as shown on each
    # site's Protected Planet page (source_url), accessed 2026-10-08. The pages do
    # not show the designation, so designation_contains is None: the designation
    # is read from the file and recorded, and the site's identity is instead
    # checked by its reported area (REP_AREA) against the page value.
    # Ground stations: the Botswana network (BIUST, Kasane, Kebonye), so downlink
    # waits reflect a Botswana-operated ground segment. ASSUMPTION: these are
    # notional receiving sites (approximate coordinates), not the operators'
    # ground networks; Sentinel-2, Landsat and PlanetScope downlink to their own
    # stations. All of them see every pass over these regions, so every wait is
    # 0 and "delivery" times are acquisition times (Cells 6, 8 say so).
    # Replaces Middle Zambezi (WDPA 555547578), whose WDPA record is a point
    # with no boundary polygon (WDPA/WDOECM Manual 1.6, p.15), so it cannot be tiled.
    "lower_zambezi": {
        "short_name": "Lower Zambezi NP",  # label on the maps (Cells 3 and 7)
        "description": "Lower Zambezi protected area (Zambia), from its official WDPA boundary.",
        "wdpa_sites": {
            7962: {"name": "Lower Zambezi", "designation_contains": None,
                   "expected_rep_area_km2": 4143.0,
                   "source_url": "https://www.protectedplanet.net/7962", "accessed": "2026-10-08"},
        },
        "ground_stations": {
            "BIUST": {"lat": -22.59, "lng": 27.14, "alt_m": 994.0},
            "Kasane": {"lat": -17.80, "lng": 25.15, "alt_m": 930.0},
            "Kebonye": {"lat": -24.84, "lng": 25.84, "alt_m": 994.0},
        },
        "tle_source": "celestrak_live",
    },
    "karoo_nature_reserve": {
        "short_name": "Karoo NR",  # label on the maps (Cells 3 and 7)
        "description": ("Karoo Nature Reserve (South Africa), from its official WDPA boundary. "
                        "32 H3 res-7 tiles with the October 2026 WDPA release (167 km2 tiled for a "
                        "143 km2 site: small sites are quantised to whole tiles)."),
        "wdpa_sites": {
            555563860: {"name": "Karoo Nature Reserve", "designation_contains": None,
                        "expected_rep_area_km2": 143.15,
                        "source_url": "https://www.protectedplanet.net/555563860", "accessed": "2026-10-08"},
        },
        "ground_stations": {
            "BIUST": {"lat": -22.59, "lng": 27.14, "alt_m": 994.0},
            "Kasane": {"lat": -17.80, "lng": 25.15, "alt_m": 930.0},
            "Kebonye": {"lat": -24.84, "lng": 25.84, "alt_m": 994.0},
        },
        "tle_source": "celestrak_live",
    },
}

# -----------------------------------------------------------------------------
# WDPA boundary loading (ROIs defined by "wdpa_sites")
# -----------------------------------------------------------------------------
# Official protected-area boundaries come from the World Database on Protected
# Areas (UNEP-WCMC and IUCN, Protected Planet). The WDPA licence forbids
# redistributing the data, so the boundary files are NOT shipped with this
# framework: download each site from https://www.protectedplanet.net/<SITE_ID>
# as a shapefile ("SHP") and put the download in ARF_BOUNDARY_DIR -- as the zip,
# or extracted, at any folder depth. Files are matched by WDPA site ID, split
# parts (_shp_0, _shp_1, ...) are combined, and the newest release is used. Cite as: UNEP-WCMC and IUCN (<year>), Protected
# Planet: The World Database on Protected Areas (WDPA) [On-line], <month/year
# of the version downloaded>, Cambridge, UK: UNEP-WCMC and IUCN.

WDPA_AREA_TOLERANCE = 0.05   # tiled area must be within 5% of the WDPA GIS_AREA ...
# ... or within WDPA_TILE_TOLERANCE tiles' worth, whichever is larger. Small sites
# are quantised to whole tiles (one res-7 tile = 5.16 km2 = 3.6% of a 143 km2
# site), so a fixed 5% fails correct boundaries. Empirical setting: for 400 random
# irregular 143 km2 polygons the largest tiling error was 4.6 tiles; 5 tiles
# covered all of them (ARF validation, 2026-10-08). Above ~3,600 km2 the 5% term
# dominates (largest error seen there: 2.1%).
WDPA_TILE_TOLERANCE = 5
WDPA_REP_AREA_TOLERANCE = 0.005   # file REP_AREA vs the value on the site page


def _default_boundary_dir():
    drive_dir = "/content/drive/MyDrive/ARF_boundaries"
    if os.path.isdir(drive_dir):
        return drive_dir
    return os.path.join(os.getcwd(), "ARF_boundaries")


ARF_BOUNDARY_DIR = globals().get("ARF_BOUNDARY_DIR") or _default_boundary_dir()


_MONTHS = {m: i for i, m in enumerate(("Jan", "Feb", "Mar", "Apr", "May", "Jun",
                                         "Jul", "Aug", "Sep", "Oct", "Nov", "Dec"), 1)}


def _wdpa_release(path):
    """(year, month) of the WDPA release in a filename like ..._Oct2026_..., or (0, 0)."""
    import re
    m = re.search(r"_([A-Z][a-z]{2})(\d{4})_", os.path.basename(path))
    return (int(m.group(2)), _MONTHS.get(m.group(1), 0)) if m else (0, 0)


def _find_wdpa_files(site_id, boundary_dir):
    """
    Every polygon shapefile part for a WDPA site under boundary_dir, at ANY depth.
    Protected Planet splits downloads into numbered parts (..._shp_0, _shp_1, ...)
    nested inside the download folder, and may deliver them as zips inside a zip;
    zips are unpacked (only this site's members) into a temporary folder. If
    several monthly releases are present, only the newest release's parts are
    returned. Returns [] if nothing is found.
    """
    import glob, zipfile, tempfile
    target = f"_{site_id}_shp-polygons.shp"
    found = [p for p in glob.glob(os.path.join(boundary_dir, "**", "*.shp"), recursive=True)
             if p.endswith(target)]
    zips = [p for p in glob.glob(os.path.join(boundary_dir, "**", "*.zip"), recursive=True)
            if f"_{site_id}_" in os.path.basename(p)]
    for zp in zips:
        out = tempfile.mkdtemp(prefix=f"wdpa_{site_id}_")
        def _unpack(zf, dest):
            for member in zf.namelist():
                base = os.path.basename(member)
                if member.endswith(".zip"):
                    import io
                    with zipfile.ZipFile(io.BytesIO(zf.read(member))) as inner:
                        _unpack(inner, os.path.join(dest, os.path.splitext(base)[0]))
                elif f"_{site_id}_shp-polygons." in base:
                    os.makedirs(dest, exist_ok=True)
                    with open(os.path.join(dest, base), "wb") as fh:
                        fh.write(zf.read(member))
        with zipfile.ZipFile(zp) as zf:
            _unpack(zf, out)
        found += [p for p in glob.glob(os.path.join(out, "**", "*.shp"), recursive=True) if p.endswith(target)]
    if not found:
        return []
    newest = max(_wdpa_release(p) for p in found)
    # De-duplicate by content: the same part may exist both extracted and inside
    # its zip; reading it twice would double-count the site's area.
    import hashlib
    unique = {}
    for p in sorted(p for p in found if _wdpa_release(p) == newest):
        with open(p, "rb") as fh:
            unique.setdefault(hashlib.sha256(fh.read()).hexdigest(), p)
    return sorted(unique.values())


def _find_wdpa_file(site_id, boundary_dir):
    """First matching file (kept for the manifest in Cell 9); see _find_wdpa_files."""
    files = _find_wdpa_files(site_id, boundary_dir)
    return files[0] if files else None


def _wdpa_sites_present(boundary_dir):
    """WDPA site IDs for which polygon shapefiles exist anywhere under boundary_dir."""
    import glob, re
    ids = set()
    for p in glob.glob(os.path.join(boundary_dir, "**", "*_shp-polygons.shp"), recursive=True):
        m = re.search(r"_(\d+)_shp-polygons\.shp$", p)
        if m:
            ids.add(int(m.group(1)))
    return sorted(ids)


def load_wdpa_site(site_id, spec, boundary_dir, h3_res, geoms_out=None):
    """
    Read one WDPA site's boundary, check it is the intended site, and tile it.
    Returns (tile_set, report_dict). If geoms_out is a list, the boundary
    polygons (GeoJSON-like) are appended to it for the maps; they are kept in
    memory only and never written to the outputs (WDPA licence). Raises ConfigurationError with download
    instructions if the file is missing, or if its designation does not match
    (e.g. a Ramsar file supplied where the World Heritage Site is required).
    """
    import shapefile   # pyshp
    paths = _find_wdpa_files(site_id, boundary_dir)
    if not paths:
        import glob
        points = [p for p in glob.glob(os.path.join(boundary_dir, "**", f"*_{site_id}_shp-points.shp"), recursive=True)]
        if points:
            # WDPA Manual 1.6, p.15: "Where boundary data are unavailable, the
            # latitude and longitude of the centremost point of the site is
            # requested ... instead." A point has no shape, so it cannot be tiled.
            raise ConfigurationError(
                f"WDPA site {site_id} ({spec['name']}) has NO boundary polygon in the WDPA: the download "
                f"contains only a point record ({os.path.basename(points[0])}). WDPA stores a point where "
                f"the data provider supplied no boundary (WDPA/WDOECM Manual 1.6, p.15), so this site's "
                f"shape is unknown and it cannot be tiled. Choose a site whose download contains "
                f"'_shp-polygons' files, or remove '{spec['name']}' from ROI_DEFINITIONS.")
        present = _wdpa_sites_present(boundary_dir) if os.path.isdir(boundary_dir) else []
        raise ConfigurationError(
            f"WDPA boundary for site {site_id} ({spec['name']}) not found anywhere under {boundary_dir}. "
            f"Download it as a shapefile from https://www.protectedplanet.net/{site_id} and place the "
            f"download (zip, or extracted folders at any depth) in that folder. "
            f"WDPA sites found there: {present if present else 'none'}.")
    path = paths[0]
    tiles, gis_area, desig, name, rep_areas = set(), 0.0, set(), set(), set()
    shp_records = []
    for p in paths:                                   # every split part of this site
        reader = shapefile.Reader(p)
        fields = [f[0] for f in reader.fields[1:]]
        shp_records += [(fields, sr) for sr in reader.iterShapeRecords()]
    for fields, shp_rec in shp_records:
        attrs = dict(zip(fields, shp_rec.record))
        rec_id = int(attrs.get("SITE_ID", attrs.get("WDPAID", -1)))
        if rec_id != int(site_id):
            raise ConfigurationError(f"{os.path.basename(path)} holds WDPA site {rec_id}, expected {site_id}.")
        desig.add(str(attrs.get("DESIG_ENG", "")))
        name.add(str(attrs.get("NAME_ENG", attrs.get("NAME", ""))))
        gis_area += float(attrs.get("GIS_AREA", 0.0))
        rep_areas.add(round(float(attrs.get("REP_AREA", 0.0)), 4))
        tiles |= set(h3.geo_to_cells(shp_rec.shape.__geo_interface__, h3_res))
        if geoms_out is not None:
            geoms_out.append(shp_rec.shape.__geo_interface__)
    # Identity check 1: reported area (site-level, repeated on every part) must
    # match the value on the site's Protected Planet page, when one is given.
    exp_rep = spec.get("expected_rep_area_km2")
    if exp_rep:
        rep = max(rep_areas) if rep_areas else 0.0
        if abs(rep - exp_rep) / exp_rep > WDPA_REP_AREA_TOLERANCE:
            raise ConfigurationError(
                f"WDPA site {site_id}: file reports REP_AREA {rep:,.2f} km2, but {spec.get('source_url', 'the site page')} "
                f"shows {exp_rep:,.2f} km2. This is probably not {spec['name']}.")
    # Identity check 2: designation, when the ROI specifies one. Otherwise the
    # file's designation is only recorded.
    if spec.get("designation_contains") is None:
        logger.info(f"  WDPA {site_id}: designation {sorted(desig)} (recorded, not pre-specified).")
    elif not any(spec["designation_contains"].lower() in d.lower() for d in desig):
        raise ConfigurationError(
            f"WDPA site {site_id} is designated {sorted(desig)}, but this ROI requires "
            f"'{spec['designation_contains']}' ({spec['name']}). Check the site ID: Protected Planet "
            f"lists several overlapping designations under similar names.")
    tiled_area = float(sum(h3.cell_area(c, unit="km^2") for c in tiles))
    deviation = (tiled_area - gis_area) / gis_area if gis_area else float("nan")
    tile_km2 = tiled_area / len(tiles) if tiles else 0.0
    tolerance = max(WDPA_AREA_TOLERANCE, WDPA_TILE_TOLERANCE * tile_km2 / gis_area if gis_area else 0.0)
    report = {"site_id": int(site_id), "name": sorted(name), "designation": sorted(desig),
              "file": os.path.basename(path), "n_parts": len(paths), "wdpa_gis_area_km2": gis_area,
              "expected_gis_area_km2": spec.get("expected_gis_area_km2"),
              "tiled_area_km2": tiled_area, "area_deviation_pct": 100 * deviation, "n_tiles": len(tiles),
              "area_tolerance_pct": 100 * tolerance, "wdpa_rep_area_km2": max(rep_areas) if rep_areas else None,
              "source_url": spec.get("source_url"), "accessed": spec.get("accessed")}
    if not abs(deviation) <= tolerance:
        raise SpatialDiscretisationError(
            f"WDPA site {site_id}: tiled area {tiled_area:,.0f} km2 differs from WDPA GIS_AREA "
            f"{gis_area:,.0f} km2 by {100 * deviation:+.1f}% (tolerance {100 * tolerance:.1f}%).")
    exp = spec.get("expected_gis_area_km2")
    if exp and abs(gis_area - exp) / exp > WDPA_AREA_TOLERANCE:
        logger.warning(f"WDPA site {site_id}: GIS_AREA {gis_area:,.0f} km2 differs from the expected "
                       f"{exp:,.0f} km2 -- the boundary may have been revised in this WDPA release.")
    return tiles, report


def discretise_wdpa_roi(roi_def, h3_res, boundary_dir=None, geoms_out=None):
    """Union of the H3 tiles of every WDPA site in roi_def['wdpa_sites'].
    geoms_out: optional list collecting the boundary polygons (see load_wdpa_site)."""
    boundary_dir = boundary_dir or ARF_BOUNDARY_DIR
    all_tiles, reports = set(), []
    for site_id, spec in roi_def["wdpa_sites"].items():
        tiles, rep = load_wdpa_site(site_id, spec, boundary_dir, h3_res, geoms_out=geoms_out)
        all_tiles |= tiles
        reports.append(rep)
        logger.info(f"  WDPA {site_id} ({', '.join(rep['name'])}; {', '.join(rep['designation'])}): "
                    f"{rep['n_tiles']:,} tiles, {rep['tiled_area_km2']:,.0f} km2 vs WDPA "
                    f"{rep['wdpa_gis_area_km2']:,.0f} km2 ({rep['area_deviation_pct']:+.2f}%)")
    return all_tiles, reports


def resolve_roi_coords(roi_name: str) -> List[Tuple[float, float]]:
    """Resolve a named ROI to its (lat, lon) polygon."""
    if roi_name not in ROI_DEFINITIONS:
        raise ConfigurationError(f"ACTIVE_ROI={roi_name!r} is not defined in ROI_DEFINITIONS.")

    roi_def = ROI_DEFINITIONS[roi_name]
    if "roi_coords" in roi_def:
        return roi_def["roi_coords"]
    elif "waypoints_latlon" in roi_def:
        return _buffer_waypoints_to_polygon(roi_def["waypoints_latlon"], roi_def["buffer_km"])
    elif "wdpa_sites" in roi_def:
        return None          # boundary polygons are loaded from WDPA files (see section 5)
    else:
        raise ConfigurationError(f"ROI {roi_name!r} lacks coordinate definitions.")

# Resolve active ROI from Cell 0
_active_roi = ACTIVE_ROI if 'ACTIVE_ROI' in globals() else "botswana_full"
roi_coords = resolve_roi_coords(_active_roi)

# =============================================================================
# 5. H3 DGGS SPATIAL DISCRETISATION
# =============================================================================

_h3_res = H3_RESOLUTION if 'H3_RESOLUTION' in globals() else 7

roi_boundary_report = None
# ROI outline for the maps (Cells 3 and 7): the official boundary polygons, or
# the ROI polygon. Kept in memory only -- WDPA boundaries must never be written
# to the outputs (the WDPA licence forbids redistributing the data).
ROI_OUTLINE_GEOMS: List[dict] = []
if "wdpa_sites" in ROI_DEFINITIONS[_active_roi]:
    # Configuration/discretisation errors here carry their own instructions.
    target_tiles, roi_boundary_report = discretise_wdpa_roi(ROI_DEFINITIONS[_active_roi], _h3_res,
                                                            geoms_out=ROI_OUTLINE_GEOMS)
    ROI_OUTLINE_SOURCE = "WDPA"
else:
    ROI_OUTLINE_GEOMS = [{"type": "Polygon", "coordinates": [[(lon, lat) for lat, lon in roi_coords]]}]
    ROI_OUTLINE_SOURCE = "ROI definition"
    try:
        geo_polygon = h3.LatLngPoly(roi_coords)
        target_tiles: Set[str] = set(h3.polygon_to_cells(geo_polygon, _h3_res))
    except Exception as e:
        raise SpatialDiscretisationError(f"Failed to discretise ROI '{_active_roi}' at H3 Res {_h3_res}: {e}") from e

_n_tiles = len(target_tiles)
if _n_tiles == 0:
    raise SpatialDiscretisationError(f"Discretisation of '{_active_roi}' returned 0 tiles.")

logger.info(f"━ ROI: {_active_roi} discretised into {_n_tiles:,} tiles at Resolution {_h3_res}.")

# =============================================================================
# 6. GROUND STATION NETWORK (Downlink Physical Constraints)
# =============================================================================

GROUND_STATIONS = ROI_DEFINITIONS[_active_roi].get("ground_stations", {})
if not GROUND_STATIONS:
    logger.warning(f"ROI {_active_roi!r} defines no ground stations; D_downlink will be zero.")

ground_station_points: Dict[str, GeodeticPoint] = {
    name: GeodeticPoint(FastMath.toRadians(d['lat']), FastMath.toRadians(d['lng']), d['alt_m'])
    for name, d in GROUND_STATIONS.items()
}

# =============================================================================
# SUMMARY & PERSISTENCE
# =============================================================================

# v1.5.2: the smallest cell ID, not next(iter(set)): string hashing is salted
# per Python process, so a set's first element (and this JSON) changed between
# identical runs.
_sample_tile = min(target_tiles)
_sample_lat, _sample_lng = h3.cell_to_latlng(_sample_tile)

_validation_results = {
    "jvm_initialised": True,
    "orekit_data_loaded": _orekit_data_loaded,
    "h3_version": h3.__version__,
    "roi_active": _active_roi,
    "roi_tiles": _n_tiles,
    "sample_tile": _sample_tile,
    "sample_tile_centroid": (_sample_lat, _sample_lng),
    "package_versions": ARF_PACKAGE_VERSIONS,
    "validated_versions": ARF_VALIDATED_VERSIONS,
    "ground_stations": len(GROUND_STATIONS),
    "wdpa_boundaries": roi_boundary_report,
    "h3_resolution": _h3_res,
}

print("\n" + "=" * 60)
print("  CELL 1: Environment Setup & Spatial Discretisation")
print("=" * 60)
print(f"  JVM Status:                Initialised ({platform.system()})")
print(f"  Orekit Data:               Loaded Successfully")
print(f"  H3 Version:                {h3.__version__}")
print(f"  Reference Frames:          ITRF 2010, WGS84 Ellipsoid")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Target ROI:                {_active_roi}")
print(f"  Discretised Tiles:         {_n_tiles:,} (Res {_h3_res})")
print(f"  Ground Stations Configured:{len(GROUND_STATIONS)} ({', '.join(GROUND_STATIONS.keys())})")
print("=" * 60)

validation_filepath = os.path.join(_out_dir, 'arf_cell1_validation.json')
try:
    with open(validation_filepath, 'w') as f:
        json.dump(_validation_results, f, indent=2, default=str)
    logger.info(f"Validation results saved to {validation_filepath}")
except Exception as e:
    logger.warning(f"Could not save validation results: {e}")

print("\n=== CELL 1 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 2: TLE ACQUISITION & DETERMINISTIC SATELLITE SELECTION
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Ingests orbital elements for the imaging fleet and builds SGP4 propagators
#   (Orekit TLEPropagator).
#
#   Real fleets (default): CelesTrak GP data as OMM JSON (catalogue numbers
#   above 99999 included), live from the newest element sets or from
#   REFERENCE_EPOCH_UTC; past windows (ANALYSIS_EPOCH) use Space-Track
#   gp_history elements as of that instant. EXCLUDE_SATELLITES (v1.5.2) leaves
#   named satellites out for a fleet sensitivity run.
#   Starlink proxy (legacy): TLE text, live (celestrak_live) or at a disaster
#   onset (spacetrack_history, e.g. the Nepal flash flood).
#
# Reproducibility:
#   Real fleets: every operational satellite with a sourced MISSION_TABLE entry,
#   from CelesTrak GP data cached for 6 h (ARF_cache), or from Space-Track
#   gp_history as of ANALYSIS_EPOCH (retrospective). The fleet list, the element
#   files and the window start are recorded (Cell 9). Runs to be compared must
#   share the window start (REFERENCE_EPOCH_UTC or ANALYSIS_EPOCH, Cell 0).
#   Starlink proxy (legacy): NORAD ID pinning of a seeded sample of N.
#
# Dependencies:
#   - Cell 0 (provides ACTIVE_N, ACTIVE_ROI, OUT_DIR)
#   - Cell 1 (provides ROI_DEFINITIONS, itrf, earth)
#
# Outputs (Globals for Downstream Pipeline):
#   - constellation: List[Dict] of active satellite objects and TLE propagators
#   - sim_epoch: Orekit AbsoluteDate marking the simulation origin
#   - CONSTELLATION_ALTITUDE_KM: Derived physical median altitude
#   - T_MAX_WAIT_MINUTES: Dynamically derived orbital period
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 2 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["2"] = _ARF_CELL_VERSION

import os
import sys
import json
import time
import logging
import urllib.request
import urllib.parse
import numpy as np
from org.orekit.propagation.analytical.tle import TLE, TLEPropagator
from org.hipparchus.util import FastMath

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logger = logging.getLogger("arf.cell2")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# =============================================================================
# CACHE & SOURCE CONFIGURATION
# =============================================================================

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()
# v1.5.0: ARF_SHARED_CACHE_DIR (set by a batch driver) shares the CelesTrak group
# lists and Space-Track element sets between runs, so a batch does not download
# the same data once per run; every cache file name carries its ROI, fleet or
# epoch, so runs cannot read each other's selections.
ARF_CACHE_DIR = globals().get("ARF_SHARED_CACHE_DIR") or os.path.join(_out_dir, "ARF_cache")
os.makedirs(ARF_CACHE_DIR, exist_ok=True)

CELESTRAK_URL = "https://celestrak.org/NORAD/elements/gp.php?GROUP=starlink&FORMAT=tle"
TLE_RAW_CACHE_PATH = os.path.join(ARF_CACHE_DIR, f"arf_raw_cache_{ACTIVE_ROI}.json")
SELECTION_CACHE_PATH = os.path.join(ARF_CACHE_DIR, f"arf_selection_{ACTIVE_ROI}_N{ACTIVE_N}.json")
TLE_CACHE_MAX_AGE_HOURS = 6

_roi_def = ROI_DEFINITIONS[ACTIVE_ROI]
TLE_SOURCE = _roi_def.get("tle_source", "celestrak_live")

# =============================================================================
# 1A. LIVE FETCH (CELESTRAK)
# =============================================================================

def fetch_tle_data(url: str = CELESTRAK_URL) -> List[Tuple[str, str, str]]:
    logger.info(f"Fetching live TLE data from: {url}")
    req = urllib.request.Request(url, headers={"User-Agent": "ARF-Research-Pipeline/1.0"})
    with urllib.request.urlopen(req, timeout=30) as resp:
        raw_text = resp.read().decode("utf-8")

    lines = [l.strip() for l in raw_text.splitlines() if l.strip()]
    records = []
    for i in range(0, len(lines), 3):
        records.append((lines[i], lines[i+1], lines[i+2]))
    return records

def load_tle_records(url=CELESTRAK_URL, cache_path=TLE_RAW_CACHE_PATH, max_age_hours=TLE_CACHE_MAX_AGE_HOURS, force_refresh=False):
    if os.path.exists(cache_path) and not force_refresh:
        with open(cache_path) as f:
            cache = json.load(f)
        age_hours = (time.time() - cache["fetched_at"]) / 3600.0
        if age_hours < max_age_hours:
            logger.info(f"Using cached TLE fetch ({age_hours:.1f}h old): {len(cache['records'])} records.")
            return [tuple(r) for r in cache["records"]]

    records = fetch_tle_data(url)
    logger.info(f"Total records fetched: {len(records)}")
    try:
        with open(cache_path, "w") as f:
            json.dump({"fetched_at": time.time(), "url": url, "records": records}, f)
    except Exception as e:
        logger.warning(f"Could not write TLE raw cache: {e}")
    return records

# =============================================================================
# 1B. HISTORICAL FETCH (SPACE-TRACK)
# =============================================================================

def _get_spacetrack_credentials():
    try:
        from google.colab import userdata
        username = userdata.get('SPACETRACK_USERNAME')
        password = userdata.get('SPACETRACK_PASSWORD')
    except Exception as e:
        raise RuntimeError("Could not read Space-Track credentials from Colab userdata. "
                           "Set SPACETRACK_USERNAME and SPACETRACK_PASSWORD in the Secrets tab.")
    if not username or not password:
        raise RuntimeError("SPACETRACK_USERNAME or SPACETRACK_PASSWORD is empty.")
    return username, password

def _spacetrack_login(username, password, base_url="https://www.space-track.org"):
    import http.cookiejar
    cookie_jar = http.cookiejar.CookieJar()
    opener = urllib.request.build_opener(urllib.request.HTTPCookieProcessor(cookie_jar))
    login_url = f"{base_url}/ajaxauth/login"
    payload = urllib.parse.urlencode({"identity": username, "password": password}).encode()
    req = urllib.request.Request(login_url, data=payload, headers={"User-Agent": "ARF-Research-Pipeline/1.0"})

    resp = opener.open(req, timeout=30)
    if resp.getcode() != 200:
        raise RuntimeError(f"Space-Track auth failed (HTTP {resp.getcode()})")
    # Space-Track signals bad credentials with HTTP 200 and a body of exactly
    # {"Login": "Failed"} (success returns an empty body plus the session
    # cookie), so the status code alone cannot detect a failed login.
    body = resp.read().decode("utf-8").strip()
    if body:
        try:
            parsed = json.loads(body)
            if isinstance(parsed, dict) and parsed.get("Login") == "Failed":
                raise RuntimeError("Space-Track authentication rejected: "
                                   "check SPACETRACK_USERNAME / SPACETRACK_PASSWORD.")
        except json.JSONDecodeError:
            pass
    return opener

def fetch_starlink_gp_history(opener, epoch_date, base_url="https://www.space-track.org", lookback_days=30):
    from datetime import datetime, timedelta
    end_dt = datetime.strptime(epoch_date, "%Y-%m-%dT%H:%M:%S.%f") if "." in epoch_date else datetime.strptime(epoch_date, "%Y-%m-%dT%H:%M:%S")
    start_dt = end_dt - timedelta(days=lookback_days)
    start_str = start_dt.strftime("%Y-%m-%dT%H:%M:%S")

    query_path = f"/basicspacedata/query/class/gp_history/OBJECT_NAME/~~STARLINK/EPOCH/{start_str}--{epoch_date}/orderby/EPOCH%20desc/format/json"
    req = urllib.request.Request(base_url + query_path, headers={"User-Agent": "ARF-Research-Pipeline/1.0"})

    with opener.open(req, timeout=120) as resp:
        parsed = json.loads(resp.read().decode("utf-8"))

    if not parsed or (isinstance(parsed, dict) and "error" in parsed):
        raise RuntimeError(f"Space-Track gp_history query error or empty result.")
    return parsed

def _parse_gp_history_epoch(epoch_str):
    from datetime import datetime
    try:
        return datetime.strptime(epoch_str, "%Y-%m-%dT%H:%M:%S.%f")
    except ValueError:
        return datetime.strptime(epoch_str, "%Y-%m-%dT%H:%M:%S")

def _dedupe_gp_history_to_latest_per_satellite(gp_history_records):
    by_epoch_desc = sorted(gp_history_records, key=lambda r: _parse_gp_history_epoch(r["EPOCH"]), reverse=True)
    by_id_stable = sorted(by_epoch_desc, key=lambda r: r["NORAD_CAT_ID"])
    seen, out = set(), []
    for rec in by_id_stable:
        nid = rec["NORAD_CAT_ID"]
        if nid not in seen:
            seen.add(nid)
            out.append((rec["OBJECT_NAME"], rec["TLE_LINE1"], rec["TLE_LINE2"]))
    return out

def load_historical_tle_records(epoch_date, cache_path=TLE_RAW_CACHE_PATH, force_refresh=False):
    if os.path.exists(cache_path) and not force_refresh:
        with open(cache_path) as f:
            cache = json.load(f)
        logger.info(f"Using cached Space-Track gp_history fetch (epoch <= {epoch_date}).")
        return [tuple(r) for r in cache["records"]]

    username, password = _get_spacetrack_credentials()
    opener = _spacetrack_login(username, password)
    raw_records = fetch_starlink_gp_history(opener, epoch_date)
    records = _dedupe_gp_history_to_latest_per_satellite(raw_records)
    logger.info(f"Deduplicated to {len(records)} satellites at historical epoch.")

    try:
        with open(cache_path, "w") as f:
            json.dump({"fetched_at": time.time(), "epoch_date": epoch_date, "records": records}, f)
    except Exception as e:
        logger.warning(f"Could not write Space-Track cache: {e}")
    return records

# =============================================================================
# 2. EXECUTE FETCH & DETERMINISTIC PINNING
# =============================================================================

if FLEET != "starlink" and TLE_SOURCE == "spacetrack_history" and not globals().get("ANALYSIS_EPOCH"):
    # A historical ROI with a real fleet is a retrospective run from the ROI's onset.
    from datetime import datetime as _dt_hist
    ANALYSIS_EPOCH = _dt_hist.fromisoformat(_roi_def["historical_epoch"]).strftime("%Y-%m-%dT%H:%M:%S.%f")
    logger.info(f"Historical ROI {ACTIVE_ROI!r} with fleet {FLEET!r}: retrospective run from {ANALYSIS_EPOCH} UTC.")
if FLEET != "starlink":
    raw_records = []        # real fleets use CelesTrak GP JSON (section 2B)
elif TLE_SOURCE == "spacetrack_history":
    _historical_epoch = _roi_def["historical_epoch"]
    logger.info(f"━ ACTIVE_ROI={ACTIVE_ROI!r} requires historical TLEs (epoch <= {_historical_epoch}).")
    raw_records = load_historical_tle_records(_historical_epoch)
else:
    raw_records = load_tle_records()

import re

# =============================================================================
# 2B. REAL EARTH-OBSERVATION FLEETS (CelesTrak GP data, JSON)
# =============================================================================
# JSON (OMM) instead of TLE text: CelesTrak cannot deliver catalogue numbers
# above 99999 as TLEs, so newly catalogued satellites would silently vanish
# from a TLE fetch. Orekit TLE objects are built from the numeric elements.

def omm_to_tle(omm, utc):
    """
    Orekit TLE from a CelesTrak OMM JSON record (GP data, FORMAT=json). Built
    from the numeric elements, not TLE text, so catalogue numbers above 99999
    are supported. OMM units: MEAN_MOTION rev/day; MEAN_MOTION_DOT and
    MEAN_MOTION_DDOT as in the TLE fields (rev/day^2, rev/day^3); angles in
    degrees; EPOCH UTC ISO-8601. Orekit expects rad/s, rad/s^2, rad/s^3, radians.
    """
    from org.orekit.propagation.analytical.tle import TLE
    from org.orekit.time import AbsoluteDate
    from jpype import JChar
    import math
    two_pi, day = 2.0 * math.pi, 86400.0
    obj_id = str(omm.get("OBJECT_ID", "") or "")            # e.g. "1998-067A"
    try:
        launch_year, rest = int(obj_id[:4]), obj_id[5:]
        launch_number = int("".join(ch for ch in rest if ch.isdigit()) or 0)
        launch_piece = "".join(ch for ch in rest if ch.isalpha()) or "A"
    except ValueError:
        launch_year, launch_number, launch_piece = 2000, 0, "A"
    return TLE(int(omm["NORAD_CAT_ID"]), JChar(str(omm.get("CLASSIFICATION_TYPE", "U"))[0]),
               launch_year, launch_number, launch_piece,
               int(omm.get("EPHEMERIS_TYPE", 0)), int(omm.get("ELEMENT_SET_NO", 999)),
               AbsoluteDate(str(omm["EPOCH"]).replace("Z", ""), utc),
               float(omm["MEAN_MOTION"]) * two_pi / day,
               # TLE/OMM derivative fields hold n-dot/2 and n-ddot/6; Orekit stores
               # the derivatives themselves (verified against Orekit's text parser).
               2.0 * float(omm["MEAN_MOTION_DOT"]) * two_pi / day ** 2,
               6.0 * float(omm["MEAN_MOTION_DDOT"]) * two_pi / day ** 3,
               float(omm["ECCENTRICITY"]), math.radians(float(omm["INCLINATION"])),
               math.radians(float(omm["ARG_OF_PERICENTER"])), math.radians(float(omm["RA_OF_ASC_NODE"])),
               math.radians(float(omm["MEAN_ANOMALY"])),
               int(omm.get("REV_AT_EPOCH", 0)), float(omm["BSTAR"]), utc)


def load_gp_json(groups, cache_dir, max_age_hours=TLE_CACHE_MAX_AGE_HOURS):
    """OMM records for CelesTrak GROUPs, cached per group; de-duplicated by
    NORAD_CAT_ID across groups (a satellite may be listed in more than one)."""
    by_id = {}
    for g in groups:
        path = os.path.join(cache_dir, f"arf_gp_{g}.json")
        recs = None
        if os.path.exists(path):
            with open(path) as f:
                cache = json.load(f)
            if (time.time() - cache["fetched_at"]) / 3600.0 < max_age_hours:
                recs = cache["records"]
                logger.info(f"Using cached GP JSON for GROUP={g}: {len(recs)} records.")
        if recs is None:
            url = f"https://celestrak.org/NORAD/elements/gp.php?GROUP={g}&FORMAT=json"
            logger.info(f"Fetching {url}")
            req = urllib.request.Request(url, headers={"User-Agent": "ARF-Research-Pipeline/1.0"})
            with urllib.request.urlopen(req, timeout=60) as resp:
                recs = json.loads(resp.read().decode("utf-8"))
            with open(path, "w") as f:
                json.dump({"fetched_at": time.time(), "url": url, "records": recs}, f)
        for r in recs:
            r = dict(r, _GROUP=g)
            by_id.setdefault(int(r["NORAD_CAT_ID"]), r)
    return list(by_id.values())


def load_eo_gp_history(live_records, epoch_iso, cache_dir, lookback_days=7.0, base_url="https://www.space-track.org"):
    """
    Orbital elements of a real fleet AS OF epoch_iso, for retrospective runs.
    Satellites: those in the current CelesTrak list (live_records) whose name
    matches a MISSION_TABLE entry. Elements: Space-Track gp_history, newest
    element set per satellite with lookback_days <= age <= 0 before epoch_iso
    (no element set from after the epoch is used). The CelesTrak name is kept,
    so mission matching is identical to live runs. Cached per fleet and epoch.
    Satellites launched after epoch_iso have no elements and drop out;
    satellites retired since then are not in the current list (stated limit).
    """
    from datetime import datetime, timedelta
    names = {int(r["NORAD_CAT_ID"]): str(r.get("OBJECT_NAME", "")).strip() for r in live_records}
    groups = {int(r["NORAD_CAT_ID"]): r.get("_GROUP") for r in live_records}
    ids = sorted(i for i, n in names.items() if _match_mission(n) is not None)
    end = datetime.fromisoformat(epoch_iso.replace("Z", ""))
    start = end - timedelta(days=lookback_days)
    path = os.path.join(cache_dir, f"arf_gp_history_{FLEET_TAG}_{end:%Y%m%dT%H%M%S}.json")
    globals()["GP_HISTORY_PATH"] = path                   # recorded in the manifest (Cell 9)
    if os.path.exists(path):
        with open(path) as f:
            recs = json.load(f)["records"]
        logger.info(f"Using cached Space-Track gp_history for {len(ids)} satellites (epoch <= {epoch_iso}).")
    else:
        if not ids:
            raise ConfigurationError("No satellite in the current CelesTrak list matches a MISSION_TABLE entry.")
        username, password = _get_spacetrack_credentials()
        opener = _spacetrack_login(username, password)
        recs = []
        for i in range(0, len(ids), 100):                  # keep each URL short
            chunk = ",".join(str(x) for x in ids[i:i + 100])
            q = (f"/basicspacedata/query/class/gp_history/NORAD_CAT_ID/{chunk}/EPOCH/"
                 f"{start:%Y-%m-%dT%H:%M:%S}--{end:%Y-%m-%dT%H:%M:%S}/orderby/EPOCH%20desc/format/json")
            req = urllib.request.Request(base_url + q, headers={"User-Agent": "ARF-Research-Pipeline/1.0"})
            with opener.open(req, timeout=120) as resp:
                part = json.loads(resp.read().decode("utf-8"))
            if isinstance(part, dict) and "error" in part:
                raise RuntimeError(f"Space-Track gp_history error: {part}")
            recs.extend(part)
        with open(path, "w") as f:
            json.dump({"fetched_at": time.time(), "epoch": epoch_iso, "norad_ids": ids, "records": recs}, f)
    latest = {}
    for r in recs:
        nid = int(r["NORAD_CAT_ID"])
        ep = _omm_epoch(r)
        if ep.replace(tzinfo=None) > end:                  # defensive: never use a later element set
            continue
        if nid not in latest or ep > _omm_epoch(latest[nid]):
            latest[nid] = r
    out = [dict(r, OBJECT_NAME=names.get(nid, r.get("OBJECT_NAME", "")), _GROUP=groups.get(nid))
           for nid, r in sorted(latest.items())]
    logger.info(f"Retrospective elements at {epoch_iso}: {len(out)} of {len(ids)} sourced satellites have an "
                f"element set within {lookback_days:g} days before the epoch.")
    return out


def _omm_orbit_km(omm, mu=398600.4418, re=6378.137):
    """(perigee_alt_km, apogee_alt_km) from OMM mean motion (rev/day) and eccentricity."""
    import math
    n = float(omm["MEAN_MOTION"]) * 2.0 * math.pi / 86400.0
    a = (mu / n ** 2) ** (1.0 / 3.0)
    e = float(omm["ECCENTRICITY"])
    return a * (1.0 - e) - re, a * (1.0 + e) - re


def swath_from_fov_km(altitude_km, fov_deg, re=6378.137):
    """Ground swath (km) of a nadir-pointing sensor with full field of view
    fov_deg at altitude_km over a sphere of radius re: twice the Earth-central
    angle to the edge of the field, lambda = arcsin((re+h)/re * sin(fov/2)) - fov/2."""
    import math
    eta = math.radians(fov_deg / 2.0)
    s = (re + altitude_km) / re * math.sin(eta)
    if s >= 1.0:
        raise ValueError(f"field of view {fov_deg} deg reaches beyond the horizon at {altitude_km} km")
    return 2.0 * re * (math.asin(s) - eta)


def _omm_epoch(omm):
    from datetime import datetime, timezone
    return datetime.fromisoformat(str(omm["EPOCH"]).replace("Z", "")).replace(tzinfo=timezone.utc)


def _match_mission(name):
    for m in MISSION_TABLE:
        if re.match(m["name_pattern"], str(name).strip(), flags=re.IGNORECASE):
            return m
    return None


def build_eo_constellation(omm_records, reference_iso, min_perigee_km=300.0, max_age_days=7.0, min_age_days=0.0,
                           exclude_names=()):
    """
    Every satellite that (1) matches a MISSION_TABLE entry, (2) has a perigee of
    at least min_perigee_km and (3) an element set at most max_age_days older
    than the reference epoch (and at most -min_age_days newer: 0 by default, -2
    when Cell 0 sets a live window start, v1.5.1). No sampling: N is the whole
    operational fleet. exclude_names (v1.5.2): OBJECT_NAMEs left out on
    purpose (EXCLUDE_SATELLITES, Cell 0), counted in stats. Returns
    (constellation, stats); stats lists unmatched names, never guessed.
    """
    import collections, jpype
    from datetime import datetime, timezone
    from org.orekit.time import TimeScalesFactory
    utc = TimeScalesFactory.getUTC()
    ref = datetime.fromisoformat(reference_iso).replace(tzinfo=timezone.utc)
    excl = {str(n).strip().upper() for n in exclude_names}
    stats = {"fetched": len(omm_records), "malformed": 0, "stale_or_future": 0, "low_perigee": 0,
             "high_apogee": 0, "unmatched_mission": 0, "excluded_by_config": 0, "operational": 0,
             "criteria": {"min_perigee_km": min_perigee_km, "max_elset_age_days": max_age_days,
                          "min_elset_age_days": min_age_days, "exclude_satellites": sorted(excl)}, "seed": None}
    unmatched, per_mission, constellation, seen_names = collections.Counter(), collections.Counter(), [], set()
    unmatched_names, excluded_found = [], []
    for omm in sorted(omm_records, key=lambda r: int(r["NORAD_CAT_ID"])):
        name = str(omm.get("OBJECT_NAME", "")).strip()
        m = _match_mission(name)
        if m is None:
            stats["unmatched_mission"] += 1
            unmatched_names.append(name)
            unmatched[re.sub(r"[\s\-]*\d+[A-Z]?$", "", name) or name] += 1
            continue
        if name.upper() in excl:
            stats["excluded_by_config"] += 1
            excluded_found.append(name)
            continue
        try:
            age = (ref - _omm_epoch(omm)).total_seconds() / 86400.0
            perigee, apogee = _omm_orbit_km(omm)
            tle = omm_to_tle(omm, utc)
        except (KeyError, ValueError, TypeError, ZeroDivisionError, jpype.JException):
            # Bad or missing fields in the record itself. Programming errors
            # (NameError etc.) are deliberately NOT caught, so they surface.
            stats["malformed"] += 1
            continue
        if not min_age_days <= age <= max_age_days:
            stats["stale_or_future"] += 1
            continue
        if perigee < min_perigee_km:
            stats["low_perigee"] += 1
            continue
        sid = name if name not in seen_names else f"{name} [{int(omm['NORAD_CAT_ID'])}]"
        seen_names.add(sid)
        per_mission[m["mission"]] += 1
        if "swath_km" in m:
            swath, fov = float(m["swath_km"]["value"]), None
        else:      # field of view given: swath at this satellite's mean altitude
            fov = float(m["fov_deg"]["value"])
            swath = swath_from_fov_km(0.5 * (perigee + apogee), fov)
        constellation.append({"id": sid, "tle": tle, "propagator": TLEPropagator.selectExtrapolator(tle),
                              "norad_id": int(omm["NORAD_CAT_ID"]), "group": omm.get("_GROUP"),
                              "mission": m["mission"], "swath_km": swath, "fov_deg": fov,
                              "min_sun_elev_deg": float(m["min_sun_elev_deg"]["value"])})
    stats["operational"] = len(constellation)
    stats["per_mission"] = dict(per_mission)
    stats["unmatched_families"] = dict(unmatched.most_common(40))
    stats["unmatched_names"] = sorted(set(unmatched_names))[:200]
    stats["excluded_names"] = sorted(excluded_found)
    stats["exclusions_not_found"] = sorted(excl - {n.upper() for n in excluded_found})
    return constellation, stats


def _parse_norad_id(line1):
    try:
        return int(line1[2:7])
    except (ValueError, IndexError):
        return None

# -----------------------------------------------------------------------------
# Historical runs: satellites OPERATIONAL AT THE ONSET, seeded random sample
# -----------------------------------------------------------------------------
# gp_history returns every Starlink object with an element set in the lookback
# window, including satellites that are deorbiting or no longer tracked.
# Taking the first N by catalogue number would pick the OLDEST satellites
# (2019-2022 launches), many of them decaying. Instead: keep satellites whose
# latest element set describes an operational orbit at the onset, then draw a
# reproducible random sample of N from them.
#   perigee >= 300 km : excludes deorbiting / re-entering objects
#   apogee  <= 700 km : excludes anomalous or non-Starlink-shell orbits
#   element set <= 7 days older than the onset : excludes stale / untracked
# Orbit size comes from the element set's own mean motion and eccentricity
# (semi-major axis a = (mu / n^2)^(1/3)), not from a single propagated point.
# Checked against Orekit's SGP4: propagated altitude dips ~7 km below the
# mean-element perigee (short-period J2 terms), so the 300 km bound admits
# orbits whose true minimum altitude is down to ~293 km.

OPERATIONAL_CRITERIA = {"min_perigee_km": 300.0, "max_apogee_km": 700.0, "max_elset_age_days": 7.0}


def _elset_epoch_utc(line1):
    """TLE line 1 epoch (cols 19-32, YYDDD.DDDDDDDD) as a UTC datetime."""
    from datetime import datetime, timedelta, timezone
    yy = int(line1[18:20])
    year = 2000 + yy if yy < 57 else 1900 + yy
    return datetime(year, 1, 1, tzinfo=timezone.utc) + timedelta(days=float(line1[20:32]) - 1.0)


def _elset_orbit_km(line2, mu=398600.4418, re=6378.137):
    """(perigee_alt_km, apogee_alt_km) from TLE line 2: eccentricity (cols 27-33,
    implied leading decimal point) and mean motion (cols 53-63, rev/day)."""
    import math
    e = float("0." + line2[26:33].strip())
    n = float(line2[52:63]) * 2.0 * math.pi / 86400.0          # rad/s
    a = (mu / n ** 2) ** (1.0 / 3.0)
    return a * (1.0 - e) - re, a * (1.0 + e) - re


def select_operational_sample(records, onset_iso, seed, criteria=OPERATIONAL_CRITERIA):
    """
    Filter records to satellites operational at onset_iso (see criteria), put
    them in catalogue-number order, and shuffle with numpy's seeded generator.
    Returns (shuffled_records, stats); the first N are the sample.
    """
    from datetime import datetime, timezone
    onset = datetime.fromisoformat(onset_iso.replace("Z", "")).replace(tzinfo=timezone.utc)
    stats = {"fetched": len(records), "malformed": 0, "stale_or_future": 0, "low_perigee": 0,
             "high_apogee": 0, "operational": 0, "criteria": dict(criteria), "seed": seed}
    keep = []
    for rec in records:
        name, l1, l2 = rec
        try:
            age_days = (onset - _elset_epoch_utc(l1)).total_seconds() / 86400.0
            perigee, apogee = _elset_orbit_km(l2)
            nid = int(l1[2:7])
        except (ValueError, IndexError, ZeroDivisionError):
            stats["malformed"] += 1
            continue
        if not (0.0 <= age_days <= criteria["max_elset_age_days"]):
            stats["stale_or_future"] += 1
        elif perigee < criteria["min_perigee_km"]:
            stats["low_perigee"] += 1
        elif apogee > criteria["max_apogee_km"]:
            stats["high_apogee"] += 1
        else:
            keep.append((nid, rec))
    keep.sort(key=lambda x: x[0])
    stats["operational"] = len(keep)
    order = np.random.default_rng(seed).permutation(len(keep))
    return [keep[i][1] for i in order], stats


def select_pinned_satellites(records, active_n, cache_path):
    if os.path.exists(cache_path):
        with open(cache_path) as f:
            pinned_ids = json.load(f)["norad_ids"]
        by_id = {nid: (name, l1, l2) for name, l1, l2 in records if (nid := _parse_norad_id(l1)) is not None}

        selected = [by_id[nid] for nid in pinned_ids if nid in by_id]
        if len(selected) < len(pinned_ids):
            logger.warning(f"Only {len(selected)}/{len(pinned_ids)} pinned satellites found in current fetch.")
        logger.info(f"Loaded {len(selected)} pinned satellites from {cache_path}.")
        return selected
    else:
        chosen = records[:active_n]
        norad_ids = [nid for name, l1, l2 in chosen if (nid := _parse_norad_id(l1)) is not None]
        try:
            with open(cache_path, "w") as f:
                json.dump({"active_n": active_n, "norad_ids": norad_ids}, f, indent=2)
        except Exception as e:
            logger.warning(f"Could not write selection cache: {e}")
        logger.info(f"Pinned {len(norad_ids)} satellites to {cache_path} for deterministic re-runs.")
        return chosen

def _newest_elset_epoch_iso(records):
    """Newest element-set epoch among the fetched records (UTC, ISO 8601)."""
    epochs = []
    for _, l1, _ in records:
        try:
            epochs.append(_elset_epoch_utc(l1))
        except (ValueError, IndexError):
            continue
    if not epochs:
        raise RuntimeError("No parseable element sets in the TLE fetch.")
    return max(epochs).strftime("%Y-%m-%dT%H:%M:%S.%f")


if FLEET == "starlink":
    # Reference epoch for both the operational filter and the simulation start:
    #   historical runs -> the disaster onset (ROI_DEFINITIONS historical_epoch)
    #   live runs       -> the newest element-set epoch in the fetch (effectively
    #                      the fetch time; reproducible because the fetch is cached)
    _reference_epoch = (_historical_epoch if TLE_SOURCE == "spacetrack_history"
                        else _newest_elset_epoch_iso(raw_records))
    REFERENCE_EPOCH_SOURCE = ("ROI historical_epoch (Space-Track gp_history)" if TLE_SOURCE == "spacetrack_history"
                              else "newest element set in the Starlink fetch (live)")
    GP_INPUT_PATHS = [TLE_RAW_CACHE_PATH]

    # Same selection method for every run: satellites operational at the reference
    # epoch, seeded random sample of N. Pin file named for this method, so a pin
    # written by the earlier first-N-by-catalogue-number method is never reused.
    SELECTION_CACHE_PATH = os.path.join(
        ARF_CACHE_DIR, f"arf_selection_{ACTIVE_ROI}_N{ACTIVE_N}_opsample_seed{RANDOM_SEED}.json")
    _candidates, _op_stats = select_operational_sample(raw_records, _reference_epoch, RANDOM_SEED)
    logger.info(f"Operational at {_reference_epoch} UTC: {_op_stats['operational']} of {_op_stats['fetched']} fetched "
                f"(excluded: {_op_stats['low_perigee']} low perigee, {_op_stats['high_apogee']} high apogee, "
                f"{_op_stats['stale_or_future']} stale, {_op_stats['malformed']} malformed). "
                f"Seeded random sample (seed={RANDOM_SEED}) of up to N={ACTIVE_N}.")
    if _op_stats["operational"] < ACTIVE_N:
        logger.warning(f"Only {_op_stats['operational']} operational satellites available; using all of them.")

    selected_records = select_pinned_satellites(_candidates, ACTIVE_N, SELECTION_CACHE_PATH)

    # =============================================================================
    # 3. OREKIT SGP4 PROPAGATOR CONSTRUCTION
    # =============================================================================

    constellation = []
    for name, l1, l2 in selected_records:
        try:
            tle = TLE(l1, l2)
            constellation.append({
                "id": name,
                "tle": tle,
                "propagator": TLEPropagator.selectExtrapolator(tle)
            })
        except Exception as e:
            logger.warning(f"Skipping {name}: {e}")

else:
    # Real Earth-observation fleet: all operational satellites with a sourced
    # mission entry (section 2B). Reference epoch = newest element set (live),
    # or ANALYSIS_EPOCH with elements as of then (retrospective).
    _gp_records = load_gp_json(CELESTRAK_GROUPS[FLEET], ARF_CACHE_DIR)
    GP_INPUT_PATHS = [os.path.join(ARF_CACHE_DIR, f"arf_gp_{g}.json") for g in CELESTRAK_GROUPS[FLEET]]
    _min_age_days = 0.0
    if globals().get("ANALYSIS_EPOCH"):
        _gp_records = load_eo_gp_history(_gp_records, ANALYSIS_EPOCH, ARF_CACHE_DIR)
        _reference_epoch = ANALYSIS_EPOCH
        REFERENCE_EPOCH_SOURCE = "analysis_epoch (retrospective: elements as of then, Space-Track gp_history)"
        GP_INPUT_PATHS.append(globals().get("GP_HISTORY_PATH"))
    elif globals().get("REFERENCE_EPOCH_UTC"):
        _reference_epoch = REFERENCE_EPOCH_UTC
        REFERENCE_EPOCH_SOURCE = "reference_epoch_utc (set in Cell 0; live elements)"
        _min_age_days = -2.0          # live elements may be up to 2 days newer than a set start
    else:
        # v1.5.1: the newest element set over BOTH real-fleet groups, whichever
        # fleet is run, so the fleets of one session share their window start.
        _ref_groups = sorted({g for gs in CELESTRAK_GROUPS.values() for g in gs})
        _ref_records = load_gp_json(_ref_groups, ARF_CACHE_DIR)
        GP_INPUT_PATHS = sorted(set(GP_INPUT_PATHS) | {os.path.join(ARF_CACHE_DIR, f"arf_gp_{g}.json")
                                                       for g in _ref_groups})
        _reference_epoch = max(_omm_epoch(r) for r in _ref_records).strftime("%Y-%m-%dT%H:%M:%S.%f")
        REFERENCE_EPOCH_SOURCE = f"newest element set over CelesTrak groups {_ref_groups} (live)"
    constellation, _op_stats = build_eo_constellation(_gp_records, _reference_epoch, min_age_days=_min_age_days,
                                                      exclude_names=globals().get("EXCLUDE_SATELLITES", ()))
    if _op_stats["exclusions_not_found"]:
        logger.warning(f"EXCLUDE_SATELLITES names not found in the fleet: {_op_stats['exclusions_not_found']} "
                       f"(names are matched exactly against CelesTrak OBJECT_NAME).")
    TLE_RAW_CACHE_PATH = os.path.join(ARF_CACHE_DIR, f"arf_gp_{CELESTRAK_GROUPS[FLEET][0]}.json")
    SELECTION_CACHE_PATH = os.path.join(ARF_CACHE_DIR, f"arf_fleet_{FLEET_TAG}.json")
    with open(SELECTION_CACHE_PATH, "w") as f:
        json.dump({"fleet": FLEET, "reference_epoch": _reference_epoch,
                   "satellites": [{"id": s["id"], "norad_id": s["norad_id"], "mission": s["mission"],
                                   "swath_km": round(s["swath_km"], 3), "fov_deg": s.get("fov_deg"),
                                   "min_sun_elev_deg": s["min_sun_elev_deg"]}
                                  for s in constellation]}, f, indent=2)
    logger.info(f"Fleet {FLEET!r}: {_op_stats['operational']} of {_op_stats['fetched']} satellites used "
                f"{_op_stats['per_mission']}; excluded {_op_stats['unmatched_mission']} with no sourced mission "
                f"entry, {_op_stats['low_perigee']} low perigee, {_op_stats['stale_or_future']} stale, "
                f"{_op_stats['malformed']} malformed"
                + (f", {_op_stats['excluded_by_config']} left out by EXCLUDE_SATELLITES "
                   f"({', '.join(_op_stats['excluded_names'])})" if _op_stats['excluded_by_config'] else "") + ".")
    if _op_stats["unmatched_families"]:
        logger.info(f"Excluded (no sourced swath -- add to MISSION_TABLE with a source to include): "
                    f"{_op_stats['unmatched_families']}; names: {_op_stats['unmatched_names'][:60]}")
    if not constellation:
        raise ConfigurationError(f"No satellite in fleet {FLEET!r} matched a MISSION_TABLE entry.")

real_epochs = [sat["tle"].getDate() for sat in constellation]
# The simulation window starts at the reference epoch: the onset for a
# disaster scenario (so only acquisitions after the event count), the newest
# element set for a live run. Each satellite is propagated from its own
# element-set epoch (at most 7 days earlier) to this instant.
from org.orekit.time import AbsoluteDate, TimeScalesFactory
sim_epoch = AbsoluteDate(_reference_epoch, TimeScalesFactory.getUTC())

# =============================================================================
# 4. COMPUTE DYNAMIC PARAMETERS (Altitude & Orbital Period)
# =============================================================================

mu_earth = 398600.4418
Re_km = 6378.137

pos_list = []
for sat in constellation:
    pv = sat["propagator"].getPVCoordinates(sim_epoch, itrf)
    r = np.sqrt(pv.getPosition().getX()**2 + pv.getPosition().getY()**2 + pv.getPosition().getZ()**2) / 1000.0
    pos_list.append(r - Re_km)

CONSTELLATION_ALTITUDE_KM = float(np.median(pos_list))
T_MAX_WAIT_MINUTES = 2 * np.pi * np.sqrt((Re_km + CONSTELLATION_ALTITUDE_KM)**3 / mu_earth) / 60.0

# =============================================================================
# SUMMARY & PERSISTENCE
# =============================================================================

_summary = {
    "n_satellites": len(constellation),
    "median_alt_km": CONSTELLATION_ALTITUDE_KM,
    "t_max_wait_min": T_MAX_WAIT_MINUTES,
    "sim_epoch": str(sim_epoch),
    "tle_source": TLE_SOURCE,
    "analysis_epoch": globals().get("ANALYSIS_EPOCH"),
    "reference_epoch": _reference_epoch,
    "reference_epoch_source": REFERENCE_EPOCH_SOURCE,
    "fleet": FLEET,
    "satellite_selection": dict(method=("operational at reference epoch, seeded random sample" if FLEET == "starlink"
                                        else "all operational satellites with a sourced MISSION_TABLE entry"),
                                **_op_stats),
    "mission_table": MISSION_TABLE if FLEET != "starlink" else None,
    "acquisition_exclusions": globals().get("ACQUISITION_EXCLUSIONS") if FLEET != "starlink" else None,   # v1.5.2
}

print("\n" + "=" * 60)
print("  CELL 2: TLE Acquisition & SGP4 Dynamics")
print("=" * 60)
# v1.5.2: real fleets read CelesTrak GP data (OMM JSON) or Space-Track history,
# and use every operational satellite -- "(Pinned)" described the Starlink
# proxy's seeded sample only.
print(f"  Orbital elements:          " + (TLE_SOURCE if FLEET == "starlink" else
                                          ("Space-Track gp_history (as of the analysis epoch)"
                                           if globals().get("ANALYSIS_EPOCH") else "CelesTrak GP data (OMM JSON)")))
print(f"  Window start:              {_reference_epoch} UTC [{REFERENCE_EPOCH_SOURCE}]")
print(f"  Simulation Epoch:          {sim_epoch}")
print(f"  Constellation Size (N):    {len(constellation)} "
      + ("(seeded sample, pinned)" if FLEET == "starlink" else "(every operational satellite with a sourced swath)"))
if _op_stats:
    print(f"  Operational at reference:  {_op_stats['operational']} of {_op_stats['fetched']} fetched "
          + (f"(seeded sample, seed={RANDOM_SEED})" if FLEET == "starlink" else f"(fleet {FLEET}: {_op_stats.get('per_mission')})"))
print(f"  Fleet Median Altitude:     {CONSTELLATION_ALTITUDE_KM:.2f} km")
print(f"  Derived Orbital Period:    {T_MAX_WAIT_MINUTES:.2f} min")
print("=" * 60)

try:
    with open(os.path.join(_out_dir, 'arf_cell2_summary.json'), 'w') as f:
        json.dump(_summary, f, indent=2)
    logger.info("Summary exported to arf_cell2_summary.json")
except Exception as e:
    logger.warning(f"Could not save Cell 2 summary: {e}")

print("\n=== CELL 2 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 3: SPATIAL VALIDATION & GROUND TRACK VISUALISATION
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Study-area map of the active ROI, drawn to journal standard:
#     (a) Regional setting -- the ROI, the ground-station network,
#         international borders and sample ground tracks of the fleet.
#     (b) The ROI itself -- its H3 tiling (the grid the mosaic is built on)
#         and its official boundary, with a scale bar and a north arrow.
#   Panel (b) is fitted to the ROI, so the mosaic is legible at every ROI
#   size, from a 32-tile reserve to an 18,000-tile complex; ground stations
#   far from the ROI appear in panel (a) only. Large tile sets get a zoom
#   inset showing the individual hexagons. Both panels are in geographic
#   coordinates (WGS 84), drawn with the equirectangular aspect of their
#   centre latitude (1/cos(latitude)), so shapes and distances are not
#   stretched.
#
#   The map helpers defined here are reused by Cell 7's mosaic map.
#
# Research Proposal Mapping:
#   - Section 3.2.1: Geodesic Grid Model (Uber H3 Tiles)
#   - Section 3.2.3: Ground Station Telemetry Anchors
#   - Section 4.1: Simulation Environment Validation
#
# Dependencies:
#   - Cell 0 (ACTIVE_N, ACTIVE_ROI, OUT_DIR, FLEET)
#   - Cell 1 (target_tiles, GROUND_STATIONS, ROI_OUTLINE_GEOMS, itrf, earth)
#   - Cell 2 (constellation, sim_epoch)
#
# Outputs (names unchanged):
#   - arf_coverage_map.png (300 dpi)
#   - arf_coverage_map.pdf (vector; tile layers above MAP_RASTER_TILES tiles
#     are embedded as a 300 dpi image to keep the file small)
#   - arf_cell3_metadata.json (reproducibility metadata and a draft caption)
#
# Data drawn on the map:
#   - ROI boundaries: World Database on Protected Areas (UNEP-WCMC and IUCN).
#     The WDPA/WDOECM Manual 1.6 (section 5.3, p. 30) asks that its citation
#     be reproduced in any publication using the WDPA; the draft caption in
#     arf_cell3_metadata.json carries it. Boundaries are drawn on the figure
#     but never written to the output files (the licence forbids
#     redistributing the data).
#   - International borders: Natural Earth 1:50m Admin 0 - Countries, v5.1.2,
#     public domain ("No permission is needed to use Natural Earth",
#     https://www.naturalearthdata.com/about/terms-of-use/). Downloaded once
#     and cached; if it cannot be downloaded the map is drawn without borders.
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

"""
ARF Spatial Validation Visualisation Module
============================================

1. **ROI mosaic grid**: the H3 cells of the ROI, filled as one dissolved
   shape (no seams between cells) with the cell outlines drawn on top while
   they are larger than a pixel; a zoom inset shows them for large ROIs.
2. **Ground stations**: the downlink stations of Cell 1, with labels.
3. **Ground tracks**: SGP4 ground tracks of a deterministic sample of the
   fleet over the first MAP_TRACK_HOURS hours, keeping the first
   MAP_TRACK_MAX_SEGMENTS crossings of the regional frame (arrows give the
   direction of flight).
"""

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 3 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["3"] = _ARF_CELL_VERSION

import os
import sys
import json
import math
import hashlib
import logging
from datetime import datetime, timezone
from typing import Dict, List, Tuple, Set, Optional, Any

import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.transforms
import matplotlib.lines as mlines
import matplotlib.patheffects as patheffects
from matplotlib.collections import PolyCollection, LineCollection
from matplotlib.path import Path as MplPath
import h3
from org.hipparchus.util import FastMath

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logger = logging.getLogger("arf.cell3")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()

# =============================================================================
# MATPLOTLIB PUBLICATION CONFIGURATION
# =============================================================================
# Times New Roman where installed, otherwise a metric-compatible Times clone
# (e.g. Liberation Serif); DejaVu Serif is the last resort. pdf.fonttype 42
# embeds the fonts in PDFs as TrueType (Type 42) instead of matplotlib's
# default Type 3, so figure text stays embedded, selectable text.

ARF_SERIF_FONTS = ['Times New Roman', 'Liberation Serif', 'Nimbus Roman', 'FreeSerif', 'DejaVu Serif']

plt.rcParams.update({
    'font.family': 'serif',
    'font.serif': ARF_SERIF_FONTS,
    'font.size': 10,
    'mathtext.fontset': 'stix',
    'axes.linewidth': 1.0,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,
    'legend.fontsize': 8,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',
    'figure.dpi': 100,
    'figure.facecolor': 'white',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# =============================================================================
# MAP HELPERS (shared with Cell 7's mosaic map)
# =============================================================================

MAP_FIG_WIDTH_IN = 7.0          # double-column figure width (about 178 mm)
MAP_ASPECT_RANGE = (0.8, 1.5)   # panel width/height is kept inside this range
MAP_RASTER_TILES = 3000         # tile layers with more tiles are rasterised in the PDF
MAP_EDGE_TILES = 3000           # cell outlines are drawn up to this many tiles (beyond: sub-pixel)
MAP_ZOOM_TILES = 1500           # above this many tiles, a zoom inset shows the hexagons
MAP_TRACK_HOURS = 24            # ground tracks: hours from the analysis start ...
MAP_TRACK_MAX_SATS = 40         # ... for an evenly spaced sample of this many satellites ...
MAP_TRACK_MAX_SEGMENTS = 8      # ... keeping the first crossings of the regional frame

MAP_COLORS = {
    "tile": "#41ab5d",          # ROI cells
    "tile_edge": "white",
    "boundary": "#1a1a1a",
    "station": "#b2182b",
    "track": "#525252",
    "extent": "#1a1a1a",
    "land": "#f0efe9",
    "water": "#e4eef6",
    "border": "#a0a0a0",
    "country_label": "#808080",
}

# Map text sizes at the printed figure width (journals ask for >= 6-7 pt).
_MAP_RC = {
    "font.family": "serif",
    "font.serif": ARF_SERIF_FONTS,
    "mathtext.fontset": "stix",
    "font.size": 8,
    "axes.titlesize": 8.5,
    "axes.titleweight": "bold",
    "axes.labelsize": 8,
    "xtick.labelsize": 7,
    "ytick.labelsize": 7,
    "legend.fontsize": 7,
    "axes.linewidth": 0.6,
    "xtick.major.width": 0.5,
    "ytick.major.width": 0.5,
    "xtick.major.size": 2.5,
    "ytick.major.size": 2.5,
    "xtick.direction": "out",
    "ytick.direction": "out",
    "savefig.dpi": 300,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
}

_HALO = [patheffects.withStroke(linewidth=2.2, foreground="white")]

NATURAL_EARTH_VERSION = "v5.1.2"
NATURAL_EARTH_URL = ("https://raw.githubusercontent.com/nvkelso/natural-earth-vector/"
                     + NATURAL_EARTH_VERSION + "/50m_cultural/ne_50m_admin_0_countries.{ext}")
_NATURAL_EARTH_BASE = "ne_50m_admin_0_countries"
_country_shape_cache: Dict[str, Any] = {}


def _km_per_deg_lon(lat_deg: float) -> float:
    """Length of one degree of longitude along the parallel at lat_deg on the
    WGS 84 ellipsoid (a = 6378.137 km, e^2 = 6.69437999014e-3): (pi/180) N cos(lat)."""
    a_km, e2 = 6378.137, 6.69437999014e-3
    s = math.sin(math.radians(lat_deg))
    return math.radians(1.0) * a_km * math.cos(math.radians(lat_deg)) / math.sqrt(1.0 - e2 * s * s)


_KM_PER_DEG_LAT = 110.574       # WGS 84 meridian degree at the equator (shortest; used only for padding)


def _cos_lat(ext) -> float:
    return max(math.cos(math.radians(0.5 * (ext[2] + ext[3]))), 0.05)


def panel_aspect(ext) -> float:
    """Displayed width/height of a map panel with extent [lon0, lon1, lat0, lat1]."""
    return (ext[1] - ext[0]) * _cos_lat(ext) / (ext[3] - ext[2])


def pad_extent(ext, frac: float = 0.06, min_pad_km: float = 1.0):
    """Grow an extent on all sides by frac of its larger side (at least min_pad_km)."""
    lon0, lon1, lat0, lat1 = ext
    mid = 0.5 * (lat0 + lat1)
    w_km = (lon1 - lon0) * _km_per_deg_lon(mid)
    h_km = (lat1 - lat0) * _KM_PER_DEG_LAT
    pad_km = max(frac * max(w_km, h_km), min_pad_km)
    dlon = pad_km / max(_km_per_deg_lon(mid), 1e-6)
    dlat = pad_km / _KM_PER_DEG_LAT
    return [lon0 - dlon, lon1 + dlon, max(lat0 - dlat, -89.0), min(lat1 + dlat, 89.0)]


def fit_aspect(ext, aspect: float):
    """Grow the shorter side of an extent (about its centre) to the given displayed aspect."""
    lon0, lon1, lat0, lat1 = ext
    if panel_aspect(ext) < aspect:
        half = 0.5 * (lat1 - lat0) * aspect / _cos_lat(ext)
        cx = 0.5 * (lon0 + lon1)
        return [cx - half, cx + half, lat0, lat1]
    half = 0.5 * (lon1 - lon0) * _cos_lat(ext) / aspect
    cy = 0.5 * (lat0 + lat1)
    return [lon0, lon1, cy - half, cy + half]


def _ring_area(ring) -> float:
    x, y = ring[:, 0], ring[:, 1]
    return 0.5 * float(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1)))


def geo_polygons(geoms) -> List[Tuple[np.ndarray, List[np.ndarray]]]:
    """[(exterior, [holes])] as (lon, lat) arrays from GeoJSON-like Polygon /
    MultiPolygon dicts (pyshp __geo_interface__, h3.cells_to_geo)."""
    out = []
    for g in geoms or []:
        if not g:
            continue
        if g["type"] == "Polygon":
            polys = [g["coordinates"]]
        elif g["type"] == "MultiPolygon":
            polys = g["coordinates"]
        else:
            continue
        for poly in polys:
            rings = []
            for r in poly:
                if len(r) < 3:
                    continue
                r = np.asarray(r, dtype=float)[:, :2]
                if not np.allclose(r[0], r[-1]):          # close open rings (e.g. ROI corner lists)
                    r = np.vstack([r, r[:1]])
                rings.append(r)
            if rings:
                out.append((rings[0], rings[1:]))
    return out


def geo_lines(geoms) -> List[np.ndarray]:
    """Every ring (exteriors and holes) of the geometries, for outlines."""
    return [r for ext_ring, holes in geo_polygons(geoms) for r in [ext_ring] + list(holes)]


def geo_path(geoms) -> Optional[MplPath]:
    """One compound Path for filling (exteriors anticlockwise, holes clockwise)."""
    verts, codes = [], []
    for ext_ring, holes in geo_polygons(geoms):
        for i, ring in enumerate([ext_ring] + list(holes)):
            if (_ring_area(ring) > 0) != (i == 0):
                ring = ring[::-1]
            verts.extend(ring.tolist())
            verts.append(ring[0].tolist())
            codes.extend([MplPath.MOVETO] + [MplPath.LINETO] * (len(ring) - 1) + [MplPath.CLOSEPOLY])
    return MplPath(verts, codes) if verts else None


def tiles_outline(tiles) -> Optional[dict]:
    """Dissolved outline of a set of H3 cells (GeoJSON-like, lon/lat)."""
    tiles = list(tiles)
    return h3.cells_to_geo(tiles) if tiles else None


def tile_polygons(tiles) -> List[np.ndarray]:
    """Hexagon vertices (lon, lat) of each H3 cell, in the order given."""
    polys = []
    for t in tiles:
        p = np.array([(lng, lat) for lat, lng in h3.cell_to_boundary(t)], dtype=float)
        if np.ptp(p[:, 0]) > 180.0:                      # cell straddles the antimeridian
            p[:, 0] = np.where(p[:, 0] < 0.0, p[:, 0] + 360.0, p[:, 0])
        polys.append(p)
    return polys


def geo_bounds(geoms=None, points=()):
    """[lon0, lon1, lat0, lat1] of geometries and (lat, lon) points."""
    xs, ys = [], []
    for r in geo_lines(geoms):
        xs.append(r[:, 0])
        ys.append(r[:, 1])
    for lat, lon in points:
        xs.append([lon])
        ys.append([lat])
    x, y = np.concatenate(xs), np.concatenate(ys)
    return [float(x.min()), float(x.max()), float(y.min()), float(y.max())]


def roi_map_extent(tiles, outline_geoms=None, pad_frac: float = 0.06):
    """Map extent fitted to the ROI (its cells and boundary), padded, with the
    displayed aspect clamped to MAP_ASPECT_RANGE."""
    ext = pad_extent(geo_bounds([tiles_outline(tiles)] + list(outline_geoms or [])), pad_frac, min_pad_km=1.0)
    lo, hi = MAP_ASPECT_RANGE
    return fit_aspect(ext, min(max(panel_aspect(ext), lo), hi))


def regional_extent(detail_ext, stations, aspect: float, context: float = 3.0,
                    min_span_km: float = 900.0, pad_frac: float = 0.12):
    """Extent for the regional panel: the ROI panel and every ground station,
    padded, at least `context` times the ROI panel and min_span_km across,
    fitted to the same displayed aspect as the ROI panel."""
    xs = [detail_ext[0], detail_ext[1]] + [gs["lng"] for gs in stations.values()]
    ys = [detail_ext[2], detail_ext[3]] + [gs["lat"] for gs in stations.values()]
    ext = pad_extent([min(xs), max(xs), min(ys), max(ys)], pad_frac, min_pad_km=50.0)
    mid = 0.5 * (ext[2] + ext[3])
    need_lon = max(context * (detail_ext[1] - detail_ext[0]), min_span_km / max(_km_per_deg_lon(mid), 1e-6))
    need_lat = max(context * (detail_ext[3] - detail_ext[2]), min_span_km / _KM_PER_DEG_LAT)
    cx, cy = 0.5 * (ext[0] + ext[1]), 0.5 * (ext[2] + ext[3])
    half_lon = 0.5 * max(ext[1] - ext[0], need_lon)
    half_lat = 0.5 * max(ext[3] - ext[2], need_lat)
    ext = fit_aspect([cx - half_lon, cx + half_lon, cy - half_lat, cy + half_lat], aspect)
    if ext[2] < -85.0 or ext[3] > 85.0:                  # keep clear of the poles
        shift = (-85.0 - ext[2]) if ext[2] < -85.0 else (85.0 - ext[3])
        ext = [ext[0], ext[1], max(ext[2] + shift, -85.0), min(ext[3] + shift, 85.0)]
    return ext


_TICK_STEPS_DEG = (1 / 60, 2 / 60, 5 / 60, 10 / 60, 15 / 60, 20 / 60, 30 / 60, 1, 2, 5, 10, 15, 20, 30)


def _fmt_deg(v: float, step: float, pos: str, neg: str) -> str:
    """24°E, 24°30′E, 32°10′S ... (degrees, or degrees and minutes when step < 1°)."""
    hemi = pos if v > 1e-9 else (neg if v < -1e-9 else "")
    a = abs(v)
    if step >= 1.0 - 1e-9:
        return f"{a:.0f}°{hemi}"
    d = int(math.floor(a + 1e-9))
    m = int(round((a - d) * 60.0))
    if m == 60:
        d, m = d + 1, 0
    return f"{d}°{m:02d}′{hemi}" if m else f"{d}°{hemi}"


def _ticks(lo: float, hi: float, max_intervals: int) -> Tuple[np.ndarray, float]:
    """Graticule ticks at a 'nice' step, with at least two ticks inside [lo, hi]."""
    steps = [s for s in _TICK_STEPS_DEG if (hi - lo) / s <= max_intervals] or [30.0]
    step = steps[0]
    for s in reversed(_TICK_STEPS_DEG[:_TICK_STEPS_DEG.index(step) + 1] if step in _TICK_STEPS_DEG else ()):
        t = np.arange(math.ceil(lo / s - 1e-9) * s, hi + 1e-9, s)
        if t.size >= 2:
            return t, s
    return np.arange(math.ceil(lo / step - 1e-9) * step, hi + 1e-9, step), step


def setup_map_axes(ax, ext, max_intervals=(4, 4), water: Optional[str] = None, labels=(True, True)):
    """Limits, equirectangular aspect, graticule ticks with degree labels."""
    ax.set_xlim(ext[0], ext[1])
    ax.set_ylim(ext[2], ext[3])
    ax.set_aspect(1.0 / _cos_lat(ext), adjustable="box")
    xt, sx = _ticks(ext[0], ext[1], max_intervals[0])
    yt, sy = _ticks(ext[2], ext[3], max_intervals[1])
    ax.set_xticks(xt)
    ax.set_yticks(yt)
    ax.set_xticklabels([_fmt_deg(v, sx, "E", "W") for v in xt] if labels[0] else [])
    ax.set_yticklabels([_fmt_deg(v, sy, "N", "S") for v in yt] if labels[1] else [])
    ax.grid(True, color="0.5", linewidth=0.3, linestyle=(0, (2, 2)), alpha=0.6)
    ax.set_axisbelow(True)
    if water:
        ax.set_facecolor(water)
    for spine in ax.spines.values():
        spine.set_linewidth(0.6)


def _nice_length(target: float) -> float:
    for v in (1000, 500, 250, 200, 100, 50, 25, 20, 10, 5, 2, 1, 0.5, 0.2, 0.1):
        if v <= target:
            return float(v)
    return 0.1


_SCALE_BAR_BOX = (0.36, 0.11)    # axes-fraction footprint of a scale bar with its labels
_NORTH_BOX = (0.09, 0.17)        # ... of the north arrow

# Places for map furniture: the four corners and the four edge midpoints.
MAP_ANCHORS = ("lower left", "lower right", "upper left", "upper right",
               "lower center", "upper center", "center left", "center right")
MAP_FURNITURE_MAX_GROW = 0.5     # the map frame may grow by up to this share of its span to clear the ROI


def _anchor_origin(anchor: str, size) -> Tuple[float, float]:
    """Lower-left (axes fraction) of a box of `size` at the named anchor, 2% from
    the frame: a corner ('lower left' ...) or an edge midpoint ('lower center',
    'center left' ...)."""
    sx, sy = size
    x = 0.02 if "left" in anchor else (0.98 - sx if "right" in anchor else 0.5 - 0.5 * sx)
    y = 0.02 if "lower" in anchor else (0.98 - sy if "upper" in anchor else 0.5 - 0.5 * sy)
    return x, y


_corner_origin = _anchor_origin      # name used up to v1.3.1


def map_obstacle(tiles, outline_geoms=None):
    """What map furniture must not cover, as one shapely geometry in lon/lat:
    the ROI's cells and its boundary polygons (None if there is nothing)."""
    from shapely.geometry import shape
    from shapely.ops import unary_union
    parts = []
    g = tiles_outline(tiles)
    if g:
        parts.append(shape(g).buffer(0))
    for og in outline_geoms or []:
        try:
            parts.append(shape(og).buffer(0))
        except Exception:
            pass
    return unary_union(parts) if parts else None


def _grow_extent(ext, side: str, g: float):
    """Extent grown on one side by g times its span on that axis, then fitted
    back into MAP_ASPECT_RANGE (growing the other axis about its centre if needed)."""
    lon0, lon1, lat0, lat1 = ext
    if side == "bottom":
        lat0 = max(lat0 - g * (lat1 - lat0), -89.0)
    elif side == "top":
        lat1 = min(lat1 + g * (lat1 - lat0), 89.0)
    elif side == "left":
        lon0 -= g * (lon1 - lon0)
    elif side == "right":
        lon1 += g * (lon1 - lon0)
    new = [lon0, lon1, lat0, lat1]
    lo, hi = MAP_ASPECT_RANGE
    return fit_aspect(new, min(max(panel_aspect(new), lo), hi))


def plan_furniture(ext, obstacle, items, points=(), max_grow: float = MAP_FURNITURE_MAX_GROW,
                   clearance: float = 0.015):
    """Positions for map furniture that hide none of the ROI. items: [(name,
    size in axes fraction, preferred anchors)]. Every combination of
    MAP_ANCHORS with no two boxes overlapping is scored by the area of
    `obstacle` (lon/lat geometry) plus small discs around `points` (lon, lat),
    widened by `clearance` (axes fraction) so furniture keeps a visible gap,
    that the boxes would cover; ties go to the preferred anchors, in item order.
    If every combination hides part of the ROI, the frame is grown on one side
    (bottom, right, top, left; by 10%, 20%, ... up to max_grow of its span) and
    the first clear arrangement is used; failing that, the least-hiding one.
    Returns (extent, {name: anchor}, report) with report = {"grown_side",
    "grown_by", "roi_hidden_pct" (share of the ROI area, with the station
    discs, inside the frame that the furniture boxes hide)}."""
    import itertools
    from shapely.geometry import box as _box, Point
    from shapely.ops import unary_union
    from shapely import affinity

    def evaluate(e):
        sx, sy = 1.0 / (e[1] - e[0]), 1.0 / (e[3] - e[2])
        obs = []
        if obstacle is not None and not obstacle.is_empty:
            obs.append(affinity.affine_transform(obstacle, [sx, 0.0, 0.0, sy, -e[0] * sx, -e[2] * sy]))
        for lon, lat in points:
            fx, fy = (lon - e[0]) * sx, (lat - e[2]) * sy
            if 0.0 <= fx <= 1.0 and 0.0 <= fy <= 1.0:
                obs.append(Point(fx, fy).buffer(0.02))
        geom = unary_union(obs).intersection(_box(0.0, 0.0, 1.0, 1.0)) if obs else None
        total = geom.area if geom is not None else 0.0
        near = geom.buffer(clearance) if geom is not None else None     # keep a small gap, too
        boxes, hidden, hidden_roi = {}, {}, {}
        for name, size, _ in items:
            for a in MAP_ANCHORS:
                x0, y0 = _anchor_origin(a, size)
                boxes[name, a] = (x0, y0, x0 + size[0], y0 + size[1])
                b = _box(*boxes[name, a])
                hidden[name, a] = b.intersection(near).area if near is not None else 0.0
                hidden_roi[name, a] = b.intersection(geom).area if geom is not None else 0.0

        def overlap(p, q):
            return min(p[2], q[2]) - max(p[0], q[0]) > 1e-9 and min(p[3], q[3]) - max(p[1], q[1]) > 1e-9
        best = None
        for combo in itertools.product(MAP_ANCHORS, repeat=len(items)):
            bs = [boxes[it[0], a] for it, a in zip(items, combo)]
            if any(overlap(bs[i], bs[j]) for i in range(len(bs)) for j in range(i + 1, len(bs))):
                continue
            cost = sum(hidden[it[0], a] for it, a in zip(items, combo))
            ranks = tuple(it[2].index(a) if a in it[2] else len(it[2]) + MAP_ANCHORS.index(a)
                          for it, a in zip(items, combo))
            key = (round(cost, 9), ranks)
            if best is None or key < best[0]:
                best = (key, combo)
        roi = sum(hidden_roi[it[0], a] for it, a in zip(items, best[1]))
        return best[0][0], (100.0 * roi / total if total > 0 else 0.0), dict(zip([it[0] for it in items], best[1]))

    tried = []
    for g in [0.0] + [round(0.1 * i, 1) for i in range(1, int(round(max_grow * 10)) + 1)]:
        for side in (("none",) if g == 0.0 else ("bottom", "right", "top", "left")):
            e = ext if g == 0.0 else _grow_extent(ext, side, g)
            cost, pct, anchors = evaluate(e)
            tried.append((round(cost, 9), pct, g, e, anchors, side))
            if cost <= 1e-7:
                return e, anchors, {"grown_side": None if g == 0.0 else side, "grown_by": g, "roi_hidden_pct": 0.0,
                                    "clearance": clearance}
    cost, pct, g, e, anchors, side = min(tried, key=lambda t: (round(t[1], 6), t[0], t[2]))
    logger.warning(f"Map furniture hides {pct:.2f}% of the ROI at best (frame grown by {g:g} on the {side}).")
    return e, anchors, {"grown_side": None if g == 0.0 else side, "grown_by": g, "roi_hidden_pct": pct,
                        "clearance": clearance}


def choose_corners(ext, lonlat: np.ndarray, items) -> Dict[str, str]:
    """Assign each map furniture item to a corner, in the order given, choosing
    the free corner whose box hides the fewest of the points `lonlat` (N x 2)
    -- ties go to the item's preferred order. items: [(name, size, preferred corners)]."""
    lonlat = np.asarray(lonlat, dtype=float).reshape(-1, 2)
    fx = (lonlat[:, 0] - ext[0]) / (ext[1] - ext[0])
    fy = (lonlat[:, 1] - ext[2]) / (ext[3] - ext[2])
    free = ["lower left", "lower right", "upper left", "upper right"]
    out = {}
    for name, size, preferred in items:
        def cost(c):
            x0, y0 = _corner_origin(c, size)
            hidden = int(((fx >= x0) & (fx <= x0 + size[0]) & (fy >= y0) & (fy <= y0 + size[1])).sum())
            return (hidden, preferred.index(c) if c in preferred else len(preferred))
        best = min(free, key=cost)
        out[name] = best
        free.remove(best)
    return out


def add_scale_bar(ax, ext, frac: float = 0.22, corner: str = "lower left", fontsize: float = 6.5,
                  compact: bool = False):
    """Two-segment scale bar, true along the parallel it is drawn on, on a
    white backing box, inside the _SCALE_BAR_BOX footprint at `corner` (any of
    MAP_ANCHORS). compact: a single bar labelled with its length only."""
    bx, by = _anchor_origin(corner, _SCALE_BAR_BOX)
    lat_b = ext[2] + (by + 0.035) * (ext[3] - ext[2])
    kpd = _km_per_deg_lon(lat_b)
    length_km = _nice_length(frac * (ext[1] - ext[0]) * kpd)
    dlon = length_km / kpd
    bar_frac = dlon / (ext[1] - ext[0])
    span = bar_frac + (0.0 if compact else 0.07)              # bar plus its right-hand label
    x0 = (bx + 0.03 if "left" in corner else
          (bx + _SCALE_BAR_BOX[0] - 0.03 - span if "right" in corner else 0.5 - 0.5 * span))
    lon_b = ext[0] + x0 * (ext[1] - ext[0])
    h = 0.012 * (ext[3] - ext[2])
    box_w = dlon + (0.03 if compact else 0.10) * (ext[1] - ext[0])
    backing = ax.add_patch(mpatches.FancyBboxPatch(
        (lon_b - 0.015 * (ext[1] - ext[0]), lat_b - 0.022 * (ext[3] - ext[2])), box_w, 0.085 * (ext[3] - ext[2]),
        boxstyle="square,pad=0", facecolor="white", edgecolor="none", alpha=0.85, zorder=8))
    segments = (("black",),) if compact else (("black",), ("white",))
    for i, (color,) in enumerate(segments):
        w = dlon / len(segments)
        ax.add_patch(mpatches.Rectangle((lon_b + i * w, lat_b), w, h, facecolor=color,
                                        edgecolor="black", linewidth=0.5, zorder=9))
    labels = (((lon_b + dlon / 2, f"{length_km:g} km", "center"),) if compact else
              ((lon_b, "0", "center"), (lon_b + dlon / 2, f"{length_km / 2:g}", "center"),
               (lon_b + dlon, f"{length_km:g} km", "left")))
    for x, txt, ha in labels:
        ax.text(x, lat_b + 1.8 * h, txt, ha=ha, va="bottom", fontsize=fontsize, zorder=10, path_effects=_HALO)
    return length_km, backing


def add_north_arrow(ax, corner: str = "upper right", fontsize: float = 7.5):
    """North arrow on a white backing box (north is up in these lat/lon panels),
    at `corner` (any of MAP_ANCHORS)."""
    x0, y0 = _anchor_origin(corner, _NORTH_BOX)
    x, y, length = x0 + 0.5 * _NORTH_BOX[0], y0 + 0.02, 0.09
    backing = ax.add_patch(mpatches.FancyBboxPatch((x0 + 0.01, y0), _NORTH_BOX[0] - 0.02, _NORTH_BOX[1],
                                                   boxstyle="square,pad=0", transform=ax.transAxes,
                                                   facecolor="white", edgecolor="none", alpha=0.85, zorder=8))
    ax.annotate("", xy=(x, y + length), xytext=(x, y), xycoords="axes fraction",
                arrowprops=dict(arrowstyle="-|>,head_length=0.5,head_width=0.25", color="black", lw=0.9),
                zorder=10)
    ax.text(x, y + length + 0.008, "N", transform=ax.transAxes, ha="center", va="bottom",
            fontsize=fontsize, fontweight="bold", zorder=10)
    return backing


class LabelPlacer:
    """Greedy collision-avoiding placement of map labels: each label takes the
    first candidate position whose text box stays inside the panel and clear
    of everything placed or registered before it."""

    def __init__(self, ax):
        self.ax = ax
        ax.apply_aspect()
        self.renderer = ax.figure.canvas.get_renderer()
        self.frame = ax.get_window_extent(self.renderer)
        self.boxes = []

    def occupy_point(self, lon: float, lat: float, radius_pt: float = 4.0):
        from matplotlib.transforms import Bbox
        x, y = self.ax.transData.transform((lon, lat))
        r = radius_pt * self.ax.figure.dpi / 72.0
        self.boxes.append(Bbox([[x - r, y - r], [x + r, y + r]]))

    def occupy_artist(self, artist):
        self.boxes.append(artist.get_window_extent(self.renderer))

    def _fits(self, bb) -> bool:
        f = self.frame
        return (f.x0 <= bb.x0 and bb.x1 <= f.x1 and f.y0 <= bb.y0 and bb.y1 <= f.y1
                and not any(bb.overlaps(b) for b in self.boxes))

    def place(self, text: str, candidates, required: bool = True, **kw):
        """candidates: [(lon, lat, dx_pt, dy_pt, ha, va)]. Returns the text, or
        None if required is False and no candidate fits."""
        for lon, lat, dx, dy, ha, va in candidates:
            t = self.ax.annotate(text, (lon, lat), xytext=(dx, dy), textcoords="offset points",
                                 ha=ha, va=va, **kw)
            bb = t.get_window_extent(self.renderer).expanded(1.04, 1.10)
            if self._fits(bb):
                self.boxes.append(bb)
                return t
            t.remove()
        if not required:
            return None
        lon, lat, dx, dy, ha, va = candidates[0]
        t = self.ax.annotate(text, (lon, lat), xytext=(dx, dy), textcoords="offset points", ha=ha, va=va, **kw)
        self.boxes.append(t.get_window_extent(self.renderer))
        return t


def _point_candidates(lon, lat, r: float = 5.0):
    """Label positions around a point marker of radius r points: right, left, above, below, diagonals."""
    return [(lon, lat, r, 0, "left", "center"), (lon, lat, -r, 0, "right", "center"),
            (lon, lat, 0, r, "center", "bottom"), (lon, lat, 0, -r, "center", "top"),
            (lon, lat, r, r, "left", "bottom"), (lon, lat, -r, r, "right", "bottom"),
            (lon, lat, r, -r, "left", "top"), (lon, lat, -r, -r, "right", "top")]


def _box_candidates(ext, gap: float = 4.0):
    """Label positions around a rectangle [lon0, lon1, lat0, lat1]."""
    xc, yc = 0.5 * (ext[0] + ext[1]), 0.5 * (ext[2] + ext[3])
    return [(ext[1], yc, gap, 0, "left", "center"), (ext[0], yc, -gap, 0, "right", "center"),
            (xc, ext[3], 0, gap, "center", "bottom"), (xc, ext[2], 0, -gap, "center", "top")]


def load_country_shapes(search_dirs=(), download_dir: Optional[str] = None) -> List[dict]:
    """Natural Earth 1:50m Admin 0 country polygons, [{'name', 'label', 'bbox',
    'rings'}]; [] if no local copy exists and the download fails. A local copy
    in any of search_dirs is used first; otherwise the files are downloaded
    from the Natural Earth repository (tag NATURAL_EARTH_VERSION) into
    download_dir."""
    base = _NATURAL_EARTH_BASE
    path = None
    for d in [x for x in search_dirs if x]:
        if all(os.path.exists(os.path.join(d, f"{base}.{e}")) for e in ("shp", "shx", "dbf")):
            path = os.path.join(d, base + ".shp")
            break
    if path is None and download_dir:
        try:
            import urllib.request
            os.makedirs(download_dir, exist_ok=True)
            for e in ("shp", "shx", "dbf"):
                part = os.path.join(download_dir, f"{base}.{e}.part")
                with urllib.request.urlopen(NATURAL_EARTH_URL.format(ext=e), timeout=60) as r, open(part, "wb") as fh:
                    fh.write(r.read())
                os.replace(part, os.path.join(download_dir, f"{base}.{e}"))
            path = os.path.join(download_dir, base + ".shp")
            logger.info(f"Natural Earth {NATURAL_EARTH_VERSION} borders downloaded to {download_dir}")
        except Exception as exc:
            logger.warning(f"Natural Earth borders unavailable ({exc}); maps are drawn without borders.")
            return []
    if path is None:
        return []
    if path in _country_shape_cache:
        return _country_shape_cache[path]
    try:
        import shapefile   # pyshp
        reader = shapefile.Reader(path)
        fields = [f[0] for f in reader.fields[1:]]
        shapes = []
        for sr in reader.iterShapeRecords():
            pts = np.asarray(sr.shape.points, dtype=float)
            if pts.size == 0:
                continue
            rec = dict(zip(fields, sr.record))
            parts = list(sr.shape.parts) + [len(pts)]
            label = (rec.get("LABEL_X"), rec.get("LABEL_Y"))
            shapes.append({"name": str(rec.get("NAME") or rec.get("ADMIN") or ""),
                           "label": label if None not in label else None,
                           "bbox": list(sr.shape.bbox),
                           "rings": [pts[parts[i]:parts[i + 1]] for i in range(len(parts) - 1)]})
        with open(path, "rb") as fh:
            sha = hashlib.sha256(fh.read()).hexdigest()
        _country_shape_cache[path] = shapes
        _country_shape_cache["_meta"] = {"path": path, "sha256": sha, "version": NATURAL_EARTH_VERSION}
        return shapes
    except Exception as exc:
        logger.warning(f"Could not read Natural Earth borders at {path} ({exc}); maps are drawn without borders.")
        return []


def _in_ext(ext, lon, lat, margin: float = 0.0) -> bool:
    mx, my = margin * (ext[1] - ext[0]), margin * (ext[3] - ext[2])
    return ext[0] + mx <= lon <= ext[1] - mx and ext[2] + my <= lat <= ext[3] - my


def draw_countries(ax, shapes, ext) -> List[dict]:
    """Land fill and borders/coastlines inside ext; returns the countries drawn."""
    vis = [s for s in shapes if not (s["bbox"][2] < ext[0] or s["bbox"][0] > ext[1]
                                     or s["bbox"][3] < ext[2] or s["bbox"][1] > ext[3])]
    rings = [r for s in vis for r in s["rings"]]
    if rings:
        ax.add_collection(PolyCollection(rings, facecolor=MAP_COLORS["land"], edgecolor="none", zorder=0.2))
        ax.add_collection(LineCollection(rings, colors=MAP_COLORS["border"], linewidths=0.45, zorder=0.7))
    return vis


def label_countries(placer: "LabelPlacer", countries, ext, fontsize: float = 6.0) -> int:
    """Country names at their Natural Earth label points, for countries with a
    sizeable part of the frame, skipping any name that would collide."""
    frame = (ext[1] - ext[0]) * (ext[3] - ext[2])
    n = 0
    for s in countries:
        if not s["label"] or not _in_ext(ext, *s["label"], margin=0.04):
            continue
        bx0, by0, bx1, by1 = s["bbox"]
        overlap = max(0.0, min(bx1, ext[1]) - max(bx0, ext[0])) * max(0.0, min(by1, ext[3]) - max(by0, ext[2]))
        if overlap < 0.03 * frame:
            continue
        t = placer.place(s["name"].upper(), [(*s["label"], 0, 0, "center", "center")], required=False,
                         fontsize=fontsize, color=MAP_COLORS["country_label"], style="italic", zorder=1.7,
                         path_effects=[patheffects.withStroke(linewidth=2.0, foreground=MAP_COLORS["land"])])
        n += t is not None
    return n


def draw_station_markers(ax, stations, ext, margin: float = 0.0) -> List[Tuple[str, float, float]]:
    """Ground-station triangles inside ext (at least `margin` of the frame from its edges)."""
    drawn = []
    for name, gs in stations.items():
        lon, lat = gs["lng"], gs["lat"]
        if _in_ext(ext, lon, lat, margin):
            ax.plot(lon, lat, marker="^", ms=5.5, mfc=MAP_COLORS["station"], mec="black", mew=0.6,
                    ls="none", zorder=6)
            drawn.append((name, lon, lat))
    return drawn


def label_stations(placer: "LabelPlacer", drawn, fontsize: float = 6.5):
    for name, lon, lat in drawn:
        placer.place(name.replace("_", " "), _point_candidates(lon, lat, 4.5), fontsize=fontsize,
                     zorder=7, path_effects=_HALO)


def map_figure(aspects, fig_w: float = MAP_FIG_WIDTH_IN, left: float = 0.52, gap: float = 0.62,
               right: float = 0.12, top: float = 0.30, bottom: float = 0.80):
    """Figure with equal-height map panels side by side whose boxes match the
    panels' displayed aspects exactly (so set_aspect never shrinks them).
    Returns (fig, axes, layout) with layout sizes in inches."""
    h = (fig_w - left - right - gap * (len(aspects) - 1)) / float(sum(aspects))
    fig_h = top + h + bottom
    fig = plt.figure(figsize=(fig_w, fig_h))
    axes, x = [], left
    for a in aspects:
        axes.append(fig.add_axes([x / fig_w, bottom / fig_h, a * h / fig_w, h / fig_h]))
        x += a * h + gap
    return fig, axes, {"fig_w": fig_w, "fig_h": fig_h, "panel_h": h, "bottom": bottom, "top": top}


def draw_tile_grid(ax, tiles, color=None, edges: Optional[bool] = None, zorder: float = 2.0,
                   rasterize: Optional[bool] = None):
    """ROI cells as one dissolved fill plus (while visible) white cell outlines."""
    tiles = list(tiles)
    path = geo_path([tiles_outline(tiles)])
    if path is not None:
        patch = mpatches.PathPatch(path, facecolor=color or MAP_COLORS["tile"], edgecolor="none", zorder=zorder)
        ax.add_patch(patch)
    if edges if edges is not None else len(tiles) <= MAP_EDGE_TILES:
        lw = 0.9 if len(tiles) <= 100 else (0.5 if len(tiles) <= 800 else 0.25)
        lines = LineCollection([np.vstack([p, p[:1]]) for p in tile_polygons(tiles)],
                               colors=MAP_COLORS["tile_edge"], linewidths=lw, zorder=zorder + 0.1)
        if rasterize if rasterize is not None else len(tiles) > MAP_RASTER_TILES:
            lines.set_rasterized(True)
        ax.add_collection(lines)


def draw_outline(ax, geoms, color=None, lw: float = 0.8, zorder: float = 4.0, ls="-", rasterized=None):
    """ROI outline. rasterized (v1.5.0): default True when the outline is a
    WDPA boundary (ROI_OUTLINE_SOURCE == "WDPA"). The WDPA licence forbids
    redistributing the data, so saved figures carry the boundary as pixels --
    as a PNG always does -- and a PDF holds no vector copy of the polygon."""
    lines = geo_lines(geoms)
    if lines:
        lc = LineCollection(lines, colors=color or MAP_COLORS["boundary"], linewidths=lw, linestyles=ls,
                            zorder=zorder)
        lc.set_gid("roi_outline")
        if globals().get("ROI_OUTLINE_SOURCE") == "WDPA" if rasterized is None else rasterized:
            lc.set_rasterized(True)
        ax.add_collection(lc)


_ZOOM_BOX = (0.34, 0.34)


def zoom_centre(tiles, n_across: int = 9) -> str:
    """The cell deepest inside the ROI (hexagon rings to the nearest cell
    outside, capped at 3x the zoom radius), ties going to the cell nearest the
    centroid of the ROI's cells -- so the zoom shows a full patch of the grid,
    not the ROI's rim, even for ROIs made of separate parts."""
    tset = tiles if isinstance(tiles, (set, frozenset, dict)) else set(tiles)
    cap = 3 * (n_across // 2 + 1)
    depth, frontier = {}, []
    for t in tset:
        if any(n not in tset for n in h3.grid_disk(t, 1)):
            depth[t] = 0
            frontier.append(t)
    d = 0
    while frontier and d < cap:
        d += 1
        nxt = []
        for t in frontier:
            for n in h3.grid_disk(t, 1):
                if n in tset and n not in depth:
                    depth[n] = d
                    nxt.append(n)
        frontier = nxt
    tl = list(tset)
    cent = np.array([h3.cell_to_latlng(t) for t in tl])
    dist2 = ((cent - cent.mean(axis=0)) ** 2).sum(axis=1)
    deep = np.array([min(depth.get(t, cap), cap) for t in tl])
    return tl[int(np.lexsort((dist2, -deep))[0])]


def add_zoom_inset(ax, tiles, ext, corner: str, outline_geoms=None, n_across: int = 9, color=None,
                   colors_by_tile: Optional[Dict[str, Any]] = None):
    """Magnified view (n_across cells wide) of the cells around zoom_centre(),
    in the given corner of `ax`, with a zoom indicator and its own scale bar.
    colors_by_tile: optional {tile: facecolor} (Cell 7's mosaic map)."""
    centre = zoom_centre(tiles, n_across)
    lat_c, lon_c = h3.cell_to_latlng(centre)
    edge_km = h3.average_hexagon_edge_length(h3.get_resolution(centre), unit="km")
    half_km = 0.5 * n_across * edge_km * math.sqrt(3.0)
    dlon, dlat = half_km / _km_per_deg_lon(lat_c), half_km / _KM_PER_DEG_LAT
    zext = fit_aspect([lon_c - dlon, lon_c + dlon, lat_c - dlat, lat_c + dlat], 1.0)
    axins = ax.inset_axes([*_anchor_origin(corner, _ZOOM_BOX), *_ZOOM_BOX])
    tset = tiles if isinstance(tiles, (set, frozenset, dict)) else set(tiles)
    near = [t for t in h3.grid_disk(centre, n_across) if t in tset]
    if colors_by_tile is None:
        draw_tile_grid(axins, near, color=color, edges=True, rasterize=False)
    else:
        polys = tile_polygons(near)
        axins.add_collection(PolyCollection(polys, facecolors=[colors_by_tile[t] for t in near],
                                            edgecolors="white", linewidths=0.6, zorder=2))
    draw_outline(axins, outline_geoms, lw=0.8)
    axins.set_xlim(zext[0], zext[1])
    axins.set_ylim(zext[2], zext[3])
    axins.set_aspect(1.0 / _cos_lat(zext), adjustable="box")
    axins.set_xticks([])
    axins.set_yticks([])
    axins.set_facecolor("white")
    for spine in axins.spines.values():
        spine.set_linewidth(0.6)
    add_scale_bar(axins, zext, frac=0.35, corner="lower left", fontsize=5.5, compact=True)
    ax.indicate_inset_zoom(axins, edgecolor="black", linewidth=0.6, alpha=1.0)
    return axins


def _wdpa_citation(reports) -> Tuple[Optional[str], Optional[str]]:
    """(full citation, short credit) for the WDPA release the boundaries came from,
    in the form the WDPA/WDOECM Manual 1.6 (section 5.3) asks for."""
    if not reports:
        return None, None
    rel = max(_wdpa_release(r.get("file", "")) for r in reports) if '_wdpa_release' in globals() else (0, 0)
    if rel == (0, 0):
        when, short = "[month/year of the version downloaded]", "WDPA"
    else:
        months = ("January", "February", "March", "April", "May", "June", "July", "August",
                  "September", "October", "November", "December")
        when = f"{months[rel[1] - 1]} {rel[0]}"
        short = f"WDPA, {months[rel[1] - 1][:3]} {rel[0]}"
    year = rel[0] if rel[0] else "[year]"
    return (f"UNEP-WCMC and IUCN ({year}), Protected Planet: The World Database on Protected Areas (WDPA) "
            f"[On-line], {when}, Cambridge, UK: UNEP-WCMC and IUCN. Available at: www.protectedplanet.net.",
            short)


def roi_outline_for_maps():
    """(geometries, legend label, source) of the ROI outline: the official
    boundary kept in memory by Cell 1, the ROI polygon, or -- if Cell 1 predates
    ROI_OUTLINE_GEOMS -- the dissolved outline of the ROI's cells."""
    geoms = globals().get("ROI_OUTLINE_GEOMS")
    if geoms:
        src = globals().get("ROI_OUTLINE_SOURCE", "ROI definition")
        return list(geoms), ("WDPA boundary" if src == "WDPA" else "ROI boundary"), src
    if globals().get("roi_coords"):
        return ([{"type": "Polygon", "coordinates": [[(lon, lat) for lat, lon in roi_coords]]}],
                "ROI boundary", "ROI definition")
    return [tiles_outline(target_tiles)], "ROI outline (cells)", "H3 cells"


def roi_short_name() -> str:
    spec = globals().get("ROI_DEFINITIONS", {}).get(ACTIVE_ROI, {})
    return spec.get("short_name") or ACTIVE_ROI.replace("_", " ").title()


# =============================================================================
# GROUND TRACK COMPUTATION
# =============================================================================

def compute_track(sat: Dict, epoch: Any, window_min: int, step_min: int) -> Tuple[List[float], List[float]]:
    """Propagates satellite state vectors and transforms them into geodetic coordinates."""
    lats, lngs = [], []
    prop = sat["propagator"]
    for m in range(0, window_min + 1, step_min):
        try:
            t = epoch.shiftedBy(float(m * 60))
            pv = prop.getPVCoordinates(t, itrf)
            geo = earth.transform(pv.getPosition(), itrf, t)
            lats.append(FastMath.toDegrees(geo.getLatitude()))
            lngs.append(FastMath.toDegrees(geo.getLongitude()))
        except Exception as e:
            logger.warning(f"Propagation anomaly at t={m}m for {sat['id']}: {e}")
            lats.append(float('nan'))
            lngs.append(float('nan'))
    return lats, lngs


def _break_at_antimeridian(lats: List[float], lngs: List[float]) -> Tuple[List[float], List[float]]:
    """Inserts NaN to lift the plotter pen when crossing the International Date Line."""
    if not lngs:
        return lats, lngs
    out_lat, out_lng = [lats[0]], [lngs[0]]
    for i in range(1, len(lngs)):
        if abs(lngs[i] - lngs[i - 1]) > 180.0:
            out_lat.append(np.nan)
            out_lng.append(np.nan)
        out_lat.append(lats[i])
        out_lng.append(lngs[i])
    return out_lat, out_lng


def frame_crossings(sats, epoch, ext, hours: float = MAP_TRACK_HOURS, max_sats: int = MAP_TRACK_MAX_SATS,
                    max_segments: int = MAP_TRACK_MAX_SEGMENTS):
    """Ground-track segments crossing ext during the first `hours` hours, for an
    evenly spaced, deterministic sample of up to max_sats satellites (1-minute
    steps). Returns the earliest max_segments as dicts (sat, t_min, lon, lat,
    inside = index range within the frame), and the sample size."""
    n = len(sats)
    if n == 0:
        return [], 0
    idx = sorted(set(np.linspace(0, n - 1, min(max_sats, n)).round().astype(int).tolist()))
    segs = []
    for i in idx:
        lats, lons = compute_track(sats[i], epoch, int(round(hours * 60)), 1)
        lats, lons = np.asarray(lats, dtype=float), np.asarray(lons, dtype=float)
        inside = ((lons >= ext[0]) & (lons <= ext[1]) & (lats >= ext[2]) & (lats <= ext[3]))
        j = 0
        while j < inside.size:
            if not inside[j]:
                j += 1
                continue
            k = j
            while k + 1 < inside.size and inside[k + 1]:
                k += 1
            a, b = max(j - 1, 0), min(k + 1, inside.size - 1)      # one step past each edge
            seg_lon, seg_lat = lons[a:b + 1], lats[a:b + 1]
            if b > a and np.all(np.isfinite(seg_lon)) and np.all(np.abs(np.diff(seg_lon)) < 180.0):
                segs.append({"sat": sats[i]["id"], "t_min": int(j), "lon": seg_lon, "lat": seg_lat,
                             "inside": (j - a, k - a)})
            j = k + 1
    segs.sort(key=lambda s: (s["t_min"], s["sat"]))
    return segs[:max_segments], len(idx)


def draw_track(ax, seg, color=None, lw: float = 0.7):
    color = color or MAP_COLORS["track"]
    ax.plot(seg["lon"], seg["lat"], color=color, lw=lw, alpha=0.85, zorder=1.5, solid_capstyle="butt")
    i0, i1 = seg["inside"]
    m = max(i0, min((i0 + i1) // 2, len(seg["lon"]) - 2))
    ax.annotate("", xy=(seg["lon"][m + 1], seg["lat"][m + 1]), xytext=(seg["lon"][m], seg["lat"][m]),
                arrowprops=dict(arrowstyle="-|>,head_length=0.45,head_width=0.22", color=color, lw=lw),
                zorder=1.6)


# =============================================================================
# FIGURE GENERATION
# =============================================================================

logger.info("Constructing study-area map...")
_roi_name = roi_short_name()
_outline_geoms, _outline_label, _outline_source = roi_outline_for_maps()
_n_tiles_map = len(target_tiles)
_roi_area_km2 = float(sum(h3.cell_area(t, unit="km^2") for t in target_tiles))
_wdpa_cite, _wdpa_short = _wdpa_citation(globals().get("roi_boundary_report"))
_tile_lonlat = np.array([h3.cell_to_latlng(t)[::-1] for t in target_tiles])        # (lon, lat)
_roi_centroid = tuple(_tile_lonlat.mean(axis=0))                                   # (lon, lat)

_detail_ext = roi_map_extent(target_tiles, _outline_geoms)
# Furniture of panel (b) goes where it hides none of the ROI (cells, boundary,
# stations); the frame grows on one side if no such place exists.
_furniture = [("scale", _SCALE_BAR_BOX, ["lower left", "lower right", "upper left", "upper right"]),
              ("north", _NORTH_BOX, ["upper right", "upper left", "lower right", "lower left"])]
if _n_tiles_map > MAP_ZOOM_TILES:
    _furniture.insert(0, ("zoom", _ZOOM_BOX, ["lower right", "upper left", "upper right", "lower left"]))
_detail_ext, _corners, _furniture_report = plan_furniture(
    _detail_ext, map_obstacle(target_tiles, _outline_geoms), _furniture,
    points=[(gs["lng"], gs["lat"]) for gs in GROUND_STATIONS.values()])
_map_aspect = panel_aspect(_detail_ext)
_regional_ext = regional_extent(_detail_ext, GROUND_STATIONS, _map_aspect)

_ne_dirs = [os.path.join(ARF_BOUNDARY_DIR, "natural_earth") if 'ARF_BOUNDARY_DIR' in globals() else None,
            os.path.join(ARF_CACHE_DIR, "natural_earth") if 'ARF_CACHE_DIR' in globals() else None]
_ne_download = next((d for d in _ne_dirs if d and os.path.isdir(os.path.dirname(d))), None)
_country_shapes = load_country_shapes(_ne_dirs, _ne_download)

with plt.rc_context(_MAP_RC):
    fig, (ax_a, ax_b), _layout = map_figure([_map_aspect, _map_aspect])

    # --- (a) Regional setting ------------------------------------------------
    setup_map_axes(ax_a, _regional_ext, max_intervals=(5, 5),
                   water=MAP_COLORS["water"] if _country_shapes else None)
    _countries = draw_countries(ax_a, _country_shapes, _regional_ext)
    try:
        _segments, _n_track_sample = frame_crossings(constellation, sim_epoch, _regional_ext)
    except Exception as exc:          # a map without tracks beats no map
        logger.warning(f"Ground tracks not drawn ({exc}).")
        _segments, _n_track_sample = [], 0
    for seg in _segments:
        draw_track(ax_a, seg)
    _roi_patch_path = geo_path([tiles_outline(target_tiles)])
    if _roi_patch_path is not None:
        ax_a.add_patch(mpatches.PathPatch(_roi_patch_path, facecolor=MAP_COLORS["tile"],
                                          edgecolor=MAP_COLORS["tile"], linewidth=0.4, zorder=3))
    draw_outline(ax_a, _outline_geoms, lw=0.5, zorder=3.5)
    _small_roi = (_detail_ext[1] - _detail_ext[0]) < 0.05 * (_regional_ext[1] - _regional_ext[0])
    if _small_roi:
        _roi_marker = ax_a.plot(_roi_centroid[0], _roi_centroid[1], marker="o", ms=9, mfc="none", mec="black",
                                mew=0.9, zorder=5)[0]
    else:
        _roi_marker = ax_a.add_patch(mpatches.Rectangle(
            (_detail_ext[0], _detail_ext[2]), _detail_ext[1] - _detail_ext[0], _detail_ext[3] - _detail_ext[2],
            fill=False, edgecolor=MAP_COLORS["extent"], linewidth=0.7, zorder=5))
    _stations_a = draw_station_markers(ax_a, GROUND_STATIONS, _regional_ext)
    _, _bar_a = add_scale_bar(ax_a, _regional_ext, corner="lower left")
    _placer_a = LabelPlacer(ax_a)
    _placer_a.occupy_artist(_roi_marker)
    _placer_a.occupy_artist(_bar_a)
    for _, lon, lat in _stations_a:
        _placer_a.occupy_point(lon, lat)
    label_stations(_placer_a, _stations_a)
    _placer_a.place(_roi_name, _point_candidates(*_roi_centroid, 7.0) if _small_roi else _box_candidates(_detail_ext),
                    fontsize=6.5, fontweight="bold", zorder=7, path_effects=_HALO)
    _n_country_labels = label_countries(_placer_a, _countries, _regional_ext)
    ax_a.set_title("(a) Regional setting", loc="left", pad=4)

    # --- (b) ROI and its H3 tiling --------------------------------------------
    setup_map_axes(ax_b, _detail_ext)
    ax_b.set_facecolor("white")
    draw_tile_grid(ax_b, target_tiles)
    draw_outline(ax_b, _outline_geoms, lw=0.8)
    _, _bar_b = add_scale_bar(ax_b, _detail_ext, corner=_corners["scale"])
    _north_b = add_north_arrow(ax_b, corner=_corners["north"])
    _zoom_corner = None
    if "zoom" in _corners:
        add_zoom_inset(ax_b, target_tiles, _detail_ext, _corners["zoom"], _outline_geoms)
        _zoom_corner = _corners["zoom"]
    _stations_b = draw_station_markers(ax_b, GROUND_STATIONS, _detail_ext, margin=0.03)
    _placer_b = LabelPlacer(ax_b)
    for artist in (_bar_b, _north_b):
        _placer_b.occupy_artist(artist)
    if _zoom_corner:
        x0, y0 = _corner_origin(_zoom_corner, _ZOOM_BOX)
        _placer_b.boxes.append(ax_b.transAxes.transform_bbox(
            matplotlib.transforms.Bbox.from_bounds(x0, y0, *_ZOOM_BOX)))
    for _, lon, lat in _stations_b:
        _placer_b.occupy_point(lon, lat)
    label_stations(_placer_b, _stations_b)
    ax_b.set_title(f"(b) {_roi_name}: H3 tiling", loc="left", pad=4)

    # --- Legend and credits ----------------------------------------------------
    _res = H3_RESOLUTION if 'H3_RESOLUTION' in globals() else h3.get_resolution(next(iter(target_tiles)))
    _handles = [mpatches.Patch(facecolor=MAP_COLORS["tile"], edgecolor="white",
                               label=f"H3 res-{_res} cells (n = {_n_tiles_map:,}; {_roi_area_km2:,.0f} km²)"),
                mlines.Line2D([], [], color=MAP_COLORS["boundary"], lw=0.8, label=_outline_label),
                mlines.Line2D([], [], color=MAP_COLORS["station"], marker="^", mec="black", mew=0.6, ms=5.5,
                              ls="none", label="Ground station (notional)")]
    if _segments:
        _handles.append(mlines.Line2D([], [], color=MAP_COLORS["track"], lw=0.7,
                                      label=f"Ground track, first {MAP_TRACK_HOURS} h (arrow: direction)"))
    _handles.append(mlines.Line2D([], [], color="black", marker="o", ms=7, mfc="none", ls="none", mew=0.9,
                                  label="Location of (b)") if _small_roi else
                    mpatches.Patch(facecolor="none", edgecolor=MAP_COLORS["extent"], lw=0.7, label="Extent of (b)"))
    if _countries:
        _handles.append(mlines.Line2D([], [], color=MAP_COLORS["border"], lw=0.6, label="Border / coastline"))
    _fw, _fh, _bot = _layout["fig_w"], _layout["fig_h"], _layout["bottom"]
    fig.legend(handles=_handles, loc="upper center", bbox_to_anchor=(0.5, (_bot - 0.24) / _fh),
               ncol=3, frameon=False, handlelength=1.8, columnspacing=1.4, handletextpad=0.5)
    _credits = [f"Boundary: {_wdpa_short} (UNEP-WCMC & IUCN)" if _wdpa_short else None,
                "Borders: Natural Earth" if _countries else None,
                "Coordinates: WGS 84"]
    fig.text(0.52 / _fw, 0.04 / _fh, " · ".join(c for c in _credits if c), fontsize=6, color="0.4",
             ha="left", va="bottom")

# =============================================================================
# EXPORT & SUMMARY
# =============================================================================

_png_path = os.path.join(_out_dir, "arf_coverage_map.png")
_pdf_path = os.path.join(_out_dir, "arf_coverage_map.pdf")
_meta_path = os.path.join(_out_dir, "arf_cell3_metadata.json")

try:
    fig.savefig(_png_path, dpi=300, bbox_inches='tight')
    fig.savefig(_pdf_path, dpi=300, bbox_inches='tight')
    logger.info(f"Cartographic outputs exported to {_out_dir}")
except Exception as e:
    logger.warning(f"Failed to save figures: {e}")

plt.show()
plt.close(fig)

_fleet_label = (f"proxy constellation (N={ACTIVE_N})" if globals().get("FLEET", "starlink") == "starlink"
                else f"{FLEET} fleet ({len(constellation)} satellites)")
# v1.5.2: the stations are notional receiving sites at approximate coordinates
# (Cell 1), not the operators' ground networks; the caption says so.
_caption = (
    f"Study area. (a) Regional setting of {_roi_name}, with the notional ground stations used for downlink in the "
    f"model ({', '.join(n.replace('_', ' ') for n in GROUND_STATIONS)}; approximate coordinates, not the "
    f"operators' ground networks)"
    + (f" and the first {len(_segments)} crossings of the frame by ground tracks of a sample of "
       f"{_n_track_sample} satellites of the {_fleet_label} during the first {MAP_TRACK_HOURS} h "
       f"from {sim_epoch} (arrows: direction of flight)" if _segments else "")
    + (". Borders and coastlines: Natural Earth 1:50m." if _countries else ".")
    + f" (b) The ROI discretised into {_n_tiles_map:,} H3 resolution-{_res} cells "
      f"({_roi_area_km2:,.0f} km²; mean cell area {_roi_area_km2 / _n_tiles_map:.2f} km²)"
    + (f" and its boundary from the World Database on Protected Areas" if _outline_source == "WDPA" else "")
    + (f"; the inset magnifies the cells near the ROI centre" if _zoom_corner else "")
    + ". Geographic coordinates (WGS 84), drawn with the equirectangular aspect of each panel's centre latitude."
)

_metadata = {
    "figure_type": "study_area_map",
    "timestamp": datetime.now(timezone.utc).isoformat(),
    "roi": ACTIVE_ROI,
    "roi_label": _roi_name,
    "h3_resolution": _res,
    "n_tiles": _n_tiles_map,
    "roi_area_km2": _roi_area_km2,
    "n_ground_stations": len(GROUND_STATIONS),
    "constellation_size": len(constellation),
    "sim_epoch": str(sim_epoch),
    "panel_b_extent_lon_lat": [round(v, 5) for v in _detail_ext],
    "panel_a_extent_lon_lat": [round(v, 4) for v in _regional_ext],
    "outline_source": _outline_source,
    "zoom_inset": _zoom_corner,
    "furniture": dict(_furniture_report, anchors=_corners),
    "ground_tracks": {"hours": MAP_TRACK_HOURS, "satellites_sampled": _n_track_sample,
                      "segments_shown": [{"sat": s["sat"], "t_min": s["t_min"]} for s in _segments]},
    "borders": dict(_country_shape_cache.get("_meta", {}), countries_drawn=len(_countries)) if _countries else None,
    "wdpa_citation": _wdpa_cite,
    "draft_caption": _caption,
}

try:
    with open(_meta_path, 'w') as f:
        json.dump(_metadata, f, indent=2, default=str)
    logger.info(f"Metadata exported to {_meta_path}")
except Exception as e:
    logger.warning(f"Could not save metadata: {e}")

print("\n" + "=" * 60)
print("  CELL 3: Spatial Validation & Visualisation Complete")
print("=" * 60)
print(f"  ROI:                       {_roi_name} ({_n_tiles_map:,} tiles, {_roi_area_km2:,.0f} km²)")
print(f"  Panel (b) extent:          {_detail_ext[1] - _detail_ext[0]:.3f}° x {_detail_ext[3] - _detail_ext[2]:.3f}°")
print(f"  Ground-track crossings:    {len(_segments)} (sample of {_n_track_sample} satellites)")
print(f"  Borders:                   {'Natural Earth ' + NATURAL_EARTH_VERSION if _countries else 'not available'}")
print(f"  Exported PNG:              arf_coverage_map.png (300 dpi)")
print(f"  Exported PDF:              arf_coverage_map.pdf")
print("=" * 60)
print("\n=== CELL 3 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 4: SPATIO-TEMPORAL ACCESS DETECTION (Coarse-to-Fine Scan)
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Implements Section 3.3.1 of the research proposal (Footprint Projection).
#   Discretises continuous SGP4 orbital ground tracks onto the H3 hexagonal grid
#   to construct the candidate Spatio-Temporal Graph event set Φ.
#
# Algorithmic Efficiency & Kinematic Correction:
#   Avoids the O(N x T x M) brute-force scan with a two-stage search:
#     1. Stage 1 (Coarse, 5-min): flags a satellite whose sub-satellite point
#        falls inside the ROI's bounding box inflated by the footprint radius
#        plus the distance a satellite can travel in HALF a coarse step (~1,100
#        km at LEO). The nearest coarse sample is then guaranteed to fall inside
#        the box for every pass, so no pass can slip between samples. Longitude
#        margins are scaled by cos(latitude); boxes are antimeridian-safe.
#     2. Stage 2 (Strips, 1-min): each whole minute near a coarse hit is a
#        candidate event (sat, m) = the continuous push-broom swath swept over
#        [m-0.5, m+0.5) min, sampled every 2 s. Tile distances are computed
#        directly against tile centres (vectorised), not by enumerating H3 rings.
#   Search radii are deliberate supersets (highest satellite's footprint,
#   lowest satellite's ground speed, padded); Cell 5 applies the exact
#   incidence-angle filter. Validated against a 0.25-s ground truth on
#   synthetic 340-570 km constellations (43/53/70/97.6 deg): 99.9% tile-access
#   recall, zero false positives; residual misses lie within ~0.07 deg of
#   the swath edge (2-s sampling tolerance).
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 4 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["4"] = _ARF_CELL_VERSION

import os
import sys
import time
import logging
from typing import Dict, List, Tuple, Set, Optional, Any
from dataclasses import dataclass
import numpy as np
import pandas as pd
import h3
from org.hipparchus.util import FastMath

logger = logging.getLogger("arf.cell4")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(asctime)s [%(name)s] %(levelname)s: %(message)s', datefmt='%H:%M:%S'))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

def _footprint_ground_radius_km(altitude_km, half_angle_deg):
    """
    Ground-range radius of a nadir-pointing sensor's footprint on a
    spherical Earth: Re * (asin((Re+h)/Re * sin(eta)) - eta). Same
    spherical-Earth model as compute_incidence (Cell 5), so this search
    radius and Cell 5's exact filter agree on where the swath edge is.
    """
    eta = np.radians(half_angle_deg)
    s = min(1.0, (Re_km + altitude_km) / Re_km * np.sin(eta))
    return Re_km * (np.arcsin(s) - eta)


@dataclass
class SensorConfig:
    swath_half_angle_deg: float = SWATH_HALF_ANGLE_DEG
    max_altitude_km: float = CONSTELLATION_ALTITUDE_KM
    min_altitude_km: float = CONSTELLATION_ALTITUDE_KM
    window_hours: int = WINDOW_HOURS
    coarse_step_minutes: int = 5
    # 2 s along-track sampling (~14 km) caps swath-edge erosion from
    # sampling at ~0.3 km -- about a fifth of one Res-7 tile edge.
    strip_sample_seconds: float = 2.0
    # Real fleets: the WIDEST swath in the fleet (km on the ground). The
    # search radius is then a superset for every satellite; Cell 5 applies
    # each satellite's own swath exactly.
    max_swath_km: Optional[float] = None

    def __post_init__(self):
        # Fastest ground-relative speed: the LOWEST satellite's ground-track
        # speed plus Earth's equatorial surface rotation speed.
        h = max(150.0, self.min_altitude_km - 15.0)
        v_orb = np.sqrt(mu_earth / (Re_km + h)) * 60.0 * Re_km / (Re_km + h)
        v_rot = 2.0 * np.pi * Re_km / 1436.068
        self.max_ground_speed_km_min = v_orb + v_rot
        if self.max_swath_km:
            # Widest half-swath +5%, plus HALF the along-track sample spacing:
            # a tile lying between two 2-s samples (~15 km apart) can be up to
            # that far from both while still on the track. Without this term
            # an 8 km Dove swath loses most tiles between samples (measured).
            self.footprint_radius_km = (1.05 * self.max_swath_km / 2.0
                                        + 0.5 * self.max_ground_speed_km_min * self.strip_sample_seconds / 60.0)
        else:
            # Footprint of the HIGHEST satellite, +15 km altitude for orbital
            # eccentricity over the window, +5%.
            self.footprint_radius_km = 1.05 * _footprint_ground_radius_km(
                self.max_altitude_km + 15.0, self.swath_half_angle_deg)
        self.coarse_margin_km = (self.footprint_radius_km
                                 + self.max_ground_speed_km_min * self.coarse_step_minutes / 2.0)
        self.strip_margin_km = self.footprint_radius_km + self.max_ground_speed_km_min * 0.5


_alts = pos_list if ('pos_list' in globals() and len(pos_list)) else [CONSTELLATION_ALTITUDE_KM]
_swaths = [s["swath_km"] for s in constellation if s.get("swath_km")] if 'constellation' in globals() else []
sensor_config = SensorConfig(max_altitude_km=float(np.max(_alts)), min_altitude_km=float(np.min(_alts)),
                             max_swath_km=(float(max(_swaths)) if _swaths else None))
logger.info(f"Footprint search radius: {sensor_config.footprint_radius_km:.1f} km | "
            f"coarse-box margin: {sensor_config.coarse_margin_km:.0f} km | "
            f"strip margin: {sensor_config.strip_margin_km:.0f} km")

# --- Geometry helpers --------------------------------------------------------
_KM_PER_DEG_LAT_MIN = 110.574   # WGS84 meridian, at the equator (smallest -> conservative)
_KM_PER_DEG_LON_EQ = 111.320    # WGS84 parallel, at the equator


def _wrap_dlon(lon, lon_ref):
    """Longitude difference wrapped into [-180, 180) -- antimeridian-safe."""
    return (np.asarray(lon, dtype=float) - lon_ref + 180.0) % 360.0 - 180.0


def _hav_km(lat1, lon1, lat2, lon2):
    p1, p2 = np.radians(lat1), np.radians(lat2)
    a = (np.sin((p2 - p1) / 2) ** 2
         + np.cos(p1) * np.cos(p2) * np.sin(np.radians(_wrap_dlon(lon2, 0) - _wrap_dlon(lon1, 0)) / 2) ** 2)
    return 2.0 * Re_km * np.arcsin(np.sqrt(np.clip(a, 0.0, 1.0)))


class _GeoBox:
    """
    Lat/lon box around a set of points, inflated by margin_km. The
    longitude margin is scaled by cos(latitude) at the box's widest-
    latitude edge (where a degree of longitude is shortest), longitudes are
    handled relative to a centre so a box may straddle +/-180, and a box
    reaching within 1 deg of a pole accepts all longitudes.
    """
    def __init__(self, lats, lons, margin_km):
        lats = np.asarray(lats, dtype=float)
        mlat = margin_km / _KM_PER_DEG_LAT_MIN
        self.lat_lo, self.lat_hi = lats.min() - mlat, lats.max() + mlat
        lat_ref = max(abs(self.lat_lo), abs(self.lat_hi))
        ref = float(np.asarray(lons, dtype=float).flat[0])
        d = _wrap_dlon(lons, ref)
        self.lon_c = ref + (d.min() + d.max()) / 2.0
        self.all_lon = lat_ref >= 89.0
        self.lon_half = (d.max() - d.min()) / 2.0
        if not self.all_lon:
            self.lon_half += margin_km / (_KM_PER_DEG_LON_EQ * np.cos(np.radians(lat_ref)))
            self.all_lon = self.lon_half >= 180.0

    def contains(self, lat, lon):
        lat = np.asarray(lat, dtype=float)
        ok = (lat >= self.lat_lo) & (lat <= self.lat_hi)
        return ok if self.all_lon else ok & (np.abs(_wrap_dlon(lon, self.lon_c)) <= self.lon_half)


# ROI tile centres, sorted by latitude so candidate tiles near a strip can be
# found with a binary search instead of a scan of every tile.
_tile_ids = np.array(sorted(target_tiles))
_ll = np.array([h3.cell_to_latlng(t) for t in _tile_ids])
_order = np.argsort(_ll[:, 0], kind="stable")
_tile_ids, _tile_lat, _tile_lon = _tile_ids[_order], _ll[_order, 0], _ll[_order, 1]
del _ll, _order


def _tiles_near(lats, lons, radius_km):
    """Indices of ROI tiles whose centre lies within radius_km (ground
    range) of at least one of the given sub-satellite points."""
    box = _GeoBox(lats, lons, radius_km)
    lo = np.searchsorted(_tile_lat, box.lat_lo, side="left")
    hi = np.searchsorted(_tile_lat, box.lat_hi, side="right")
    idx = np.arange(lo, hi)
    if idx.size and not box.all_lon:
        idx = idx[np.abs(_wrap_dlon(_tile_lon[idx], box.lon_c)) <= box.lon_half]
    if idx.size == 0:
        return idx
    d = _hav_km(np.asarray(lats)[:, None], np.asarray(lons)[:, None],
                _tile_lat[idx][None, :], _tile_lon[idx][None, :]).min(axis=0)
    return idx[d <= radius_km]


def _subsatellite(prop, t_min):
    """(geodetic lat deg, lon deg, geocentric radius km) at t_min."""
    t = sim_epoch.shiftedBy(float(t_min * 60.0))
    pos = prop.getPVCoordinates(t, itrf).getPosition()
    geo = earth.transform(pos, itrf, t)
    return (FastMath.toDegrees(geo.getLatitude()), FastMath.toDegrees(geo.getLongitude()),
            pos.getNorm() / 1000.0)


_roi_box = _GeoBox(_tile_lat, _tile_lon, 0.0)
coarse_box = _GeoBox(_tile_lat, _tile_lon, sensor_config.coarse_margin_km)
strip_box = _GeoBox(_tile_lat, _tile_lon, sensor_config.strip_margin_km)

# (sat, t_min) -> (lats, lons, radii_km) of the strip's sub-satellite samples;
# Cell 5 reuses these instead of re-propagating.
strip_samples = {}

_ACCESS_COLUMNS = ["sat", "t_min", "lat", "lon", "tiles_hit"]


def detect_access_events(constellation, config):
    win_min = config.window_hours * 60
    step = config.coarse_step_minutes
    coarse_times = list(range(0, win_min + 1, step))
    if coarse_times[-1] != win_min:
        coarse_times.append(win_min)
    offsets = np.arange(-0.5, 0.5, config.strip_sample_seconds / 60.0)
    if config.max_swath_km:
        # Real fleets: include the strip's end point (= next strip's first
        # sample) so the track arcs of consecutive strips join with no gap.
        offsets = np.append(offsets, 0.5)

    events = []
    n_coarse_hits = n_candidate_min = n_strips = 0
    for sat in constellation:
        prop = sat["propagator"]
        # Stage 1: coarse scan against the inflated box.
        hits = []
        for m in coarse_times:
            lat, lon, _ = _subsatellite(prop, m)
            if coarse_box.contains(lat, lon):
                hits.append(m)
        n_coarse_hits += len(hits)
        if not hits:
            continue
        # Stage 2: every minute within one coarse step of a hit.
        minutes = sorted({mm for m_c in hits
                          for mm in range(max(0, m_c - step), min(win_min, m_c + step) + 1)})
        n_candidate_min += len(minutes)
        for m in minutes:
            mid_lat, mid_lon, _ = _subsatellite(prop, m)
            if not strip_box.contains(mid_lat, mid_lon):
                continue            # no part of this minute's strip can reach the ROI
            n_strips += 1
            ts = np.clip(m + offsets, 0.0, float(win_min))
            s = np.array([_subsatellite(prop, t) for t in ts])
            near = _tiles_near(s[:, 0], s[:, 1], config.footprint_radius_km)
            if near.size:
                strip_samples[(sat["id"], m)] = (s[:, 0], s[:, 1], s[:, 2])
                events.append({"sat": sat["id"], "t_min": m, "lat": mid_lat, "lon": mid_lon,
                               "tiles_hit": int(near.size)})

    logger.info(f"Stage 1: {n_coarse_hits} coarse hits | Stage 2: {n_candidate_min} candidate "
                f"minutes, {n_strips} strips sampled, {len(events)} events")
    return pd.DataFrame(events, columns=_ACCESS_COLUMNS)


logger.info(f"Starting Access Scan for {len(constellation)} satellites...")
t0 = time.time()
access_events_df = detect_access_events(constellation, sensor_config)
_t_scan = time.time() - t0
logger.info(f"Scan Complete in {_t_scan:.2f}s. "
            f"Found {len(access_events_df)} continuous access strips.")
display(access_events_df.head())

import os, json
_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()
access_events_df.to_csv(os.path.join(_out_dir, "arf_access_events.csv"), index=False)
with open(os.path.join(_out_dir, "arf_cell4_summary.json"), "w") as f:
    json.dump({
        "n_satellites": len(constellation),
        "n_access_strips": int(len(access_events_df)),
        "footprint_search_radius_km": float(sensor_config.footprint_radius_km),
        "coarse_box_margin_km": float(sensor_config.coarse_margin_km),
        "strip_margin_km": float(sensor_config.strip_margin_km),
        "coarse_step_minutes": sensor_config.coarse_step_minutes,
        "strip_sample_seconds": sensor_config.strip_sample_seconds,
        "altitude_range_km": [float(sensor_config.min_altitude_km), float(sensor_config.max_altitude_km)],
        "scan_seconds": round(_t_scan, 2),
    }, f, indent=2)



In [ ]:
# =============================================================================
# CELL 5: PHYSICAL CONSTRAINTS & REFINEMENT (Incidence Angle & Downlink)
# Algorithmic Reconstruction Framework (ARF)
# =============================================================================
#
# Description:
#   Implements physical Earth Observation (EO) constraints for the Spatio-Temporal
#   Selection Function (Equation 4 of the research proposal):
#     1. A_nadir(i, j): Precise off-nadir incidence angle calculation & FOV filter.
#     2. D_downlink(i): Ground station line-of-sight visibility & wait-time reward.
#
# Kinematic Strip Adaptation:
#   Each Cell 4 event is a one-minute push-broom strip. A tile is imaged when
#   the ground track passes abeam of it (closest point of approach, CPA), so
#   its off-nadir angle comes from its great-circle distance to the track:
#   real fleets use the distance to the arcs between the strip's 2-s samples
#   (exact between samples), and each tile observation belongs to the ONE
#   strip that contains its CPA (v1.5.1; checked against an independent SGP4
#   propagation, xval_sgp4.py); the Starlink proxy keeps the nearest sample.
#   The samples are reused from Cell 4 rather than re-propagated. Sunlit tiles
#   only (each mission's limit), and passes inside a sourced acquisition
#   exclusion (Cell 0, v1.5.2) are dropped.
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 5 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["5"] = _ARF_CELL_VERSION

import os
import sys
import time
import logging
from typing import Dict, List, Tuple, Set, Optional, Any
from dataclasses import dataclass

import numpy as np
import pandas as pd
from org.orekit.frames import TopocentricFrame
from org.orekit.bodies import GeodeticPoint
from org.hipparchus.util import FastMath
from org.orekit.utils import Constants
import h3

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logger = logging.getLogger("arf.cell5")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# =============================================================================
# CUSTOM EXCEPTIONS
# =============================================================================

class RefinementError(Exception):
    """Raised when geometric or downlink refinement fails."""
    pass

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()

# =============================================================================
# REFINEMENT CONFIGURATION
# =============================================================================

@dataclass
class RefinementConfig:
    """Physical refinement parameters."""
    downlink_min_elevation_deg: float = 15.0
    downlink_coarse_step_min: int = 5
    t_max_wait_minutes: float = 95.0
    swath_half_angle_deg: float = 9.22
    wgs84_re_km: float = Constants.WGS84_EARTH_EQUATORIAL_RADIUS / 1000.0

    def __post_init__(self):
        if 'DOWNLINK_MIN_ELEVATION_DEG' in globals():
            self.downlink_min_elevation_deg = DOWNLINK_MIN_ELEVATION_DEG
        if 'T_MAX_WAIT_MINUTES' in globals():
            self.t_max_wait_minutes = T_MAX_WAIT_MINUTES
        if 'SWATH_HALF_ANGLE_DEG' in globals():
            self.swath_half_angle_deg = SWATH_HALF_ANGLE_DEG

refine_config = RefinementConfig()
_real_fleet5 = globals().get("FLEET", "starlink") != "starlink"
logger.info(f"Refinement Parameters: Min Elevation = {refine_config.downlink_min_elevation_deg}°, "
            f"T_max = {refine_config.t_max_wait_minutes:.1f} min, "
            + ("swath edge: each satellite's own (MISSION_TABLE)" if _real_fleet5 else
               f"Swath Half-Angle = {refine_config.swath_half_angle_deg}° (Starlink proxy)"))

# =============================================================================
# SATELLITE ACCESS HELPERS
# =============================================================================

def get_satellite_id(sat: Any) -> str:
    if hasattr(sat, 'name'):
        return sat.name
    elif isinstance(sat, dict):
        return sat.get('id', sat.get('name', 'Unknown'))
    return str(sat)

def get_satellite_propagator(sat: Any) -> Any:
    if hasattr(sat, 'propagator'):
        return sat.propagator
    elif isinstance(sat, dict):
        return sat.get('propagator')
    raise RefinementError(f"No propagator found in {type(sat).__name__}")

# =============================================================================
# GROUND STATION TOPOCENTRIC FRAMES
# =============================================================================

gs_frames: Dict[str, TopocentricFrame] = {}
for name, gs in GROUND_STATIONS.items():
    gp = GeodeticPoint(
        FastMath.toRadians(gs["lat"]),
        FastMath.toRadians(gs["lng"]),
        gs["alt_m"]
    )
    gs_frames[name] = TopocentricFrame(earth, gp, name)

# =============================================================================
# DOWNLINK WINDOW DETECTION
# =============================================================================

def _visibility_radius_km(altitude_km: float, min_elev_deg: float) -> float:
    """
    Ground (great-circle) distance from a station within which a satellite at
    altitude_km appears at or above min_elev_deg:
        lambda = arccos(Re * cos(E) / (Re + h)) - E.
    At E = 0 this is the horizon distance (~2,559 km at 550 km altitude).
    """
    E = np.radians(min_elev_deg)
    return float(Re_km * (np.arccos(Re_km * np.cos(E) / (Re_km + altitude_km)) - E))


def detect_downlink_windows(
    constellation: List[Any],
    sim_epoch: Any,
    window_hours: int,
    min_elev_deg: float = refine_config.downlink_min_elevation_deg,
    coarse_step_min: int = refine_config.downlink_coarse_step_min,
    fine_step_sec: int = 10,
    verbose: bool = True
) -> pd.DataFrame:
    """
    Records every satellite-to-ground-station contact minute at or above the
    elevation mask.

    Stage 1 (coarse, every coarse_step_min): flag a station when the
    sub-satellite point lies within the visibility radius of the HIGHEST
    satellite (+15 km altitude pad) plus the distance any satellite can travel
    in half a coarse step. The nearest coarse sample to any contact instant is
    then guaranteed to be flagged, so no pass can fall between samples. (A
    previous elevation-slack test -- flag at >= mask - 5 deg -- missed 2-3% of
    passes: low passes that clear the mask but never exceed mask - 5 deg at a
    5-minute sample.)

    Stage 2 (fine, every fine_step_sec within half a coarse step of a flag): exact
    Orekit topocentric elevation. A contact at time t is recorded against the
    next whole minute, ceil(t), so sub-minute passes are kept and the wait
    derived from this table is never under-estimated. Each satellite is
    propagated once per time step for all stations. Resolution limit: a contact
    above the mask for less than fine_step_sec can be missed -- validated
    residuals were 3-9 s grazing passes, too brief to be usable downlinks.
    """
    window_minutes = window_hours * 60
    win_sec = window_minutes * 60
    step_sec = coarse_step_min * 60
    max_alt = sensor_config.max_altitude_km + 15.0
    reach_km = (_visibility_radius_km(max_alt, min_elev_deg)
                + sensor_config.max_ground_speed_km_min * coarse_step_min / 2.0)
    gs_ll = {n: (float(GROUND_STATIONS[n]["lat"]), float(GROUND_STATIONS[n]["lng"])) for n in gs_frames}

    t0 = time.time()
    n_coarse_hits = n_fine_samples = 0
    best = {}                                    # (sat, gs, minute) -> max elevation
    coarse_secs = list(range(0, win_sec + 1, step_sec))
    if coarse_secs[-1] != win_sec:
        coarse_secs.append(win_sec)

    for sat in constellation:
        sat_id = get_satellite_id(sat)
        propagator = get_satellite_propagator(sat)

        # Stage 1: one propagation per coarse step, tested against all stations.
        windows = {n: [] for n in gs_frames}
        for ts in coarse_secs:
            lat, lon, _ = _subsatellite(propagator, ts / 60.0)
            for n, (gla, glo) in gs_ll.items():
                if _hav_km(lat, lon, gla, glo) <= reach_km:
                    # Every contact instant lies within step/2 of a flagged coarse
                    # sample, so +/-(step/2 + one fine step) is sufficient.
                    half = step_sec // 2 + fine_step_sec
                    windows[n].append((max(0, ts - half), min(win_sec, ts + half)))
                    n_coarse_hits += 1
        windows = {n: w for n, w in windows.items() if w}
        if not windows:
            continue

        # Stage 2: exact elevation, one propagation per fine sample.
        fine_secs = sorted({s for w in windows.values() for lo, hi in w
                            for s in range(lo, hi + 1, fine_step_sec)})
        n_fine_samples += len(fine_secs)
        for ts in fine_secs:
            t = sim_epoch.shiftedBy(float(ts))
            pos = propagator.getPVCoordinates(t, itrf).getPosition()
            for n, w in windows.items():
                if not any(lo <= ts <= hi for lo, hi in w):
                    continue
                elev = FastMath.toDegrees(gs_frames[n].getElevation(pos, itrf, t))
                if elev >= min_elev_deg:
                    key = (sat_id, n, -(-ts // 60))       # ceil to the next whole minute
                    if elev > best.get(key, -90.0):
                        best[key] = elev

    if best:
        df = pd.DataFrame([{"sat": s, "gs": g, "t_min": int(m), "elev": e}
                           for (s, g, m), e in sorted(best.items())])
    else:
        df = pd.DataFrame(columns=["sat", "gs", "t_min", "elev"])

    if verbose:
        logger.info(f"Downlink scan: {n_coarse_hits} coarse hits, {n_fine_samples} fine samples, "
                    f"{time.time() - t0:.1f}s | reach radius {reach_km:.0f} km")
        logger.info(f"Downlink Opportunities: {len(df)} contact minutes identified across "
                    f"{df['sat'].nunique() if len(df) > 0 else 0} satellites.")

    return df

logger.info("Scanning for ground station downlink contact windows...")
_window_hours = WINDOW_HOURS if 'WINDOW_HOURS' in globals() else 48

downlink_events_df = detect_downlink_windows(
    constellation=constellation,
    sim_epoch=sim_epoch,
    window_hours=_window_hours
)

_downlink_lookup = (
    downlink_events_df.groupby("sat")["t_min"]
    .apply(lambda x: sorted(x.unique()))
    .to_dict()
    if len(downlink_events_df) > 0 else {}
)

def time_to_next_downlink(sat_id: str, event_t_min: int) -> Optional[int]:
    """Minutes from event_t_min to the satellite's first contact minute at or
    after it (None if none in the window). Binary search (v1.5.2; was a
    linear scan, same result) over the sorted contact minutes."""
    import bisect
    minutes = _downlink_lookup.get(sat_id, [])
    i = bisect.bisect_left(minutes, event_t_min)
    return int(minutes[i] - event_t_min) if i < len(minutes) else None

def downlink_reward(t_wait_min: Optional[int], t_max_min: float = refine_config.t_max_wait_minutes) -> float:
    if t_wait_min is None:
        return 0.0
    return max(0.0, 1.0 - t_wait_min / t_max_min)

# =============================================================================
# VECTORISED SPHERICAL TRIGONOMETRY (A_nadir)
# =============================================================================

def _incidence_angle_deg_vec(ground_distance_km: np.ndarray, Re_km: float, Rs_km: float) -> np.ndarray:
    """
    Sensor off-nadir angle (deg) to a ground point at the given great-circle
    distance from the sub-satellite point, for a satellite at geocentric
    radius Rs_km over a sphere of radius Re_km. Points beyond the horizon are
    returned as +inf: without that guard, arcsin(Re*sin(gamma)/slant) turns
    small again on the far side of the Earth and would report a spurious
    near-nadir angle for an unobservable point.
    """
    gamma = np.asarray(ground_distance_km, dtype=float) / Re_km
    slant_range = np.sqrt(Rs_km**2 + Re_km**2 - 2.0 * Rs_km * Re_km * np.cos(gamma))
    sin_eta = np.clip(Re_km * np.sin(gamma) / slant_range, -1.0, 1.0)
    eta = np.degrees(np.arcsin(sin_eta))
    return np.where(gamma < np.arccos(Re_km / Rs_km), eta, np.inf)


def solar_elevation_deg(t_utc, lat_deg, lon_deg):
    """
    Solar elevation (degrees above the horizon, geometric, no refraction) at
    ground points (lat_deg, lon_deg, east-positive) and UTC datetime(s) t_utc,
    by NOAA GML "General Solar Position Calculations"
    (https://gml.noaa.gov/grad/solcalc/solareqns.PDF): fractional year,
    equation of time, declination, true solar time, hour angle, zenith.
    Vectorised over arrays of points; t_utc a datetime or an array of them.
    """
    t = np.atleast_1d(np.asarray(t_utc, dtype="datetime64[s]"))
    years = t.astype("datetime64[Y]")
    doy = (t.astype("datetime64[D]") - years).astype(int) + 1
    yr = years.astype(int) + 1970
    ndays = np.where((yr % 4 == 0) & ((yr % 100 != 0) | (yr % 400 == 0)), 366, 365)
    sec = (t - t.astype("datetime64[D]")).astype(int)
    hour = sec / 3600.0
    g = 2.0 * np.pi / ndays * (doy - 1 + (hour - 12.0) / 24.0)
    eqtime = 229.18 * (0.000075 + 0.001868 * np.cos(g) - 0.032077 * np.sin(g)
                       - 0.014615 * np.cos(2 * g) - 0.040849 * np.sin(2 * g))
    decl = (0.006918 - 0.399912 * np.cos(g) + 0.070257 * np.sin(g) - 0.006758 * np.cos(2 * g)
            + 0.000907 * np.sin(2 * g) - 0.002697 * np.cos(3 * g) + 0.00148 * np.sin(3 * g))
    tst = hour * 60.0 + eqtime + 4.0 * np.asarray(lon_deg)           # timezone 0 (UTC)
    ha = np.radians(tst / 4.0 - 180.0)
    lat = np.radians(lat_deg)
    cos_zen = np.sin(lat) * np.sin(decl) + np.cos(lat) * np.cos(decl) * np.cos(ha)
    return 90.0 - np.degrees(np.arccos(np.clip(cos_zen, -1.0, 1.0)))


# Per-satellite sensor geometry and daylight limit (real fleets); the Starlink
# proxy has neither and keeps the uniform half-angle with no daylight limit.
_sat_meta = {s["id"]: s for s in constellation} if 'constellation' in globals() else {}


def _min_track_distance_km(s_lat, s_lon, s_rs, c_lat, c_lon, re_km, exclude_caps=True):
    """
    Minimum great-circle distance (km) from each ground point (c_lat, c_lon) to
    the sub-satellite GROUND TRACK through the strip samples -- to the arcs
    between consecutive samples, not just the samples. Point sampling misses a
    tile lying between samples (2-s samples are ~15 km apart), which is
    negligible for a ~80 km footprint but loses whole tiles for an 8 km swath.

    exclude_caps (v1.5.1): a push-broom or frame imager sees a ground point when
    the track passes ABEAM of it (its closest point of approach, CPA). A point
    whose CPA lies before the strip's first sample or after its last is imaged
    in the neighbouring strip, so its distance here is +inf: the foot of its
    perpendicular on the great circle of the first (last) arc lies before the
    first (after the last) sample. Up to v1.5.0 such points took their distance
    to the end sample, so every strip also claimed a half-disc ("end cap") of
    radius half the swath beyond each end: 73-80% of the Sentinel-2 tile
    observations of the 12 October CKGR runs appeared in two consecutive strips
    (checked against an independent SGP4 propagation). Samples clipped to the
    window edges are merged first, so a strip at t = 0 or at the window end ends
    there.
    Returns (distance_km, geocentric radius of the nearest arc) per point;
    distance +inf where the CPA lies outside the strip (exclude_caps).
    """
    def unit(lat, lon):
        la, lo = np.radians(lat), np.radians(lon)
        return np.stack([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)], axis=-1)
    tol = 1e-12                                                              # rad: rounding at arc ends
    P = unit(np.asarray(s_lat, float), np.asarray(s_lon, float))              # (k, 3) samples
    rs = np.asarray(s_rs, float)
    if exclude_caps and len(P) > 1:
        # merge repeated samples (strips clipped at the window edges)
        keep = np.r_[True, np.linalg.norm(np.diff(P, axis=0), axis=1) > 1e-15]
        P, rs = P[keep], rs[keep]
    C = unit(np.asarray(c_lat, float), np.asarray(c_lon, float))              # (m, 3) points
    best = np.arccos(np.clip(C @ P.T, -1.0, 1.0))                            # (m, k) to samples
    best_ang, best_idx = best.min(axis=1), best.argmin(axis=1)
    rs_best = rs[best_idx]
    for i in range(len(P) - 1):
        A, B = P[i], P[i + 1]
        n = np.cross(A, B)
        nn = np.linalg.norm(n)
        if nn < 1e-15:
            continue
        n = n / nn
        sin_xt = C @ n                                                        # cross-track
        F = C - np.outer(sin_xt, n)                                           # foot on the great circle
        t_arc = tol if exclude_caps else 0.0                                  # legacy: exact v1.5.0 test
        on_arc = ((np.cross(A, F) @ n) >= -t_arc) & ((np.cross(F, B) @ n) >= -t_arc)
        ang = np.abs(np.arcsin(np.clip(sin_xt, -1.0, 1.0)))
        better = on_arc & (ang < best_ang)
        best_ang = np.where(better, ang, best_ang)
        rs_best = np.where(better, 0.5 * (rs[i] + rs[i + 1]), rs_best)
    if exclude_caps and len(P) > 1:
        def _outside(A, B, before):
            n = np.cross(A, B)
            nn = np.linalg.norm(n)
            if nn < 1e-15:
                return np.zeros(len(C), dtype=bool)
            n = n / nn
            F = C - np.outer(C @ n, n)
            return ((np.cross(A, F) @ n) < -tol) if before else ((np.cross(F, B) @ n) < -tol)
        outside = _outside(P[0], P[1], True) | _outside(P[-2], P[-1], False)
        best_ang = np.where(outside, np.inf, best_ang)
    return best_ang * re_km, rs_best


def _sim_epoch_datetime64():
    """sim_epoch (Orekit AbsoluteDate) as numpy datetime64 UTC."""
    from org.orekit.time import TimeScalesFactory
    return np.datetime64(sim_epoch.toString(TimeScalesFactory.getUTC()).replace("Z", "")[:26])


def _acquisition_exclusion_windows():
    """v1.5.2: ACQUISITION_EXCLUSIONS (Cell 0) as {satellite id: [(start_min,
    end_min, entry index)]}, minutes from the analysis start. A satellite id
    may carry a ' [NORAD id]' suffix (Cell 2, duplicate names); the name before
    it is matched."""
    import re as _re5
    out = {}
    excl = globals().get("ACQUISITION_EXCLUSIONS") or []
    if not excl or not _sat_meta or globals().get("FLEET", "starlink") == "starlink":
        return out
    t0 = _sim_epoch_datetime64()
    for i, ex in enumerate(excl):
        a = float((np.datetime64(ex["start_utc"]) - t0) / np.timedelta64(1, "s")) / 60.0
        b = float((np.datetime64(ex["end_utc"]) - t0) / np.timedelta64(1, "s")) / 60.0
        for sid in _sat_meta:
            if _re5.match(ex["name_pattern"], str(sid).split(" [")[0].strip(), flags=_re5.IGNORECASE):
                out.setdefault(sid, []).append((a, b, i))
    return out


def refine_access_events(
    access_events_df: pd.DataFrame,
    constellation: List[Any],
    sim_epoch: Any,
    swath_half_angle_deg: float = refine_config.swath_half_angle_deg,
) -> pd.DataFrame:
    """
    Exact incidence-angle FOV filter and downlink reward for each strip event.
    Uses the strip samples Cell 4 recorded (strip_samples); constellation and
    sim_epoch are kept in the signature for API compatibility. Distances use
    the same sphere (Re_km = WGS84 equatorial) as the incidence geometry.
    v1.5.2: passes inside a sourced acquisition exclusion (Cell 0) are dropped
    after the geometric and daylight filters and counted; the end-cap count of
    the summary is taken after the daylight filter, so it equals the rows the
    v1.5.0 rule would have added to this table.
    """
    # Accumulated as arrays (one chunk per event), not one dict per row: at
    # N=1000 country scale this table reaches tens of millions of rows.
    ev_sat, ev_t, ev_n, ev_d, ev_w, ev_theta = [], [], [], [], [], []
    tile_chunks, angle_chunks, sun_chunks = [], [], []
    global refine_daylight_stats
    refine_daylight_stats = {"events_dark": 0, "rows_dark": 0, "rows_lit": 0, "rows_end_cap_excluded": 0,
                             "rows_acquisition_excluded": 0, "passes_acquisition_excluded": 0,
                             "acquisition_exclusions_hit": {}}
    exclude_caps = not globals().get("LEGACY_STRIP_END_CAPS", False)        # v1.5.1 (Cell 0)
    _t0_utc = _sim_epoch_datetime64() if any(m.get("min_sun_elev_deg") is not None for m in _sat_meta.values()) else None
    _excl_win = _acquisition_exclusion_windows()
    logger.info(f"Evaluating physical constraints across {len(access_events_df):,} continuous strip events...")
    for row in access_events_df.itertuples(index=False):
        sat_id, t_min = row.sat, int(row.t_min)
        s_lat, s_lon, s_rs = strip_samples[(sat_id, t_min)]
        meta = _sat_meta.get(sat_id, {})
        excluded_by = next((i for a, b, i in _excl_win.get(sat_id, ()) if a <= t_min < b), None)
        cap_ok = None
        if meta.get("swath_km"):
            # Real fleets: exact distance to the ground-track arcs. The search
            # radius (Cell 4) includes half the sample spacing, so a tile
            # between samples is still a candidate.
            cand = _tiles_near(s_lat, s_lon, sensor_config.footprint_radius_km)
            if cand.size == 0:
                continue
            c_lat, c_lon = _tile_lat[cand], _tile_lon[cand]
            d, rs_near = _min_track_distance_km(s_lat, s_lon, s_rs, c_lat, c_lon, refine_config.wgs84_re_km,
                                                exclude_caps=exclude_caps)
            a_nadir = np.full(cand.size, np.inf)            # +inf: closest approach outside this strip
            fin = np.isfinite(d)
            if fin.any():
                a_nadir[fin] = _incidence_angle_deg_vec(d[fin], refine_config.wgs84_re_km, rs_near[fin])
            if exclude_caps and (~fin).any():
                # The end-cap observations the v1.5.0 rule would have kept,
                # found with that rule itself (legacy distance and the radius of
                # its nearest sample); counted below, after the daylight filter.
                d_old, rs_old = _min_track_distance_km(s_lat, s_lon, s_rs, c_lat[~fin], c_lon[~fin],
                                                       refine_config.wgs84_re_km, exclude_caps=False)
                th_end = float(_incidence_angle_deg_vec(meta["swath_km"] / 2.0, refine_config.wgs84_re_km,
                                                        float(np.mean(s_rs))))
                cap_ok = np.zeros(cand.size, dtype=bool)
                cap_ok[~fin] = _incidence_angle_deg_vec(d_old, refine_config.wgs84_re_km, rs_old) <= th_end
        else:
            # Starlink proxy: nearest strip sample, unchanged so earlier results
            # reproduce exactly (swath-edge error ~0.3 km against a ~80 km footprint).
            cand = _tiles_near(s_lat, s_lon, sensor_config.footprint_radius_km)
            if cand.size == 0:
                continue
            c_lat, c_lon = _tile_lat[cand], _tile_lon[cand]
            a_nadir = np.full(cand.size, np.inf)
            for la, lo, rs in zip(s_lat, s_lon, s_rs):
                d = _hav_km(la, lo, c_lat, c_lon)
                a_nadir = np.minimum(a_nadir, _incidence_angle_deg_vec(d, refine_config.wgs84_re_km, rs))
        if meta.get("swath_km"):
            # Off-nadir angle of this satellite's swath edge (half the swath on
            # the ground) at its mean geocentric radius over the strip.
            theta = float(_incidence_angle_deg_vec(meta["swath_km"] / 2.0, refine_config.wgs84_re_km,
                                                   float(np.mean(s_rs))))
        else:
            theta = swath_half_angle_deg
        within_fov = a_nadir <= theta
        if cap_ok is not None and cap_ok.any() and excluded_by is None:
            # v1.5.2: counted only where the v1.5.0 rule would also have kept
            # the row (sunlit), so the count equals the rows it would add.
            if meta.get("min_sun_elev_deg") is not None:
                _t_cap = _t0_utc + np.timedelta64(int(round(t_min * 60)), "s")
                _sun_cap = solar_elevation_deg(_t_cap, c_lat[cap_ok], c_lon[cap_ok])
                refine_daylight_stats["rows_end_cap_excluded"] += int((_sun_cap >= meta["min_sun_elev_deg"]).sum())
            else:
                refine_daylight_stats["rows_end_cap_excluded"] += int(cap_ok.sum())
        if not within_fov.any():
            continue
        sun = np.full(cand.size, np.nan)
        if meta.get("min_sun_elev_deg") is not None:
            # Optical imagers: drop tiles where the Sun is below the mission's
            # limit at the event minute (NOAA solar position, max error 0.46 deg).
            t_utc = _t0_utc + np.timedelta64(int(round(t_min * 60)), "s")
            sun[within_fov] = solar_elevation_deg(t_utc, c_lat[within_fov], c_lon[within_fov])
            lit = within_fov & (sun >= meta["min_sun_elev_deg"])
            refine_daylight_stats["rows_dark"] += int(within_fov.sum() - lit.sum())
            refine_daylight_stats["rows_lit"] += int(lit.sum())
            if not lit.any():
                refine_daylight_stats["events_dark"] += 1
                continue
            within_fov = lit
        if excluded_by is not None:
            # v1.5.2: inside a sourced acquisition exclusion (Cell 0): the
            # satellite is tracked but did not acquire, so the pass is dropped.
            _n_ex = int(within_fov.sum())
            refine_daylight_stats["rows_acquisition_excluded"] += _n_ex
            refine_daylight_stats["passes_acquisition_excluded"] += 1
            _hit = refine_daylight_stats["acquisition_exclusions_hit"].setdefault(str(excluded_by), {"passes": 0, "rows": 0})
            _hit["passes"] += 1
            _hit["rows"] += _n_ex
            continue

        # Downlink Reward (constant per event)
        d_wait = time_to_next_downlink(sat_id, t_min)
        d_reward = downlink_reward(d_wait)
        tile_chunks.append(cand[within_fov].astype(np.int32))
        angle_chunks.append(a_nadir[within_fov])
        sun_chunks.append(sun[within_fov])
        ev_theta.append(theta)
        ev_sat.append(sat_id); ev_t.append(t_min); ev_n.append(int(within_fov.sum()))
        ev_d.append(float(d_reward)); ev_w.append(np.nan if d_wait is None else float(d_wait))

    if len(tile_chunks) == 0:
        logger.warning("No tile events survived the physical FOV filter.")
        return pd.DataFrame(columns=["sat", "tile", "t_min", "A_nadir_deg", "D_downlink", "T_wait_min",
                                     "theta_max_deg", "sun_elev_deg"])

    # Compact columns: categorical sat/tile (tile categories = Cell 4's _tile_ids,
    # so codes are tile indices), int32 minutes, float64 physics values.
    counts = np.array(ev_n, dtype=np.int64)
    sat_cats = sorted(set(ev_sat))
    sat_code = {s: i for i, s in enumerate(sat_cats)}
    return pd.DataFrame({
        "sat": pd.Categorical.from_codes(np.repeat(np.array([sat_code[s] for s in ev_sat], dtype=np.int32), counts),
                                         categories=sat_cats),
        "tile": pd.Categorical.from_codes(np.concatenate(tile_chunks), categories=_tile_ids),
        "t_min": np.repeat(np.array(ev_t, dtype=np.int32), counts),
        "A_nadir_deg": np.concatenate(angle_chunks).astype(np.float64),
        "D_downlink": np.repeat(np.array(ev_d, dtype=np.float64), counts),
        "T_wait_min": np.repeat(np.array(ev_w, dtype=np.float64), counts),
        "theta_max_deg": np.repeat(np.array(ev_theta, dtype=np.float64), counts),   # swath-edge angle, this satellite
        "sun_elev_deg": np.concatenate(sun_chunks).astype(np.float64),             # NaN: no daylight limit
    })

# =============================================================================
# EXECUTION & POST-REFINEMENT ANALYSIS
# =============================================================================

assert 'access_events_df' in globals(), "access_events_df missing. Execute Cell 4."
assert 'constellation' in globals(), "constellation missing. Execute Cell 2."

logger.info("\nRefining continuous access strips with CPA incidence angles and downlink rewards...")
t0 = time.time()
refined_events_df = refine_access_events(
    access_events_df=access_events_df,
    constellation=constellation,
    sim_epoch=sim_epoch
)
t_refine = time.time() - t0
logger.info(f"Refinement complete in {t_refine:.2f}s ({len(refined_events_df):,} verified tile-access records).")

# =============================================================================
# EXPORT & SUMMARY
# =============================================================================

_refined_csv = os.path.join(_out_dir, 'arf_refined_events.csv')
_downlink_csv = os.path.join(_out_dir, 'arf_downlink_events.csv')
_summary_json = os.path.join(_out_dir, 'arf_cell5_summary.json')

try:
    refined_events_df.to_csv(_refined_csv, index=False)
    downlink_events_df.to_csv(_downlink_csv, index=False)
    logger.info(f"Refined datasets exported to {_refined_csv} and {_downlink_csv}")
except Exception as e:
    logger.warning(f"Could not save refined event CSVs: {e}")

_summary = {
    "n_access_events_input": len(access_events_df),
    "n_refined_events": len(refined_events_df),
    "n_unique_tiles": refined_events_df["tile"].nunique() if len(refined_events_df) > 0 else 0,
    "n_unique_sats": refined_events_df["sat"].nunique() if len(refined_events_df) > 0 else 0,
    "a_nadir_mean": float(refined_events_df['A_nadir_deg'].mean()) if len(refined_events_df) > 0 else 0.0,
    "d_downlink_mean": float(refined_events_df['D_downlink'].mean()) if len(refined_events_df) > 0 else 0.0,
    "n_downlink_windows": len(downlink_events_df),
    "refinement_time_s": t_refine,
    # v1.5.1: each tile observation belongs to the strip of its closest approach
    "strip_end_caps": "legacy (v1.5.0 and earlier)" if globals().get("LEGACY_STRIP_END_CAPS", False) else "excluded",
    # v1.5.2: counted after the daylight filter = rows the v1.5.0 rule would add
    "rows_end_cap_excluded": int(refine_daylight_stats.get("rows_end_cap_excluded", 0)),
    "rows_below_sun_limit": int(refine_daylight_stats.get("rows_dark", 0)),
    # v1.5.2: passes dropped by sourced acquisition exclusions (Cell 0)
    "rows_acquisition_excluded": int(refine_daylight_stats.get("rows_acquisition_excluded", 0)),
    "passes_acquisition_excluded": int(refine_daylight_stats.get("passes_acquisition_excluded", 0)),
    "acquisition_exclusions_hit": [
        {**{k: globals()["ACQUISITION_EXCLUSIONS"][int(i)][k] for k in ("name_pattern", "start_utc", "end_utc",
                                                                         "source_url")}, **v}
        for i, v in sorted(refine_daylight_stats.get("acquisition_exclusions_hit", {}).items())],
    "theta_max_deg_range": ([float(refined_events_df["theta_max_deg"].min()), float(refined_events_df["theta_max_deg"].max())]
                            if len(refined_events_df) > 0 else None),
}

try:
    with open(_summary_json, 'w') as f:
        json.dump(_summary, f, indent=2, default=str)
    logger.info(f"Cell 5 summary exported to {_summary_json}")
except Exception as e:
    logger.warning(f"Could not save Cell 5 summary: {e}")

print("\n" + "=" * 60)
print("  CELL 5: Physical Constraints & Refinement Complete")
print("=" * 60)
print(f"  Input Access Strips:       {len(access_events_df):,}")
print(f"  Refined Tile Records:      {len(refined_events_df):,}")
print(f"  Unique Valid Tiles Hit:    {_summary['n_unique_tiles']:,} / {len(target_tiles):,}")
# v1.5.2: real fleets have one swath-edge angle per satellite (Sentinel-2 ~10.3
# deg, Landsat ~7.5 deg, Doves ~1.5-2 deg); 9.22 deg is the Starlink proxy's.
_theta_txt = (f"swath edge {_summary['theta_max_deg_range'][0]:.2f}-{_summary['theta_max_deg_range'][1]:.2f}° by satellite"
              if _real_fleet5 and _summary["theta_max_deg_range"] else f"Max: {refine_config.swath_half_angle_deg}°")
print(f"  Mean Incidence Angle:      {_summary['a_nadir_mean']:.2f}° ({_theta_txt})")
print(f"  Mean Downlink Reward:      {_summary['d_downlink_mean']:.3f} / 1.000")
print(f"  Downlink Contact Windows:  {len(downlink_events_df):,}")
if _summary["rows_end_cap_excluded"]:
    print(f"  End-cap rows not counted:  {_summary['rows_end_cap_excluded']:,} (v1.5.0 rule; each tile kept in the "
          f"strip of its closest approach)")
if _summary["passes_acquisition_excluded"]:
    print(f"  Acquisition exclusions:    {_summary['passes_acquisition_excluded']} passes, "
          f"{_summary['rows_acquisition_excluded']:,} rows dropped (sourced, Cell 0)")
print("=" * 60)
print("\n=== CELL 5 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 6: SUBMODULAR GREEDY SELECTION WITH FAIL-SIGNAL GAP-FILLING
# =============================================================================
# Builds the candidate events (one-minute acquisition strips, Cell 5) and
# selects images:
#   - coverage-first: greedy on sum over tiles of the best utility, k images
#   - time-aware: greedy on the time-weighted coverage sum_t max_e w_e u_e,t
#     (TIME_AWARE_OBJECTIVE, Cell 0); both objectives are monotone submodular
#   - under cloud, the causal Fail-Signal selector: each image's cloud is known
#     only once it arrives; the plan for the remaining budget is revised from
#     what has been seen (CLOUD_PLANNING), then compensation (gap-fill)
#   - the clairvoyant greedy (cloud known in advance), reported for reference
# Utility of a tile observation: u = max(0, 1 - A_nadir / theta_max) * D_downlink
# (1 at nadir, 0 at the swath edge; D_downlink from Cell 5). This form was
# chosen for this pipeline (a design choice, not taken from a source).
# Scope of the (1 - 1/e) guarantee (v1.5.2 wording): it holds for the k-step
# greedy on a FIXED objective -- coverage-first and the facility time-aware
# objective, in clear sky or with the cloud known in advance (the oracle
# reference). The causal Fail-Signal selector re-plans after each image and
# adds compensatory images beyond k, so no approximation bound is claimed for
# it; its expected-coverage planner is itself a greedy on a monotone
# submodular function, but with the time-aware weighting it scores gain times
# weight (as the legacy objective did), a heuristic with no bound.
# Provenance: the cell was rebuilt in October 2026 from the contracts of the
# cells that use it (event keys (sat, t_min), cloud mask keyed by (event, tile)),
# when the working notebook was found to have lost it.
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 6 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["6"] = _ARF_CELL_VERSION

logger = logging.getLogger("arf.cell6")

import random
import heapq   # Fail-Signal gap-fill (lazy-greedy max-heap)

class EventStore:
    """
    Compact store of candidate events (compressed sparse rows). Event e owns
    rows offsets[e]:offsets[e+1] of the parallel arrays tile_idx (int32 index
    into _tile_ids, Cell 4) and util (float64 utility). Keys are (sat, t_min)
    in sorted order -- the same order the previous dict-of-dicts had -- and
    each event's downlink wait is kept once rather than once per tile.

    Memory: ~12 bytes per tile-access row instead of ~100+ for a nested
    Python dict, which is what lets N=1000 country-scale runs fit in Colab.

    Read access used by downstream cells still works as before:
    len(event_data), event_data.keys(), key in event_data, and
    event_data[key].items() (materialised on demand for that one event).
    """
    def __init__(self, keys, offsets, tile_idx, util, waits):
        self.keys_ = list(keys)
        self.index = {k: i for i, k in enumerate(self.keys_)}
        self.offsets = np.asarray(offsets, dtype=np.int64)
        self.tile_idx = np.asarray(tile_idx, dtype=np.int32)
        self.util = np.asarray(util, dtype=np.float64)
        self.wait_lookup = dict(zip(self.keys_, waits))
        self.counts = np.diff(self.offsets)
        self.n_rows = int(self.offsets[-1]) if len(self.offsets) else 0
        # v1.5.2: every event owns at least one row. The selectors sum per event
        # with np.add.reduceat(x, offsets[:-1]), which for an empty segment
        # returns the NEXT row's value instead of 0 (NumPy reduceat semantics),
        # so an empty event would silently get another event's gain.
        if len(self.keys_) and not (self.counts.size == len(self.keys_) and (self.counts >= 1).all()
                                    and self.n_rows == self.tile_idx.size == self.util.size):
            raise ValueError("EventStore: every event needs >= 1 row and offsets must match the row arrays.")

    def __len__(self):
        return len(self.keys_)

    def __contains__(self, key):
        return key in self.index

    def keys(self):
        return list(self.keys_)

    def rows(self, e):
        return slice(int(self.offsets[e]), int(self.offsets[e + 1]))

    def __getitem__(self, key):
        s = self.rows(self.index[key])
        return dict(zip(_tile_ids[self.tile_idx[s]], self.util[s].tolist()))


# Rank of each ROI tile in sorted-ID order: reproduces the previous cloud
# mask's "sorted(tiles)" draw order exactly, without sorting strings per event.
_tile_string_rank = np.empty(len(_tile_ids), dtype=np.int64)
_tile_string_rank[np.argsort(_tile_ids, kind="stable")] = np.arange(len(_tile_ids))

# total_target_tiles is the module-level denominator run_submodular_greedy_selection's
# history entries use for coverage_pct. Cell 7 also computes this (as a
# convenience alias, using refined_events_df's tile-reach as a cross-check),
# but Cell 6 needs it available now, from Cell 1's target_tiles.
total_target_tiles = len(target_tiles)


def build_event_data(refined_df, theta_max=SWATH_HALF_ANGLE_DEG):
    """
    Build the EventStore from the refined events table (Cell 5's output).
    Utility combines incidence-angle geometry (1 at nadir, 0 at
    theta_max=SWATH_HALF_ANGLE_DEG -- the same swath-edge cutoff Cell 5
    already filters on) with D_downlink (already a 0..1 reward from Cell 5):
        utility = max(0, 1 - A_nadir/theta_max) * D_downlink
    Events are ordered by (sat, t_min); rows keep their Cell 5 order within
    each event, so greedy tie-breaks and sums match the previous dict form.
    """
    n = len(refined_df)
    if n == 0:
        return EventStore([], np.zeros(1, np.int64), np.zeros(0, np.int32), np.zeros(0), [])

    sat = refined_df["sat"]
    if not isinstance(sat.dtype, pd.CategoricalDtype):
        sat = sat.astype("category")
    sat = sat.cat.reorder_categories(sorted(sat.cat.categories))   # lexicographic key order
    sat_code = sat.cat.codes.to_numpy(np.int64)
    t_min = refined_df["t_min"].to_numpy(np.int64)

    tile = refined_df["tile"]
    if (isinstance(tile.dtype, pd.CategoricalDtype) and len(tile.cat.categories) == len(_tile_ids)
            and bool((np.asarray(tile.cat.categories) == _tile_ids).all())):
        tidx = tile.cat.codes.to_numpy(np.int32)
    else:
        pos = {t: i for i, t in enumerate(_tile_ids)}
        tidx = np.fromiter((pos[t] for t in tile), dtype=np.int32, count=n)

    # Real fleets: normalise to EACH satellite's own swath-edge angle.
    theta = (refined_df["theta_max_deg"].to_numpy(np.float64) if "theta_max_deg" in refined_df.columns
             else theta_max)
    util = ((1.0 - refined_df["A_nadir_deg"].to_numpy(np.float64) / theta).clip(min=0.0)
            * refined_df["D_downlink"].to_numpy(np.float64))
    wait = refined_df["T_wait_min"].to_numpy(np.float64)

    order = np.lexsort((np.arange(n), t_min, sat_code))            # stable within an event
    sc, tm = sat_code[order], t_min[order]
    starts = np.flatnonzero(np.r_[True, (sc[1:] != sc[:-1]) | (tm[1:] != tm[:-1])])
    cats = list(sat.cat.categories)
    keys = [(cats[sc[i]], int(tm[i])) for i in starts]
    w = wait[order][starts]
    waits = [None if np.isnan(x) else float(x) for x in w]
    return EventStore(keys, np.r_[starts, n], tidx[order], util[order], waits)


event_data = build_event_data(refined_events_df)
logger.info(f"Built event_data: {len(event_data)} candidate (sat, t_min) events, "
            f"{event_data.n_rows:,} tile-access rows.")


class CloudMask:
    """
    Per-row cloud flags aligned with an EventStore. .get((event_key, tile))
    is kept for downstream cells (Cell 7B), backed by a one-event cache.
    """
    def __init__(self, store, flags):
        self.store = store
        self.flags = np.asarray(flags, dtype=bool)
        self.n_cloudy = int(self.flags.sum())
        self._cache_e, self._cache = None, {}

    def __len__(self):
        return len(self.flags)

    def get(self, key, default=False):
        event_key, tile = key
        e = self.store.index.get(event_key)
        if e is None:
            return default
        if e != self._cache_e:
            s = self.store.rows(e)
            self._cache = dict(zip(_tile_ids[self.store.tile_idx[s]], self.flags[s].tolist()))
            self._cache_e = e
        return self._cache.get(tile, default)


# =============================================================================
# CLOUD MODEL (v1.5.0): see CLOUD_MODEL and CLOUD_PROBABILITY_SOURCE in Cell 0
# =============================================================================

def _default_climatology_dir():
    if os.path.isdir("/content/drive/MyDrive"):
        return "/content/drive/MyDrive/ARF_climatology"
    return os.path.join(os.getcwd(), "ARF_climatology")


CLIMATOLOGY_DIR = globals().get("CLIMATOLOGY_DIR") or _default_climatology_dir()
EARTHENV_URL = "https://data.earthenv.org/cloud/MODCF_monthlymean_{month:02d}.tif"
EARTHENV_CITATION = ("Wilson, A. M. & Jetz, W. (2016). Remotely sensed high-resolution global cloud dynamics for "
                     "predicting ecosystem and biodiversity distributions. PLoS Biol 14(3): e1002415, "
                     "doi:10.1371/journal.pbio.1002415. Data: EarthEnv Global 1-km Cloud Frequency v1 "
                     "(https://www.earthenv.org/cloud), licence CC BY-NC 4.0.")


def _tile_latlng():
    """(lat, lng) arrays of the ROI tile centres, aligned with _tile_ids (cached)."""
    key = (len(_tile_ids), str(_tile_ids[0]) if len(_tile_ids) else "", str(_tile_ids[-1]) if len(_tile_ids) else "")
    c = globals().get("_tile_latlng_cache")
    if c is None or c[0] != key:
        ll = np.array([h3.cell_to_latlng(t) for t in _tile_ids], dtype=np.float64).reshape(-1, 2)
        c = (key, ll[:, 0].copy(), ll[:, 1].copy())
        globals()["_tile_latlng_cache"] = c
    return c[1], c[2]


def _climatology_month():
    """(month, how it was chosen) for the per-tile cloud climatology."""
    if globals().get("CLOUD_CLIMATOLOGY_MONTH") is not None:
        return int(CLOUD_CLIMATOLOGY_MONTH), "set in Cell 0"
    if globals().get("ANALYSIS_EPOCH"):
        return int(str(ANALYSIS_EPOCH)[5:7]), "month of ANALYSIS_EPOCH"
    try:
        return int(str(_sim_epoch_datetime64())[5:7]), "month of the simulation start"
    except Exception:
        from datetime import datetime as _dt, timezone as _tz
        return _dt.now(_tz.utc).month, "current month (simulation start unavailable)"


def earthenv_cloud_probability(month, lat, lng, path=None):
    """
    Per-tile cloud probability from the EarthEnv 2000-2014 monthly mean cloud
    frequency (Wilson & Jetz 2016): the mean of the valid ~1-km pixels in the
    3 x 3 block around each tile centre (about one H3 res-7 tile, 5.2 km2).
    Stored values 0-10,000 are percent x 100; values above 10,000 are fill
    (EarthEnv data description, Cloud_DataDescription.pdf). Tiles with no valid
    pixel fall back to CLOUD_PROBABILITY and are counted. The file is read from
    CLIMATOLOGY_DIR, downloaded there once if missing. Returns (p, info).
    """
    try:
        import rasterio
    except ImportError:                     # optional dependency, only for cloud_probability_source="earthenv"
        import subprocess, sys as _sys
        subprocess.check_call([_sys.executable, "-m", "pip", "install", "-q", "rasterio"])
        import rasterio
    from rasterio.windows import Window
    path = path or os.path.join(CLIMATOLOGY_DIR, f"MODCF_monthlymean_{month:02d}.tif")
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path) or ".", exist_ok=True)
        import urllib.request
        url = EARTHENV_URL.format(month=month)
        logger.info(f"Downloading the EarthEnv cloud climatology (month {month:02d}) to {path} (one time) ...")
        urllib.request.urlretrieve(url, path + ".part")
        os.replace(path + ".part", path)
    lat, lng = np.asarray(lat, dtype=np.float64), np.asarray(lng, dtype=np.float64)
    with rasterio.open(path) as src:
        epsg = src.crs.to_epsg() if src.crs is not None else 4326
        if epsg != 4326:
            from rasterio.warp import transform as _warp
            xs, ys = _warp("EPSG:4326", src.crs, lng.tolist(), lat.tolist())
            xs, ys = np.asarray(xs), np.asarray(ys)
        else:
            xs, ys = lng, lat
        inv = ~src.transform
        cols, rows = inv * (xs, ys)
        rows, cols = np.floor(rows).astype(np.int64), np.floor(cols).astype(np.int64)
        r0, c0 = max(int(rows.min()) - 1, 0), max(int(cols.min()) - 1, 0)
        r1, c1 = min(int(rows.max()) + 2, src.height), min(int(cols.max()) + 2, src.width)
        if r0 >= r1 or c0 >= c1:
            raise RuntimeError(f"The ROI lies outside {path}.")
        a = src.read(1, window=Window(c0, r0, c1 - c0, r1 - r0)).astype(np.float64)
        nodata = src.nodata
    bad = (a > 10000) | (a < 0) | ~np.isfinite(a)
    if nodata is not None:
        bad |= a == nodata
    a[bad] = np.nan
    vals = []
    for di in (-1, 0, 1):
        for dj in (-1, 0, 1):
            rr = np.clip(rows - r0 + di, 0, a.shape[0] - 1)
            cc = np.clip(cols - c0 + dj, 0, a.shape[1] - 1)
            vals.append(a[rr, cc])
    with np.errstate(all="ignore"):
        pct = np.nanmean(np.vstack(vals), axis=0) * 0.01           # % cloudy days
    miss = ~np.isfinite(pct)
    p = np.where(miss, float(CLOUD_PROBABILITY), pct / 100.0)
    p = np.clip(p, 0.001, 0.999)
    info = {"source": "earthenv", "month": int(month), "path": path, "citation": EARTHENV_CITATION,
            "n_tiles": int(p.size), "n_tiles_without_data": int(miss.sum()),
            "mean": float(p.mean()), "min": float(p.min()), "max": float(p.max())}
    return p, info


def cloud_probability_per_tile():
    """Marginal cloud probability of every ROI tile (aligned with _tile_ids) and how it was set."""
    if globals().get("CLOUD_PROBABILITY_SOURCE", "constant") == "earthenv":
        month, how = _climatology_month()
        lat, lng = _tile_latlng()
        p, info = earthenv_cloud_probability(month, lat, lng)
        info["month_from"] = how
        return p, info
    return (np.full(len(_tile_ids), float(CLOUD_PROBABILITY)),
            {"source": "constant", "value": float(CLOUD_PROBABILITY)})


def _gauss_kernel1d(s_cells, truncate=4.0):
    """Normalised 1-D Gaussian kernel, as scipy.ndimage.gaussian_filter1d builds it."""
    radius = int(truncate * s_cells + 0.5)
    x = np.arange(-radius, radius + 1, dtype=np.float64)
    k = np.exp(-0.5 * (x / s_cells) ** 2)
    return k / k.sum()


class LatentCloudField:
    """
    Unit-variance Gaussian field on a local equirectangular plane around the ROI
    with correlation exp(-d^2 / L^2) (e-folding distance L = length_km): white
    noise filtered by a separable Gaussian kernel of standard deviation L/2,
    on a grid of spacing min(L/6, 10 km), padded by 4.5 kernel widths and
    wrapped (opposite ROI edges are then >= 4.5 L apart through the wrap).
    Values at tile centres are bilinear interpolations; their exact variance
    (computed from the discrete kernel) rescales each tile's threshold, so the
    marginal cloud probability of every tile is exactly its p.
    """
    def __init__(self, length_km):
        from scipy.ndimage import correlate1d
        from scipy.special import ndtri
        self._correlate1d, self._ndtri = correlate1d, ndtri
        lat, lng = _tile_latlng()
        R = 6371.0088
        lat0, lng0 = np.radians(lat.mean()), np.radians(lng.mean())
        x = R * (np.radians(lng) - lng0) * np.cos(lat0)
        y = R * (np.radians(lat) - lat0)
        self.length_km = float(length_km)
        sigma = self.length_km / 2.0
        self.dx = min(self.length_km / 6.0, 10.0)
        pad = 4.5 * sigma
        x0, y0 = x.min() - pad, y.min() - pad
        self.nx = int(np.ceil((x.max() + pad - x0) / self.dx)) + 2
        self.ny = int(np.ceil((y.max() + pad - y0) / self.dx)) + 2
        fx, fy = (x - x0) / self.dx, (y - y0) / self.dx
        self.ix, self.iy = np.floor(fx).astype(np.int64), np.floor(fy).astype(np.int64)
        wx, wy = fx - self.ix, fy - self.iy
        self.k1 = _gauss_kernel1d(sigma / self.dx)
        ss = float((self.k1 ** 2).sum())
        self.norm = ss                                       # sd of the 2-D filtered noise = sum(k1^2)
        c1 = float((self.k1[:-1] * self.k1[1:]).sum() / ss)  # correlation at a lag of one cell
        var_x = (1 - wx) ** 2 + wx ** 2 + 2 * (1 - wx) * wx * c1
        var_y = (1 - wy) ** 2 + wy ** 2 + 2 * (1 - wy) * wy * c1
        self.sd = np.sqrt(var_x * var_y)
        self.w00, self.w10 = (1 - wx) * (1 - wy), wx * (1 - wy)
        self.w01, self.w11 = (1 - wx) * wy, wx * wy

    def draw(self, rng):
        """One independent unit-variance field."""
        z = rng.standard_normal((self.ny, self.nx))
        z = self._correlate1d(z, self.k1, axis=0, mode="wrap")
        z = self._correlate1d(z, self.k1, axis=1, mode="wrap")
        return z / self.norm

    def at_tiles(self, z, tiles):
        ix, iy = self.ix[tiles], self.iy[tiles]
        return (self.w00[tiles] * z[iy, ix] + self.w10[tiles] * z[iy, ix + 1]
                + self.w01[tiles] * z[iy + 1, ix] + self.w11[tiles] * z[iy + 1, ix + 1])

    def thresholds(self, p_tile):
        """Per-tile threshold on the interpolated field giving marginal probability p_tile."""
        return self._ndtri(1.0 - np.asarray(p_tile, dtype=np.float64)) * self.sd


def _latent_field(length_km):
    cache = globals().setdefault("_latent_field_cache", {})
    key = (float(length_km), len(_tile_ids), str(_tile_ids[0]) if len(_tile_ids) else "",
           str(_tile_ids[-1]) if len(_tile_ids) else "")
    if key not in cache:
        cache[key] = LatentCloudField(length_km)
    return cache[key]


def build_cloud_mask(event_data, cloud_probability, random_seed, model=None, length_km=None, time_h=None,
                     p_tile=None):
    """
    Deterministic cloud mask over every (event, tile) row (seeded).
    model "iid" (v1.4.0 and earlier): events visited in sorted key order and
      tiles in sorted-ID order, drawing from random.Random(random_seed) -- the
      exact draw sequence of every earlier run when the probability is constant.
    model "correlated" (v1.5.0): the latent field (LatentCloudField) at each
      distinct pass time, evolved between times as AR(1),
      Z(t2) = rho Z(t1) + sqrt(1 - rho^2) E, rho = exp(-(t2 - t1) / time_h),
      E an independent field; time_h = 0 makes pass times independent. Passes at
      the same minute see the same clouds. Drawn from numpy's default_rng(seed).
    p_tile: per-tile probability (aligned with _tile_ids); default from
      CLOUD_PROBABILITY_SOURCE (constant cloud_probability, or the climatology).
    """
    model = model or globals().get("CLOUD_MODEL", "iid")
    store = event_data
    flags = np.zeros(store.n_rows, dtype=bool)
    if p_tile is None:
        p_tile = (CLOUD_P_TILE if globals().get("CLOUD_PROBABILITY_SOURCE", "constant") != "constant"
                  and "CLOUD_P_TILE" in globals() else None)
    if model == "iid":
        rng = random.Random(random_seed)
        for e in sorted(range(len(store)), key=lambda i: store.keys_[i]):
            s = store.rows(e)
            rows = np.arange(s.start, s.stop)
            rows = rows[np.argsort(_tile_string_rank[store.tile_idx[s]], kind="stable")]
            if p_tile is None:
                flags[rows] = [rng.random() < cloud_probability for _ in range(rows.size)]
            else:
                pr = p_tile[store.tile_idx[rows]]
                flags[rows] = [rng.random() < pr[i] for i in range(rows.size)]
        return CloudMask(store, flags)
    if store.n_rows == 0:
        return CloudMask(store, flags)
    length_km = float(CLOUD_CORR_LENGTH_KM if length_km is None else length_km)
    time_h = float(CLOUD_CORR_TIME_H if time_h is None else time_h)
    field = _latent_field(length_km)
    p = np.full(len(_tile_ids), float(cloud_probability)) if p_tile is None else np.asarray(p_tile, dtype=np.float64)
    thr = field.thresholds(p)
    t_event = np.array([k[1] for k in store.keys_], dtype=np.float64)
    t_row = np.repeat(t_event, store.counts)
    times, inv = np.unique(t_row, return_inverse=True)
    order = np.argsort(inv, kind="stable")
    bounds = np.searchsorted(inv[order], np.arange(len(times) + 1))
    rng = np.random.default_rng(random_seed)
    z, t_prev = None, None
    for i, t in enumerate(times):
        e = field.draw(rng)
        if z is None:
            z = e
        else:
            rho = 0.0 if time_h <= 0 else float(np.exp(-(t - t_prev) / (60.0 * time_h)))
            z = rho * z + np.sqrt(max(0.0, 1.0 - rho * rho)) * e
        t_prev = t
        rows = order[bounds[i]:bounds[i + 1]]
        tiles = store.tile_idx[rows]
        flags[rows] = field.at_tiles(z, tiles) > thr[tiles]
    return CloudMask(store, flags)


CLOUD_P_TILE, cloud_climatology_info = cloud_probability_per_tile()
cloud_model_info = {"model": CLOUD_MODEL, "probability": cloud_climatology_info}
if CLOUD_MODEL == "correlated":
    _f = _latent_field(CLOUD_CORR_LENGTH_KM)
    cloud_model_info.update(length_km=float(CLOUD_CORR_LENGTH_KM), time_h=float(CLOUD_CORR_TIME_H),
                            grid_spacing_km=float(_f.dx), grid_cells=int(_f.nx * _f.ny),
                            reference="Wood, R. & Field, P. R. (2011). The distribution of cloud horizontal sizes. "
                                      "J. Climate 24, 4800-4816, doi:10.1175/2011JCLI4056.1 (scale range); "
                                      "the length and time scales are assumptions, swept in Cell 7C.")
cloud_mask = build_cloud_mask(event_data, CLOUD_PROBABILITY, RANDOM_SEED)
logger.info(f"Built cloud_mask ({CLOUD_MODEL}"
            + (f", L={CLOUD_CORR_LENGTH_KM:g} km, T={CLOUD_CORR_TIME_H:g} h" if CLOUD_MODEL == "correlated" else "")
            + f"): {cloud_mask.n_cloudy}/{len(cloud_mask)} tile-events cloud-obscured "
            + (f"({CLOUD_PROBABILITY:.0%} target)." if cloud_climatology_info["source"] == "constant" else
               f"(climatology month {cloud_climatology_info['month']:02d}, mean p = {cloud_climatology_info['mean']:.2f})."))


# =============================================================================
# DELIVERY ORDER AND THE TIME SCALE OF THE TIME-AWARE WEIGHT (v1.4.0)
# =============================================================================
# Delivery time = t_min + downlink wait. An image with no downlink contact in
# the window has no delivery time (and zero utility). Sorting on a NaN key
# leaves Python's sort unordered (sorted([3.0, nan, 1.0]) == [3.0, nan, 1.0]),
# so every delivery-order sort uses delivery_sort_key, which puts such images
# last. Real and test runs so far had no such image, so their results are
# unchanged; the fix matters for ground stations far from the ROI.
#
# Time-aware objective: each pass's marginal gain is weighted by
#     exp(-rate * t_min / tau).
# Up to v1.3.1 tau was one orbital period (T_MAX_WAIT_MINUTES). That suits a
# fleet that covers the ROI within about an orbit, but for a sparse fleet whose
# passes need a day to cover it, every pass after the first orbit is weighted
# almost to nothing (exp(-3 * 1440 / 94) ~ 1e-20), so the budget goes on the
# earliest small passes. Real run, CKGR + Okavango, Landsat + Sentinel-2 +
# PlanetScope, 2026-10-12: 91.0% clear-sky coverage at k = 30, against 100%
# for coverage-first. LATENCY_DECAY_SCALE (Cell 0):
#   "fleet" (default): tau = max(T_MAX_WAIT_MINUTES, F), where F is the
#       physical floor of the passes being selected from: the delivery time at
#       which ALL of them together cover H1_REACHABLE_COVERAGE_PCT of the tiles
#       they can reach, from clear-sky geometry (pass predictions, known before
#       any image is taken: no cloud information is used). A pass delivered at
#       the floor is weighted exp(-rate). If the fleet reaches that level within
#       one orbital period, tau is the orbital period, exactly as before.
#   "fixed": tau = T_MAX_WAIT_MINUTES (v1.3.1).
# tau is computed per EventStore, so each onset of Cell 7C's analysis uses the
# floor of its own passes.

def delivery_sort_key(store):
    """Sort key for (sat, t_min) keys of `store` by delivery time; images with
    no downlink in the window (no delivery time) sort last."""
    def key(k):
        w = store.wait_lookup.get(k)
        return np.inf if w is None or np.isnan(w) else k[1] + w
    return key


def floor_time(store, level_pct):
    """Physical floor: with ALL events of `store` in delivery order, the delivery
    time (min) at which they first cover level_pct of the tiles reachable from
    the store (utility > 0; clear-sky geometry). None if nothing is reachable.
    Same rule as Cell 7C's coverage_times(store, None, (level_pct,),
    denominator=reachable_tile_count(store, None))."""
    if len(store) == 0:
        return None
    usable = store.util > 0.0
    reach = int(np.unique(store.tile_idx[usable]).size)
    if reach == 0:
        return None
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n_seen = 0
    for k in sorted(store.keys_, key=delivery_sort_key(store)):
        w = store.wait_lookup.get(k)
        if w is None or np.isnan(w):
            break                                  # undelivered images sort last
        s = store.rows(store.index[k])
        t = store.tile_idx[s][usable[s]]
        new = t[~seen[t]]
        if new.size:
            seen[new] = True
            n_seen += new.size
            if 100.0 * n_seen / reach >= level_pct - 1e-9:
                return float(k[1] + w)
    return None


def decay_time_scale(store):
    """tau (minutes) of the time-aware weight for selections from `store`
    (see the section header). Cached on the store."""
    t_orbit = float(T_MAX_WAIT_MINUTES)
    if globals().get("LATENCY_DECAY_SCALE", "fleet") == "fixed":
        return t_orbit
    level = float(globals().get("H1_REACHABLE_COVERAGE_PCT", 95.0))
    cache = store.__dict__.setdefault("_tau_cache", {})
    if (level, t_orbit) not in cache:
        f = floor_time(store, level)
        cache[(level, t_orbit)] = t_orbit if f is None else max(t_orbit, f)
    return cache[(level, t_orbit)]


def time_aware_weights(store, latency_decay_rate):
    """Per-event weight exp(-rate * t / tau), t = DELIVERY time (t_min + downlink
    wait; v1.5.1 -- v1.5.0 used t_min, identical whenever every wait is 0, as in
    all runs so far), tau = decay_time_scale(store). An image never delivered has
    zero utility, so its weight does not matter (set to 0). Weights are floored
    at 1e-300: exp() underflows to exactly 0 beyond ~745 time scales (e.g. a
    16-day window on the fixed one-orbit scale), which would make a NEW tile
    unselectable under the facility objective."""
    tau = decay_time_scale(store)
    d = event_delivery_times(store)
    w = np.where(np.isfinite(d), np.exp(-latency_decay_rate * np.where(np.isfinite(d), d, 0.0) / tau), 0.0)
    return np.where(np.isfinite(d), np.maximum(w, 1e-300), 0.0)


def run_submodular_greedy_selection(
    events_dict, max_steps, cloud_mask=None, gap_fill=True,
    time_aware=False, latency_decay_rate=1.0,
):
    """
    Weighted maximum coverage, cardinality-constrained greedy, with
    Fail-Signal gap-filling and an optional time-aware objective.

    Implements the Nemhauser-Wolsey-Fisher (1978) greedy algorithm for
    submodular function maximisation. Selects the event with the
    highest marginal utility gain at each step, for exactly max_steps
    iterations, unless every remaining candidate can add zero new
    coverage (true information exhaustion). The (1-1/e) approximation
    guarantee applies directly to this unmodified cardinality-
    constrained weighted maximum coverage formulation.

    TIME-AWARE MODE (time_aware=True), each pass weighted
    w_e = exp(-latency_decay_rate * t_min / tau), tau =
    decay_time_scale(events_dict) (one orbital period, or the fleet's
    physical floor when that is longer -- see the section above):
      TIME_AWARE_OBJECTIVE "facility" (v1.5.0 default): greedy on the
        time-weighted coverage f(S) = sum_t max_{e in S} w_e * u_e,t, a
        weighted facility-location function -- monotone submodular, because
        the marginal gain of e, sum_t max(0, w_e u_e,t - max_{S} w u), can only
        shrink as S grows. The k-step greedy therefore attains at least
        (1 - 1/e) of the best k-pass value of f (Nemhauser, Wolsey & Fisher
        1978), and Cell 7C checks it against the exact ILP optimum of f.
        The greedy stops when no pass adds to f (a new tile always adds,
        since w_e > 0).
      "legacy" (v1.4.0): unweighted marginal gain times w_e. Those scores are
        not marginal gains of any one set function (two passes covering the
        same tile, weights 1 and 0.5, give the pair a value of 1 in one order
        and 0.5 in the other), so no guarantee applies.
    Reported coverage and quality (history, covered utility) are always the
    unweighted best utility per tile.

    FAIL-SIGNAL GAP-FILL: when cloud_mask is not None and gap_fill is
    True, a lazy-greedy set-cover search (Minoux 1978 acceleration --
    a max-heap of marginal new-tile counts, lazily re-validated on
    pop) fills any tiles left uncovered after the main k-step loop,
    always adding the event covering the most still-uncovered tiles,
    stopping when nothing remaining can cover anything outstanding.
    Never fires when cloud_mask is None (a clear-sky run has nothing
    to compensate for).

    IMPLEMENTATION: each step evaluates every event's marginal gain in one
    vectorised pass over the EventStore rows (gather covered utility,
    subtract, clip at zero, segment-sum). Ties go to the first event in
    key order, as in a sequential scan. Only per-tile state (covered
    utility, one float per ROI tile) is updated between steps.

    Args:
        events_dict: EventStore (Cell 6) -- event (sat, t_min) -> tiles, utilities
        max_steps: cardinality constraint k
        cloud_mask: optional CloudMask aligned with events_dict
        gap_fill: whether to run Fail-Signal gap-filling
        time_aware: bias selection toward earlier t_min
        latency_decay_rate: decay strength for time_aware mode

    Returns:
        (selected_events, covered_tiles_utility,
         cumulative_downlinked_tiles, history)
    """
    store = events_dict
    n_events = len(store)
    selected_events = []
    history = []
    cumulative_downlinked_tiles = 0
    covered = np.zeros(len(_tile_ids), dtype=np.float64)       # best utility per ROI tile
    if n_events == 0:
        return selected_events, {}, cumulative_downlinked_tiles, history

    tile_idx, starts = store.tile_idx, store.offsets[:-1]
    util = np.where(cloud_mask.flags, 0.0, store.util) if cloud_mask is not None else store.util
    selected = np.zeros(n_events, dtype=bool)
    weight = time_aware_weights(store, latency_decay_rate) if time_aware else None
    facility = time_aware and globals().get("TIME_AWARE_OBJECTIVE", "facility") == "facility"
    if facility:
        util_w = util * np.repeat(weight, store.counts)               # w_e * u_e,t per row
        covered_w = np.zeros(len(_tile_ids), dtype=np.float64)       # best weighted utility per tile
    buf = np.empty(store.n_rows, dtype=np.float64)

    for step in range(max_steps):
        if facility:
            np.take(covered_w, tile_idx, out=buf)
            np.subtract(util_w, buf, out=buf)
            np.maximum(buf, 0.0, out=buf)
            gains = np.add.reduceat(buf, starts)                       # marginal gain of f
            # A gain counts only if it exceeds 1e-12 in the pass's own utility
            # units (gain / w_e): rounding noise is ignored, while a new tile --
            # gain >= w_e * u -- always counts, however small w_e is.
            score = np.where(gains > 1e-12 * weight, gains, -np.inf)
            tol = 0.0
        else:
            np.take(covered, tile_idx, out=buf)
            np.subtract(util, buf, out=buf)
            np.maximum(buf, 0.0, out=buf)
            gains = np.add.reduceat(buf, starts)
            score = gains * weight if time_aware else gains
            tol = 1e-12
        score = np.where(selected, -np.inf, score)
        e = int(np.argmax(score))

        # Stop ONLY on exhaustion of the objective: coverage-first and legacy
        # time-aware stop when nothing adds real (unweighted) coverage; the
        # facility objective when nothing adds to f.
        if not np.isfinite(score[e]) or gains[e] <= tol:
            logger.info(f"  Stopped at step {step}: no remaining event adds "
                        f"to the objective (true ceiling for this dataset).")
            break

        selected[e] = True
        best_event = store.keys_[e]
        selected_events.append(best_event)
        cumulative_downlinked_tiles += int(store.counts[e])
        s = store.rows(e)
        np.maximum.at(covered, tile_idx[s], util[s])
        if facility:
            np.maximum.at(covered_w, tile_idx[s], util_w[s])

        cov_mask = covered > 0.0
        n_covered = int(cov_mask.sum())
        mean_quality = float(covered[cov_mask].sum() / n_covered) if n_covered else 0.0
        history.append({
            'step': step + 1,
            'selected': best_event,
            'coverage_pct': 100 * n_covered / total_target_tiles,
            'mean_quality': mean_quality,
        })

    # --- Fail-Signal gap-fill (lazy-greedy, heap-accelerated) ---
    if gap_fill and cloud_mask is not None and len(selected_events) > 0:
        in_access = np.zeros(len(_tile_ids), dtype=bool)
        in_access[tile_idx] = True
        remaining = in_access & ~(covered > 0.0)

        if remaining.any():
            n_uncovered_initial = int(remaining.sum())
            logger.info(f"  Gap-Filling: {n_uncovered_initial} tiles uncovered after greedy selection.")

            eligible = (store.util > 0.0) & ~cloud_mask.flags
            counts = np.add.reduceat((eligible & remaining[tile_idx]).astype(np.int64), starts)
            heap = [(-int(c), store.keys_[i]) for i, c in enumerate(counts) if c > 0 and not selected[i]]
            heapq.heapify(heap)

            gap_fill_count = 0
            while remaining.any() and heap:
                neg_cnt, event_key = heapq.heappop(heap)
                e = store.index[event_key]
                if selected[e]:
                    continue
                s = store.rows(e)
                true_cnt = int((eligible[s] & remaining[tile_idx[s]]).sum())
                if true_cnt == 0:
                    continue
                if heap and true_cnt < -heap[0][0]:
                    heapq.heappush(heap, (-true_cnt, event_key))
                    continue

                selected[e] = True
                selected_events.append(event_key)
                gap_fill_count += 1
                cumulative_downlinked_tiles += int(store.counts[e])
                t, u = tile_idx[s], util[s]
                improve = u > covered[t]
                covered[t[improve]] = u[improve]
                remaining[t[improve]] = False

            if remaining.any():
                logger.warning(f"  Gap-fill exhausted candidate events with "
                                f"{int(remaining.sum())} tiles still uncovered.")
            if gap_fill_count:
                logger.info(f"  Added {gap_fill_count} compensatory passes (Fail-Signal gap-fill).")

    covered_tiles_utility = {_tile_ids[i]: float(covered[i]) for i in np.flatnonzero(covered > 0.0)}
    return selected_events, covered_tiles_utility, cumulative_downlinked_tiles, history


# =============================================================================
# REALISTIC CLOUD KNOWLEDGE (non-clairvoyant selection under cloud)
# =============================================================================
# run_submodular_greedy_selection with a cloud mask chooses passes KNOWING
# which tiles each pass will see clear -- information no real system has
# before the image arrives. simulate_adaptive_selection is the causal
# version: plan on clear-sky geometry, reveal cloud when each image arrives,
# re-plan on failure (Fail-Signal). See CLOUD_KNOWLEDGE in Cell 0.
last_adaptive_run = {}


def event_delivery_times(store):
    """Delivery time (t_min + downlink wait, minutes) per event; +inf where
    no ground-station contact follows (such events have zero utility)."""
    return np.array([k[1] + (np.inf if store.wait_lookup.get(k) is None else store.wait_lookup[k])
                     for k in store.keys_], dtype=np.float64)


def _greedy_plan(store, state, candidates, budget, weight, util_plan=None):
    """Greedy utility-weighted max-coverage plan of up to `budget` events from
    `candidates` (bool per event), starting from per-tile best utility
    `state` (not modified). Uses CLEAR-SKY utilities only. Same selection
    rule and tie-breaking as run_submodular_greedy_selection. util_plan:
    per-row utilities to plan on (the time-weighted w * u of the facility
    objective, with `state` the best weighted utility per tile); default the
    clear-sky utilities, scored legacy-style (gain times weight)."""
    tile_idx, starts = store.tile_idx, store.offsets[:-1]
    util = store.util if util_plan is None else util_plan
    tol = 1e-12 if util_plan is None else 0.0
    state = state.copy()
    cand = candidates.copy()
    plan = []
    buf = np.empty(store.n_rows, dtype=np.float64)
    for _ in range(budget):
        np.take(state, tile_idx, out=buf)
        np.subtract(util, buf, out=buf)
        np.maximum(buf, 0.0, out=buf)
        gains = np.add.reduceat(buf, starts)
        if util_plan is None:
            score = gains * weight if weight is not None else gains
        else:                                      # facility objective: see run_submodular_greedy_selection
            score = np.where(gains > 1e-12 * weight, gains, -np.inf)
        score = np.where(cand, score, -np.inf)
        e = int(np.argmax(score))
        if not np.isfinite(score[e]) or gains[e] <= tol:
            break
        cand[e] = False
        plan.append(e)
        s = store.rows(e)
        np.maximum.at(state, tile_idx[s], util[s])
    return plan


def _expected_plan(store, uncovered, candidates, budget, p_tile, weight=None):
    """
    Greedy plan of up to `budget` events from `candidates` (bool per event)
    maximising the EXPECTED number of `uncovered` tiles (bool per ROI tile)
    that end up with a cloud-free usable view, each tile cloudy with
    probability p_tile in every pass independently (the planner's assumption;
    the simulated cloud may be correlated). The gain of a pass is the sum over
    its usable uncovered tiles of P(not yet covered by the plan) * (1 - p) --
    the greedy on sum_t [1 - prod_{e covering t} p_t], a monotone submodular
    function. weight: optional per-event multiplier (time-aware objective):
    the score is then gain times weight, which is not the marginal gain of any
    one set function, so with a weight this is a heuristic without the
    (1 - 1/e) bound (v1.5.2 note).
    """
    tile_idx, starts = store.tile_idx, store.offsets[:-1]
    usable = store.util > 0.0
    p = np.asarray(p_tile, dtype=np.float64)
    q = np.where(uncovered, 1.0, 0.0)
    one_minus_p = 1.0 - p
    cand = candidates.copy()
    plan = []
    for _ in range(int(budget)):
        g = np.where(usable, q[tile_idx] * one_minus_p[tile_idx], 0.0)
        raw = np.add.reduceat(g, starts)
        gains = raw * weight if weight is not None else raw
        gains = np.where(cand & (raw > 1e-12), gains, -np.inf)
        e = int(np.argmax(gains))
        if not np.isfinite(gains[e]):
            break
        cand[e] = False
        plan.append(e)
        s = store.rows(e)
        t = np.unique(tile_idx[s][usable[s]])
        q[t] *= p[t]
    return plan


def _planning_probabilities():
    pt = globals().get("CLOUD_P_TILE")
    return pt if pt is not None else np.full(len(_tile_ids), float(globals().get("CLOUD_PROBABILITY", 0.35)))


def simulate_adaptive_selection(events_dict, max_steps, cloud_mask, gap_fill=True,
                                time_aware=False, latency_decay_rate=1.0, planning=None, p_tile=None):
    """
    Causal (non-clairvoyant) selection under cloud. Same return values as
    run_submodular_greedy_selection. No decision uses the cloud flags of an
    image before it arrives.

    planning "expected" (v1.5.0 default, CLOUD_PLANNING in Cell 0), budget
    max_steps: plan the budget to maximise the EXPECTED coverage of the tiles
    still missing (_expected_plan, from each tile's cloud probability p_tile --
    climatology, not the realised cloud), time-weighted if time_aware. Execute
    in delivery order; each image's cloud is revealed when it arrives; after
    every arrival re-plan the remaining budget over images not yet arrived,
    from the coverage actually achieved. A planned new tile that comes back
    cloudy is a Fail-Signal (counted in `replans`).
    planning "clear" (v1.4.0): plan greedily on clear-sky geometry (the
    time-aware objective if time_aware) and re-plan only on a Fail-Signal (a
    tile a planned observation should have improved stays below its clear-sky
    utility). In clear sky there are no failures, so the selected set equals
    the clear-sky greedy plan.
    Phase 2 (gap_fill), both: after the budget, take each arriving image that
    clearly covers a still-uncovered accessible tile (decided on arrival).
    """
    store = events_dict
    n = len(store)
    if n == 0:
        last_adaptive_run.update(replans=0, compensatory=0)
        return [], {}, 0, []
    planning = planning or globals().get("CLOUD_PLANNING", "clear")
    tile_idx, util, flags = store.tile_idx, store.util, cloud_mask.flags
    realized = np.where(flags, 0.0, util)
    d = event_delivery_times(store)
    arrives = np.isfinite(d)
    weight = time_aware_weights(store, latency_decay_rate) if time_aware else None    # clear-sky geometry only
    facility = time_aware and globals().get("TIME_AWARE_OBJECTIVE", "facility") == "facility"
    covered = np.zeros(len(_tile_ids), dtype=np.float64)
    if facility:                       # plan and re-plan on f = sum_t max w u (Cell 0)
        _wr = np.repeat(weight, store.counts)
        util_plan, realized_w = util * _wr, realized * _wr
        covered_w = np.zeros(len(_tile_ids), dtype=np.float64)
    else:
        util_plan, realized_w, covered_w = None, None, covered
    taken = np.zeros(n, dtype=bool)
    selected, history = [], []
    downlinked, replans, n_comp = 0, 0, 0
    tau, budget = -np.inf, max_steps

    if planning == "expected":
        p_plan = np.asarray(p_tile if p_tile is not None else _planning_probabilities(), dtype=np.float64)
        usable = util > 0.0
        got = np.zeros(len(_tile_ids), dtype=bool)               # tiles with a cloud-free usable view so far
        plan = _expected_plan(store, ~got, arrives, budget, p_plan, weight)
        while budget > 0 and plan:
            i = min(range(len(plan)), key=lambda j: (d[plan[j]], j))   # next planned image to arrive
            e = plan.pop(i)
            tau = d[e]
            s = store.rows(e)
            t = tile_idx[s]
            replans += int(np.any(flags[s] & usable[s] & ~got[t]))    # a planned new tile came back cloudy
            np.maximum.at(covered, t, realized[s])
            got[t[realized[s] > 0.0]] = True
            taken[e] = True
            selected.append(store.keys_[e])
            downlinked += int(store.counts[e])
            budget -= 1
            cov_mask = covered > 0.0
            n_cov = int(cov_mask.sum())
            history.append({"step": len(history) + 1, "selected": store.keys_[e],
                            "coverage_pct": 100 * n_cov / total_target_tiles,
                            "mean_quality": float(covered[cov_mask].sum() / n_cov) if n_cov else 0.0})
            if budget > 0:
                plan = _expected_plan(store, ~got, (~taken) & arrives & (d >= tau), budget, p_plan, weight)

    plan = [] if planning == "expected" else _greedy_plan(store, covered_w, arrives, budget, weight, util_plan)
    while budget > 0 and plan:
        i = min(range(len(plan)), key=lambda j: (d[plan[j]], j))    # next planned image to arrive
        e = plan.pop(i)
        tau = d[e]
        s = store.rows(e)
        t = tile_idx[s]
        if facility:
            failed = bool(np.any(flags[s] & (util_plan[s] > covered_w[t] + 1e-12)))
            np.maximum.at(covered_w, t, realized_w[s])
        else:
            failed = bool(np.any(flags[s] & (util[s] > covered[t] + 1e-12)))
        np.maximum.at(covered, t, realized[s])
        taken[e] = True
        selected.append(store.keys_[e])
        downlinked += int(store.counts[e])
        budget -= 1
        cov_mask = covered > 0.0
        n_cov = int(cov_mask.sum())
        history.append({"step": len(history) + 1, "selected": store.keys_[e],
                        "coverage_pct": 100 * n_cov / total_target_tiles,
                        "mean_quality": float(covered[cov_mask].sum() / n_cov) if n_cov else 0.0})
        if failed and budget > 0:
            replans += 1
            plan = _greedy_plan(store, covered_w, (~taken) & arrives & (d >= tau), budget, weight, util_plan)

    if gap_fill and selected:
        accessible = np.zeros(len(_tile_ids), dtype=bool)
        accessible[tile_idx[util > 0.0]] = True
        remaining = accessible & ~(covered > 0.0)
        if remaining.any():
            for e in np.argsort(d, kind="stable"):
                if taken[e] or not arrives[e] or d[e] < tau:
                    continue
                s = store.rows(e)
                t, r = tile_idx[s], realized[s]
                hit = (r > 0.0) & remaining[t]
                if hit.any():
                    np.maximum.at(covered, t, r)
                    remaining[t[hit]] = False
                    taken[e] = True
                    selected.append(store.keys_[e])
                    downlinked += int(store.counts[e])
                    n_comp += 1
                    if not remaining.any():
                        break
            if remaining.any():
                logger.info(f"  Compensation ran out of arriving images with {int(remaining.sum())} tiles uncovered.")
    last_adaptive_run.update(replans=replans, compensatory=n_comp)
    covered_tiles_utility = {_tile_ids[i]: float(covered[i]) for i in np.flatnonzero(covered > 0.0)}
    return selected, covered_tiles_utility, downlinked, history


def select_passes(events_dict, max_steps, cloud_mask=None, gap_fill=True,
                  time_aware=False, latency_decay_rate=1.0):
    """The selector every cell uses. Clear sky (no mask) or CLOUD_KNOWLEDGE ==
    "oracle": run_submodular_greedy_selection unchanged. Otherwise the causal
    simulate_adaptive_selection."""
    if cloud_mask is None or globals().get("CLOUD_KNOWLEDGE", "realistic") == "oracle":
        return run_submodular_greedy_selection(events_dict, max_steps, cloud_mask=cloud_mask, gap_fill=gap_fill,
                                               time_aware=time_aware, latency_decay_rate=latency_decay_rate)
    return simulate_adaptive_selection(events_dict, max_steps, cloud_mask, gap_fill=gap_fill,
                                       time_aware=time_aware, latency_decay_rate=latency_decay_rate)


logger.info(f"=== Running clear-sky selection (k={MAX_SELECTION_STEPS}) ===")
sel_events_clear, cov_clear, down_clear, hist_clear = run_submodular_greedy_selection(
    event_data, MAX_SELECTION_STEPS, cloud_mask=None, gap_fill=True
)
logger.info(f"Clear-sky: {len(sel_events_clear)} events, "
            f"{hist_clear[-1]['coverage_pct']:.2f}% coverage" if hist_clear else "Clear-sky: no events selected")

logger.info(f"=== Running cloud/Fail-Signal selection (k={MAX_SELECTION_STEPS}, "
            f"cloud knowledge: {globals().get('CLOUD_KNOWLEDGE', 'realistic')}) ===")
sel_events_cloud, cov_cloud, down_cloud, hist_cloud = select_passes(
    event_data, MAX_SELECTION_STEPS, cloud_mask=cloud_mask, gap_fill=True
)
_adaptive_cloud_stats = dict(last_adaptive_run)
logger.info(f"Cloud: {len(sel_events_cloud)} events selected, "
            f"{100 * len(cov_cloud) / total_target_tiles:.2f}% final coverage"
            + (f" ({_adaptive_cloud_stats.get('replans', 0)} re-plans, "
               f"{_adaptive_cloud_stats.get('compensatory', 0)} compensatory passes)"
               if globals().get("CLOUD_KNOWLEDGE", "realistic") != "oracle" else ""))
# Clairvoyant reference, reported for transparency: the same greedy selector
# KNOWING the cloud (not a strict upper bound, since greedy is not optimal).
_oracle_sel_cloud, _oracle_cov_cloud, _, _ = run_submodular_greedy_selection(
    event_data, MAX_SELECTION_STEPS, cloud_mask=cloud_mask, gap_fill=True)


def compute_selection_latency(selected_events, refined_df):
    """
    SECONDARY METRIC -- last selected delivery: the delivery time
    (t_min + T_wait_min) of the LAST-arriving selected event. Reported
    alongside H1 but NOT the H1 metric: once coverage saturates, the greedy
    keeps adding events for quality gains, so this grows with the budget k
    even though the mosaic stopped changing. See compute_completion_latency.
    """
    if not selected_events:
        return None
    # EventStore keeps one wait per event; a DataFrame is still accepted.
    wait_lookup = (refined_df.wait_lookup if hasattr(refined_df, "wait_lookup") else
                   refined_df[["sat", "t_min", "T_wait_min"]]
                   .drop_duplicates(subset=["sat", "t_min"])
                   .set_index(["sat", "t_min"])["T_wait_min"]
                   .to_dict())
    waits = [wait_lookup.get((sat, t_min), 0) for sat, t_min in selected_events]
    delivery_times = [t_min + (np.nan if w is None else w) for (sat, t_min), w in zip(selected_events, waits)]
    return max(delivery_times)


def compute_completion_latency(selected_events, store, cloud_mask=None):
    """
    H1 METRIC -- mosaic completion time: with the selected events ordered by
    delivery time (t_min + T_wait_min), the delivery time at which the
    mosaic reaches its final coverage, i.e. of the last event that adds a
    tile not already delivered. A tile counts as covered under the same
    rule the selection uses: utility > 0 and not cloud-obscured. Events
    that only improve quality on already-covered tiles do not delay it.
    Returns None if no selected event covers anything.
    """
    if not selected_events:
        return None
    util = np.where(cloud_mask.flags, 0.0, store.util) if cloud_mask is not None else store.util
    def _delivery(key):
        w = store.wait_lookup.get(key)
        return key[1] + (np.nan if w is None else w)
    seen = np.zeros(len(_tile_ids), dtype=bool)
    completion = None
    for key in sorted(selected_events, key=delivery_sort_key(store)):   # stable: ties keep selection order
        s = store.rows(store.index[key])
        t = store.tile_idx[s][util[s] > 0.0]
        new = t[~seen[t]]
        if new.size:
            seen[new] = True
            completion = _delivery(key)
    return completion


latency_clear_min = compute_completion_latency(sel_events_clear, event_data)
latency_cloud_min = compute_completion_latency(sel_events_cloud, event_data, cloud_mask)
last_delivery_clear_min = compute_selection_latency(sel_events_clear, event_data)
last_delivery_cloud_min = compute_selection_latency(sel_events_cloud, event_data)
_fmt = lambda v: f"{v:.1f} min" if v is not None else "N/A (no events selected)"
logger.info(f"H1 latency (mosaic completion): clear={_fmt(latency_clear_min)}, cloud={_fmt(latency_cloud_min)} | "
            f"last selected delivery: clear={_fmt(last_delivery_clear_min)}, cloud={_fmt(last_delivery_cloud_min)}")

# Time-aware objective (decay LATENCY_DECAY_RATE), reported alongside the
# coverage-first objective above. Same budget, same cloud mask, same gap-fill.
LATENCY_DECAY_TAU_MIN = decay_time_scale(event_data)
logger.info(f"=== Time-aware selection (decay={LATENCY_DECAY_RATE}, tau={LATENCY_DECAY_TAU_MIN:.1f} min "
            f"[{globals().get('LATENCY_DECAY_SCALE', 'fleet')} scale; orbital period {float(T_MAX_WAIT_MINUTES):.1f} min], "
            f"k={MAX_SELECTION_STEPS}) ===")
sel_ta_clear, cov_ta_clear, down_ta_clear, hist_ta_clear = run_submodular_greedy_selection(
    event_data, MAX_SELECTION_STEPS, cloud_mask=None, gap_fill=True,
    time_aware=True, latency_decay_rate=LATENCY_DECAY_RATE)
sel_ta_cloud, cov_ta_cloud, down_ta_cloud, hist_ta_cloud = select_passes(
    event_data, MAX_SELECTION_STEPS, cloud_mask=cloud_mask, gap_fill=True,
    time_aware=True, latency_decay_rate=LATENCY_DECAY_RATE)
_oracle_sel_ta_cloud, _oracle_cov_ta_cloud, _, _ = run_submodular_greedy_selection(
    event_data, MAX_SELECTION_STEPS, cloud_mask=cloud_mask, gap_fill=True,
    time_aware=True, latency_decay_rate=LATENCY_DECAY_RATE)
ta_latency_clear_min = compute_completion_latency(sel_ta_clear, event_data)
ta_latency_cloud_min = compute_completion_latency(sel_ta_cloud, event_data, cloud_mask)
_mean_q = lambda cov: float(np.mean(list(cov.values()))) if cov else 0.0
logger.info(f"H1 latency, time-aware: clear={_fmt(ta_latency_clear_min)}, cloud={_fmt(ta_latency_cloud_min)} | "
            f"mean quality clear: coverage-first {_mean_q(cov_clear):.3f}, time-aware {_mean_q(cov_ta_clear):.3f}")

_cell6_summary = {
    "n_candidate_events": len(event_data),
    "selected_events_clear": len(sel_events_clear),
    "selected_events_cloud": len(sel_events_cloud),
    "coverage_clear_pct": hist_clear[-1]['coverage_pct'] if hist_clear else 0.0,
    "coverage_cloud_pct": 100 * len(cov_cloud) / total_target_tiles,
}
_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()
with open(os.path.join(_out_dir, 'arf_cell6_summary.json'), 'w') as f:
    json.dump(_cell6_summary, f)


def _selection_table(selected_events, history):
    """One row per selected event, in selection order: greedy steps carry the
    coverage/quality after that step; gap-fill events follow with stage 'gap_fill'."""
    rows = [{"step": h["step"], "stage": "greedy", "sat": h["selected"][0], "t_min": h["selected"][1],
             "coverage_pct": h["coverage_pct"], "mean_quality": h["mean_quality"]} for h in history]
    for sat, t in selected_events[len(history):]:
        rows.append({"step": None, "stage": "gap_fill", "sat": sat, "t_min": t,
                     "coverage_pct": None, "mean_quality": None})
    return pd.DataFrame(rows, columns=["step", "stage", "sat", "t_min", "coverage_pct", "mean_quality"])


_selection_table(sel_events_clear, hist_clear).to_csv(
    os.path.join(_out_dir, "arf_selection_history_clear.csv"), index=False)
_selection_table(sel_events_cloud, hist_cloud).to_csv(
    os.path.join(_out_dir, "arf_selection_history_cloud.csv"), index=False)

# Record in scenario_results (the cross-scenario ledger from Cell 0)
# for the hypothesis validators (Cell 8) and any later comparison
# across runs. Keyed by (ACTIVE_ROI, SCENARIO_N): N for the Starlink proxy,
# the fleet tag for real fleets (v1.5.2: fleet, swath case, window, start and
# left-out satellites), so scenarios run in one kernel never share an entry.
def _scenario_key(active_roi=None, active_n=None):
    """Shared key for scenario_results, used consistently across Cell
    6, 6B, 7, and 8 so every cell keys and reads the same entry."""
    return (active_roi if active_roi is not None else ACTIVE_ROI,
            active_n if active_n is not None else globals().get("SCENARIO_N", ACTIVE_N))

if 'scenario_results' not in globals():
    scenario_results = {}
# v1.5.2: this run's entry starts empty. Cells 6B, 7C and 7D add to it; with
# setdefault, a re-run of the same scenario kept fields of the previous run that
# the new run did not overwrite (e.g. its onset analysis, if 7C was skipped).
scenario_results[_scenario_key()] = {}
_n_covered_clear = len(cov_clear)
_redundancy_clear = (down_clear / _n_covered_clear) if _n_covered_clear else float('nan')
_n_covered_cloud = len(cov_cloud)
_redundancy_cloud = (down_cloud / _n_covered_cloud) if _n_covered_cloud else float('nan')

# H3 redundancy is measured AT THE POINT THE COVERAGE REQUIREMENT IS MET, not
# over all k passes: once coverage is complete the greedy keeps selecting passes
# to refine image quality, so all-k redundancy grows with the budget (e.g. 30.0
# when each of 30 passes covers a whole small ROI) whatever the efficiency of
# reaching coverage. _H3_COVERAGE_PCT matches Cell 8's H3_COVERAGE_THRESHOLD_PCT
# (Cell 8 runs later, so it is restated here and checked there).
_H3_COVERAGE_PCT = 95.0


def redundancy_at_coverage(selected, store, mask, level_pct):
    """
    Tile observations downlinked by the selection, in selection order, up to
    and including the pass at which coverage first reaches level_pct, divided
    by the unique tiles covered at that point. Every observation of a pass
    counts, cloud-obscured ones included (they are downlinked too); a tile
    counts as covered under the usual rule (utility > 0, not obscured).
    Returns (redundancy or None if the level is never reached, passes used).
    """
    util = np.where(mask.flags, 0.0, store.util) if mask is not None else store.util
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n_seen = n_obs = 0
    for i, key in enumerate(selected, 1):
        s = store.rows(store.index[key])
        t = store.tile_idx[s]
        n_obs += t.size
        new = t[(util[s] > 0.0) & ~seen[t]]
        seen[new] = True
        n_seen += new.size
        if n_seen and 100.0 * n_seen / total_target_tiles >= level_pct - 1e-9:
            return n_obs / n_seen, i
    return None, len(selected)


_h3 = {
    "h3_redundancy_clear": redundancy_at_coverage(sel_events_clear, event_data, None, _H3_COVERAGE_PCT),
    "h3_redundancy_cloud": redundancy_at_coverage(sel_events_cloud, event_data, cloud_mask, _H3_COVERAGE_PCT),
    "ta_h3_redundancy_clear": redundancy_at_coverage(sel_ta_clear, event_data, None, _H3_COVERAGE_PCT),
    "ta_h3_redundancy_cloud": redundancy_at_coverage(sel_ta_cloud, event_data, cloud_mask, _H3_COVERAGE_PCT),
}
# Under the independent per-observation cloud model, a tile needs n looks with
# 1 - p^n >= level to be seen clearly with that probability; averaged over
# tiles this bounds the expected redundancy at that coverage from below.
# (v1.5.1) Valid for the independent ("iid") cloud model with a constant
# probability only; correlated cloud needs more looks, so no floor is reported.
_cloud_redundancy_floor = (float(np.log(1.0 - _H3_COVERAGE_PCT / 100.0) / np.log(CLOUD_PROBABILITY))
                           if 0.0 < CLOUD_PROBABILITY < 1.0 and _H3_COVERAGE_PCT < 100.0
                           and globals().get("CLOUD_MODEL", "iid") == "iid"
                           and globals().get("CLOUD_PROBABILITY_SOURCE", "constant") == "constant" else None)
logger.info(f"H3 redundancy at {_H3_COVERAGE_PCT:g}% coverage: clear "
            f"{_h3['h3_redundancy_clear'][0]} ({_h3['h3_redundancy_clear'][1]} passes), cloud "
            f"{_h3['h3_redundancy_cloud'][0]} ({_h3['h3_redundancy_cloud'][1]} passes; "
            f"model floor ~{_cloud_redundancy_floor})")
scenario_results[_scenario_key()].update({
    'candidate_events': len(event_data),
    'selection_budget_k': MAX_SELECTION_STEPS,
    'coverage_clear_pct': hist_clear[-1]['coverage_pct'] if hist_clear else 0.0,
    'coverage_cloud_pct': 100 * _n_covered_cloud / total_target_tiles,
    'redundancy_clear': _redundancy_clear,           # over all selected passes (budget-driven)
    'redundancy_cloud': _redundancy_cloud,
    'h3_coverage_pct': _H3_COVERAGE_PCT,              # H3: redundancy when this coverage is first reached
    'h3_redundancy_clear': _h3['h3_redundancy_clear'][0],
    'h3_passes_clear': _h3['h3_redundancy_clear'][1],
    'h3_redundancy_cloud': _h3['h3_redundancy_cloud'][0],
    'h3_passes_cloud': _h3['h3_redundancy_cloud'][1],
    'ta_h3_redundancy_clear': _h3['ta_h3_redundancy_clear'][0],
    'ta_h3_redundancy_cloud': _h3['ta_h3_redundancy_cloud'][0],
    'cloud_redundancy_floor': _cloud_redundancy_floor,
    'selected_events_clear': len(sel_events_clear),
    'selected_events_cloud': len(sel_events_cloud),
    'reachable_tiles_pct': 100 * int(np.unique(event_data.tile_idx).size) / total_target_tiles,
    # Tiles with at least one USABLE clear-sky observation (utility > 0): the
    # most any selection could cover. H3 is not evaluable below its threshold.
    'reachable_usable_clear_pct': 100 * int(np.unique(event_data.tile_idx[event_data.util > 0.0]).size)
                                  / total_target_tiles,
    'gap_fill_enabled': True,
    'latency_clear_min': latency_clear_min,          # H1: mosaic completion time
    'latency_cloud_min': latency_cloud_min,
    'last_delivery_clear_min': last_delivery_clear_min,   # secondary
    'last_delivery_cloud_min': last_delivery_cloud_min,
    'mean_quality_clear': _mean_q(cov_clear),
    'mean_quality_cloud': _mean_q(cov_cloud),
    'latency_decay_rate': LATENCY_DECAY_RATE,
    'latency_decay_scale': globals().get("LATENCY_DECAY_SCALE", "fleet"),
    'latency_decay_tau_min': LATENCY_DECAY_TAU_MIN,   # time scale of the time-aware weight (full window)
    'orbital_period_min': float(T_MAX_WAIT_MINUTES),
    'ta_latency_clear_min': ta_latency_clear_min,     # H1, time-aware objective
    'ta_latency_cloud_min': ta_latency_cloud_min,
    'ta_coverage_clear_pct': 100 * len(cov_ta_clear) / total_target_tiles,
    'ta_coverage_cloud_pct': 100 * len(cov_ta_cloud) / total_target_tiles,
    'ta_mean_quality_clear': _mean_q(cov_ta_clear),
    'ta_mean_quality_cloud': _mean_q(cov_ta_cloud),
    'cloud_knowledge': globals().get("CLOUD_KNOWLEDGE", "realistic"),
    'time_aware_objective': globals().get("TIME_AWARE_OBJECTIVE", "facility"),   # v1.5.0
    'cloud_planning': globals().get("CLOUD_PLANNING", "clear"),                 # v1.5.0
    'cloud_model': globals().get("cloud_model_info"),                           # v1.5.0
    # v1.5.0: True when no selected-or-candidate image waits for a downlink
    # (every wait 0), so "delivery time" is the acquisition time.
    'acquisition_time_only': bool(np.all(np.nan_to_num(np.array(list(event_data.wait_lookup.values()),
                                                                 dtype=np.float64), nan=1.0) == 0.0))
                             if len(event_data) else None,
    'cloud_replans': _adaptive_cloud_stats.get("replans"),
    'cloud_compensatory_passes': _adaptive_cloud_stats.get("compensatory"),
    # v1.5.1: the window start; scenarios are comparable only if it is the same
    'reference_epoch': globals().get("_reference_epoch"),
    'reference_epoch_source': globals().get("REFERENCE_EPOCH_SOURCE"),
    # Reference: the same greedy selectors given every image's cloud in advance
    # (clairvoyant greedy). Not a strict upper bound: greedy is not optimal (v1.5.0 note).
    'oracle_coverage_cloud_pct': 100 * len(_oracle_cov_cloud) / total_target_tiles,
    'oracle_latency_cloud_min': compute_completion_latency(_oracle_sel_cloud, event_data, cloud_mask),
    'oracle_ta_coverage_cloud_pct': 100 * len(_oracle_cov_ta_cloud) / total_target_tiles,
    'oracle_ta_latency_cloud_min': compute_completion_latency(_oracle_sel_ta_cloud, event_data, cloud_mask),
})


def build_sweep_results(event_data, refined_df, cloud_mask, k_values=None):
    """
    Sweep coverage and latency across a range of selection budgets k,
    independent of MAX_SELECTION_STEPS -- feeds Cell 7's Figure 3
    (Reconstruction Latency Analysis) and Figure 4 (Coverage vs.
    Budget Trade-off), which show how these metrics move as the
    budget itself changes, not just their value at the one k Cell 6
    ran with.

    Uses gap_fill=False so the sweep reflects the raw greedy
    coverage/latency tradeoff at each k, not values inflated by the
    separate Fail-Signal gap-fill step (which exists to guarantee
    full coverage under cloud, not to characterise the tradeoff
    curve).
    """
    if k_values is None:
        _ceiling = len(event_data)
        k_values = sorted(k for k in {5, 10, 15, 20, 25, 30, 35, 40, 50, 60, 70} if k <= _ceiling)

    results = []
    if not k_values:
        return results
    # Greedy selection is incremental: the events chosen for budget k are
    # exactly the first k of any longer run (and a run that exhausts early
    # stops at the same step either way). One run at the largest k therefore
    # yields every smaller budget's selection and coverage exactly.
    k_max = max(k_values)
    _, _, _, hist_c_full = run_submodular_greedy_selection(
        event_data, max_steps=k_max, cloud_mask=None, gap_fill=False)
    _realistic = globals().get("CLOUD_KNOWLEDGE", "realistic") != "oracle"
    if not _realistic:
        _, _, _, hist_cl_full = run_submodular_greedy_selection(
            event_data, max_steps=k_max, cloud_mask=cloud_mask, gap_fill=False)

    for k in k_values:
        hist_c = hist_c_full[:k]
        # Realistic cloud: re-planning depends on the budget, so a smaller k is
        # NOT a prefix of a larger one -- each k is simulated on its own.
        hist_cl = (select_passes(event_data, k, cloud_mask=cloud_mask, gap_fill=False)[3]
                   if _realistic else hist_cl_full[:k])
        sel_c = [h['selected'] for h in hist_c]
        cov_c_pct = hist_c[-1]['coverage_pct'] if hist_c else 0.0
        cov_cl_pct = hist_cl[-1]['coverage_pct'] if hist_cl else 0.0
        latency_c = compute_completion_latency(sel_c, event_data)
        last_c = compute_selection_latency(sel_c, event_data)

        results.append({
            'k': k,
            'coverage_clear_pct': cov_c_pct,
            'coverage_cloud_pct': cov_cl_pct,
            'latency_clear_min': latency_c if latency_c is not None else 0,       # H1 metric
            'last_delivery_clear_min': last_c if last_c is not None else 0,       # secondary
        })
    return results


sweep_results = build_sweep_results(event_data, refined_events_df, cloud_mask)
pd.DataFrame(sweep_results).to_csv(os.path.join(_out_dir, "arf_coverage_sweep.csv"), index=False)
logger.info(f"Built sweep_results: {len(sweep_results)} k-values "
            f"({sweep_results[0]['k']}-{sweep_results[-1]['k']}) for Cell 7's Figures 3/4."
            if sweep_results else "Built sweep_results: empty (no valid k values for this event_data size).")

print("\n=== CELL 6 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 6B: RE-DERIVE SELECTION BUDGET (k) AGAINST THE CURRENT SCENARIO
# =============================================================================
# Cell 6 runs with a fixed selection budget k = MAX_SELECTION_STEPS
# (Cell 0). This cell independently re-derives a data-driven k for
# whichever scenario (ACTIVE_ROI, ACTIVE_N) is currently loaded, using
# the Kneedle elbow method (Satopaa, Albrecht, Irwin & Raghavan 2011, ICDCS
# Workshops, 166-171, doi:10.1109/ICDCSW.2011.20; here its chord-distance
# form) on a real coverage-vs-k sweep -- the point of diminishing returns for
# this specific candidate-event set, not an assumption carried over from a
# different scenario's derivation.
# This matters because candidate-event counts vary substantially
# between scenarios (region size, fleet density, and -- for
# spacetrack_history ROIs -- how many satellites existed at the
# historical epoch all affect this), so a k tuned for one scenario is
# not automatically appropriate for another.
#
# Run this after Cell 6 (needs event_data, run_submodular_greedy_selection,
# total_target_tiles already in the kernel namespace). Results are
# recorded in scenario_results under this run's scenario key (ROI and
# N or fleet tag, Cell 6), alongside Cell 6's own entry, so re-running
# this cell for several scenarios builds up a comparable ledger.
# The swept budgets stop at 100 (or the number of candidate events): a
# fleet that saturates beyond k = 100 shows no knee here (v1.5.2 note).
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 6B is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["6B"] = _ARF_CELL_VERSION

import numpy as np


def compute_elbow_point(k_values, coverage_values):
    """Kneedle chord-distance method -- point of maximum perpendicular
    distance from the line connecting the first and last (k, coverage)
    points.

    Degenerate cases -- fewer than 3 points, or a flat curve (coverage
    already saturated at the smallest budget swept, as for a small ROI a
    single strip can cover) -- have no knee. Normalising a flat curve
    divides 0 by 0, and argmax over the resulting NaNs silently returns
    index 0, reporting an arbitrary "elbow". Instead, return the smallest k
    that reaches the maximum coverage (the saturation budget).
    """
    k_arr = np.array(k_values, dtype=float)
    cov_arr = np.array(coverage_values, dtype=float)
    if k_arr.size < 3 or np.ptp(k_arr) == 0 or np.ptp(cov_arr) == 0:
        i = int(np.argmax(cov_arr >= cov_arr.max() - 1e-9))
        return int(k_arr[i]), float(cov_arr[i]), np.zeros_like(cov_arr)

    k_norm = (k_arr - k_arr.min()) / (k_arr.max() - k_arr.min())
    cov_norm = (cov_arr - cov_arr.min()) / (cov_arr.max() - cov_arr.min())

    x1, y1 = k_norm[0], cov_norm[0]
    x2, y2 = k_norm[-1], cov_norm[-1]

    distances = np.abs((y2 - y1) * k_norm - (x2 - x1) * cov_norm + x2 * y1 - y2 * x1) / \
                np.sqrt((y2 - y1) ** 2 + (x2 - x1) ** 2)

    elbow_idx = int(np.argmax(distances))
    return int(k_arr[elbow_idx]), float(cov_arr[elbow_idx]), distances


print(f"=== RE-DERIVING SELECTION BUDGET FOR ROI={ACTIVE_ROI!r}, "
      f"{'N=' + str(ACTIVE_N) if globals().get('FLEET', 'starlink') == 'starlink' else 'fleet=' + str(FLEET)} ===")
print(f"Candidate events available: {len(event_data)}")

# Sweep k from small values up to full exhaustion (capped for speed --
# same size-aware cap used elsewhere in this pipeline).
_sweep_ceiling = min(len(event_data), 100)
k_values = sorted(set(
    list(range(2, min(20, _sweep_ceiling) + 1, 2)) +
    list(range(20, _sweep_ceiling + 1, 5)) +
    [_sweep_ceiling]
))
k_values = [k for k in k_values if k <= len(event_data)]

print(f"Sweeping k = {k_values}")

# One run at the largest budget: greedy is incremental, so budget k's
# coverage is the k-th entry of this history (see build_sweep_results).
_, _, _, _hist_full = (run_submodular_greedy_selection(event_data, max_steps=max(k_values))
                       if k_values else (None, None, None, []))
coverage_at_k = [(_hist_full[:k][-1]['coverage_pct'] if _hist_full[:k] else 0.0) for k in k_values]
# Exact saturation budget: first greedy step reaching the run's final coverage.
saturation_k = (next(h['step'] for h in _hist_full
                     if h['coverage_pct'] >= _hist_full[-1]['coverage_pct'] - 1e-9)
                if _hist_full else None)

print("\nk     coverage%")
for k, cov in zip(k_values, coverage_at_k):
    print(f"{k:<5} {cov:.2f}")

if k_values:
    elbow_k, elbow_cov, distances = compute_elbow_point(k_values, coverage_at_k)
else:
    elbow_k, elbow_cov, distances = None, 0.0, np.zeros(0)

print(f"\n=== ELBOW POINT: ROI={ACTIVE_ROI!r} ===")
print(f"k = {elbow_k}, coverage = {elbow_cov:.2f}%")
print(f"Saturation budget (first k reaching final coverage "
      f"{_hist_full[-1]['coverage_pct'] if _hist_full else 0.0:.2f}%): k = {saturation_k}")
if k_values and not np.any(distances):
    print("(No diminishing-returns knee: coverage is flat across the swept budgets, "
          "so the elbow reported is the saturation budget, not a Kneedle knee.)")
print(f"(Configured MAX_SELECTION_STEPS for this run: {MAX_SELECTION_STEPS} -- "
      f"compare against the elbow above to see whether the fixed budget "
      f"Cell 6 actually used is under-, over-, or well-matched to this "
      f"scenario's real diminishing-returns point.)")

# Record in the ledger, keyed the same way Cell 6 records its own
# results, so both entries for this (ROI, N) merge into one record
# rather than each cell silently overwriting the other's fields.
if 'scenario_results' not in globals():
    scenario_results = {}
scenario_results.setdefault(_scenario_key(), {})
scenario_results[_scenario_key()]['real_elbow_k'] = elbow_k
scenario_results[_scenario_key()]['real_elbow_coverage_pct'] = elbow_cov
scenario_results[_scenario_key()]['real_k_sweep'] = dict(zip(k_values, coverage_at_k))
scenario_results[_scenario_key()]['saturation_k'] = saturation_k

print(f"\n=== CELL 6B COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 7: MANUSCRIPT GRAPH GENERATOR
# =============================================================================
#
# This module generates publication-quality figures for the LaTeX manuscript
# from the simulation results produced by Cell 6. It produces four key
# visualisations, drawn at IEEE column widths (v1.5.2):
#   1. Submodular Coverage Progression (Scenario A vs. B)
#   2. Mean Observation Quality Progression
#   3. Latency vs. Budget, in hours, against the physical floor and the H1a
#      margin (v1.5.2; was the falsified 10-minute H1 target, in minutes)
#   4. Coverage vs. Budget (k) Trade-off
#   7. Mosaic map: which image supplied each tile, and when
#
# Dependencies:
#   - Cell 0: ARFConfig (provides ACTIVE_N, etc.)
#   - Cell 1: target_tiles
#   - Cell 5: refined_events_df
#   - Cell 6: hist_clear, hist_cloud, sweep_results, latency metrics
#
# Outputs:
#   - arf_manuscript_figures.png (Composite raster preview)
#   - arf_manuscript_figures.pdf (Vector output for LaTeX)
#   - Individual figure files (PNG/PDF per subplot)
# =============================================================================
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
#
# References:
#   [9] Hunter, J.D. (2007). "Matplotlib: A 2D Graphics Environment."
#       Computing in Science & Engineering, 9(3), 90-95.
#   [21] Nemhauser, G.L., Wolsey, L.A. & Fisher, M.L. (1978). "An analysis
#        of approximations for maximizing submodular set functions—I."
#        Mathematical Programming, 14, 265–294.
# =============================================================================

"""
ARF Manuscript Graph Generator
================================

This module generates publication-quality figures for the research
manuscript: Times New Roman (or a metric-compatible clone), embedded
TrueType fonts, 300 dpi PNG plus vector PDF, drawn at IEEE column widths
(3.5 in single, 7.16 in double) with 8-9 pt text (v1.5.2).

Figure Descriptions:
    1. **Coverage Progression**: Shows how cumulative spatial coverage
       increases with each greedy selection step, comparing clear-sky
       (Scenario A) and cloud-obscured (Scenario B) conditions. The
       geometric ceiling is annotated.

    2. **Quality Progression**: Shows how the mean observation quality
       of the covered tiles evolves with each selection step.

    3. **Latency vs. Budget**: clear-sky completion time and time to the
       H1a level (95% of reachable tiles) for each budget k, in hours,
       against the physical floor and the floor plus the H1a margin.

    4. **Coverage vs. Budget Trade-off**: Shows the coverage achieved
       for different cardinality constraints (k), justifying the
       choice of MAX_SELECTION_STEPS.

Part of ARF (the cell's release is _ARF_CELL_VERSION below).
"""

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 7 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["7"] = _ARF_CELL_VERSION

import os
import sys
import logging
from typing import Dict, List, Tuple, Optional, Any

import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import numpy as np

# =============================================================================
# LOGGING CONFIGURATION
# =============================================================================

logger = logging.getLogger("arf.cell7")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter(
        '%(asctime)s [%(name)s] %(levelname)s: %(message)s',
        datefmt='%H:%M:%S'
    ))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

# =============================================================================
# CUSTOM EXCEPTIONS
# =============================================================================

class FigureGenerationError(Exception):
    """Raised when figure generation fails."""
    pass


# =============================================================================
# MATPLOTLIB CONFIGURATION
# =============================================================================
# Publication-quality settings for LaTeX manuscript integration.

plt.rcParams.update({
    # Font configuration (same fallback list as Cell 3: Times New Roman, else a
    # metric-compatible Times clone such as Liberation Serif on Colab)
    'font.family': 'serif',
    'font.serif': ['Times New Roman', 'Liberation Serif', 'Nimbus Roman', 'FreeSerif', 'DejaVu Serif'],
    'font.size': 10,
    'mathtext.fontset': 'stix',

    # Axis configuration
    'axes.linewidth': 1.0,
    'axes.labelsize': 11,
    'axes.titlesize': 12,
    'axes.titleweight': 'bold',

    # Tick configuration
    'xtick.direction': 'in',
    'ytick.direction': 'in',
    'xtick.major.width': 0.8,
    'ytick.major.width': 0.8,
    'xtick.labelsize': 9,
    'ytick.labelsize': 9,

    # Legend configuration
    'legend.fontsize': 8,
    'legend.framealpha': 0.9,
    'legend.edgecolor': '0.8',

    # Figure configuration
    'figure.dpi': 100,
    'figure.facecolor': 'white',
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
    # Embed fonts in PDFs as TrueType (Type 42) rather than matplotlib's default Type 3
    'pdf.fonttype': 42,
    'ps.fonttype': 42,
})

# =============================================================================
# COLOR PALETTE
# =============================================================================
# Consistent color scheme across all figures.

COLORS = {
    'clear_sky': '#1b9e77',       # Teal green
    'cloud_cover': '#d95f02',     # Orange
    'ceiling': '#7570b3',         # Purple
    'guarantee': '#e7298a',       # Pink (unused since the (1-1/e) line was removed)
    'level': '#e7298a',           # Pink: time to the H1a level (v1.5.2)
    'h1_target': '#66a61e',       # Green: floor + H1a margin (v1.5.2; was the 10-min target)
    'grid': '#cccccc',            # Light gray
}


# =============================================================================
# DATA EXTRACTION
# =============================================================================

# Verify required globals exist
_required_globals = {
    'hist_clear': 'Cell 6',
    'hist_cloud': 'Cell 6',
    'refined_events_df': 'Cell 5',
    'target_tiles': 'Cell 1',
}

_missing = []
for var, cell in _required_globals.items():
    if var not in globals():
        _missing.append(f"{var} (from {cell})")

if _missing:
    raise FigureGenerationError(
        f"Missing required globals: {', '.join(_missing)}. "
        f"Please run the required cells first."
    )

# --- Extract Real Ceiling Line ---
# Derived from actual access-event geometry, not the greedy algorithm: tiles
# with at least one USABLE clear-sky observation (utility > 0), the same rule
# every coverage figure uses (v1.5.2; was any refined row, which also counted
# rows of zero utility -- a pass with no downlink -- identical in all runs so far).
max_achievable_tiles = (int(np.unique(event_data.tile_idx[event_data.util > 0.0]).size)
                        if 'event_data' in globals() else refined_events_df['tile'].nunique())
total_target_tiles = len(target_tiles)
max_achievable_pct = 100 * max_achievable_tiles / total_target_tiles

logger.info(f"Max achievable coverage (geometric ceiling): "
            f"{max_achievable_tiles} / {total_target_tiles} tiles = "
            f"{max_achievable_pct:.2f}%")

# --- Extract Selection History Metrics ---
steps_clear = [h['step'] for h in hist_clear]
cov_clear_pct = [h['coverage_pct'] for h in hist_clear]
qual_clear = [h['mean_quality'] for h in hist_clear]

steps_cloud = [h['step'] for h in hist_cloud]
cov_cloud_pct = [h['coverage_pct'] for h in hist_cloud]
qual_cloud = [h['mean_quality'] for h in hist_cloud]

# --- Extract Sweep Results (if available) ---
sweep_data = None
if 'sweep_results' in globals() and len(sweep_results) > 0:
    sweep_data = sweep_results
    logger.info(f"Coverage sweep data available: {len(sweep_data)} k-values")

# --- Extract Latency Data ---
latency_clear = scenario_results.get(_scenario_key(), {}).get('latency_clear_min')
latency_cloud = scenario_results.get(_scenario_key(), {}).get('latency_cloud_min')

# --- H1a reference for the latency figure (v1.5.2) ---------------------------
# The original H1 target (mosaic complete within 10 min) was falsified by the
# real-fleet runs and is kept only in the reports. The latency figure instead
# shows the quantities H1a tests: the time at which the k-image selection's
# delivered mosaic reaches H1_REACHABLE_COVERAGE_PCT of the reachable tiles,
# the physical floor for that level (every pass in the window, floor_time of
# Cell 6) and the floor plus the H1a margin (one orbital period at the fleet's
# median altitude unless set in Cell 0 -- the same rule as Cell 7C). Full
# window, clear sky, coverage-first greedy (Cell 6's sweep); Cell 7C tests H1a
# over shifted onsets with the earliest-completion cover.
_H1L7 = float(globals().get("H1_REACHABLE_COVERAGE_PCT", 95.0))
_reach7 = int(np.unique(event_data.tile_idx[event_data.util > 0.0]).size) if len(event_data) else 0
_need7 = int(np.ceil(_H1L7 / 100.0 * _reach7 - 1e-9)) if _reach7 else 0
_floor7 = floor_time(event_data, _H1L7) if ('floor_time' in globals() and _reach7) else None
_margin7 = (float(H1_FLOOR_MARGIN_MIN_CONFIG) if globals().get("H1_FLOOR_MARGIN_MIN_CONFIG") is not None else
            float(2.0 * np.pi * np.sqrt((Re_km + CONSTELLATION_ALTITUDE_KM) ** 3 / mu_earth) / 60.0))


def time_to_level(selected, store, need):
    """Delivery time (min) at which the images in `selected`, in delivery order
    (ties in selection order), first cover `need` tiles with a usable clear-sky
    observation (utility > 0); None if they never do."""
    if need <= 0 or not selected:
        return None
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n = 0
    for key in sorted(selected, key=delivery_sort_key(store)):
        w = store.wait_lookup.get(key)
        if w is None or np.isnan(w):
            break                                          # undelivered images sort last
        s = store.rows(store.index[key])
        t = store.tile_idx[s][store.util[s] > 0.0]
        new = t[~seen[t]]
        if new.size:
            seen[new] = True
            n += new.size
            if n >= need:
                return float(key[1] + w)
    return None


_t_level_by_k = {}
if sweep_data:
    _k_all = [d['k'] for d in sweep_data]
    _hist7 = run_submodular_greedy_selection(event_data, max(_k_all), cloud_mask=None, gap_fill=False)[3]
    _sel7 = [h['selected'] for h in _hist7]          # greedy is incremental: budget k = the first k
    _t_level_by_k = {k: time_to_level(_sel7[:k], event_data, _need7) for k in _k_all}


# =============================================================================
# FIGURE STYLE (v1.5.2): drawn at the printed size
# =============================================================================
# IEEE Author Center, "Resolution and Size": one column 3.5 in (88.9 mm), two
# columns 7.16 in (182 mm), colour and greyscale above 300 dpi; "File
# Formatting": type "approximately 9-10 point when viewed at full size", fonts
# embedded. Earlier versions drew 7 x 5 in single figures (and a 14 x 10 in
# composite) with 10 pt text, which prints at about 5 pt once fitted to a
# column. Each figure is now drawn at its column width with 8-9 pt text.
IEEE_ONE_COL_IN, IEEE_TWO_COL_IN = 3.5, 7.16
_IEEE_RC = {"font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "axes.titleweight": "bold",
            "xtick.labelsize": 8, "ytick.labelsize": 8, "legend.fontsize": 8, "legend.framealpha": 0.9,
            "lines.linewidth": 1.2, "lines.markersize": 3.5, "axes.linewidth": 0.6,
            "xtick.major.width": 0.6, "ytick.major.width": 0.6, "pdf.fonttype": 42, "ps.fonttype": 42}


def _save_figure(fig, prefix):
    for fmt in ('png', 'pdf'):
        filepath = os.path.join(OUT_DIR if 'OUT_DIR' in globals() else os.getcwd(), f"{prefix}.{fmt}")
        fig.savefig(filepath, dpi=300, bbox_inches='tight')
        logger.info(f"Saved: {filepath}")


def _grid(ax):
    ax.grid(alpha=0.3, linestyle='--', linewidth=0.5, color=COLORS['grid'])
    # v1.5.3: every figure that calls this counts images (k) along x: whole-number ticks
    # (a run with three candidate images showed k = 1.0, 1.5, 2.0, ...)
    from matplotlib.ticker import MaxNLocator
    ax.xaxis.set_major_locator(MaxNLocator(integer=True))


def _no_sweep_note(ax):
    """v1.5.3: say why a budget panel (Figures 3 and 4) is empty: the sweep
    starts at k = 5, and a window with fewer candidate images has no budget to
    sweep (open data alone, CKGR + Okavango, 15 October 2026: 3 images)."""
    _ed = globals().get("event_data")
    n = len(_ed) if _ed is not None else 0
    ax.text(0.5, 0.5, f"No budget sweep: the window offers {n} candidate image{'' if n == 1 else 's'},\n"
                      f"fewer than the smallest budget swept (k = 5)", transform=ax.transAxes,
            ha="center", va="center", fontsize=7.5, color="0.3")


# =============================================================================
# FIGURE 1: COVERAGE PROGRESSION
# =============================================================================
# The (1-1/e) guarantee is NOT drawn: the NWF (1978) bound compares the greedy
# with OPT_k, the best coverage achievable with exactly k selections, not with
# the geometric ceiling drawn here; OPT_k is not efficiently computable, so no
# fixed line on this axis represents the guarantee (Cell 7C measures the gap
# exactly with an ILP on sub-problems).

def _draw_coverage_progression(ax, steps_clear, cov_clear_pct, steps_cloud, cov_cloud_pct, max_achievable_pct,
                               cloud_label='Scenario B (Cloud / Fail-Signal)'):
    ax.plot(steps_clear, cov_clear_pct, color=COLORS['clear_sky'], linestyle='-', marker='o',
            label='Scenario A (Clear Sky)', zorder=3)
    ax.plot(steps_cloud, cov_cloud_pct, color=COLORS['cloud_cover'], linestyle='--', marker='s',
            label=cloud_label, zorder=3)
    ax.axhline(y=max_achievable_pct, color=COLORS['ceiling'], linestyle=':', linewidth=1.0, alpha=0.8,
               label=f'Geometric Ceiling ({max_achievable_pct:.1f}%)', zorder=2)
    ax.set_xlabel('Greedy Selection Step (k)')
    ax.set_ylabel('Cumulative Spatial Coverage (%)')
    ax.legend(loc='lower right')
    _grid(ax)
    ax.set_ylim(bottom=0, top=min(100, max_achievable_pct * 1.1))


def plot_coverage_progression(steps_clear, cov_clear_pct, steps_cloud, cov_cloud_pct, max_achievable_pct,
                              save_prefix: str = "arf_fig1"):
    """Cumulative coverage after each greedy step, clear sky and cloud, with
    the geometric ceiling. One IEEE column wide."""
    with plt.rc_context(_IEEE_RC):
        fig, ax = plt.subplots(figsize=(IEEE_ONE_COL_IN, 2.7))
        _draw_coverage_progression(ax, steps_clear, cov_clear_pct, steps_cloud, cov_cloud_pct, max_achievable_pct)
        ax.set_title('Submodular Coverage Progression')
        fig.tight_layout()
        _save_figure(fig, save_prefix)
    return fig, ax


# =============================================================================
# FIGURE 2: QUALITY PROGRESSION
# =============================================================================

def _draw_quality_progression(ax, steps_clear, qual_clear, steps_cloud, qual_cloud,
                              cloud_label='Scenario B (Cloud / Fail-Signal)'):
    ax.plot(steps_clear, qual_clear, color=COLORS['clear_sky'], linestyle='-', marker='o',
            label='Scenario A (Clear Sky)', zorder=3)
    ax.plot(steps_cloud, qual_cloud, color=COLORS['cloud_cover'], linestyle='--', marker='s',
            label=cloud_label, zorder=3)
    ax.set_xlabel('Greedy Selection Step (k)')
    ax.set_ylabel('Mean Observation Quality (Q)')
    ax.legend(loc='lower right')
    _grid(ax)
    ax.set_ylim(bottom=0, top=1.05)


def plot_quality_progression(steps_clear, qual_clear, steps_cloud, qual_cloud, save_prefix: str = "arf_fig2"):
    """Mean utility of the covered tiles after each greedy step. One column wide."""
    with plt.rc_context(_IEEE_RC):
        fig, ax = plt.subplots(figsize=(IEEE_ONE_COL_IN, 2.7))
        _draw_quality_progression(ax, steps_clear, qual_clear, steps_cloud, qual_cloud)
        ax.set_title('Observation Quality Progression')
        fig.tight_layout()
        _save_figure(fig, save_prefix)
    return fig, ax


# =============================================================================
# FIGURE 3: LATENCY AGAINST THE BUDGET (H1a reference, hours)
# =============================================================================

def _draw_latency(ax, sweep_data, t_level_by_k, floor_min, margin_min, level_pct):
    """Curves and floor lines of Figure 3 (and composite panel (c)). Returns
    the legend handles and labels; the caller puts the legend under the axes
    (v1.5.3: inside them it covered the curves).
    v1.5.3: the y-axis spans the plotted values, the floor and the floor plus
    the margin, instead of starting at 0 h: in a window whose first passes
    come a day in (15 October 2026: every value between 24 and 26 h) the
    curves lay pressed against the top of the axes. A budget with no
    completion time is left out instead of being drawn at 0 h."""
    ys = []
    if sweep_data:
        k_vals = [d['k'] for d in sweep_data]
        y_done = [np.nan if d.get('latency_clear_min') is None else d['latency_clear_min'] / 60.0
                  for d in sweep_data]
        ax.plot(k_vals, y_done, color=COLORS['clear_sky'], linestyle='-', marker='o',
                label='Mosaic complete (own final coverage)', zorder=3)
        ys += y_done
        kk = [k for k in k_vals if t_level_by_k.get(k) is not None]
        if kk:
            y_lvl = [t_level_by_k[k] / 60.0 for k in kk]
            ax.plot(kk, y_lvl, color=COLORS['level'], linestyle='-', marker='^',
                    label=f'{level_pct:g}% of reachable tiles', zorder=3)
            ys += y_lvl
    if floor_min is not None:
        # v1.5.3: the floor and the margin are drawn over the curves (zorder 4,
        # was 2): the curve of the H1a level often lies on the floor.
        ax.axhline(floor_min / 60.0, color='k', linestyle=':', linewidth=1.0,
                   label=f'Physical floor, {level_pct:g}% (all passes)', zorder=4)
        ax.axhline((floor_min + margin_min) / 60.0, color=COLORS['h1_target'], linestyle='--', linewidth=1.0,
                   label=f'Floor + {margin_min:.0f} min (H1a margin)', zorder=4)
        ys += [floor_min / 60.0, (floor_min + margin_min) / 60.0]
    ax.set_xlabel('Selection Budget (k)')
    ax.set_ylabel('Time from analysis start (h)')
    _grid(ax)
    if not sweep_data:
        _no_sweep_note(ax)
    ys = [float(y) for y in ys if y is not None and np.isfinite(y)]
    if ys:
        lo, hi = min(ys), max(ys)
        pad = max(0.08 * (hi - lo), 0.1)                # at least 6 min above and below the values
        ax.set_ylim(max(lo - pad, 0.0), hi + pad)
    return ax.get_legend_handles_labels()


def plot_latency_analysis(sweep_data, t_level_by_k, floor_min, margin_min, level_pct, save_prefix: str = "arf_fig3"):
    """
    Clear-sky latency of the coverage-first greedy against the budget k, in
    hours: completion time (each budget's own final coverage) and the time to
    the H1a level of the reachable tiles, with the physical floor for that
    level and the floor plus the H1a margin. (Until v1.5.1 this figure drew
    the falsified 10-minute target, in minutes.) One column wide; from v1.5.3
    the y-axis spans the data and the legend sits under the axes.
    """
    with plt.rc_context(_IEEE_RC):
        fig, ax = plt.subplots(figsize=(IEEE_ONE_COL_IN, 3.45))
        handles, labels = _draw_latency(ax, sweep_data, t_level_by_k, floor_min, margin_min, level_pct)
        ax.set_title('Latency vs. Budget (clear sky)')
        fig.tight_layout(rect=(0, 0.22 if handles else 0, 1, 1))
        if handles:
            fig.legend(handles, labels, loc='lower center', bbox_to_anchor=(0.5, 0.0), ncol=1, frameon=False,
                       fontsize=7.5)
        _save_figure(fig, save_prefix)
    return fig, ax


# =============================================================================
# FIGURE 4: COVERAGE VS. BUDGET TRADE-OFF
# =============================================================================

def _draw_budget(ax, sweep_data, max_achievable_pct, cloud_label='Scenario B (Cloud / Fail-Signal)'):
    if sweep_data:
        k_vals = [d['k'] for d in sweep_data]
        ax.plot(k_vals, [d['coverage_clear_pct'] for d in sweep_data], color=COLORS['clear_sky'], linestyle='-',
                marker='o', label='Scenario A (Clear Sky)', zorder=3)
        ax.plot(k_vals, [d['coverage_cloud_pct'] for d in sweep_data], color=COLORS['cloud_cover'], linestyle='--',
                marker='s', label=cloud_label, zorder=3)
    ax.axhline(y=max_achievable_pct, color=COLORS['ceiling'], linestyle=':', linewidth=1.0, alpha=0.8,
               label=f'Geometric Ceiling ({max_achievable_pct:.1f}%)', zorder=2)
    if not sweep_data:
        _no_sweep_note(ax)
    ax.set_xlabel('Selection Budget (k)')
    ax.set_ylabel('Coverage (%)')
    ax.legend(loc='lower right')
    _grid(ax)
    ax.set_ylim(bottom=0, top=min(100, max_achievable_pct * 1.1))


def plot_coverage_budget_tradeoff(sweep_data, max_achievable_pct, save_prefix: str = "arf_fig4"):
    """Coverage at each budget k (no gap-fill), clear sky and cloud, with the
    geometric ceiling (no (1-1/e) line: see Figure 1). One column wide."""
    with plt.rc_context(_IEEE_RC):
        fig, ax = plt.subplots(figsize=(IEEE_ONE_COL_IN, 2.7))
        _draw_budget(ax, sweep_data, max_achievable_pct)
        ax.set_title('Coverage vs. Budget Trade-off')
        fig.tight_layout()
        _save_figure(fig, save_prefix)
    return fig, ax


# =============================================================================
# GENERATE ALL FIGURES
# =============================================================================

logger.info("Generating manuscript figures...")
fig1, ax1 = plot_coverage_progression(steps_clear, cov_clear_pct, steps_cloud, cov_cloud_pct, max_achievable_pct,
                                      save_prefix="arf_fig1_coverage_progression")
fig2, ax2 = plot_quality_progression(steps_clear, qual_clear, steps_cloud, qual_cloud,
                                     save_prefix="arf_fig2_quality_progression")
fig3, ax3 = plot_latency_analysis(sweep_data, _t_level_by_k, _floor7, _margin7, _H1L7,
                                  save_prefix="arf_fig3_latency_analysis")
fig4, ax4 = plot_coverage_budget_tradeoff(sweep_data, max_achievable_pct, save_prefix="arf_fig4_coverage_budget")

plt.show()
# v1.5.1: release the four single figures once shown (batches of runs otherwise
# accumulate open figures: matplotlib warns beyond 20)
for _f in [globals().get(n) for n in ("fig1", "fig2", "fig3", "fig4")]:
    if _f is not None:
        plt.close(_f)


# =============================================================================
# COMPOSITE FIGURE (for manuscript): two IEEE columns wide
# =============================================================================

logger.info("Generating composite manuscript figure...")
with plt.rc_context(_IEEE_RC):
    fig_composite, axes = plt.subplots(2, 2, figsize=(IEEE_TWO_COL_IN, 6.0))
    _draw_coverage_progression(axes[0, 0], steps_clear, cov_clear_pct, steps_cloud, cov_cloud_pct,
                               max_achievable_pct, cloud_label='Scenario B (Cloud)')
    axes[0, 0].set_title('(a) Submodular Coverage Progression', loc='left')
    _draw_quality_progression(axes[0, 1], steps_clear, qual_clear, steps_cloud, qual_cloud,
                              cloud_label='Scenario B (Cloud)')
    axes[0, 1].set_title('(b) Observation Quality Progression', loc='left')
    _h3c, _l3c = _draw_latency(axes[1, 0], sweep_data, _t_level_by_k, _floor7, _margin7, _H1L7)
    axes[1, 0].set_title('(c) Latency vs. Budget (clear sky)', loc='left')
    _draw_budget(axes[1, 1], sweep_data, max_achievable_pct, cloud_label='Scenario B (Cloud)')
    axes[1, 1].set_title('(d) Coverage vs. Budget Trade-off', loc='left')
    # v1.5.3: panel (c)'s legend under the panels, from panel (c)'s left edge
    fig_composite.tight_layout(rect=(0, 0.07 if _h3c else 0, 1, 1))
    if _h3c:
        fig_composite.legend(_h3c, _l3c, loc='lower left', ncol=2, frameon=False, fontsize=7.5,
                             bbox_to_anchor=(axes[1, 0].get_position().x0, 0.0))
    _save_figure(fig_composite, "arf_manuscript_figures")

plt.show()
plt.close(fig_composite)


# =============================================================================
# FIGURE 7: MOSAIC MAP (which image supplied each tile, and when)
# =============================================================================
# The mosaic the selector builds from the Cell 6 selections, drawn on the
# ROI's H3 cells. Each covered tile is coloured by the delivery time (pass
# time + downlink wait, hours from the analysis start) of the FIRST selected
# image that covers it under the selection's own rule (utility > 0, not
# cloud-obscured): the mosaic as it stands when it is complete -- Cell 6's
# H1 completion time is the latest of these times; later images can only
# improve quality. Thin dark lines separate tiles supplied by different images
# (seamlines). Grey: no image in the window could cover the tile. Hatched: an
# image in the window could have covered it, but none of the selected images
# did. (a) clear sky; (b) cloud, with the configured cloud knowledge.
# Uses the map helpers of Cell 3. Also writes arf_mosaic_tiles.csv (one row per
# tile: delivery time and source image in each panel), which loads into any GIS
# through the H3 cell IDs.

MOSAIC_MAP_POLICY = "time_aware"     # "time_aware" (the selector H1 tests) or "coverage_first"
MOSAIC_COLORMAP = "viridis"          # perceptually uniform and colour-vision-deficiency safe
MOSAIC_NONE_COLOR = "#bdbdbd"        # tiles with no usable image in the window
MOSAIC_MISS_EDGE = "#6e6e6e"         # hatching of reachable tiles left out of the mosaic
MOSAIC_MAX_SEAM_RINGS = 400          # seamlines are drawn only up to this many image fragments per
                                     # panel; beyond it (e.g. a cloud-fragmented mosaic) colour alone is legible


def tile_first_delivery(selected, store, mask=None):
    """Per ROI tile (index into _tile_ids): delivery time (minutes) of the first
    selected image that covers it (utility > 0, not cloud-obscured) and that
    image's position in delivery order; NaN / -1 where no selected image does.
    Images are ordered exactly as in compute_completion_latency (delivery time,
    ties keep selection order). Returns (first, source, ordered keys)."""
    util = np.where(mask.flags, 0.0, store.util) if mask is not None else store.util

    def _delivery(key):
        w = store.wait_lookup.get(key)
        return key[1] + (np.nan if w is None else w)

    first = np.full(len(_tile_ids), np.nan)
    source = np.full(len(_tile_ids), -1, dtype=np.int64)
    order = sorted(selected, key=delivery_sort_key(store)) if 'delivery_sort_key' in globals() else \
        sorted(selected, key=_delivery)
    for j, key in enumerate(order):
        d = _delivery(key)
        if np.isnan(d):
            continue
        s = store.rows(store.index[key])
        t = store.tile_idx[s][util[s] > 0.0]
        new = t[np.isnan(first[t])]
        first[new] = d
        source[new] = j
    return first, source, order


def reachable_tiles(store, mask=None):
    """Tiles with at least one usable observation among ALL images (as Cell 7C's reachable_tile_count)."""
    util = np.where(mask.flags, 0.0, store.util) if mask is not None else store.util
    out = np.zeros(len(_tile_ids), dtype=bool)
    out[store.tile_idx[util > 0.0]] = True
    return out


def _hour_bounds(t_max_h: float, max_bins: int = 8):
    """Colour-bin edges in hours from 0 to a 'nice' value >= t_max_h."""
    for step in (0.25, 0.5, 1, 2, 3, 4, 6, 12, 24, 48, 96, 168):
        if t_max_h / step <= max_bins:
            top = step * max(1, math.ceil(t_max_h / step - 1e-9))
            return np.arange(0.0, top + 1e-9, step)
    return np.linspace(0.0, max(t_max_h, 1.0), max_bins + 1)


def plot_mosaic_map(store, mask, sel_clear, sel_cloud, policy_label: str, out_dir: str,
                    stem: str = "arf_fig7_mosaic_map"):
    """Two-panel mosaic map (clear sky, cloud) and the per-tile CSV. Returns a summary dict."""
    from matplotlib.colors import BoundaryNorm
    tiles = list(_tile_ids)
    first_c, src_c, order_c = tile_first_delivery(sel_clear, store)
    first_k, src_k, order_k = tile_first_delivery(sel_cloud, store, mask)
    reach_c, reach_k = reachable_tiles(store), reachable_tiles(store, mask)

    outline_geoms, outline_label, outline_source = roi_outline_for_maps()
    ext = roi_map_extent(target_tiles, outline_geoms)
    items = [("scale", _SCALE_BAR_BOX, ["lower right", "lower left", "upper left", "upper right"]),
             ("north", _NORTH_BOX, ["upper right", "upper left", "lower right", "lower left"])]
    if "plan_furniture" in globals():              # v1.4.0: placed off the ROI, frame grown if needed
        ext, corners, furniture = plan_furniture(ext, map_obstacle(target_tiles, outline_geoms), items)
    else:                                          # Cell 3 of v1.3.1
        corners = choose_corners(ext, np.array([h3.cell_to_latlng(t)[::-1] for t in tiles]), items)
        furniture = None
    aspect = panel_aspect(ext)
    polys = tile_polygons(tiles)
    done = np.concatenate([first_c[~np.isnan(first_c)], first_k[~np.isnan(first_k)]]) / 60.0
    bounds = _hour_bounds(float(done.max()) if done.size else float(globals().get("WINDOW_HOURS", 48)))
    cmap = plt.get_cmap(MOSAIC_COLORMAP)
    norm = BoundaryNorm(bounds, cmap.N)
    big = len(tiles) > MAP_RASTER_TILES
    knowledge = globals().get("CLOUD_KNOWLEDGE", "realistic")
    _cm = globals().get("cloud_model_info") or {}
    _cm_txt = (f"correlated, L = {_cm.get('length_km'):g} km, T = {_cm.get('time_h'):g} h, "
               if _cm.get("model") == "correlated" else "independent, " if _cm.get("model") else "")
    # v1.5.3: the cloud settings moved from panel (b)'s title, which made the map
    # 7.75 in wide (two IEEE columns: 7.16 in), to the footer.
    cloud_title = "(b) Cloud: " + ("Fail-Signal re-planning" if knowledge != "oracle" else "cloud known in advance")
    cloud_note = (f"Panel (b) cloud: {_cm_txt}p = {CLOUD_PROBABILITY:g}; "
                  + ("each image's cloud is seen on arrival and the plan is updated (Fail-Signal)"
                     if knowledge != "oracle" else "cloud known to the selector in advance"))
    panels = [("(a) Clear sky", first_c, src_c, order_c, reach_c, sel_clear),
              (cloud_title, first_k, src_k, order_k, reach_k, sel_cloud)]

    summary = {}
    with plt.rc_context(_MAP_RC):
        fig, axes, layout = map_figure([aspect, aspect], top=0.52, bottom=1.12)
        for i, (ax, (title, first, src, order, reach, sel)) in enumerate(zip(axes, panels)):
            setup_map_axes(ax, ext)
            ax.set_facecolor("white")
            cov = ~np.isnan(first)
            miss = reach & ~cov
            none = ~reach & ~cov
            if cov.any():
                pc = PolyCollection([polys[j] for j in np.flatnonzero(cov)], array=first[cov] / 60.0,
                                    cmap=cmap, norm=norm, edgecolors="face", linewidths=0.2, zorder=2)
                pc.set_rasterized(big)
                ax.add_collection(pc)
            if miss.any():
                pc = PolyCollection([polys[j] for j in np.flatnonzero(miss)], facecolors="white",
                                    edgecolors=MOSAIC_MISS_EDGE, linewidths=0.0, hatch="//////", zorder=2)
                pc.set_rasterized(big)
                ax.add_collection(pc)
            if none.any():
                pc = PolyCollection([polys[j] for j in np.flatnonzero(none)], facecolors=MOSAIC_NONE_COLOR,
                                    edgecolors="face", linewidths=0.2, zorder=2)
                pc.set_rasterized(big)
                ax.add_collection(pc)
            seams = []
            for j in np.unique(src[src >= 0]):
                seams += geo_lines([tiles_outline(_tile_ids[src == j])])
            seams_drawn = 0 < len(seams) <= MOSAIC_MAX_SEAM_RINGS
            if seams_drawn:
                lc = LineCollection(seams, colors="0.12", linewidths=0.45 if len(tiles) <= MAP_EDGE_TILES else 0.35,
                                    zorder=3)
                ax.add_collection(lc)
            draw_outline(ax, outline_geoms, lw=0.8, zorder=4)
            if i == 0:
                add_scale_bar(ax, ext, corner=corners["scale"])
                add_north_arrow(ax, corner=corners["north"])
            n_src = int(np.unique(src[src >= 0]).size)
            pct = 100.0 * float(cov.sum()) / len(tiles)
            t_done = float(np.nanmax(first)) / 60.0 if cov.any() else None
            reach_pct = 100.0 * float(reach.sum()) / len(tiles)
            stats = (f"{n_src} of {len(sel)} selected images supply tiles"
                     + ("" if seams_drawn or not seams else f" (seamlines omitted: {len(seams):,} fragments)") + "\n"
                     + (f"{pct:.1f}% of tiles covered ({reach_pct:.1f}% reachable); complete at {t_done:.1f} h"
                        if t_done is not None else f"No tile covered ({reach_pct:.1f}% reachable)"))
            ax.text(0.0, 1.012, stats, transform=ax.transAxes, ha="left", va="bottom", fontsize=6.3,
                    linespacing=1.3, color="0.15")
            ax.set_title(title, loc="left", pad=21)
            summary["clear" if i == 0 else "cloud"] = {
                "n_selected": len(sel), "n_source_images": n_src, "tiles_covered_pct": pct,
                "tiles_reachable_pct": reach_pct, "mosaic_complete_h": t_done,
                "image_fragments": len(seams), "seamlines_drawn": seams_drawn}

        # Colour bar (left) and legend (right) under the panels.
        fw, fh, bot = layout["fig_w"], layout["fig_h"], layout["bottom"]
        cax = fig.add_axes([0.52 / fw, (bot - 0.52) / fh, 2.7 / fw, 0.085 / fh])
        cb = fig.colorbar(plt.cm.ScalarMappable(norm=norm, cmap=cmap), cax=cax, orientation="horizontal",
                          ticks=bounds, spacing="proportional")
        cb.ax.tick_params(labelsize=6.5, width=0.5, length=2.5)
        cb.outline.set_linewidth(0.5)
        cb.set_ticklabels([f"{b:g}" for b in bounds])
        # v1.5.0: with no downlink wait anywhere, delivery time IS acquisition time; say so.
        _acq_only = bool(np.all(np.nan_to_num(np.array(list(store.wait_lookup.values()), dtype=np.float64),
                                              nan=1.0) == 0.0)) if len(store) else False
        cb.set_label(("Acquisition" if _acq_only else "Delivery") + " time of the first usable image "
                     "(h from analysis start)", fontsize=7, labelpad=2)
        handles = [mpatches.Patch(facecolor="white", edgecolor=MOSAIC_MISS_EDGE, hatch="//////", lw=0.4,
                                  label="Reachable, not in mosaic"),
                   mpatches.Patch(facecolor=MOSAIC_NONE_COLOR, edgecolor="none", label="No usable image in window"),
                   mlines.Line2D([], [], color="0.12", lw=0.6, label="Image boundary (seamline)"),
                   mlines.Line2D([], [], color=MAP_COLORS["boundary"], lw=0.8, label=outline_label)]
        fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.50, (bot - 0.36) / fh), ncol=2,
                   frameon=False, handlelength=1.8, columnspacing=1.2, handletextpad=0.5, fontsize=7)
        cite, short = _wdpa_citation(globals().get("roi_boundary_report")) if '_wdpa_citation' in globals() else (None, None)
        credits = [policy_label, f"Boundary: {short} (UNEP-WCMC & IUCN)" if short else None, "Coordinates: WGS 84"]
        fig.text(0.52 / fw, 0.04 / fh, cloud_note + "\n" + " · ".join(c for c in credits if c), fontsize=6,
                 color="0.4", ha="left", va="bottom", linespacing=1.45)
        for fmt in ("png", "pdf"):
            fig.savefig(os.path.join(out_dir, f"{stem}.{fmt}"), dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig)

    # Per-tile table (H3 IDs are derived grid cells, not WDPA data).
    def _src_cols(src, order):
        sat = np.array([order[j][0] if j >= 0 else "" for j in src], dtype=object)
        tmin = np.array([order[j][1] if j >= 0 else -1 for j in src], dtype=np.int64)
        return sat, tmin
    sat_c, t_c = _src_cols(src_c, order_c)
    sat_k, t_k = _src_cols(src_k, order_k)
    pd.DataFrame({"tile": _tile_ids, "reachable_clear": reach_c, "first_delivery_min_clear": first_c,
                  "source_sat_clear": sat_c, "source_t_min_clear": t_c,
                  "reachable_cloud": reach_k, "first_delivery_min_cloud": first_k,
                  "source_sat_cloud": sat_k, "source_t_min_cloud": t_k}).to_csv(
        os.path.join(out_dir, "arf_mosaic_tiles.csv"), index=False)
    summary.update(policy=policy_label, colour_bins_h=[float(b) for b in bounds], outline_source=outline_source,
                   extent_lon_lat=[round(v, 5) for v in ext],
                   furniture=None if furniture is None else dict(furniture, anchors=corners))
    return summary


_mosaic_summary = None
_map_helpers = ("roi_map_extent", "map_figure", "setup_map_axes", "tile_polygons", "roi_outline_for_maps",
                "add_scale_bar", "add_north_arrow", "draw_outline", "tiles_outline", "geo_lines")
if all(h in globals() for h in _map_helpers):
    if MOSAIC_MAP_POLICY == "time_aware":
        _mosaic_sel = (sel_ta_clear, sel_ta_cloud)
        _tau = globals().get("LATENCY_DECAY_TAU_MIN")
        _policy_label = (f"Selector: time-aware greedy (k = {MAX_SELECTION_STEPS}, decay {LATENCY_DECAY_RATE:g}"
                         + ("" if _tau is None else f", time scale {_tau / 60:.1f} h") + ") with gap-filling")
    else:
        _mosaic_sel = (sel_events_clear, sel_events_cloud)
        _policy_label = f"Selector: coverage-first greedy (k = {MAX_SELECTION_STEPS}) with gap-filling"
    try:
        _mosaic_summary = plot_mosaic_map(event_data, cloud_mask, *_mosaic_sel, _policy_label,
                                          OUT_DIR if 'OUT_DIR' in globals() else os.getcwd())
        logger.info("Saved: arf_fig7_mosaic_map.png/.pdf and arf_mosaic_tiles.csv")
    except Exception as e:
        logger.warning(f"Mosaic map not generated: {e}")
else:
    logger.warning("Mosaic map skipped: run Cell 3 first (it defines the map helpers this figure uses).")


# =============================================================================
# FIGURE METADATA
# =============================================================================

_metadata = {
    "figures_generated": [
        "arf_fig1_coverage_progression",
        "arf_fig2_quality_progression",
        "arf_fig3_latency_analysis",
        "arf_fig4_coverage_budget",
        "arf_manuscript_figures (composite)",
    ] + (["arf_fig7_mosaic_map"] if _mosaic_summary else []),
    "mosaic_map": _mosaic_summary,
    # Satellites actually simulated (Cell 2). Until v1.3.1 this was ACTIVE_N,
    # which real-fleet runs leave at its default (1000) while the fleet is all
    # operational satellites of the chosen fleet (e.g. 97).
    "n_constellation": len(constellation) if 'constellation' in globals() else 'unknown',
    "fleet": globals().get("FLEET", "starlink"),
    "scenario": str(globals().get("SCENARIO_N", globals().get("ACTIVE_N", "unknown"))),
    "max_achievable_pct": max_achievable_pct,
    # "guarantee_pct" removed: max_achievable_pct * (1 - 1/e) is not a
    # valid guarantee figure -- see plot_coverage_progression's
    # docstring. The qualitative (1-1/e) guarantee claim (without a
    # fabricated percentage) is still recorded correctly under
    # "approximation_guarantee" in the hypothesis_validation export.
    "final_coverage_clear": cov_clear_pct[-1] if cov_clear_pct else None,
    "final_coverage_cloud": cov_cloud_pct[-1] if cov_cloud_pct else None,
    "latency_clear_min": latency_clear,
    "latency_cloud_min": latency_cloud,
    # v1.5.2: what Figure 3 draws (full window, clear sky, coverage-first)
    "figure3": {"h1_level_pct_of_reachable": _H1L7, "reachable_tiles": _reach7,
                "physical_floor_min": _floor7, "h1a_margin_min": _margin7,
                "time_to_level_min_by_k": {str(k): v for k, v in _t_level_by_k.items()},
                "figure_widths_in": {"single": IEEE_ONE_COL_IN, "composite": IEEE_TWO_COL_IN}},
    "h1_target_min": 10.0,
    # 10.0 matches H1_LATENCY_THRESHOLD_MIN (Cell 8) -- duplicated here
    # rather than referenced, since Cell 8 runs after this cell and
    # that constant doesn't exist yet at this point. If the H1
    # threshold ever changes, both this line and Cell 8's constant
    # need updating together.
    "h1_supported": latency_clear is not None and latency_clear < 10.0,
    "h1_supported_note": "original H1 (< 10 min) only; H1a/H1b are in arf_hypothesis_validation.json (Cell 8)",
}

# Save metadata for audit trail
try:
    import json
    with open(os.path.join(OUT_DIR if 'OUT_DIR' in globals() else os.getcwd(), 'arf_cell7_metadata.json'), 'w') as f:
        json.dump(_metadata, f, indent=2, default=str)
    logger.info("Figure metadata saved to arf_cell7_metadata.json")
except Exception as e:
    logger.warning(f"Could not save metadata: {e}")


# =============================================================================
# SUMMARY
# =============================================================================

print("\n" + "=" * 70)
print("  CELL 7: Manuscript Graph Generator")
print("=" * 70)
print(f"  Figures Generated:")
print(f"    1. Coverage Progression (Scenario A vs. B)")
print(f"    2. Quality Progression (Scenario A vs. B)")
print(f"    3. Latency vs. Budget (hours; physical floor and H1a margin)")
print(f"    4. Coverage vs. Budget Trade-off")
print(f"    5. Composite Manuscript Figure (2x2, {IEEE_TWO_COL_IN} in wide)")
print(f"    7. Mosaic Map ({MOSAIC_MAP_POLICY.replace('_', '-')} selector)" if _mosaic_summary else "    7. Mosaic Map: not generated (see log)")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Geometric Ceiling:         {max_achievable_pct:.2f}%")
# v1.5.2: "is not None" (a latency of 0.0 min is a valid value and was not printed)
if cov_clear_pct:
    print(f"  Final Coverage (Clear):    {cov_clear_pct[-1]:.2f}%")
if cov_cloud_pct:
    print(f"  Final Coverage (Cloud):    {cov_cloud_pct[-1]:.2f}%")
if latency_clear is not None:
    print(f"  Latency (Clear):           {latency_clear:.1f} min ({latency_clear / 60:.1f} h)")
if latency_cloud is not None:
    print(f"  Latency (Cloud):           {latency_cloud:.1f} min ({latency_cloud / 60:.1f} h)")
if _floor7 is not None:
    print(f"  Floor to {_H1L7:g}% reachable: {_floor7:.1f} min; H1a margin {_margin7:.1f} min")
print(f"  Original H1 (< 10 min):    {'SUPPORTED' if _metadata.get('h1_supported') else 'NOT SUPPORTED'}"
      f"  (superseded, kept for the record; H1a/H1b: Cell 8)")
print(f"  ─────────────────────────────────────────────────────────")
print(f"  Output Formats:            PNG (300 DPI), PDF (vector)")
print(f"  Composite Figure:          arf_manuscript_figures.png/pdf")
print("=" * 70)

print(f"\n=== CELL 7 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 7B: COVERAGE-VS-TIME CURVE (honest check on H1 before writing it off)
# Algorithmic Reconstruction Framework (ARF)
# Run after Cell 6 (needs sel_events_clear, sel_events_cloud, event_data,
# refined_events_df, total_target_tiles already in the kernel namespace).
# =============================================================================
#
# H1 (Cell 6/8) is the mosaic COMPLETION time: when delivered coverage
# reaches its final value. This cell adds the partial milestones (time to
# 25/50/75/90% coverage), useful when a region is usable before it is
# complete. Its final-coverage time equals the H1 figure, because tiles
# are counted under the same rule (utility > 0, not cloud-obscured).
#
# This reorders the SELECTED events by actual delivery time (t_min +
# T_wait_min) — NOT greedy selection order, which has no temporal meaning
# — and tracks cumulative coverage against real elapsed time. Verified
# against a deliberately scrambled test case before being run for real:
# correctly re-sorts regardless of the order events were greedily chosen in.
# Output (v1.5.2): arf_coverage_vs_time.csv (both curves); images that are
# never delivered (no downlink in the window) are left out.

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 7B is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["7B"] = _ARF_CELL_VERSION

import numpy as np


def compute_coverage_vs_time(selected_events, event_data, refined_events_df,
                               total_target_tiles, cloud_mask=None):
    # Build (sat, t_min) -> T_wait_min lookup from the refined dataset
    # (T_wait_min is constant per event, computed once in Cell 5)
    wait_lookup = (event_data.wait_lookup if hasattr(event_data, "wait_lookup") else
                   refined_events_df[["sat", "t_min", "T_wait_min"]]
                   .drop_duplicates(subset=["sat", "t_min"])
                   .set_index(["sat", "t_min"])["T_wait_min"]
                   .to_dict())

    # v1.5.2: an image with no downlink in the window (wait None or NaN) is
    # never delivered, so it is left out of the curve; before, None crashed
    # the addition and NaN left the sort unordered (Cell 6, delivery_sort_key).
    delivery_times = []
    for event_key in selected_events:
        sat, t_min = event_key
        t_wait = wait_lookup.get((sat, t_min), 0)
        if t_wait is None or (isinstance(t_wait, float) and np.isnan(t_wait)):
            continue
        delivery_times.append((t_min + t_wait, event_key))
    delivery_times.sort(key=lambda x: x[0])       # stable: ties keep selection order

    covered_tiles = set()
    curve = []
    for delivery_time, event_key in delivery_times:
        for tile, util in event_data[event_key].items():
            if util <= 0.0:                 # same coverage rule as the selection (Cell 6)
                continue
            if cloud_mask is not None and cloud_mask.get((event_key, tile), False):
                continue
            covered_tiles.add(tile)
        coverage_pct = len(covered_tiles) / total_target_tiles * 100
        curve.append({"delivery_time_min": delivery_time, "coverage_pct": coverage_pct,
                      "event": event_key})

    return curve


def time_to_reach(curve, target_pct):
    for point in curve:
        if point["coverage_pct"] >= target_pct:
            return point["delivery_time_min"]
    return None  # never reached


print("=== COVERAGE-VS-TIME (Clear-Sky) ===")
curve_clear = compute_coverage_vs_time(sel_events_clear, event_data, refined_events_df, total_target_tiles)
final_cov_clear = curve_clear[-1]["coverage_pct"] if curve_clear else 0.0

print(f"{'delivery_time_min':<20}{'coverage_pct':<15}")
for pt in curve_clear:
    print(f"{pt['delivery_time_min']:<20}{pt['coverage_pct']:.2f}")

print(f"\nFinal coverage: {final_cov_clear:.2f}%")
for pct in [25, 50, 75, 90]:
    if pct <= final_cov_clear:
        t = time_to_reach(curve_clear, pct)
        print(f"Time to reach {pct}% coverage: {t} min ({t/60:.1f} hours)")

print("\n=== COVERAGE-VS-TIME (Cloud/Fail-Signal) ===")
curve_cloud = compute_coverage_vs_time(sel_events_cloud, event_data, refined_events_df,
                                         total_target_tiles, cloud_mask=cloud_mask)
final_cov_cloud = curve_cloud[-1]["coverage_pct"] if curve_cloud else 0.0

print(f"{'delivery_time_min':<20}{'coverage_pct':<15}")
for pt in curve_cloud:
    print(f"{pt['delivery_time_min']:<20}{pt['coverage_pct']:.2f}")

print(f"\nFinal coverage: {final_cov_cloud:.2f}%")
for pct in [25, 50, 75, 90]:
    if pct <= final_cov_cloud:
        t = time_to_reach(curve_cloud, pct)
        print(f"Time to reach {pct}% coverage: {t} min ({t/60:.1f} hours)")

# v1.5.2: the two curves as a file (they were only printed), one row per
# delivered image of the Cell 6 selections (coverage-first, gap-fill included).
import os as _os7b
import pandas as _pd7b
_pd7b.DataFrame(
    [{"sky": sky, "delivery_time_min": p["delivery_time_min"], "coverage_pct": p["coverage_pct"],
      "sat": p["event"][0], "t_min": p["event"][1]}
     for sky, curve in (("clear", curve_clear), ("cloud", curve_cloud)) for p in curve],
    columns=["sky", "delivery_time_min", "coverage_pct", "sat", "t_min"]).to_csv(
    _os7b.path.join(OUT_DIR if 'OUT_DIR' in globals() else _os7b.getcwd(), "arf_coverage_vs_time.csv"), index=False)
print("Saved arf_coverage_vs_time.csv")

print("\n=== CELL 7B COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 7C: OBJECTIVE COMPARISON & ONSET-ROBUSTNESS ANALYSIS (H1)
# =============================================================================
# Two analyses of the H1 latency (mosaic completion time, Cell 6):
#
# 1. DECAY SWEEP: the time-aware objective (Cell 6) at each rate in
#    DECAY_SWEEP, plus rate 0 -- which weights every event equally and is
#    therefore identical to the coverage-first objective. Reports latency
#    against final coverage and mean quality, so LATENCY_DECAY_RATE is one
#    reported point on a trade-off curve, not an unexplained tuned value.
#
# 2. ONSET ROBUSTNESS: one onset time is one sample of orbital geometry; the
#    first pass may arrive early or late by chance. The request time is
#    shifted from 0 to ONSET_SHIFT_HOURS in ONSET_SHIFT_STEP_MIN steps, and for
#    each shift only the passes in the following
#    (WINDOW_HOURS - ONSET_SHIFT_HOURS) hours are used, times re-expressed
#    from the shifted onset. For each shift it records:
#      - best possible: earliest delivery time at which ALL passes so far
#        together reach full reachable coverage (a physical lower bound,
#        independent of budget k and of any algorithm)
#      - coverage-first and time-aware selections (k = MAX_SELECTION_STEPS)
#    under clear sky and the Fail-Signal cloud mask. The cloud mask is the
#    same per-(event, tile) mask as Cell 6, not redrawn per shift.
#
# 3. BASELINES: first-come-first-served (passes in delivery order, kept if
#    they add a new tile) and random selection (median of
#    RANDOM_BASELINE_DRAWS seeded draws), compared with both greedy objectives
#    on the full window and in the onset-robustness analysis.
#
# 4. OPTIMALITY GAP: on ILP_INSTANCES sub-problems (ILP_HORIZON_HOURS of
#    passes, budget ILP_BUDGET_K, clear sky), the exact maximum-utility
#    selection from a mixed-integer program (SciPy/HiGHS) -- the objective the
#    greedy approximates -- reported as each policy's utility / optimum.
#
# EQUAL-COVERAGE LATENCY: completion time is measured to each policy's OWN
# final coverage, so a policy that covers less can look faster (under cloud,
# a budget-limited policy may stop well short of full coverage). Every policy
# and the physical floor are therefore also timed to common coverage levels,
# COVERAGE_TARGETS_PCT (default 95% and 99% of ROI tiles); a policy that never
# reaches a level is recorded as not reaching it. Figure 5 plots the highest
# level. At equal coverage no policy can beat the physical floor.
#
# All policy comparisons in this cell use the budget k WITHOUT the
# Fail-Signal gap-fill: gap-fill adds compensatory passes beyond k, which the
# baselines have no counterpart for, so including it would favour the greedy
# variants. Cell 6 reports H1 with gap-fill; where gap-fill never triggers
# (every tile covered within k) the two agree.
#
# H1 FROM v1.4.0 (tested in Cell 8), per onset, for the level L =
# H1_REACHABLE_COVERAGE_PCT of the tiles reachable in that onset's horizon:
#   H1a, latency non-inferiority: time-aware time to L minus the floor's time
#       to L <= H1_FLOOR_MARGIN_MIN (the pre-specified revised H1).
#   H1b, image economy: images first-come-first-served (FCFS, no budget) has
#       taken delivery of when its mosaic reaches L, against the images of the
#       earliest-completion cover (ECC, budget k), which must also reach L
#       within the margin of the floor. FCFS keeps every image that adds a new
#       tile, so its mosaic at any time is the union of all delivered images:
#       it reaches L exactly at the floor. Onsets where FCFS needs one image
#       cannot show economy and are counted separately.
#   Quality at L (reported, not tested): mean utility of the tiles covered
#       when each policy's delivered mosaic first reaches L.
# A share of onsets is NOT EVALUABLE below H1_MIN_EVALUABLE_ONSETS (Cell 0),
# and is reported with its exact one-sided 95% lower confidence bound
# (Clopper & Pearson 1934). Consecutive onsets share most of their passes, so
# they are not independent trials: the bound is optimistic and the minimum
# count is necessary, not sufficient.
#
# Cost: 2 greedy selections + baselines per shift and sky. Trivial for small
# ROIs; for large ROIs at N=1000, increase ONSET_SHIFT_STEP_MIN or set
# onset_robustness_enabled=False in Cell 0 to trade resolution for time.
#
# Outputs (OUT_DIR): arf_decay_sweep.csv, arf_policy_comparison.csv,
#   arf_onset_robustness.csv, arf_onset_robustness_summary.json,
#   arf_fig5_onset_robustness.png/.pdf, arf_fig9_h1a_gap.png/.pdf (v1.5.2),
#   arf_ilp_gap.csv, arf_cost_latency_frontier.csv, arf_fig6_cost_latency.png/.pdf,
#   arf_cloud_robustness.csv, arf_cloud_robustness_summary.json,
#   arf_cloud_sensitivity.csv, arf_fig8_cloud_robustness.png/.pdf
# =============================================================================
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 7C is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["7C"] = _ARF_CELL_VERSION

import os
import json
import math
import logging
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

logger = logging.getLogger("arf.cell7c")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(asctime)s [%(name)s] %(levelname)s: %(message)s', datefmt='%H:%M:%S'))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()
# Same H1 threshold as Cell 8's H1_LATENCY_THRESHOLD_MIN (Cell 8 runs later,
# so it is restated here; change both together).
_H1_TARGET_MIN = 10.0

# Revised H1 margin: one orbital period at the fleet's median altitude, unless
# set explicitly in Cell 0. T = 2*pi*sqrt(a^3/mu) (Kepler's third law).
H1_ORBITAL_PERIOD_MIN = float(2.0 * np.pi * np.sqrt((Re_km + CONSTELLATION_ALTITUDE_KM) ** 3 / mu_earth) / 60.0)
H1_FLOOR_MARGIN_MIN = (H1_ORBITAL_PERIOD_MIN if globals().get("H1_FLOOR_MARGIN_MIN_CONFIG") is None
                       else float(H1_FLOOR_MARGIN_MIN_CONFIG))
H1_MARGIN_SOURCE = ("derived: one orbital period at the fleet median altitude "
                    f"{CONSTELLATION_ALTITUDE_KM:.1f} km" if globals().get("H1_FLOOR_MARGIN_MIN_CONFIG") is None
                    else "set in Cell 0")
_H1R = float(globals().get("H1_REACHABLE_COVERAGE_PCT", 95.0))
_H1MIN = float(globals().get("H1_MIN_ONSET_PCT", 90.0))

# Evaluability (v1.4.0). Clopper, C. J. & Pearson, E. S. (1934). The use of
# confidence or fiducial limits illustrated in the case of the binomial.
# Biometrika 26(4), 404-413, doi:10.1093/biomet/26.4.404. The exact one-sided
# lower bound for x successes in n trials at confidence 1 - alpha is the alpha
# quantile of Beta(x, n - x + 1); with x = n it is alpha**(1/n).
H1_CONFIDENCE_ALPHA = 0.05


def cp_lower_bound(x, n, alpha=H1_CONFIDENCE_ALPHA):
    """Exact (Clopper-Pearson) one-sided lower confidence bound, as a fraction,
    for x successes in n trials; 0 if x == 0, None if n == 0 or SciPy is missing."""
    if not n:
        return None
    if x <= 0:
        return 0.0
    try:
        from scipy.stats import beta
    except ImportError:
        return None
    return float(beta.ppf(alpha, x, n - x + 1))


def min_onsets_for(p0_pct, alpha=H1_CONFIDENCE_ALPHA):
    """Smallest n for which n successes in n trials give a one-sided lower bound
    >= p0_pct: alpha**(1/n) >= p0, i.e. n >= ln(alpha) / ln(p0). None for p0 = 100%
    (no finite sample reaches it)."""
    p0 = float(p0_pct) / 100.0
    if p0 >= 1.0:
        return None
    return int(math.ceil(math.log(alpha) / math.log(p0) - 1e-12))


H1_MIN_EVALUABLE_ONSETS = (int(H1_MIN_EVALUABLE_ONSETS_CONFIG)
                           if globals().get("H1_MIN_EVALUABLE_ONSETS_CONFIG") is not None
                           else (min_onsets_for(_H1MIN) or 1))
H1_EVALUABILITY_SOURCE = ("set in Cell 0" if globals().get("H1_MIN_EVALUABLE_ONSETS_CONFIG") is not None else
                          f"derived: smallest n with {H1_CONFIDENCE_ALPHA:g}**(1/n) >= {_H1MIN / 100:g} "
                          f"(all-pass Clopper-Pearson one-sided {100 * (1 - H1_CONFIDENCE_ALPHA):g}% lower bound)")


def subset_by_time(store, mask, t0, t1):
    """
    EventStore (and aligned CloudMask) of the events with t0 <= t_min < t1,
    with times re-expressed relative to t0. Rows and cloud flags are copied
    from the full store, so utilities and cloud cover are unchanged.
    """
    keep = [e for e, k in enumerate(store.keys_) if t0 <= k[1] < t1]
    if not keep:
        empty = EventStore([], np.zeros(1, np.int64), np.zeros(0, np.int32), np.zeros(0), [])
        return empty, (CloudMask(empty, np.zeros(0, bool)) if mask is not None else None)
    rows = np.concatenate([np.arange(store.offsets[e], store.offsets[e + 1]) for e in keep])
    counts = store.counts[keep]
    sub = EventStore([(store.keys_[e][0], store.keys_[e][1] - t0) for e in keep],
                     np.r_[0, np.cumsum(counts)], store.tile_idx[rows], store.util[rows],
                     [store.wait_lookup[store.keys_[e]] for e in keep])
    return sub, (CloudMask(sub, mask.flags[rows]) if mask is not None else None)


def best_possible_completion(store, mask=None):
    """
    Physical lower bound on mosaic completion: with ALL events ordered by
    delivery time, the delivery time at which their union first reaches the
    full coverage reachable from this store (utility > 0, not cloud-obscured).
    Returns (minutes, coverage_pct), or (None, 0.0) if nothing is reachable.
    """
    if len(store) == 0:
        return None, 0.0
    util = np.where(mask.flags, 0.0, store.util) if mask is not None else store.util
    reachable = np.unique(store.tile_idx[util > 0.0]).size
    if reachable == 0:
        return None, 0.0
    deliveries = []
    for e, (sat, t) in enumerate(store.keys_):
        w = store.wait_lookup.get((sat, t))
        if w is not None and not np.isnan(w):
            deliveries.append((t + w, e))
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n_seen = 0
    for d, e in sorted(deliveries):
        s = store.rows(e)
        t = store.tile_idx[s][util[s] > 0.0]
        new = t[~seen[t]]
        if new.size:
            seen[new] = True
            n_seen += new.size
            if n_seen == reachable:
                return float(d), 100.0 * reachable / total_target_tiles
    return None, 100.0 * n_seen / total_target_tiles


def _run_objective(store, mask, decay):
    """Selection at budget MAX_SELECTION_STEPS (no gap-fill -- see header) with
    the time-aware objective at `decay` (decay 0 == coverage-first). Returns
    completion latency, final coverage % and final mean quality of the
    covered tiles."""
    sel, cov, _, _ = select_passes(
        store, MAX_SELECTION_STEPS, cloud_mask=mask, gap_fill=False,
        time_aware=decay > 0, latency_decay_rate=decay)
    lat = compute_completion_latency(sel, store, mask)
    q = float(np.mean(list(cov.values()))) if cov else 0.0
    return lat, 100.0 * len(cov) / total_target_tiles, q


def _effective_util(store, mask):
    """Per-row utility with cloud-obscured rows zeroed (same rule as Cell 6)."""
    return np.where(mask.flags, 0.0, store.util) if mask is not None else store.util


def _delivery_time(store, key):
    w = store.wait_lookup.get(key)
    return key[1] + (np.nan if w is None else w)


def select_fcfs(store, mask, k):
    """
    First-come-first-served baseline: take passes in order of delivery time
    (t_min + downlink wait), keeping any pass that adds at least one tile not
    yet covered, until k passes are kept. The obvious operational rule: no
    optimisation, just "use the next pass that shows something new".
    Returns (selected_keys, covered utility per tile index).
    """
    util = _effective_util(store, mask)
    best = np.zeros(len(_tile_ids))
    selected = []
    for key in sorted(store.keys_, key=delivery_sort_key(store)):
        if len(selected) >= k:
            break
        s = store.rows(store.index[key])
        t, u = store.tile_idx[s], util[s]
        if np.any((u > 0.0) & (best[t] == 0.0)):
            selected.append(key)
            np.maximum.at(best, t, u)
    return selected, best


def select_random(store, mask, k, rng):
    """Random baseline: k passes drawn uniformly (without replacement) from those
    that could cover at least one tile. With realistic cloud knowledge the draw
    uses clear-sky geometry (which passes turn out cloudy is not known in
    advance); coverage is then scored with the cloud mask."""
    util = _effective_util(store, mask)
    draw_util = (store.util if mask is not None and globals().get("CLOUD_KNOWLEDGE", "realistic") != "oracle"
                 else util)
    useful = [e for e in range(len(store)) if np.any(draw_util[store.rows(e)] > 0.0)]
    pick = rng.choice(len(useful), size=min(k, len(useful)), replace=False) if useful else []
    best = np.zeros(len(_tile_ids))
    selected = []
    for i in pick:
        e = useful[int(i)]
        s = store.rows(e)
        np.maximum.at(best, store.tile_idx[s], util[s])
        selected.append(store.keys_[e])
    return selected, best


def solve_ilp_max_utility(store, mask, k, time_limit_s=60.0, max_pairs=300_000, weight=None):
    """
    Exact budgeted maximum-utility selection (the objective Cell 6's greedy
    approximates), solved as a mixed-integer program with SciPy's HiGHS.
    weight (per event, v1.5.0): solve the time-weighted coverage instead,
    u[e,t] replaced by weight[e] * u[e,t] -- the exact optimum of the
    facility objective of the time-aware selector (Cell 0):
        maximise   sum_{e,t} u[e,t] * z[e,t]
        subject to sum_e z[e,t] <= 1          (a tile takes its utility once)
                   z[e,t] <= x[e]             (only from a selected pass)
                   sum_e x[e] <= k            (budget)
                   x binary, 0 <= z <= 1
    Given binary x, the optimal z simply assigns each tile to its best selected
    pass, so the objective equals the greedy's sum of best utilities per tile.
    Returns dict(status, objective, bound, selected, seconds, n_pairs).
    """
    import time as _time
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import coo_matrix, vstack
    util = _effective_util(store, mask)
    scale = 1.0
    if weight is not None:
        w = np.asarray(weight, dtype=np.float64)
        has = np.add.reduceat((util > 0.0).astype(np.int64), store.offsets[:-1]) > 0 if len(store) else np.zeros(0, bool)
        # v1.5.1: weights normalised to a maximum of 1 over the passes that can
        # contribute, so the solver's absolute tolerances (1e-6 to 1e-7) do not
        # swallow a small objective; the objective is scaled back on return.
        scale = float(w[has].max()) if has.any() and w[has].max() > 0 else 1.0
        util = util * np.repeat(w / scale, store.counts)
    keep = util > 0.0
    pair_event = np.repeat(np.arange(len(store)), store.counts)[keep]
    pair_tile = store.tile_idx[keep]
    pair_u = util[keep]
    n_e, n_p = len(store), int(keep.sum())
    if n_p == 0:
        return {"status": "empty", "objective": 0.0, "bound": 0.0, "selected": [], "seconds": 0.0, "n_pairs": 0}
    if n_p > max_pairs:
        return {"status": f"skipped (>{max_pairs} pairs)", "objective": None, "bound": None,
                "selected": [], "seconds": 0.0, "n_pairs": n_p}
    _, tile_row = np.unique(pair_tile, return_inverse=True)
    n_t = int(tile_row.max()) + 1
    c = np.concatenate([np.zeros(n_e), -pair_u])
    budget = coo_matrix((np.ones(n_e), (np.zeros(n_e), np.arange(n_e))), shape=(1, n_e + n_p))
    per_tile = coo_matrix((np.ones(n_p), (tile_row, n_e + np.arange(n_p))), shape=(n_t, n_e + n_p))
    link = coo_matrix((np.r_[np.ones(n_p), -np.ones(n_p)],
                       (np.r_[np.arange(n_p), np.arange(n_p)], np.r_[n_e + np.arange(n_p), pair_event])),
                      shape=(n_p, n_e + n_p))
    A = vstack([budget, per_tile, link]).tocsr()
    ub = np.r_[k, np.ones(n_t), np.zeros(n_p)]
    t0 = _time.time()
    # v1.5.1: relative MIP gap 1e-9 (HiGHS default 1e-4: "optimal" then meant
    # within 0.01% of the optimum)
    res = milp(c, constraints=LinearConstraint(A, -np.inf, ub), bounds=Bounds(0, 1),
               integrality=np.r_[np.ones(n_e), np.zeros(n_p)],
               options={"time_limit": float(time_limit_s), "mip_rel_gap": 1e-9})
    secs = _time.time() - t0
    if res.x is None:
        return {"status": f"no solution ({res.message})", "objective": None, "bound": None,
                "selected": [], "seconds": secs, "n_pairs": n_p}
    chosen = [store.keys_[e] for e in np.flatnonzero(res.x[:n_e] > 0.5)]
    bound = getattr(res, "mip_dual_bound", None)
    return {"status": "optimal" if res.status == 0 else "time limit (best found)",
            "objective": float(-res.fun) * scale, "bound": None if bound is None else float(-bound) * scale,
            "selected": chosen, "seconds": secs, "n_pairs": n_p}


def solve_ilp_min_images(store, mask, need, time_limit_s=60.0, max_pairs=300_000):
    """
    Exact minimum number of images covering at least `need` tiles (a tile is
    covered by any usable observation), as a mixed-integer program (HiGHS):
        minimise sum_e x[e]
        subject to y[t] <= sum_{e covers t} x[e],  sum_t y[t] >= need,
                   x, y binary.
    The check for ECC's greedy cover step. Returns dict(status, images, seconds, n_pairs).
    """
    import time as _time
    from scipy.optimize import milp, LinearConstraint, Bounds
    from scipy.sparse import coo_matrix, vstack
    util = _effective_util(store, mask)
    keep = util > 0.0
    pair_event = np.repeat(np.arange(len(store)), store.counts)[keep]
    pair_tile = store.tile_idx[keep]
    n_e, n_p = len(store), int(keep.sum())
    if n_p == 0:
        return {"status": "empty", "images": None, "seconds": 0.0, "n_pairs": 0}
    if n_p > max_pairs:
        return {"status": f"skipped (>{max_pairs} pairs)", "images": None, "seconds": 0.0, "n_pairs": n_p}
    _, tile_row = np.unique(pair_tile, return_inverse=True)
    n_t = int(tile_row.max()) + 1
    if n_t < need:
        return {"status": "infeasible", "images": None, "seconds": 0.0, "n_pairs": n_p}
    c = np.r_[np.ones(n_e), np.zeros(n_t)]
    cover = coo_matrix((np.r_[-np.ones(n_p), np.ones(n_t)],
                        (np.r_[tile_row, np.arange(n_t)], np.r_[pair_event, n_e + np.arange(n_t)])),
                       shape=(n_t, n_e + n_t))
    total = coo_matrix((np.ones(n_t), (np.zeros(n_t), n_e + np.arange(n_t))), shape=(1, n_e + n_t))
    t0 = _time.time()
    res = milp(c, constraints=[LinearConstraint(cover.tocsr(), -np.inf, 0.0), LinearConstraint(total.tocsr(), need, np.inf)],
               bounds=Bounds(0, 1), integrality=np.ones(n_e + n_t), options={"time_limit": float(time_limit_s)})
    secs = _time.time() - t0
    if res.x is None or res.status != 0:
        return {"status": "time limit or no solution", "images": None, "seconds": secs, "n_pairs": n_p}
    return {"status": "optimal", "images": int(round(res.fun)), "seconds": secs, "n_pairs": n_p}


def reachable_tile_count(store, mask):
    """Tiles with at least one usable observation (utility > 0, not
    cloud-obscured) among ALL events in this store: the most any policy
    could cover from these passes."""
    util = _effective_util(store, mask)
    return int(np.unique(store.tile_idx[util > 0.0]).size)


def coverage_times(store, mask, targets, selected=None, denominator=None):
    """
    Delivery time at which delivered coverage first reaches each target (% of
    ROI tiles), with events ordered by delivery time (t_min + downlink wait).
    selected=None uses ALL events -- the physical floor for that coverage
    level; otherwise only the selected events. A tile counts under the same
    rule as everywhere else (utility > 0, not cloud-obscured). Returns
    {target: minutes or None if never reached}. denominator: tile count the
    percentages refer to (default: all ROI tiles; the revised H1 passes the
    reachable-tile count).
    """
    util = _effective_util(store, mask)
    denom = total_target_tiles if denominator is None else denominator
    if denom <= 0:
        return {tg: None for tg in targets}
    keys = store.keys_ if selected is None else selected
    order = sorted(keys, key=delivery_sort_key(store))
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n_seen, out, pending = 0, {}, sorted(targets)
    for key in order:
        if not pending:
            break
        d = _delivery_time(store, key)
        if np.isnan(d):
            continue
        s = store.rows(store.index[key])
        t = store.tile_idx[s][util[s] > 0.0]
        new = t[~seen[t]]
        if new.size:
            seen[new] = True
            n_seen += new.size
            pct = 100.0 * n_seen / denom
            while pending and pct >= pending[0] - 1e-9:
                out[pending.pop(0)] = float(d)
    out.update({tg: None for tg in pending})
    return out


def _count_cover(store, util_eff, events, k, need):
    """Greedy maximum coverage, UNWEIGHTED (a tile counts once it has any usable
    observation: utility > 0, not cloud-obscured), using at most k of
    `events` (event indices, in delivery order -- ties go to the earlier
    image). Stops as soon as `need` tiles are covered.
    Returns (tiles covered, chosen event indices in greedy order)."""
    events = np.asarray(events, dtype=np.int64)
    if events.size == 0 or k <= 0:
        return 0, []
    counts = store.counts[events]
    rows = np.concatenate([np.arange(store.offsets[e], store.offsets[e + 1]) for e in events])
    starts = np.r_[0, np.cumsum(counts)[:-1]]
    t = store.tile_idx[rows]
    ok = util_eff[rows] > 0.0
    covered = np.zeros(len(_tile_ids), dtype=bool)
    avail = np.ones(events.size, dtype=bool)
    chosen, n_cov = [], 0
    for _ in range(k):
        gain = np.add.reduceat((ok & ~covered[t]).astype(np.int64), starts)
        gain[~avail] = -1
        j = int(np.argmax(gain))
        if gain[j] <= 0:
            break
        avail[j] = False
        chosen.append(int(events[j]))
        sl = slice(int(starts[j]), int(starts[j] + counts[j]))
        tt = t[sl][ok[sl]]
        n_cov += int((~covered[tt]).sum())
        covered[tt] = True
        if n_cov >= need:
            break
    return n_cov, chosen


def earliest_completion(store, mask, k, target_pct, denominator=None):
    """
    Earliest-completion cover (ECC). The earliest delivery time t at which at
    most k images DELIVERED BY t cover target_pct of the tiles (of the ROI, or
    of `denominator` tiles), found by binary search over delivery times with
    a greedy set cover at each probe; the floor (all images) bounds the search
    from below. Uses only images already delivered at t, so under cloud it is
    causal: their cloud is known by then (from the image itself).
    Returns (completion time = latest delivery among the chosen images or
    None if never reached, chosen keys, images used). The greedy cover step
    is approximate (Chvatal 1979: at most H(n) times the minimum number of
    images), so the time is the earliest at which the GREEDY finds a cover.
    Bisection assumes that once the greedy succeeds with the images delivered
    by t it also succeeds with those delivered by any later t. Greedy maximum
    coverage is not monotone in its ground set in general, so this is an
    assumption (v1.5.2 note): it held at every onset of the October 2026 runs
    (bisection equal to a linear scan, test_ecc_monotone.py), and a violation
    could only make the reported time LATER than the greedy's earliest success
    -- every reported time is achieved by its returned images -- so H1a can
    only be understated by it, never overstated.
    """
    util_eff = _effective_util(store, mask)
    denom = total_target_tiles if denominator is None else denominator
    if denom <= 0 or len(store) == 0:
        return None, [], 0
    need = int(np.ceil(target_pct / 100.0 * denom - 1e-9))
    d = event_delivery_times(store)
    order = np.argsort(d, kind="stable")
    order = order[np.isfinite(d[order])]
    times = d[order]
    seen = np.zeros(len(_tile_ids), dtype=bool)
    n_seen, lo = 0, None
    for pos, e in enumerate(order):                       # floor: all delivered images
        s = store.rows(e)
        tt = store.tile_idx[s][util_eff[s] > 0.0]
        new = tt[~seen[tt]]
        seen[new] = True
        n_seen += new.size
        if n_seen >= need:
            lo = pos
            break
    if lo is None:
        return None, [], 0

    def probe(pos):
        end = int(np.searchsorted(times, times[pos], side="right"))   # all images delivered by times[pos]
        n_cov, chosen = _count_cover(store, util_eff, order[:end], k, need)
        return n_cov >= need, chosen

    hi = len(order) - 1
    ok_hi, chosen_hi = probe(hi)
    if not ok_hi:
        return None, [], 0
    best = chosen_hi
    a, b = lo, hi
    while a < b:
        mid = (a + b) // 2
        ok_mid, ch = probe(mid)
        if ok_mid:
            b, best = mid, ch
        else:
            a = mid + 1
    # Invariant: probe(b) succeeded and `best` is its cover, so at exit (a == b)
    # `best` is the cover at the earliest successful delivery time found.
    return float(max(d[e] for e in best)), [store.keys_[e] for e in best], len(best)


def _ecc_best_effort(store, mask, k):
    """ECC when no target is reachable: the greedy k-image cover of every
    delivered image (maximum coverage by the end of the window)."""
    util_eff = _effective_util(store, mask)
    d = event_delivery_times(store)
    order = np.argsort(d, kind="stable")
    order = order[np.isfinite(d[order])]
    _, chosen = _count_cover(store, util_eff, order, k, need=10 ** 12)
    return [store.keys_[e] for e in chosen]


def level_stats(store, mask, selected, need):
    """The delivered mosaic of `selected` (images in delivery order, ties in
    selection order, as compute_completion_latency) when it first covers `need`
    tiles (usable: utility > 0, not cloud-obscured): (delivery time of the image
    at which that happens, images delivered up to and including it, mean
    utility of the tiles covered then). (None, None, None) if never."""
    if need <= 0 or not selected:
        return None, None, None
    util = _effective_util(store, mask)
    best = np.zeros(len(_tile_ids))
    n_cov = 0
    for i, key in enumerate(sorted(selected, key=delivery_sort_key(store)), 1):
        d = _delivery_time(store, key)
        if np.isnan(d):
            break                                   # undelivered images sort last
        s = store.rows(store.index[key])
        t, u = store.tile_idx[s], util[s]
        n_cov += int(np.unique(t[(u > 0.0) & (best[t] == 0.0)]).size)
        np.maximum.at(best, t, u)
        if n_cov >= need:
            cov = best > 0.0
            return float(d), i, float(best[cov].mean())
    return None, None, None


def mosaic_quality(store, mask, selected):
    """Mean utility of the tiles covered by `selected` (best usable observation per tile)."""
    util = _effective_util(store, mask)
    best = np.zeros(len(_tile_ids))
    for key in selected:
        s = store.rows(store.index[key])
        np.maximum.at(best, store.tile_idx[s], util[s])
    cov = best > 0.0
    return float(best[cov].mean()) if cov.any() else None


def facility_value(store, mask, selected, weight):
    """Time-weighted coverage f(S) = sum_t max_{e in S} w_e * u_e,t (usable observations)."""
    util = _effective_util(store, mask)
    best = np.zeros(len(_tile_ids))
    for key in selected:
        e = store.index[key]
        s = store.rows(e)
        np.maximum.at(best, store.tile_idx[s], weight[e] * util[s])
    return float(best.sum())


def realized_coverage(store, mask, selected):
    """Tiles with a usable observation (utility > 0, not cloud-obscured) among `selected`."""
    util = _effective_util(store, mask)
    seen = np.zeros(len(_tile_ids), dtype=bool)
    for key in selected:
        s = store.rows(store.index[key])
        seen[store.tile_idx[s][util[s] > 0.0]] = True
    return int(seen.sum())


def plan_expected_coverage(store, k, p_tile, weight=None):
    """
    Open-loop plan (v1.5.0): k passes chosen IN ADVANCE, with no cloud
    observation, by the same planner the adaptive selector uses
    (_expected_plan, Cell 6): expected coverage under cloud from pass geometry
    and each tile's cloud probability, passes treated as independent,
    time-weighted when `weight` is given. The adaptive selector differs only
    in re-planning from what it has seen, so the comparison isolates the value
    of acting on cloud observations.
    """
    if len(store) == 0 or k <= 0:
        return []
    d = event_delivery_times(store)
    plan = _expected_plan(store, np.ones(len(_tile_ids), dtype=bool), np.isfinite(d), int(k), p_tile, weight)
    return [store.keys_[e] for e in plan]


def paired_summary(d, alpha=0.05):
    """Mean of paired differences with its one-sided (1 - alpha) lower confidence bound
    (Student t, n - 1 degrees of freedom), SD, and the share of positive / negative pairs."""
    d = np.asarray([x for x in d if x is not None and np.isfinite(x)], dtype=np.float64)
    n = int(d.size)
    if n == 0:
        return {"n": 0, "mean": None, "sd": None, "lcb": None, "pct_positive": None, "pct_negative": None}
    m = float(d.mean())
    sd = float(d.std(ddof=1)) if n > 1 else 0.0
    if n > 1:
        from scipy.stats import t as _t
        lcb = m - float(_t.ppf(1.0 - alpha, n - 1)) * sd / math.sqrt(n)
    else:
        lcb = None
    return {"n": n, "mean": m, "sd": sd, "lcb": lcb, "alpha": alpha,
            "pct_positive": float(100.0 * (d > 1e-9).mean()), "pct_negative": float(100.0 * (d < -1e-9).mean())}


def _metrics(selected, store, mask):
    """Completion latency, coverage %, mean quality and per-target coverage
    times of one selection (budget-limited, no gap-fill)."""
    util = _effective_util(store, mask)
    best = np.zeros(len(_tile_ids))
    for key in selected:
        s = store.rows(store.index[key])
        np.maximum.at(best, store.tile_idx[s], util[s])
    covered = best > 0.0
    return {"lat": compute_completion_latency(selected, store, mask),
            "cov": 100.0 * int(covered.sum()) / total_target_tiles,
            "q": float(best[covered].mean()) if covered.any() else 0.0,
            "t": coverage_times(store, mask, COVERAGE_TARGETS_PCT, selected)}


POLICIES = ("coverage_first", "time_aware", "fcfs", "random", "earliest_completion")


def _policy_selections(store, mask, policy, seed=None):
    """One selection (deterministic policies) or RANDOM_BASELINE_DRAWS seeded
    selections (random), each at budget MAX_SELECTION_STEPS, no gap-fill."""
    k = MAX_SELECTION_STEPS
    if policy in ("coverage_first", "time_aware"):
        rate = 0.0 if policy == "coverage_first" else LATENCY_DECAY_RATE
        sel, _, _, _ = select_passes(store, k, cloud_mask=mask, gap_fill=False,
                                                       time_aware=rate > 0, latency_decay_rate=rate)
        return [sel]
    if policy == "fcfs":
        return [select_fcfs(store, mask, k)[0]]
    rng = np.random.default_rng(RANDOM_SEED if seed is None else seed)
    return [select_random(store, mask, k, rng)[0] for _ in range(RANDOM_BASELINE_DRAWS)]


def _median_or_none(values):
    """Median treating None as 'never' (+inf); None if the median is never."""
    a = np.array([np.inf if v is None else v for v in values], dtype=float)
    m = float(np.median(a)) if a.size else np.inf
    return None if np.isinf(m) else m


def _policy_outcome(store, mask, policy, seed=None):
    """Metrics of one policy: completion latency, coverage %, mean quality and
    time to each coverage target. Random: median over its seeded draws."""
    if len(store) == 0:
        return {"lat": None, "cov": 0.0, "q": 0.0, "t": {tg: None for tg in COVERAGE_TARGETS_PCT}}
    if policy == "earliest_completion":
        # One ECC run per coverage target (it stops once its target is met);
        # coverage/quality/latency are those of the highest target reached,
        # or of the best k-image cover if none is reached.
        t, best_sel = {}, None
        for tg in sorted(COVERAGE_TARGETS_PCT):
            tc, sel, _ = earliest_completion(store, mask, MAX_SELECTION_STEPS, tg)
            t[tg] = tc
            if sel:
                best_sel = sel
        m = _metrics(best_sel if best_sel is not None else _ecc_best_effort(store, mask, MAX_SELECTION_STEPS),
                     store, mask)
        m["t"] = t
        return m
    ms = [_metrics(sel, store, mask) for sel in _policy_selections(store, mask, policy, seed)]
    if len(ms) == 1:
        return ms[0]
    return {"lat": _median_or_none([m["lat"] for m in ms]),
            "cov": float(np.median([m["cov"] for m in ms])),
            "q": float(np.median([m["q"] for m in ms])),
            "t": {tg: _median_or_none([m["t"][tg] for m in ms]) for tg in COVERAGE_TARGETS_PCT}}


# =============================================================================
# 1. DECAY SWEEP
# =============================================================================
_sweep_rates = [0.0] + sorted(set(float(x) for x in DECAY_SWEEP) - {0.0})
decay_sweep_rows = []
for rate in _sweep_rates:
    lc, cc, qc = _run_objective(event_data, None, rate)
    ll, cl, ql = _run_objective(event_data, cloud_mask, rate)
    decay_sweep_rows.append({
        "decay_rate": rate, "objective": "coverage-first" if rate == 0 else "time-aware",
        "h1_latency_clear_min": lc, "coverage_clear_pct": cc, "mean_quality_clear": qc,
        "h1_latency_cloud_min": ll, "coverage_cloud_pct": cl, "mean_quality_cloud": ql,
    })
decay_sweep_df = pd.DataFrame(decay_sweep_rows)
decay_sweep_df.to_csv(os.path.join(_out_dir, "arf_decay_sweep.csv"), index=False)
logger.info(f"Decay sweep over {len(_sweep_rates)} rates saved to arf_decay_sweep.csv")

# =============================================================================
# 1B. POLICY COMPARISON (full window, budget k, no gap-fill)
# =============================================================================
policy_rows = []
for sky, m in (("clear", None), ("cloud", cloud_mask)):
    floor = coverage_times(event_data, m, COVERAGE_TARGETS_PCT)
    policy_rows.append({"sky": sky, "policy": "best_possible", "h1_latency_min": None, "coverage_pct": None,
                        "mean_quality": None, **{f"t{tg:g}_min": floor[tg] for tg in COVERAGE_TARGETS_PCT}})
    for name in POLICIES:
        o = _policy_outcome(event_data, m, name)
        policy_rows.append({"sky": sky, "policy": name, "h1_latency_min": o["lat"],
                            "coverage_pct": o["cov"], "mean_quality": o["q"],
                            **{f"t{tg:g}_min": o["t"][tg] for tg in COVERAGE_TARGETS_PCT}})
policy_df = pd.DataFrame(policy_rows)
policy_df.to_csv(os.path.join(_out_dir, "arf_policy_comparison.csv"), index=False)

# =============================================================================
# 1C. COST-LATENCY FRONTIER (earliest-completion cover at each image budget)
# =============================================================================
# For a budget of k images: the earliest time a mosaic of at most k delivered
# images reaches the target. k -> infinity is the floor. Targets: the revised-
# H1 level of REACHABLE tiles, and each COVERAGE_TARGETS_PCT level of the ROI.
# Grid up to the number of candidate images, plus the run's own budget k always
# (a budget above the image count behaves like the image count).
_frontier_k = sorted({k for k in (1, 2, 3, 5, 8, 10, 15, 20, 25, 30, 40, 50, 75, 100, 150, 200)
                      if k <= len(event_data)} | {MAX_SELECTION_STEPS})
frontier_rows = []
_h1r_level = float(globals().get("H1_REACHABLE_COVERAGE_PCT", 95.0))
for sky, m in (("clear", None), ("cloud", cloud_mask)):
    _reach = reachable_tile_count(event_data, m)
    targets = [("reachable", _h1r_level, _reach)] + [("roi", tg, total_target_tiles) for tg in COVERAGE_TARGETS_PCT]
    for kind, tg, denom in targets:
        floor_t = coverage_times(event_data, m, (tg,), denominator=denom)[tg] if denom else None
        for k in _frontier_k:
            tc, _, n_used = earliest_completion(event_data, m, k, tg, denominator=denom) if denom else (None, [], 0)
            frontier_rows.append({"sky": sky, "target": kind, "target_pct": tg, "denominator_tiles": denom,
                                  "k": k, "t_complete_min": tc, "images_used": n_used if tc is not None else None,
                                  "floor_min": floor_t})
frontier_df = pd.DataFrame(frontier_rows)
frontier_df.to_csv(os.path.join(_out_dir, "arf_cost_latency_frontier.csv"), index=False)
logger.info(f"Cost-latency frontier over k={_frontier_k} saved to arf_cost_latency_frontier.csv")

# v1.5.2: two IEEE columns wide (7.16 in) with 8-9 pt text (was 7.5 x 4.6 in).
with plt.rc_context({"font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "xtick.labelsize": 8,
                     "ytick.labelsize": 8, "legend.fontsize": 8, "axes.linewidth": 0.6,
                     "pdf.fonttype": 42, "ps.fonttype": 42}):
    fig6, ax6 = plt.subplots(figsize=(7.16, 3.0))
    for sky, col in (("clear", "#1b9e77"), ("cloud", "#7570b3")):
        f = frontier_df[(frontier_df.sky == sky) & (frontier_df.target == "reachable")]
        ok = f.t_complete_min.notna()
        if ok.any():
            ax6.plot(f.k[ok], f.t_complete_min[ok] / 60.0, marker="o", ms=3.5, lw=1.2, color=col,
                     label=f"{'Clear sky' if sky == 'clear' else 'Cloud (realised)'}: ECC")
            fl = f.floor_min.dropna()
            if len(fl):
                ax6.axhline(fl.iloc[0] / 60.0, color=col, ls=":", lw=1.0,
                            label=f"{'Clear' if sky == 'clear' else 'Cloud'} floor (all images)")
    ax6.set_xscale("log")
    ax6.set_xlabel("Image budget k (images downloaded / processed / purchased)")
    ax6.set_ylabel(f"Time to {_h1r_level:g}% of reachable tiles (h)")
    _scen6 = globals().get("SCENARIO_N", ACTIVE_N)
    ax6.set_title(f"Cost-latency frontier: {roi_short_name() if 'roi_short_name' in globals() else ACTIVE_ROI}, "
                  f"{'N=' + str(_scen6) if isinstance(_scen6, int) else _scen6}", loc="left")
    ax6.grid(alpha=0.3, which="both", lw=0.5)
    ax6.legend()
    fig6.tight_layout()
    for fmt in ("png", "pdf"):
        fig6.savefig(os.path.join(_out_dir, f"arf_fig6_cost_latency.{fmt}"), dpi=300, bbox_inches="tight")
    plt.show()
    plt.close(fig6)

# =============================================================================
# 2. ONSET ROBUSTNESS
# =============================================================================
onset_robustness_summary = {}
if ONSET_ROBUSTNESS_ENABLED:
    _horizon = (WINDOW_HOURS - ONSET_SHIFT_HOURS) * 60
    _shifts = list(range(0, ONSET_SHIFT_HOURS * 60 + 1, ONSET_SHIFT_STEP_MIN))
    logger.info(f"Onset robustness: {len(_shifts)} onsets over {ONSET_SHIFT_HOURS} h, "
                f"{_horizon / 60:.0f} h of passes each, decay={LATENCY_DECAY_RATE}")
    onset_rows = []
    for d in _shifts:
        sub, sub_mask = subset_by_time(event_data, cloud_mask, d, d + _horizon)
        row = {"onset_shift_min": d, "n_events": len(sub),
               "decay_tau_min": decay_time_scale(sub) if len(sub) else None}   # time-aware weight scale here
        for sky, m in (("clear", None), ("cloud", sub_mask)):
            row[f"best_possible_{sky}_min"], row[f"best_possible_{sky}_cov_pct"] = best_possible_completion(sub, m)
            for tg, v in coverage_times(sub, m, COVERAGE_TARGETS_PCT).items():
                row[f"best_possible_{sky}_t{tg:g}_min"] = v
            for name in POLICIES:
                o = _policy_outcome(sub, m, name, seed=RANDOM_SEED + d)
                row[f"{name}_{sky}_min"], row[f"{name}_{sky}_cov_pct"] = o["lat"], o["cov"]
                for tg, v in o["t"].items():
                    row[f"{name}_{sky}_t{tg:g}_min"] = v
            # Revised H1: time to H1_REACHABLE_COVERAGE_PCT of the tiles reachable
            # in this onset's horizon, floor vs each deterministic policy.
            n_reach = reachable_tile_count(sub, m)
            row[f"reachable_{sky}_tiles"] = n_reach
            row[f"best_possible_{sky}_t{_H1R:g}r_min"] = coverage_times(sub, m, (_H1R,), denominator=n_reach)[_H1R]
            # H1a/H1b (v1.4.0): time, images and mosaic quality when each
            # policy's delivered mosaic first covers `need` reachable tiles.
            need = int(np.ceil(_H1R / 100.0 * n_reach - 1e-9)) if n_reach else 0
            for name in ("coverage_first", "time_aware", "fcfs"):
                sel = _policy_selections(sub, m, name)[0] if len(sub) else []
                row[f"{name}_{sky}_t{_H1R:g}r_min"] = (coverage_times(sub, m, (_H1R,), sel, denominator=n_reach)[_H1R]
                                                       if n_reach else None)
                if name != "fcfs":
                    _, n_img, q = level_stats(sub, m, sel, need) if n_reach else (None, None, None)
                    row[f"{name}_{sky}_images{_H1R:g}r"], row[f"{name}_{sky}_quality{_H1R:g}r"] = n_img, q
            # FCFS without a budget: the images it takes to reach the level (at the floor).
            t_f, n_f, q_f = (level_stats(sub, m, select_fcfs(sub, m, len(sub))[0], need) if n_reach
                             else (None, None, None))
            row[f"fcfs_unbudgeted_{sky}_t{_H1R:g}r_min"] = t_f
            row[f"fcfs_unbudgeted_{sky}_images{_H1R:g}r"], row[f"fcfs_unbudgeted_{sky}_quality{_H1R:g}r"] = n_f, q_f
            t_e, sel_e, n_e = (earliest_completion(sub, m, MAX_SELECTION_STEPS, _H1R, denominator=n_reach) if n_reach
                               else (None, [], 0))
            row[f"earliest_completion_{sky}_t{_H1R:g}r_min"] = t_e
            row[f"earliest_completion_{sky}_images{_H1R:g}r"] = n_e if t_e is not None else None
            row[f"earliest_completion_{sky}_quality{_H1R:g}r"] = mosaic_quality(sub, m, sel_e) if t_e is not None else None
        onset_rows.append(row)
    onset_df = pd.DataFrame(onset_rows)
    onset_df.to_csv(os.path.join(_out_dir, "arf_onset_robustness.csv"), index=False)

    def _stats(col):
        x = pd.to_numeric(onset_df[col], errors="coerce").to_numpy(dtype=float)
        ok = ~np.isnan(x)
        return {
            "median_min": float(np.median(x[ok])) if ok.any() else None,
            "p90_min": float(np.percentile(x[ok], 90)) if ok.any() else None,
            "max_min": float(np.max(x[ok])) if ok.any() else None,
            # Onsets where the series never completes count as NOT meeting H1.
            "pct_onsets_under_h1": float(100.0 * np.mean(ok & (np.nan_to_num(x, nan=np.inf) < _H1_TARGET_MIN))),
            "n_onsets": int(len(x)), "n_completed": int(ok.sum()),
            "pct_onsets_reached": float(100.0 * ok.mean()) if len(x) else 0.0,
        }

    _taus = pd.to_numeric(onset_df["decay_tau_min"], errors="coerce").dropna()
    onset_robustness_summary = {
        "h1_target_min": _H1_TARGET_MIN, "latency_decay_rate": LATENCY_DECAY_RATE,
        "latency_decay_scale": globals().get("LATENCY_DECAY_SCALE", "fleet"),
        "decay_tau_min_over_onsets": ({"median": float(_taus.median()), "min": float(_taus.min()),
                                       "max": float(_taus.max())} if len(_taus) else None),
        "onset_shift_hours": ONSET_SHIFT_HOURS, "onset_shift_step_min": ONSET_SHIFT_STEP_MIN,
        "horizon_hours": _horizon / 60,
        "series": {f"{name}_{sky}": _stats(f"{name}_{sky}_min")
                   for sky in ("clear", "cloud") for name in ("best_possible",) + POLICIES},
        # Equal-coverage series: time to each common coverage level.
        "coverage_targets_pct": list(COVERAGE_TARGETS_PCT),
        "equal_coverage_series": {f"{name}_{sky}_t{tg:g}": _stats(f"{name}_{sky}_t{tg:g}_min")
                                  for tg in COVERAGE_TARGETS_PCT for sky in ("clear", "cloud")
                                  for name in ("best_possible",) + POLICIES},
    }
    # Revised H1 per policy and sky: share of onsets (with any reachable tile)
    # where the policy reaches _H1R% of reachable tiles within the margin of
    # the floor. An onset where the policy never reaches it counts as a miss.
    def _h1_revised(name, sky):
        n_ok = n_eval = 0
        gaps = []
        for _, r in onset_df.iterrows():
            if not r.get(f"reachable_{sky}_tiles"):
                continue
            fl = r.get(f"best_possible_{sky}_t{_H1R:g}r_min")
            if fl is None or pd.isna(fl):
                continue
            n_eval += 1
            v = r.get(f"{name}_{sky}_t{_H1R:g}r_min")
            if v is not None and not pd.isna(v):
                gaps.append(float(v) - float(fl))
                n_ok += int(float(v) - float(fl) <= H1_FLOOR_MARGIN_MIN + 1e-9)
        lcb = cp_lower_bound(n_ok, n_eval)
        return {"n_onsets_evaluated": n_eval, "n_within_margin": n_ok,
                "pct_within_margin": (100.0 * n_ok / n_eval) if n_eval else None,
                "pct_within_margin_lcb95": None if lcb is None else 100.0 * lcb,
                "evaluable": n_eval >= H1_MIN_EVALUABLE_ONSETS,
                "median_gap_min": float(np.median(gaps)) if gaps else None,
                "p90_gap_min": float(np.percentile(gaps, 90)) if gaps else None,
                "pct_reached": (100.0 * len(gaps) / n_eval) if n_eval else None}

    _evaluability = {"min_evaluable_onsets": H1_MIN_EVALUABLE_ONSETS, "source": H1_EVALUABILITY_SOURCE,
                     "confidence_alpha": H1_CONFIDENCE_ALPHA,
                     "reference": "Clopper, C. J. & Pearson, E. S. (1934). Biometrika 26(4), 404-413, "
                                  "doi:10.1093/biomet/26.4.404",
                     "caveat": "consecutive onsets share most of their passes, so they are not independent "
                               "trials: the lower bound is optimistic and the minimum count necessary, not sufficient"}
    onset_robustness_summary["h1_revised"] = {
        "framing": ("H1a: latency non-inferiority to the physical floor. Pre-specified 2026-10-09 for the "
                    "time-aware selector; from v1.5.0 tested on the earliest-completion cover (ECC), the selector "
                    "H1b also tests, with the time-aware selector reported (series time_aware_*)."),
        "primary_series": "earliest_completion",
        # v1.5.1: one window's onsets share most of their passes, so they are not
        # independent trials; within a run the status and its bound describe this
        # window. The inferential test treats windows as the unit (confirmatory
        # protocol, arf_confirmatory_analysis.py).
        "inference": ("descriptive for this window: consecutive onsets share passes, so the Clopper-Pearson bound, "
                      "which assumes independent trials, overstates the evidence; test across independent windows"),
        "statement": (f"At >= {_H1MIN:g}% of onsets the earliest-completion cover reaches {_H1R:g}% of "
                      f"reachable tiles within {H1_FLOOR_MARGIN_MIN:.1f} min of the physical floor, "
                      f"using at most {MAX_SELECTION_STEPS} images (clear sky)."),
        "reachable_coverage_pct": _H1R, "floor_margin_min": H1_FLOOR_MARGIN_MIN,
        "floor_margin_source": H1_MARGIN_SOURCE, "min_onset_pct": _H1MIN,
        "budget_k": MAX_SELECTION_STEPS, **_evaluability,
        "series": {f"{name}_{sky}": _h1_revised(name, sky)
                   for sky in ("clear", "cloud")
                   for name in ("time_aware", "coverage_first", "fcfs", "earliest_completion")},
    }

    def _num(r, col):
        v = r.get(col)
        return None if v is None or pd.isna(v) else float(v)

    # H1b: image economy of ECC over FCFS at the level, ECC within the margin.
    def _h1b(sky):
        n_eval = n_ok = n_one = 0
        n_e_all, n_f_all, ratios = [], [], []
        for _, r in onset_df.iterrows():
            fl = _num(r, f"best_possible_{sky}_t{_H1R:g}r_min")
            n_f = _num(r, f"fcfs_unbudgeted_{sky}_images{_H1R:g}r")
            if not r.get(f"reachable_{sky}_tiles") or fl is None or n_f is None:
                continue
            if n_f < 2:
                n_one += 1                              # one image: no room for economy
                continue
            n_eval += 1
            t_e = _num(r, f"earliest_completion_{sky}_t{_H1R:g}r_min")
            n_e = _num(r, f"earliest_completion_{sky}_images{_H1R:g}r")
            n_f_all.append(n_f)
            if t_e is not None and n_e is not None:
                n_e_all.append(n_e)
                ratios.append(n_e / n_f)
                n_ok += int(t_e - fl <= H1_FLOOR_MARGIN_MIN + 1e-9 and n_e < n_f)
        lcb = cp_lower_bound(n_ok, n_eval)
        return {"n_onsets_evaluated": n_eval, "n_onsets_one_image": n_one, "n_success": n_ok,
                "pct_success": (100.0 * n_ok / n_eval) if n_eval else None,
                "pct_success_lcb95": None if lcb is None else 100.0 * lcb,
                "evaluable": n_eval >= H1_MIN_EVALUABLE_ONSETS,
                "median_images_ecc": float(np.median(n_e_all)) if n_e_all else None,
                "median_images_fcfs": float(np.median(n_f_all)) if n_f_all else None,
                "median_image_ratio_ecc_to_fcfs": float(np.median(ratios)) if ratios else None,
                "max_images_ecc": float(np.max(n_e_all)) if n_e_all else None}

    onset_robustness_summary["h1b_image_economy"] = {
        "framing": "H1b: image economy (added 2026-10-12, after the 15 runs of that day; confirmatory on fresh data only)",
        # v1.5.1: one window's onsets share most of their passes, so they are not
        # independent trials; within a run the status and its bound describe this
        # window. The inferential test treats windows as the unit (confirmatory
        # protocol, arf_confirmatory_analysis.py).
        "inference": ("descriptive for this window: consecutive onsets share passes, so the Clopper-Pearson bound, "
                      "which assumes independent trials, overstates the evidence; test across independent windows"),

        "statement": (f"At >= {_H1MIN:g}% of onsets where first-come-first-served needs two or more images to "
                      f"reach {_H1R:g}% of reachable tiles, the earliest-completion cover reaches that level within "
                      f"{H1_FLOOR_MARGIN_MIN:.1f} min of the physical floor using fewer images than first-come-"
                      f"first-served, with at most {MAX_SELECTION_STEPS} images (clear sky)."),
        "reachable_coverage_pct": _H1R, "floor_margin_min": H1_FLOOR_MARGIN_MIN, "min_onset_pct": _H1MIN,
        "budget_k": MAX_SELECTION_STEPS, **_evaluability,
        "series": {sky: _h1b(sky) for sky in ("clear", "cloud")},
    }

    # Mosaic quality when each policy first reaches the level (reported, not tested).
    def _quality(sky):
        out = {}
        for name in ("time_aware", "coverage_first", "fcfs_unbudgeted", "earliest_completion"):
            q = pd.to_numeric(onset_df[f"{name}_{sky}_quality{_H1R:g}r"], errors="coerce").dropna()
            out[name] = {"median": float(q.median()) if len(q) else None, "n_onsets": int(len(q))}
        for name in ("time_aware", "earliest_completion"):
            a = pd.to_numeric(onset_df[f"{name}_{sky}_quality{_H1R:g}r"], errors="coerce")
            b = pd.to_numeric(onset_df[f"fcfs_unbudgeted_{sky}_quality{_H1R:g}r"], errors="coerce")
            ok = a.notna() & b.notna() & (b > 0)
            out[f"{name}_vs_fcfs"] = {
                "n_onsets": int(ok.sum()),
                "median_difference": float((a[ok] - b[ok]).median()) if ok.any() else None,
                "median_relative_difference_pct": float((100.0 * (a[ok] - b[ok]) / b[ok]).median()) if ok.any() else None,
                "pct_onsets_higher": float(100.0 * (a[ok] > b[ok] + 1e-12).mean()) if ok.any() else None}
        return out

    onset_robustness_summary["quality_at_level"] = {
        "note": (f"Mean utility of the tiles covered when each policy's delivered mosaic first reaches {_H1R:g}% "
                 f"of reachable tiles; first-come-first-served without a budget. Reported, not tested."),
        "series": {sky: _quality(sky) for sky in ("clear", "cloud")},
    }
    with open(os.path.join(_out_dir, "arf_onset_robustness_summary.json"), "w") as f:
        json.dump(onset_robustness_summary, f, indent=2)

    # Figure 5: empirical CDF, across the evaluated onsets, of the time from the
    # onset to the H1a level (H1_REACHABLE_COVERAGE_PCT of the tiles reachable in
    # that onset's horizon): the selectors against the physical floor, like for
    # like. An onset where a selector never reaches the level leaves its curve
    # below 100%.
    # v1.5.3: until v1.5.2 the level was the highest COVERAGE_TARGETS_PCT of ALL
    # ROI tiles (99%), which no curve reaches when the window's passes cannot
    # (15 October 2026: 65-89% of the ROI reachable), so both panels were empty;
    # hours now on a linear axis over the onset horizon (was symlog), one legend
    # under the panels. The random baseline has no time to the reachable level
    # (Cell 7C records it at the ROI targets only), so it is not drawn here.
    _rc7c = {"font.size": 9, "axes.labelsize": 9, "axes.titlesize": 9, "xtick.labelsize": 8, "ytick.labelsize": 8,
             "legend.fontsize": 7.5, "axes.linewidth": 0.6, "pdf.fonttype": 42, "ps.fonttype": 42}
    from matplotlib.ticker import MaxNLocator as _MaxNLocator
    _sel_styles = {"earliest_completion": ("Earliest-completion cover (H1a)", "#e7298a", "-"),
                   "time_aware": ("Time-aware greedy", "#1b9e77", "-"),
                   "coverage_first": ("Coverage-first greedy", "#d95f02", "--"),
                   "fcfs": ("First-come-first-served (budget k)", "#7570b3", "-.")}

    def _legend_under(fig, axes, ncol, order):
        """One legend for both panels, under them, in the given order of labels
        (a series may appear in one panel only)."""
        hl = {}
        for _ax in axes:
            for _h, _l in zip(*_ax.get_legend_handles_labels()):
                hl.setdefault(_l, _h)
        labs = [l for l in order if l in hl] + [l for l in hl if l not in order]
        if labs:
            fig.legend([hl[l] for l in labs], labs, loc="lower center", bbox_to_anchor=(0.5, 0.0), ncol=ncol,
                       frameon=False)

    _floor_lab = "Physical floor (all passes)"
    _hz_h = float(WINDOW_HOURS - ONSET_SHIFT_HOURS)
    with plt.rc_context(_rc7c):
        fig5, axes5 = plt.subplots(1, 2, figsize=(7.16, 3.35), sharey=True)
        for ax, sky in zip(axes5, ("clear", "cloud")):
            fl = pd.to_numeric(onset_df[f"best_possible_{sky}_t{_H1R:g}r_min"], errors="coerce")
            n_ev = int(fl.notna().sum())
            at_floor, never = [], []
            # The floor as a wide grey band under the selectors (a selector at the
            # floor runs inside it); the H1a selector drawn last, on top.
            for name, (lab, col, ls, lw) in [("best_possible", (_floor_lab, "0.72", "-", 4.0))] + \
                    [(n, (*_sel_styles[n], 1.3)) for n in reversed(list(_sel_styles))]:
                v = pd.to_numeric(onset_df[f"{name}_{sky}_t{_H1R:g}r_min"], errors="coerce")
                x = np.sort(v[fl.notna() & v.notna()].to_numpy(dtype=float)) / 60.0
                if x.size and n_ev:
                    ax.step(np.r_[0.0, x], 100.0 * np.arange(0, x.size + 1) / n_ev, where="post", color=col,
                            ls=ls, lw=lw, label=lab, solid_capstyle="butt")
                    _v, _f = v[fl.notna()], fl[fl.notna()]
                    if name != "best_possible" and _v.notna().all() and bool(np.all(np.abs(_v - _f) <= 1e-9)):
                        at_floor.insert(0, lab.split(" (")[0])
                elif n_ev:
                    never.insert(0, lab.split(" (")[0])
            _notes = (["At the floor at every onset:"] + at_floor if at_floor else []) + \
                     (["Never at the level:"] + never if never else [])
            if _notes:
                ax.text(0.98, 0.04, "\n".join(_notes), transform=ax.transAxes, ha="right", va="bottom", fontsize=7,
                        color="0.25")
            ax.set_xlim(0, _hz_h)
            ax.xaxis.set_major_locator(_MaxNLocator(nbins=7, steps=[1, 2, 3, 4, 6, 10]))
            ax.set_ylim(0, 102)
            ax.set_xlabel(f"Time from onset to {_H1R:g}% of reachable tiles (h)")
            ax.set_title(f"({'a' if sky == 'clear' else 'b'}) {'Clear sky' if sky == 'clear' else 'Cloud / Fail-Signal'}"
                         f" ({n_ev} onsets)", loc="left")
            ax.grid(alpha=0.3, lw=0.5)
        axes5[0].set_ylabel("Share of onsets (%)")
        fig5.tight_layout(rect=(0, 0.15, 1, 1))
        _legend_under(fig5, axes5, ncol=3, order=[_floor_lab] + [s[0] for s in _sel_styles.values()])
        for fmt in ("png", "pdf"):
            fig5.savefig(os.path.join(_out_dir, f"arf_fig5_onset_robustness.{fmt}"), dpi=300, bbox_inches="tight")
        plt.show()

        # Figure 9 (v1.5.2): the H1a quantity itself -- per onset, the time to
        # H1_REACHABLE_COVERAGE_PCT of the reachable tiles minus the physical
        # floor's time to it, against the H1a margin. Empirical CDF over the
        # evaluated onsets; an onset where a selector never reaches the level
        # leaves its curve below 100%.
        # v1.5.3: minutes on a linear axis that ends a quarter past the margin
        # (was hours on a symlog axis, where gaps of a few minutes against a
        # 94-min margin lay on the y-axis); the axis starts just left of 0 so a
        # selector at the floor (gap 0) shows off the spine; the largest gap of
        # each selector (or how many onsets fall beyond the axis or never reach
        # the level) is written on the panel; one legend under the panels.
        _xmax9 = max(1.25 * H1_FLOOR_MARGIN_MIN, 10.0)
        _margin_lab = f"H1a margin ({H1_FLOOR_MARGIN_MIN:.0f} min)"
        fig9, axes9 = plt.subplots(1, 2, figsize=(7.16, 3.15), sharey=True)
        for ax, sky in zip(axes9, ("clear", "cloud")):
            fl = pd.to_numeric(onset_df[f"best_possible_{sky}_t{_H1R:g}r_min"], errors="coerce")
            n_ev = int(fl.notna().sum())
            notes = []
            for name in reversed(list(_sel_styles)):               # the H1a selector last, on top
                lab, col, ls = _sel_styles[name]
                v = pd.to_numeric(onset_df[f"{name}_{sky}_t{_H1R:g}r_min"], errors="coerce")
                gap = np.sort(((v - fl)[fl.notna() & v.notna()]).to_numpy(dtype=float))
                if gap.size and n_ev:
                    ax.step(np.r_[0.0, gap], 100.0 * np.arange(0, gap.size + 1) / n_ev, where="post", color=col,
                            ls=ls, lw=1.3, label=lab)
                n_out = n_ev - int((gap <= _xmax9).sum())
                if n_ev:
                    notes.insert(0, f"{lab.split(' (')[0]}: " + (f"largest {round(float(gap.max()), 1):g} min"
                                                                 if not n_out else
                                                                 f"{n_out} beyond {_xmax9:.0f} min or never"))
            ax.axvline(H1_FLOOR_MARGIN_MIN, color="k", ls=":", lw=1.0, label=_margin_lab)
            if notes:
                ax.text(0.97 * H1_FLOOR_MARGIN_MIN, 0.04, "Gap to the floor, per selector\n" + "\n".join(notes),
                        transform=ax.get_xaxis_transform(), ha="right", va="bottom", fontsize=7, color="0.25")
            ax.set_xlim(-0.03 * _xmax9, _xmax9)
            ax.set_ylim(0, 102)
            ax.set_xlabel(f"Time to {_H1R:g}% of reachable tiles minus the floor (min)")
            ax.set_title(f"({'a' if sky == 'clear' else 'b'}) {'Clear sky' if sky == 'clear' else 'Cloud'}"
                         f" ({n_ev} onsets)", loc="left")
            ax.grid(alpha=0.3, lw=0.5)
        axes9[0].set_ylabel("Share of onsets (%)")
        fig9.tight_layout(rect=(0, 0.15, 1, 1))
        _legend_under(fig9, axes9, ncol=3, order=[s[0] for s in _sel_styles.values()] + [_margin_lab])
        for fmt in ("png", "pdf"):
            fig9.savefig(os.path.join(_out_dir, f"arf_fig9_h1a_gap.{fmt}"), dpi=300, bbox_inches="tight")
        plt.show()
        plt.close(fig5)
        plt.close(fig9)
else:
    logger.info("Onset robustness analysis disabled (onset_robustness_enabled=False in Cell 0).")

# =============================================================================
# 3. OPTIMALITY GAP (exact ILP on small sub-problems, clear sky)
# =============================================================================
ilp_rows, ilp_summary = [], {}
_ILP_MIN_HORIZON_MIN = 30
if ILP_ENABLED:
    _span = max(0, WINDOW_HOURS - ILP_HORIZON_HOURS) * 60
    for d in sorted(set(int(x) for x in np.linspace(0, _span, ILP_INSTANCES))):
        # Large ROIs: halve the window until the instance fits ILP_MAX_PAIRS
        # (down to _ILP_MIN_HORIZON_MIN), so the gap stays measurable; the
        # window actually used is recorded per instance.
        h_min = ILP_HORIZON_HOURS * 60
        sub, _ = subset_by_time(event_data, None, d, d + h_min)
        while int((sub.util > 0.0).sum()) > ILP_MAX_PAIRS and h_min > _ILP_MIN_HORIZON_MIN:
            h_min = max(_ILP_MIN_HORIZON_MIN, h_min // 2)
            sub, _ = subset_by_time(event_data, None, d, d + h_min)
        if len(sub) == 0:
            continue
        res = solve_ilp_max_utility(sub, None, ILP_BUDGET_K, ILP_TIME_LIMIT_S, ILP_MAX_PAIRS)
        row = {"shift_min": d, "horizon_min": h_min, "n_events": len(sub), "n_pairs": res["n_pairs"],
               "ilp_status": res["status"],
               "ilp_seconds": round(res["seconds"], 2), "ilp_objective": res["objective"], "ilp_bound": res["bound"],
               "ilp_latency_min": compute_completion_latency(res["selected"], sub) if res["selected"] else None}
        _sel_ta = []
        for name, rate in (("coverage_first", 0.0), ("time_aware", LATENCY_DECAY_RATE)):
            _sel, cov, _, _ = run_submodular_greedy_selection(sub, ILP_BUDGET_K, gap_fill=False,
                                                              time_aware=rate > 0, latency_decay_rate=rate)
            row[f"{name}_objective"] = float(sum(cov.values()))
            if rate > 0:
                _sel_ta = _sel
        row["fcfs_objective"] = float(select_fcfs(sub, None, ILP_BUDGET_K)[1].sum())
        # v1.5.0: the time-aware selector against the exact optimum of ITS OWN
        # objective, the time-weighted coverage (ILP with u replaced by w * u).
        _w = time_aware_weights(sub, LATENCY_DECAY_RATE)
        _rw = solve_ilp_max_utility(sub, None, ILP_BUDGET_K, ILP_TIME_LIMIT_S, ILP_MAX_PAIRS, weight=_w)
        row.update({"tw_ilp_status": _rw["status"], "tw_ilp_objective": _rw["objective"],
                    "time_aware_tw_objective": facility_value(sub, None, _sel_ta, _w)})
        row["time_aware_tw_ratio"] = (row["time_aware_tw_objective"] / _rw["objective"]
                                      if _rw["status"] == "optimal" and _rw["objective"] else None)
        # Ratios only against a PROVEN optimum. A time-limited solve's best-found
        # solution can be arbitrarily poor and its dual bound loose, so such
        # instances are recorded but excluded from the gap statistics.
        _optimal = res["status"] == "optimal" and res["objective"]
        for name in ("coverage_first", "time_aware", "fcfs"):
            row[f"{name}_ratio"] = row[f"{name}_objective"] / res["objective"] if _optimal else None
        # ECC's greedy cover step vs the exact minimum number of images, for the
        # revised-H1 coverage level of this window's reachable tiles.
        _lvl = float(globals().get("H1_REACHABLE_COVERAGE_PCT", 95.0))
        _R = reachable_tile_count(sub, None)
        _need = int(np.ceil(_lvl / 100.0 * _R - 1e-9))
        _tg, _, _ng = earliest_completion(sub, None, len(sub), _lvl, denominator=_R) if _R else (None, [], 0)
        _ex = solve_ilp_min_images(sub, None, _need, ILP_TIME_LIMIT_S, ILP_MAX_PAIRS) if _R else {"status": "empty", "images": None}
        row.update({"ecc_target_pct": _lvl, "ecc_greedy_images": _ng if _tg is not None else None,
                    "ecc_exact_images": _ex["images"], "ecc_exact_status": _ex["status"]})
        ilp_rows.append(row)
    ilp_df = pd.DataFrame(ilp_rows)
    ilp_df.to_csv(os.path.join(_out_dir, "arf_ilp_gap.csv"), index=False)
    _solved = ilp_df[ilp_df["ilp_status"] == "optimal"] if len(ilp_df) else ilp_df
    ilp_summary = {
        "instances": int(len(ilp_df)), "solved": int(len(_solved)),
        "optimal": int((ilp_df["ilp_status"] == "optimal").sum()) if len(ilp_df) else 0,
        "time_limited": int(ilp_df["ilp_status"].astype(str).str.startswith("time limit").sum()) if len(ilp_df) else 0,
        "horizon_hours": ILP_HORIZON_HOURS, "budget_k": ILP_BUDGET_K,
        "horizons_used_min": sorted(int(h) for h in ilp_df["horizon_min"].unique()) if len(ilp_df) else [],
        **{f"{n}_ratio_{s}": (float(getattr(_solved[f"{n}_ratio"], s)()) if len(_solved) else None)
           for n in ("coverage_first", "time_aware", "fcfs") for s in ("median", "min")},
        # v1.5.0: time-aware greedy / exact optimum of the time-weighted coverage
        "time_aware_objective": globals().get("TIME_AWARE_OBJECTIVE", "facility"),
        "tw_solved": int((ilp_df["tw_ilp_status"] == "optimal").sum()) if len(ilp_df) else 0,
        **{f"time_aware_tw_ratio_{s}": (float(getattr(ilp_df["time_aware_tw_ratio"].dropna(), s)())
                                        if len(ilp_df) and ilp_df["time_aware_tw_ratio"].notna().any() else None)
           for s in ("median", "min")},
        "guarantee_note": ("facility objective: the k-step greedy attains >= 1 - 1/e of the optimum of the time-"
                           "weighted coverage (Nemhauser, Wolsey & Fisher 1978); the ratio above is the measured value"
                           if globals().get("TIME_AWARE_OBJECTIVE", "facility") == "facility" else
                           "legacy objective: no guarantee; the ratio is measured against the time-weighted coverage "
                           "optimum for comparison only"),
        **(lambda _e: {"ecc_instances_exact": int(len(_e)),
                       "ecc_matches_exact": int((_e["ecc_greedy_images"] == _e["ecc_exact_images"]).sum()),
                       "ecc_max_extra_images": (int((_e["ecc_greedy_images"] - _e["ecc_exact_images"]).max())
                                                if len(_e) else None)})(
            ilp_df[(ilp_df.get("ecc_exact_status") == "optimal") & ilp_df["ecc_greedy_images"].notna()]
            if len(ilp_df) and "ecc_exact_status" in ilp_df else pd.DataFrame(columns=["ecc_greedy_images", "ecc_exact_images"])),
    }

# =============================================================================
# 4. CLOUD ROBUSTNESS OVER CLOUD REALISATIONS (H2, v1.5.0)
# =============================================================================
# One cloud realisation is one random draw, so a comparison of policies under
# cloud needs many. For CLOUD_SEEDS realisations (seeds RANDOM_SEED,
# RANDOM_SEED + 1, ...; the model of Cell 0), full window, for each greedy
# objective (time-aware -- the primary -- and coverage-first), four plans with
# the SAME number of images n:
#   adaptive          the Fail-Signal selector as configured (select_passes,
#                     CLOUD_PLANNING, gap-fill on): learns each image's cloud
#                     when it arrives and re-plans; n = the images it uses
#   open-loop (exp.)  n images chosen in advance by the SAME planner without
#                     any cloud observation (plan_expected_coverage, expected
#                     coverage from the tiles' cloud probabilities, time-
#                     weighted for the time-aware objective)
#   open-loop (clear) n images chosen in advance by the clear-sky objective
#                     (no cloud information at all)
#   oracle            n images chosen by the greedy knowing every image's cloud
#                     (clairvoyant reference; greedy, so not a strict upper bound)
# and, for the record, the v1.4.0 Fail-Signal (clear-sky planning, re-plan on
# failure) with its own image count and an expected-coverage open-loop plan of
# that size.
# H2 (v1.5.0): the mean paired difference in coverage, adaptive minus
# open-loop (expected), has a one-sided 95% lower confidence bound above zero
# (Student t over the realisations) -- the value of acting on what the images
# show, the planner held fixed. NOT EVALUABLE when the selector never saw a
# failure or compensated in any realisation, when even the oracle never
# beats the open-loop plan (no room for adaptation), or under
# CLOUD_KNOWLEDGE="oracle". Each realisation also records the floor, ECC and
# FCFS under cloud at the revised-H1 level (H1a/H1b under cloud, secondary).
cloud_seed_rows, cloud_robustness_summary = [], {}
cloud_sensitivity_rows = []
_cs_n = int(globals().get("CLOUD_SEEDS", 20))
_ptile = globals().get("CLOUD_P_TILE")
if _ptile is None:
    _ptile = np.full(len(_tile_ids), float(CLOUD_PROBABILITY))
_OBJECTIVES = (("time_aware", float(LATENCY_DECAY_RATE)), ("coverage_first", 0.0))
_PLANS = ("adaptive", "openloop_expected", "openloop_clear", "oracle")
_PLAN_LABELS = {"adaptive": "Fail-Signal (adaptive)", "openloop_expected": "Open-loop (expected coverage)",
                "openloop_clear": "Open-loop (clear-sky plan)", "oracle": "Greedy with the cloud known"}


def _cloud_trial(m, objectives=None, with_h1=True):
    """One cloud realisation: the four plans for each objective (default
    _OBJECTIVES), plus the floor, ECC and FCFS at the revised-H1 level of the
    tiles reachable under that cloud."""
    objectives = _OBJECTIVES if objectives is None else objectives
    row = {"cloudy_rows_pct": 100.0 * m.n_cloudy / max(1, len(m))}
    reach = reachable_tile_count(event_data, m)
    row["reachable_tiles"] = reach
    for obj, rate in objectives:
        ta = rate > 0
        w = time_aware_weights(event_data, rate) if ta else None
        sel_a, _, _, _ = select_passes(event_data, MAX_SELECTION_STEPS, cloud_mask=m, gap_fill=True,
                                       time_aware=ta, latency_decay_rate=rate)
        stats = dict(last_adaptive_run)
        n = len(sel_a)
        plans = {"adaptive": sel_a,
                 "openloop_expected": plan_expected_coverage(event_data, n, _ptile, w),
                 "openloop_clear": run_submodular_greedy_selection(event_data, n, cloud_mask=None, gap_fill=False,
                                                                   time_aware=ta, latency_decay_rate=rate)[0],
                 "oracle": run_submodular_greedy_selection(event_data, n, cloud_mask=m, gap_fill=False,
                                                           time_aware=ta, latency_decay_rate=rate)[0]}
        if globals().get("CLOUD_KNOWLEDGE", "realistic") != "oracle":
            sel_f = simulate_adaptive_selection(event_data, MAX_SELECTION_STEPS, m, gap_fill=True, time_aware=ta,
                                                latency_decay_rate=rate, planning="clear")[0]
            plans["failsignal_v140"] = sel_f
            plans["openloop_expected_v140n"] = plan_expected_coverage(event_data, len(sel_f), _ptile, w)
        for tag, sel in plans.items():
            c = realized_coverage(event_data, m, sel)
            row[f"{obj}_{tag}_images"] = len(sel)
            row[f"{obj}_{tag}_cov_pct"] = 100.0 * c / total_target_tiles
            row[f"{obj}_{tag}_reach_pct"] = (100.0 * c / reach) if reach else None
        row[f"{obj}_replans"] = int(stats.get("replans", 0))
        row[f"{obj}_compensatory"] = int(stats.get("compensatory", 0))
    if with_h1:
        need = int(np.ceil(_H1R / 100.0 * reach - 1e-9)) if reach else 0
        fl = coverage_times(event_data, m, (_H1R,), denominator=reach)[_H1R] if reach else None
        te, _, ne = earliest_completion(event_data, m, MAX_SELECTION_STEPS, _H1R, denominator=reach) if reach else (None, [], 0)
        tf, nf, _ = (level_stats(event_data, m, select_fcfs(event_data, m, len(event_data))[0], need) if reach
                     else (None, None, None))
        row.update({f"floor_t{_H1R:g}r_min": fl, f"ecc_t{_H1R:g}r_min": te,
                    f"ecc_images{_H1R:g}r": ne if te is not None else None,
                    f"fcfs_t{_H1R:g}r_min": tf, f"fcfs_images{_H1R:g}r": nf})
    return row


def _numcol(df, c):
    return (pd.to_numeric(df[c], errors="coerce").to_numpy(dtype=float) if c in df.columns
            else np.full(len(df), np.nan))


def _h2_summary(df, obj):
    A, X, C, O = (_numcol(df, f"{obj}_{t}_cov_pct") for t in _PLANS)
    adapted = int(np.nansum(_numcol(df, f"{obj}_replans")) + np.nansum(_numcol(df, f"{obj}_compensatory")))
    room = bool(np.any(O > X + 1e-9))
    d_x, d_c = paired_summary(A - X), paired_summary(A - C)
    captured = [(a - x) / (o - x) for a, x, o in zip(A, X, O) if np.isfinite(o - x) and o - x > 1e-9]
    reasons = []
    if globals().get("CLOUD_KNOWLEDGE", "realistic") == "oracle":
        reasons.append("cloud knowledge is 'oracle', so the selector does not adapt")
    if adapted == 0:
        reasons.append("the selector never re-planned or compensated in any realisation (nothing to adapt)")
    if not room:
        reasons.append("even the greedy that knows the cloud never beats the open-loop plan with the same images "
                       "(no room to adapt)")
    evaluable = not reasons and d_x["n"] >= 2
    status = ("NOT EVALUABLE" if not evaluable else
              ("SUPPORTED" if d_x["lcb"] is not None and d_x["lcb"] > 0 else "NOT SUPPORTED"))
    F, XF = _numcol(df, f"{obj}_failsignal_v140_cov_pct"), _numcol(df, f"{obj}_openloop_expected_v140n_cov_pct")
    return {"status": status, "evaluable": evaluable, "not_evaluable_reasons": reasons,
            "n_realisations": int(len(A)),
            "planning": globals().get("CLOUD_PLANNING", "clear"),
            "coverage_pct_mean": {t: (float(np.nanmean(v)) if np.isfinite(v).any() else None)
                                  for t, v in zip(_PLANS, (A, X, C, O))},
            "images_mean": float(np.nanmean(_numcol(df, f"{obj}_adaptive_images"))) if len(df) else None,
            "difference_vs_openloop_expected": d_x, "difference_vs_openloop_clear": d_c,
            "v140_failsignal": ({"coverage_pct_mean": float(np.nanmean(F)),
                                 "images_mean": float(np.nanmean(_numcol(df, f"{obj}_failsignal_v140_images"))),
                                 "difference_vs_openloop_expected_same_images": paired_summary(F - XF)}
                                if np.isfinite(F).any() else None),
            "share_of_oracle_gain_captured_median": float(np.median(captured)) if captured else None,
            "replans_total": int(np.nansum(_numcol(df, f"{obj}_replans"))),
            "compensatory_total": int(np.nansum(_numcol(df, f"{obj}_compensatory")))}


def _h1_under_cloud(df):
    fl, te = _numcol(df, f"floor_t{_H1R:g}r_min"), _numcol(df, f"ecc_t{_H1R:g}r_min")
    ne, nf = _numcol(df, f"ecc_images{_H1R:g}r"), _numcol(df, f"fcfs_images{_H1R:g}r")
    ok = np.isfinite(fl)
    within = ok & np.isfinite(te) & (te - fl <= H1_FLOOR_MARGIN_MIN + 1e-9)
    elig = ok & np.isfinite(nf) & (nf >= 2)
    econ = elig & within & np.isfinite(ne) & (ne < nf)
    return {"n_realisations": int(ok.sum()),
            "pct_ecc_within_margin": float(100.0 * within.sum() / ok.sum()) if ok.sum() else None,
            "n_realisations_fcfs_two_plus_images": int(elig.sum()),
            "pct_ecc_fewer_images_within_margin": float(100.0 * econ.sum() / elig.sum()) if elig.sum() else None,
            "median_images_ecc": float(np.median(ne[elig & np.isfinite(ne)])) if (elig & np.isfinite(ne)).any() else None,
            "median_images_fcfs": float(np.median(nf[elig])) if elig.any() else None}


if _cs_n >= 2 and len(event_data):
    logger.info(f"Cloud robustness: {_cs_n} cloud realisations ({globals().get('CLOUD_MODEL', 'iid')} model) ...")
    for _i in range(_cs_n):
        _m = build_cloud_mask(event_data, CLOUD_PROBABILITY, RANDOM_SEED + _i)
        cloud_seed_rows.append({"seed": RANDOM_SEED + _i, **_cloud_trial(_m)})
    cloud_seed_df = pd.DataFrame(cloud_seed_rows)
    cloud_seed_df.to_csv(os.path.join(_out_dir, "arf_cloud_robustness.csv"), index=False)
    cloud_robustness_summary = {
        "design": ("Per cloud realisation, four plans with the same number of images: the Fail-Signal adaptive "
                   "selector; the same expected-coverage planner run once in advance with no cloud observation "
                   "(open-loop); an open-loop plan on clear-sky geometry; and the greedy given the cloud in advance. Also "
                   "the v1.4.0 Fail-Signal (clear-sky planning). Coverage = tiles with a usable cloud-free "
                   "observation, % of ROI tiles."),
        "statement": ("H2 (v1.5.0): with the same number of images, the Fail-Signal adaptive selector covers more of "
                      "the region under cloud than the same planner run in advance without seeing any image (open-"
                      "loop, maximising expected coverage from the tiles' cloud probabilities): the mean paired "
                      "difference over cloud realisations has a one-sided 95% lower confidence bound above zero."),
        "cloud_planning": globals().get("CLOUD_PLANNING", "clear"),
        "primary_objective": "time_aware",
        "cloud_model": globals().get("cloud_model_info", {"model": globals().get("CLOUD_MODEL", "iid")}),
        "seeds": [int(RANDOM_SEED), int(RANDOM_SEED + _cs_n - 1)],
        "h2": {obj: _h2_summary(cloud_seed_df, obj) for obj, _ in _OBJECTIVES},
        "h1_under_cloud": _h1_under_cloud(cloud_seed_df),
    }

    # =========================================================================
    # 5. CLOUD SENSITIVITY: the correlation length and time (one at a time)
    # =========================================================================
    _combos = [(float(L), None if float(L) == 0.0 else float(CLOUD_CORR_TIME_H))
               for L in globals().get("CLOUD_SWEEP_LENGTHS_KM", ())]
    _combos += [(float(CLOUD_CORR_LENGTH_KM), float(T)) for T in globals().get("CLOUD_SWEEP_TIMES_H", ())
                if float(T) != float(CLOUD_CORR_TIME_H)]
    _combos = list(dict.fromkeys(_combos))
    _sw_n = int(globals().get("CLOUD_SWEEP_SEEDS", 10))
    if _combos and _sw_n >= 2:
        logger.info(f"Cloud sensitivity: {len(_combos)} settings x {_sw_n} realisations ...")
        for L, T in _combos:
            _rows = []
            for _i in range(_sw_n):
                _m = (build_cloud_mask(event_data, CLOUD_PROBABILITY, RANDOM_SEED + _i, model="iid") if L == 0.0 else
                      build_cloud_mask(event_data, CLOUD_PROBABILITY, RANDOM_SEED + _i, model="correlated",
                                       length_km=L, time_h=T))
                _rows.append(_cloud_trial(_m, objectives=_OBJECTIVES[:1], with_h1=False))
            _d = pd.DataFrame(_rows)
            _A, _X, _C, _O = (_numcol(_d, f"time_aware_{t}_cov_pct") for t in _PLANS)
            _ps = paired_summary(_A - _X)
            cloud_sensitivity_rows.append({
                "model": "iid" if L == 0.0 else "correlated", "length_km": L, "time_h": T,
                "primary": (globals().get("CLOUD_MODEL") == "correlated" and L == float(CLOUD_CORR_LENGTH_KM)
                            and T == float(CLOUD_CORR_TIME_H)),
                "n_realisations": int(len(_d)), "images_mean": float(np.nanmean(_numcol(_d, "time_aware_adaptive_images"))),
                "adaptive_cov_pct": float(np.nanmean(_A)), "openloop_expected_cov_pct": float(np.nanmean(_X)),
                "openloop_clear_cov_pct": float(np.nanmean(_C)), "oracle_cov_pct": float(np.nanmean(_O)),
                "diff_mean": _ps["mean"], "diff_lcb95": _ps["lcb"], "diff_pct_positive": _ps["pct_positive"],
                "replans_total": int(np.nansum(_numcol(_d, "time_aware_replans"))),
                "compensatory_total": int(np.nansum(_numcol(_d, "time_aware_compensatory")))})
        pd.DataFrame(cloud_sensitivity_rows).to_csv(os.path.join(_out_dir, "arf_cloud_sensitivity.csv"), index=False)
        cloud_robustness_summary["sensitivity"] = cloud_sensitivity_rows
    with open(os.path.join(_out_dir, "arf_cloud_robustness_summary.json"), "w") as f:
        json.dump(cloud_robustness_summary, f, indent=2, default=lambda o: o.item() if hasattr(o, "item") else str(o))

    # Figure 8 (IEEE double-column width): (a) coverage of the four plans with
    # the same images over the realisations; (b), (c) the H2 difference against
    # the correlation length and time (mean, with its one-sided 95% lower bound).
    _cols8 = {"adaptive": "#2a78d6", "openloop_expected": "#eb6834", "openloop_clear": "#1baf7a",
              "oracle": "#8a8986"}
    _n_pan = 3 if cloud_sensitivity_rows else 1
    with plt.rc_context({"font.size": 8, "axes.titlesize": 8, "axes.labelsize": 8, "xtick.labelsize": 7,
                         "ytick.labelsize": 7, "legend.fontsize": 7, "axes.spines.top": False,
                         "axes.spines.right": False}):
        # v1.5.3: 7.0 in (was 7.16): with the tight bounding box the saved figure grew to 7.18 in when
        # panels (b) and (c) carry longer tick labels (open data alone, 15 October 2026)
        fig8, axs8 = plt.subplots(1, _n_pan, figsize=(7.0, 2.6),
                                  gridspec_kw={"width_ratios": [1.4, 1, 1][:_n_pan]})
        axs8 = np.atleast_1d(axs8)
        ax = axs8[0]
        _rng8 = np.random.default_rng(0)
        for j, tag in enumerate(_PLANS):
            v = _numcol(cloud_seed_df, f"time_aware_{tag}_cov_pct")
            v = v[np.isfinite(v)]
            if not v.size:
                continue
            ax.boxplot([v], positions=[j], widths=0.5, showfliers=False, medianprops={"color": "#0b0b0b", "lw": 1.2},
                       boxprops={"color": "#52514e", "lw": 0.8}, whiskerprops={"color": "#52514e", "lw": 0.8},
                       capprops={"color": "#52514e", "lw": 0.8})
            ax.scatter(j + _rng8.uniform(-0.13, 0.13, v.size), v, s=9, color=_cols8[tag], alpha=0.85, lw=0, zorder=3)
        ax.set_xticks(range(len(_PLANS)))
        ax.set_xticklabels(["Fail-Signal\n(adaptive)", "Open-loop\n(expected)", "Open-loop\n(clear sky)",
                            "Greedy,\ncloud known"])
        ax.set_ylabel("Coverage under cloud (% of tiles)")
        _h2p = cloud_robustness_summary["h2"]["time_aware"]
        ax.set_title(f"(a) Same images, {_cs_n} cloud realisations", loc="left")
        ax.grid(axis="y", alpha=0.3, lw=0.5)
        if _n_pan == 3:
            _sens = pd.DataFrame(cloud_sensitivity_rows)
            for ax, (key, sub_df, xlab) in zip(axs8[1:], (
                    ("length_km", _sens[(_sens.model == "iid") | (_sens.time_h == float(CLOUD_CORR_TIME_H))],
                     f"Correlation length (km), time {CLOUD_CORR_TIME_H:g} h"),
                    ("time_h", _sens[(_sens.model == "correlated") & (_sens.length_km == float(CLOUD_CORR_LENGTH_KM))],
                     f"Correlation time (h), length {CLOUD_CORR_LENGTH_KM:g} km"))):
                sub_df = sub_df.sort_values(key, na_position="first")
                labels = ["iid" if (key == "length_km" and r.model == "iid") else f"{r[key]:g}" for _, r in sub_df.iterrows()]
                xs = np.arange(len(sub_df))
                mean = sub_df["diff_mean"].to_numpy(dtype=float)
                lcb = sub_df["diff_lcb95"].to_numpy(dtype=float)
                ax.axhline(0.0, color="#52514e", lw=0.8)
                ax.vlines(xs, lcb, mean, color="#2a78d6", lw=1.6)
                ax.plot(xs, mean, "o", color="#2a78d6", ms=4)
                prim = sub_df["primary"].to_numpy(dtype=bool)
                if prim.any():
                    ax.plot(xs[prim], mean[prim], "o", mfc="none", mec="#0b0b0b", ms=7, mew=1.0)
                ax.set_xticks(xs)
                ax.set_xticklabels(labels)
                ax.set_xlabel(xlab)
                ax.grid(axis="y", alpha=0.3, lw=0.5)
            axs8[1].set_ylabel("Adaptive minus open-loop\n(coverage points)")
            axs8[1].set_title("(b) Sensitivity to cloud size", loc="left")
            axs8[2].set_title("(c) Sensitivity to cloud persistence", loc="left")
        fig8.tight_layout()
        for fmt in ("png", "pdf"):
            fig8.savefig(os.path.join(_out_dir, f"arf_fig8_cloud_robustness.{fmt}"), dpi=300, bbox_inches="tight")
        plt.show()

if 'scenario_results' in globals():
    scenario_results.setdefault(_scenario_key(), {})
    scenario_results[_scenario_key()]["decay_sweep"] = decay_sweep_rows
    scenario_results[_scenario_key()]["onset_robustness"] = onset_robustness_summary
    scenario_results[_scenario_key()]["policy_comparison"] = policy_rows
    scenario_results[_scenario_key()]["ilp_gap"] = ilp_summary
    scenario_results[_scenario_key()]["cost_latency_frontier"] = frontier_rows
    scenario_results[_scenario_key()]["cloud_robustness"] = cloud_robustness_summary
    scenario_results[_scenario_key()]["cloud_model"] = globals().get("cloud_model_info")

print("\n" + "=" * 70)
print("  CELL 7C: Objective Comparison & Onset Robustness (H1)")
print("=" * 70)
print(decay_sweep_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
print("\n  Policy comparison (full window, budget k, no gap-fill):")
print(policy_df.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
if ilp_summary:
    print(f"\n  Optimality gap ({ilp_summary['solved']}/{ilp_summary['instances']} instances solved, "
          f"{ilp_summary['optimal']} to proven optimality; {ILP_HORIZON_HOURS} h of passes, k={ILP_BUDGET_K}, clear sky):")
    for n in ("coverage_first", "time_aware", "fcfs"):
        mi, md = ilp_summary[f"{n}_ratio_min"], ilp_summary[f"{n}_ratio_median"]
        print(f"    {n:16s} utility / optimum: median {md:.4f}, worst {mi:.4f}" if md is not None else f"    {n}: n/a")
if onset_robustness_summary:
    print(f"\n  {len(onset_df)} onsets | superseded H1 target {_H1_TARGET_MIN:g} min (kept for the record)")
    print(f"  {'series':26s} {'median':>8s} {'p90':>8s} {'max':>8s} {'% <10min':>8s}")
    for k, s in onset_robustness_summary["series"].items():
        f = lambda v: f"{v:8.0f}" if v is not None else "     n/a"
        print(f"  {k:26s} {f(s['median_min'])} {f(s['p90_min'])} {f(s['max_min'])} {s['pct_onsets_under_h1']:7.1f}%")
    _a = onset_robustness_summary["h1_revised"]["series"]["earliest_completion_clear"]
    _at = onset_robustness_summary["h1_revised"]["series"]["time_aware_clear"]
    _b = onset_robustness_summary["h1b_image_economy"]["series"]["clear"]
    _p = lambda v: "n/a" if v is None else f"{v:.1f}%"
    print(f"\n  H1a (clear): ECC within {H1_FLOOR_MARGIN_MIN:.1f} min of the floor at {_a['n_within_margin']}/"
          f"{_a['n_onsets_evaluated']} onsets ({_p(_a['pct_within_margin'])}; 95% lower bound "
          f"{_p(_a['pct_within_margin_lcb95'])}; evaluable from {H1_MIN_EVALUABLE_ONSETS} onsets); "
          f"time-aware {_p(_at['pct_within_margin'])}")
    print(f"  H1b (clear): ECC fewer images than FCFS, within the margin, at {_b['n_success']}/{_b['n_onsets_evaluated']} "
          f"onsets ({_p(_b['pct_success'])}); median images ECC {_b['median_images_ecc']} vs FCFS "
          f"{_b['median_images_fcfs']}; {_b['n_onsets_one_image']} onsets need one image")
if cloud_robustness_summary:
    for _obj in ("time_aware", "coverage_first"):
        _h = cloud_robustness_summary["h2"][_obj]
        _dx = _h["difference_vs_openloop_expected"]
        _f = lambda v, n=2: "n/a" if v is None else f"{v:.{n}f}"
        print(f"  H2 ({_obj}{', primary' if _obj == 'time_aware' else ''}): {_h['status']} -- adaptive minus "
              f"open-loop (expected), same images: mean {_f(_dx['mean'])} points, one-sided 95% lower bound "
              f"{_f(_dx['lcb'])} over {_dx['n']} realisations; mean images {_f(_h['images_mean'], 1)}")
print("=" * 70)
print("\n=== CELL 7C COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 7D: VALIDATION AGAINST REAL ACQUISITIONS (Sentinel-2 and Landsat archives)
# =============================================================================
# Do the passes ARF predicts correspond to images the satellites actually took?
# For every open-data satellite in the fleet (Sentinel-2, Landsat 8/9), the
# predicted overpasses of the ROI are matched against the scenes in the public
# Earth Search STAC catalogue (Element 84; https://earth-search.aws.element84.com/v1)
# for the same window: same platform, overpass times within
# VALIDATION_MATCH_TOL_MIN. Reported: precision (predicted overpasses that were
# really acquired), recall (real acquisitions that were predicted), how much of
# each matched pass's predicted ROI coverage lies inside the real scenes, and
# the real scene cloud cover (eo:cloud_cover, %) as an observed counterpart of
# the cloud model. Item fields used (checked on live items, 2026-10-09):
# properties.datetime, properties.platform ("sentinel-2b", "landsat-9"),
# properties["eo:cloud_cover"], and the item geometry.
#
# Only the part of the window older than each platform's archive lag is
# evaluated (scenes take time to reach the archive; v1.5.2: per platform,
# VALIDATION_ARCHIVE_LAG_H_BY_PLATFORM, with precision and recall reported per
# platform as well as pooled). PlanetScope has no open archive, so its passes
# are not validated here. Needs internet access; if the catalogue cannot be
# reached the cell records "skipped" and the run goes on.
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 7D is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["7D"] = _ARF_CELL_VERSION

import re
import json
import urllib.request
from datetime import datetime, timedelta, timezone
import numpy as np
import pandas as pd
import h3

logger = logging.getLogger("arf.cell7d")
_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()

VALIDATION_STAC_URL = "https://earth-search.aws.element84.com/v1"
# Both Sentinel-2 L2A collections are queried; the same acquisition appearing
# in both is counted once (grouped into one overpass by platform and time).
VALIDATION_COLLECTIONS = {"Sentinel-2": ["sentinel-2-c1-l2a", "sentinel-2-l2a"], "Landsat 8/9": ["landsat-c2-l2"]}
VALIDATION_MATCH_TOL_MIN = 15.0      # same overpass if times differ by at most this
VALIDATION_GROUP_GAP_MIN = 10.0      # scenes/minutes closer than this belong to one overpass
VALIDATION_ARCHIVE_LAG_H = 48.0      # ignore the most recent part of the window (Sentinel-2; default)
# v1.5.2: per platform. The Landsat collection queried is Collection 2 Level-2.
# USGS (https://www.usgs.gov/landsat-missions/landsat-collection-2-level-2-
# science-products, accessed 2026-10-14): Level-2 products "are typically
# available within 24-72 hours" after a scene is processed into Tier 1 or 2;
# "For Landsat 8 OLI/TIRS, it takes approximately 14-16 days to process to a
# Tier 1 or Tier 2 product"; Landsat 9 takes "approximately 3 days to process
# to a Level-2 product". (USGS's Level-1 page gives 3 to 10 days for Landsat 8
# tiering and its generation timeline 4 to 11 days for Level-2; the longest
# statement is used, so a pass is never counted as missed while it may still be
# in processing.) With the single 48 h lag, recent Landsat passes that were
# acquired but not yet published counted as false predictions. Lags: Landsat 9,
# 3 days + 1 day margin; Landsat 8, 16 + 3 days. Sentinel-2 keeps 48 h (the
# Sentinel-2 passes of the October 2026 runs were in the Earth Search catalogue
# within that time).
VALIDATION_ARCHIVE_LAG_H_BY_PLATFORM = {"landsat-9": 96.0, "landsat-8": 456.0}


def _archive_lag_h(platform):
    return float(VALIDATION_ARCHIVE_LAG_H_BY_PLATFORM.get(platform, VALIDATION_ARCHIVE_LAG_H))


def _stac_platform(sat):
    """STAC platform string for a fleet satellite, or None if not validated."""
    name, mission = str(sat.get("id", "")).upper(), str(sat.get("mission", ""))
    m = re.match(r"^SENTINEL[- ]?2([A-Z])", name)
    if mission.startswith("Sentinel-2") and m:
        return f"sentinel-2{m.group(1).lower()}"
    m = re.match(r"^LANDSAT[- ]?([89])", name)
    if mission.startswith("Landsat") and m:
        return f"landsat-{m.group(1)}"
    return None


def stac_search(collections, bbox, start, end, url=VALIDATION_STAC_URL, limit=100, max_pages=100):
    """All items of `collections` intersecting bbox [W, S, E, N] with datetime in
    [start, end] (UTC datetimes), following the STAC API "next" links."""
    body = {"collections": list(collections), "bbox": list(bbox), "limit": limit,
            "datetime": f"{start:%Y-%m-%dT%H:%M:%SZ}/{end:%Y-%m-%dT%H:%M:%SZ}"}
    href, method, items = url.rstrip("/") + "/search", "POST", []
    for _ in range(max_pages):
        data = json.dumps(body).encode() if method == "POST" else None
        req = urllib.request.Request(href, data=data, method=method,
                                     headers={"Content-Type": "application/json", "User-Agent": "ARF-Research-Pipeline/1.0"})
        with urllib.request.urlopen(req, timeout=60) as resp:
            page = json.loads(resp.read().decode("utf-8"))
        items.extend(page.get("features", []))
        nxt = next((l for l in page.get("links", []) if l.get("rel") == "next"), None)
        if not nxt:
            break
        href, method = nxt["href"], str(nxt.get("method", "GET")).upper()
        if method == "POST":
            body = nxt.get("body", body) if nxt.get("merge") is not True else {**body, **nxt.get("body", {})}
    return items


def _item_tiles(geometry, roi_tiles, res):
    """ROI tiles inside a GeoJSON Polygon / MultiPolygon footprint."""
    polys = ([geometry["coordinates"]] if geometry.get("type") == "Polygon"
             else geometry.get("coordinates", []) if geometry.get("type") == "MultiPolygon" else [])
    cells = set()
    for poly in polys:
        outer = [(lat, lon) for lon, lat in poly[0]]
        holes = [[(lat, lon) for lon, lat in ring] for ring in poly[1:]]
        try:
            cells |= set(h3.polygon_to_cells(h3.LatLngPoly(outer, *holes), res))
        except Exception:
            continue
    return cells & roi_tiles


def _group_overpasses(records, gap_min):
    """records: (platform, datetime, tiles, cloud or None) -> overpasses (one per
    platform and time cluster): platform, time (median), tiles (union), cloud
    (mean of the scenes' eo:cloud_cover), n_scenes."""
    out = []
    for plat in sorted({r[0] for r in records}):
        rs = sorted((r for r in records if r[0] == plat), key=lambda r: r[1])
        cluster = []
        for r in rs + [None]:
            if cluster and (r is None or (r[1] - cluster[-1][1]).total_seconds() / 60.0 > gap_min):
                times = sorted(c[1] for c in cluster)
                clouds = [c[3] for c in cluster if c[3] is not None]
                out.append({"platform": plat, "time": times[len(times) // 2],
                            "tiles": set().union(*(c[2] for c in cluster)),
                            "cloud_pct": float(np.mean(clouds)) if clouds else None, "n_scenes": len(cluster)})
                cluster = []
            if r is not None:
                cluster.append(r)
    return out


def validate_acquisitions(refined_df, fleet, sim_start, window_h, roi_tiles, res,
                          search_fn=stac_search, platform_fn=_stac_platform, now=None):
    """Predicted vs real overpasses for the open-data satellites. Returns (summary, rows)."""
    now = now or datetime.now(timezone.utc).replace(tzinfo=None)
    plat_of = {s["id"]: platform_fn(s) for s in fleet}
    plat_of = {k: v for k, v in plat_of.items() if v}
    w_end = sim_start + timedelta(hours=window_h)
    # v1.5.2: each platform is evaluated up to (now - its archive lag)
    t_end_of = {p: min(w_end, now - timedelta(hours=_archive_lag_h(p))) for p in set(plat_of.values())}
    t_end = max(t_end_of.values()) if t_end_of else min(w_end, now - timedelta(hours=VALIDATION_ARCHIVE_LAG_H))
    base = {"stac_url": VALIDATION_STAC_URL, "match_tolerance_min": VALIDATION_MATCH_TOL_MIN,
            "archive_lag_h": VALIDATION_ARCHIVE_LAG_H, "window_start_utc": f"{sim_start:%Y-%m-%dT%H:%M:%S}",
            "archive_lag_h_by_platform": {p: _archive_lag_h(p) for p in sorted(t_end_of)},
            "evaluated_until_utc_by_platform": {p: f"{t:%Y-%m-%dT%H:%M:%S}" for p, t in sorted(t_end_of.items())},
            "evaluated_until_utc": f"{t_end:%Y-%m-%dT%H:%M:%S}", "platforms": sorted(set(plat_of.values()))}
    if not plat_of:
        return {**base, "status": "skipped: no Sentinel-2 or Landsat 8/9 satellite in this fleet"}, []
    live = {p for p, t in t_end_of.items() if t > sim_start}       # platforms with any archive-ready time
    if not live:
        _lag = min(_archive_lag_h(p) for p in t_end_of)
        return {**base, "status": f"skipped: window not yet older than {_lag:g} h "
                                  f"(re-run Cell 7D after {sim_start + timedelta(hours=_lag):%Y-%m-%d %H:%M} UTC)"}, []

    # Predicted overpasses: refined events of the validated satellites, grouped by time.
    pred_recs = []
    r = refined_df[refined_df["sat"].astype(str).isin(plat_of)]
    for (sat, t_min), g in r.groupby(["sat", "t_min"], observed=True):
        t = sim_start + timedelta(minutes=float(t_min))
        p = plat_of[str(sat)]
        if p in live and t <= t_end_of[p]:
            pred_recs.append((p, t, set(g["tile"].astype(str)), None))
    predicted = _group_overpasses(pred_recs, VALIDATION_GROUP_GAP_MIN)

    # Real overpasses: catalogue scenes touching the ROI tiles.
    lat_lon = np.array([h3.cell_to_latlng(t) for t in roi_tiles])
    bbox = [float(lat_lon[:, 1].min()) - 0.1, float(lat_lon[:, 0].min()) - 0.1,
            float(lat_lon[:, 1].max()) + 0.1, float(lat_lon[:, 0].max()) + 0.1]
    want = set(plat_of.values())
    real_recs, n_items = [], 0
    try:
        for mission, cols in VALIDATION_COLLECTIONS.items():
            if not any(p.startswith("sentinel-2" if mission == "Sentinel-2" else "landsat") for p in want):
                continue
            for it in search_fn(cols, bbox, sim_start, t_end):
                n_items += 1
                p = it.get("properties", {})
                plat = str(p.get("platform", "")).lower()
                if plat not in want:
                    continue
                tiles = _item_tiles(it.get("geometry") or {}, roi_tiles, res)
                if not tiles:
                    continue
                dt = datetime.fromisoformat(str(p["datetime"]).replace("Z", "+00:00")).astimezone(timezone.utc).replace(tzinfo=None)
                if plat not in live or dt > t_end_of[plat]:
                    continue                       # v1.5.2: beyond this platform's archive-ready time
                cc = p.get("eo:cloud_cover")
                real_recs.append((plat, dt, tiles, None if cc is None else float(cc)))
    except Exception as e:                         # no network, catalogue down, ...
        return {**base, "status": f"skipped: catalogue query failed ({type(e).__name__}: {e})"}, []
    real = _group_overpasses(real_recs, VALIDATION_GROUP_GAP_MIN)

    # One-to-one matching, nearest in time first.
    pairs = sorted(((abs((a["time"] - b["time"]).total_seconds()) / 60.0, i, j)
                    for i, a in enumerate(predicted) for j, b in enumerate(real) if a["platform"] == b["platform"]))
    used_p, used_r, match = set(), set(), []
    for dtm, i, j in pairs:
        if dtm <= VALIDATION_MATCH_TOL_MIN and i not in used_p and j not in used_r:
            used_p.add(i); used_r.add(j); match.append((i, j, dtm))
    rows = []
    for i, j, dtm in match:
        a, b = predicted[i], real[j]
        ov = len(a["tiles"] & b["tiles"])
        rows.append({"kind": "matched", "platform": a["platform"], "predicted_utc": f"{a['time']:%Y-%m-%dT%H:%M}",
                     "actual_utc": f"{b['time']:%Y-%m-%dT%H:%M:%S}", "dt_min": round(dtm, 2),
                     "predicted_tiles": len(a["tiles"]), "actual_tiles": len(b["tiles"]), "overlap_tiles": ov,
                     "cloud_pct": b["cloud_pct"], "scenes": b["n_scenes"]})
    for i, a in enumerate(predicted):
        if i not in used_p:
            rows.append({"kind": "predicted_only", "platform": a["platform"], "predicted_utc": f"{a['time']:%Y-%m-%dT%H:%M}",
                         "actual_utc": None, "dt_min": None, "predicted_tiles": len(a["tiles"]), "actual_tiles": None,
                         "overlap_tiles": None, "cloud_pct": None, "scenes": 0})
    for j, b in enumerate(real):
        if j not in used_r:
            rows.append({"kind": "actual_only", "platform": b["platform"], "predicted_utc": None,
                         "actual_utc": f"{b['time']:%Y-%m-%dT%H:%M:%S}", "dt_min": None, "predicted_tiles": None,
                         "actual_tiles": len(b["tiles"]), "overlap_tiles": None, "cloud_pct": b["cloud_pct"],
                         "scenes": b["n_scenes"]})
    m = [x for x in rows if x["kind"] == "matched"]
    pt = sum(x["predicted_tiles"] for x in m)
    at = sum(x["actual_tiles"] for x in m)
    ov = sum(x["overlap_tiles"] for x in m)
    clouds = [b["cloud_pct"] for b in real if b["cloud_pct"] is not None]
    # v1.5.2: per platform -- a satellite that is tracked but not acquiring
    # (e.g. Sentinel-2A on its extension-campaign plan) shows up as predicted
    # overpasses with no match, which a pooled precision hides.
    by_platform = {}
    for plat in sorted(t_end_of):
        n_p = sum(1 for a in predicted if a["platform"] == plat)
        n_r = sum(1 for b in real if b["platform"] == plat)
        n_m = sum(1 for x in m if x["platform"] == plat)
        by_platform[plat] = {"evaluated": plat in live, "archive_lag_h": _archive_lag_h(plat),
                             "predicted_overpasses": n_p, "actual_overpasses": n_r, "matched": n_m,
                             "precision": (n_m / n_p) if n_p else None, "recall": (n_m / n_r) if n_r else None,
                             "evaluated_until_utc": f"{t_end_of[plat]:%Y-%m-%dT%H:%M:%S}" if plat in live else None}
    summary = {**base, "status": "ok", "catalogue_items_returned": n_items, "by_platform": by_platform,
               "predicted_overpasses": len(predicted), "actual_overpasses": len(real), "matched": len(m),
               "precision": (len(m) / len(predicted)) if predicted else None,
               "recall": (len(m) / len(real)) if real else None,
               "median_time_offset_min": float(np.median([x["dt_min"] for x in m])) if m else None,
               "predicted_tiles_confirmed_pct": (100.0 * ov / pt) if pt else None,
               "actual_tiles_predicted_pct": (100.0 * ov / at) if at else None,
               "actual_cloud_cover_mean_pct": float(np.mean(clouds)) if clouds else None,
               "actual_cloud_cover_median_pct": float(np.median(clouds)) if clouds else None,
               "model_cloud_probability_pct": 100.0 * CLOUD_PROBABILITY,
               "note": "PlanetScope passes are not validated (no open archive)."}
    return summary, rows


_val_cols = ["kind", "platform", "predicted_utc", "actual_utc", "dt_min", "predicted_tiles", "actual_tiles",
             "overlap_tiles", "cloud_pct", "scenes"]
if globals().get("FLEET", "starlink") == "starlink":
    acquisition_validation = {"status": "skipped: the Starlink proxy has no imaging archive"}
    _val_rows = []
else:
    _sim_start = datetime.fromisoformat(str(_sim_epoch_datetime64())[:26])
    acquisition_validation, _val_rows = validate_acquisitions(
        refined_events_df, constellation, _sim_start, WINDOW_HOURS, set(target_tiles), H3_RESOLUTION)
pd.DataFrame(_val_rows, columns=_val_cols).to_csv(os.path.join(_out_dir, "arf_acquisition_validation.csv"), index=False)
with open(os.path.join(_out_dir, "arf_acquisition_validation_summary.json"), "w") as f:
    json.dump(acquisition_validation, f, indent=2, default=str)
scenario_results.setdefault(_scenario_key(), {})["acquisition_validation"] = acquisition_validation
logger.info(f"Acquisition validation: {acquisition_validation['status']}"
            + ("" if acquisition_validation["status"] != "ok" else
               f" | precision {acquisition_validation['precision']}, recall {acquisition_validation['recall']}"))
print("\n=== CELL 7D COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 8: HYPOTHESIS VALIDATION (H1/H2/H3)
# =============================================================================
# Statuses (SUPPORTED / NOT SUPPORTED / NOT EVALUABLE) for every scenario in
# the ledger, from the analyses of Cells 6 and 7C:
#   H1 = H1a and H1b (Cell 7C onset analysis, clear sky; level = 95% of the
#        tiles reachable in each onset's horizon):
#     H1a latency non-inferiority: the earliest-completion cover (ECC, k images)
#        reaches the level within one orbital period of the physical floor at
#        >= 90% of onsets. ECC aims at the floor by construction, so H1a checks
#        that the budget allows it; it does not compare policies (the time-aware
#        selector's result is reported alongside).
#     H1b image economy: at >= 90% of the onsets where first-come-first-served
#        needs two or more images, ECC meets the H1a criterion with fewer images.
#     The onsets of one window share most of their passes, so these statuses
#     describe the window; the confirmatory test treats windows as the unit.
#   H2 Fail-Signal adaptivity (Cell 7C, cloud realisations): with the same
#        number of images, the adaptive selector covers more under cloud than
#        the same planner run in advance with no cloud observation (one-sided
#        95% lower bound of the mean paired difference > 0).
#   H3 redundancy below 1.5 when 95% coverage is first reached (clear sky);
#        NOT EVALUABLE when the passes cannot reach 95%.
#   Overall status of H1a, H1b, H2 and (from v1.5.2) H3: SUPPORTED only if
#   every evaluable scenario in the ledger is; NOT EVALUABLE if none is.
# Kept for the record: the original H1 (completion < 10 min) and the original
# H2 (cloud coverage > 50% of clear-sky coverage, which a plan with no
# Fail-Signal passes whenever p < 0.5, so it could not fail).
# Provenance: rebuilt in October 2026 when the working notebook was found to
# have lost this cell; two original criteria were corrected then (H2 tested
# coverage > 0; H3 used OR instead of AND).
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 8 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["8"] = _ARF_CELL_VERSION

logger = logging.getLogger("arf.cell8")

H1_LATENCY_THRESHOLD_MIN = 10.0
H2_MAINTENANCE_THRESHOLD_PCT = 50.0
H3_COVERAGE_THRESHOLD_PCT = 95.0
H3_REDUNDANCY_THRESHOLD = 1.5


def _scenario_label(key):
    """Format a scenario_results key -- (active_roi, active_n) -- as a
    readable label for validator assessments. Falls back to str() for
    any legacy int-only key, so this doesn't hard-crash against an
    older-format ledger loaded from a previous run."""
    if isinstance(key, tuple) and len(key) == 2:
        roi, n = key
        return f"ROI={roi}, N={n}" if isinstance(n, int) else f"ROI={roi}, fleet={n}"
    return f"N={key}"


def validate_h1_latency(scenario_results, threshold_min=H1_LATENCY_THRESHOLD_MIN,
                        latency_key='latency_clear_min', label="H1 (Latency Reduction)"):
    """H1 (Latency Reduction): SUPPORTED if latency_clear_min -- the mosaic
    completion time, i.e. when delivered coverage reaches its final value --
    is < threshold_min for at least one tested scenario."""
    assessments = {}
    any_under = False
    for scenario_k, data in scenario_results.items():
        latency = data.get(latency_key)
        under = latency is not None and latency < threshold_min
        assessments[_scenario_label(scenario_k)] = {"latency_clear_min": latency, "under_threshold": under}
        if under:
            any_under = True
    return {
        "hypothesis": label,
        "overall_status": "SUPPORTED ✅" if any_under else "NOT SUPPORTED ❌",
        "threshold_min": threshold_min,
        "assessments": assessments,
    }


def validate_h2_reliability_fail_signal(scenario_results, maintenance_threshold_pct=H2_MAINTENANCE_THRESHOLD_PCT):
    """H2 (Reliability via Fail-Signal): SUPPORTED if cloud-scenario coverage
    maintains > maintenance_threshold_pct of clear-sky coverage for at
    least one tested scenario."""
    assessments = {}
    any_maintained = False
    for scenario_k, data in scenario_results.items():
        cc = data.get('coverage_clear_pct', 0.0)
        cl = data.get('coverage_cloud_pct', 0.0)
        maintenance_pct = (cl / cc * 100) if cc > 0 else 0.0
        maintained = maintenance_pct > maintenance_threshold_pct
        assessments[_scenario_label(scenario_k)] = {
            "coverage_clear_pct": cc, "coverage_cloud_pct": cl,
            "coverage_maintenance_pct": maintenance_pct,
        }
        if maintained:
            any_maintained = True
    return {
        "hypothesis": "H2 (Reliability via Fail-Signal)",
        "overall_status": "SUPPORTED ✅" if any_maintained else "NOT SUPPORTED ❌",
        "maintenance_threshold_pct": maintenance_threshold_pct,
        "assessments": assessments,
    }


def validate_h3_redundancy_minimisation(scenario_results,
                                         coverage_threshold_pct=H3_COVERAGE_THRESHOLD_PCT,
                                         redundancy_threshold=H3_REDUNDANCY_THRESHOLD):
    """H3 (Redundancy Minimisation): SUPPORTED if coverage_clear_pct >
    coverage_threshold_pct AND h3_redundancy_clear -- the redundancy at the
    point coverage_threshold_pct is first reached (Cell 6) -- is below
    redundancy_threshold,
    TOGETHER, for at least one tested scenario.
    NOT EVALUABLE (v1.4.0) for a scenario whose passes cannot reach
    coverage_threshold_pct at all (usable clear-sky reach below it): the
    redundancy at that coverage is then undefined, whatever the selector does.
    Overall (v1.5.2): SUPPORTED only if every evaluable scenario is; NOT
    EVALUABLE if none is -- the same rule as H1a, H1b and H2 (until v1.5.1 a
    single supported scenario sufficed, the rule of the original H1/H2)."""
    assessments = {}
    any_supported = any_evaluable = False
    for scenario_k, data in scenario_results.items():
        cov = data.get('coverage_clear_pct', 0.0)
        if data.get('h3_coverage_pct') not in (None, coverage_threshold_pct):
            logger.warning(f"H3: Cell 6 measured redundancy at {data.get('h3_coverage_pct')}% coverage but "
                           f"H3_COVERAGE_THRESHOLD_PCT is {coverage_threshold_pct}%; update _H3_COVERAGE_PCT in Cell 6.")
        reach = data.get('reachable_usable_clear_pct', data.get('reachable_tiles_pct'))
        red = data.get('h3_redundancy_clear')
        evaluable = reach is None or reach >= coverage_threshold_pct - 1e-9
        supported = evaluable and red is not None and (cov > coverage_threshold_pct) and (red < redundancy_threshold)
        assessments[_scenario_label(scenario_k)] = {
            "status": ("NOT EVALUABLE" if not evaluable else ("SUPPORTED" if supported else "NOT SUPPORTED")),
            "coverage_clear_pct": cov, "reachable_clear_pct": reach,
            "redundancy_clear": float('inf') if red is None else red,
            "passes_to_coverage": data.get('h3_passes_clear'),
            "redundancy_all_passes": data.get('redundancy_clear'), "and_condition": supported,
        }
        any_evaluable |= evaluable
        any_supported |= supported
    _ev = [a["status"] == "SUPPORTED" for a in assessments.values() if a["status"] != "NOT EVALUABLE"]
    return {
        "hypothesis": "H3 (Redundancy Minimisation)",
        "overall_status": ("NOT EVALUABLE" if not _ev else ("SUPPORTED ✅" if all(_ev) else "NOT SUPPORTED ❌")),
        "overall_rule": "SUPPORTED only if every evaluable scenario is (v1.5.2)",
        "coverage_threshold_pct": coverage_threshold_pct,
        "redundancy_threshold": redundancy_threshold,
        "assessments": assessments,
    }


h1_result = validate_h1_latency(scenario_results)
h1_ta_result = validate_h1_latency(scenario_results, latency_key='ta_latency_clear_min',
                                   label="H1 (Latency Reduction) -- time-aware objective")
h2_result = validate_h2_reliability_fail_signal(scenario_results)
h3_result = validate_h3_redundancy_minimisation(scenario_results)
# v1.5.3: the superseded hypotheses carry the label wherever they are printed or
# saved. Until v1.5.2 the console and the pipeline log printed, e.g., "H2
# (Reliability via Fail-Signal): SUPPORTED" with nothing to say that this test
# was replaced (and passes whenever p < 0.5).
for _r, _note in ((h1_result, "superseded by H1a and H1b: the original < 10 min completion target, falsified "
                              "by the real-fleet runs of 2026-10-09"),
                  (h1_ta_result, "superseded by H1a and H1b: the < 10 min completion target for the "
                                 "time-aware selector"),
                  (h2_result, "superseded by H2 (v1.5.0, comparative): a plan with no Fail-Signal passes "
                              "this ratio test whenever p < 0.5")):
    _r["superseded"], _r["superseded_note"] = True, _note


def _share_status(n_eval, pct, min_pct, min_n):
    """Status of an 'at >= min_pct of onsets' test: NOT EVALUABLE below min_n
    evaluated onsets (see Cell 7C, evaluability), else SUPPORTED / NOT SUPPORTED."""
    if not n_eval or (min_n is not None and n_eval < min_n):
        return "NOT EVALUABLE"
    return "SUPPORTED" if pct is not None and pct >= min_pct - 1e-9 else "NOT SUPPORTED"


def _overall(statuses):
    """SUPPORTED only if every evaluable scenario is; NOT EVALUABLE if none is."""
    verdicts = [s == "SUPPORTED" for s in statuses if s in ("SUPPORTED", "NOT SUPPORTED")]
    return "NOT EVALUABLE" if not verdicts else ("SUPPORTED ✅" if all(verdicts) else "NOT SUPPORTED ❌")


def validate_h1_revised(scenario_results):
    """H1a, latency non-inferiority: SUPPORTED for a scenario if the primary
    selector -- the earliest-completion cover from v1.5.0 (the time-aware
    selector in v1.4.0 runs, whose summaries carry no primary_series) -- is
    within the floor margin at >= min_onset_pct of the evaluated onsets, clear
    sky. NOT EVALUABLE if fewer onsets were evaluated than min_evaluable_onsets.
    The time-aware selector's status is reported alongside. Overall: SUPPORTED
    only if every evaluable scenario is supported."""
    assessments = {}
    for k, d in scenario_results.items():
        h = (d.get("onset_robustness") or {}).get("h1_revised")
        if not h:
            assessments[_scenario_label(k)] = {"status": "NOT RUN"}
            continue
        prim = h.get("primary_series", "time_aware")
        s = h["series"][f"{prim}_clear"]
        st = _share_status(s["n_onsets_evaluated"], s["pct_within_margin"], h["min_onset_pct"],
                           h.get("min_evaluable_onsets"))
        s_ta = h["series"]["time_aware_clear"]
        st_ta = _share_status(s_ta["n_onsets_evaluated"], s_ta["pct_within_margin"], h["min_onset_pct"],
                              h.get("min_evaluable_onsets"))
        assessments[_scenario_label(k)] = {"status": st, "primary_selector": prim, "status_time_aware": st_ta,
                                           "statement": h["statement"],
                                           "floor_margin_source": h["floor_margin_source"],
                                           "min_evaluable_onsets": h.get("min_evaluable_onsets"),
                                           **{f"{n}": h["series"][n] for n in h["series"]}}
    return {"hypothesis": "H1a: latency non-inferiority (earliest-completion cover vs physical floor, reachable tiles)",
            "overall_status": _overall(a["status"] for a in assessments.values()), "assessments": assessments,
            "note": "Revised after the original < 10 min H1 was falsified by real-fleet runs (2026-10-09, then for "
                    "the time-aware selector); tested on the earliest-completion cover from v1.5.0 (2026-10-12), so "
                    "that H1a and H1b concern one selector. Confirmatory only on data not used to write it. Every "
                    "policy meets the floor margin when the budget allows (first-come-first-served by construction), "
                    "so H1a shows the selector loses no time, not that it beats the baselines; H1b carries the "
                    "comparison. Baseline rates are listed."}


def validate_h1b_image_economy(scenario_results):
    """H1b, image economy (added 2026-10-12): SUPPORTED for a scenario if, at
    >= min_onset_pct of the onsets where first-come-first-served needs two or
    more images to reach the level, the earliest-completion cover reaches it
    within the floor margin with fewer images (clear sky). NOT EVALUABLE below
    min_evaluable_onsets such onsets."""
    assessments = {}
    for k, d in scenario_results.items():
        h = (d.get("onset_robustness") or {}).get("h1b_image_economy")
        if not h:
            assessments[_scenario_label(k)] = {"status": "NOT RUN"}
            continue
        s = h["series"]["clear"]
        st = _share_status(s["n_onsets_evaluated"], s["pct_success"], h["min_onset_pct"], h.get("min_evaluable_onsets"))
        assessments[_scenario_label(k)] = {"status": st, "statement": h["statement"],
                                           "min_evaluable_onsets": h.get("min_evaluable_onsets"),
                                           "clear": h["series"]["clear"], "cloud": h["series"]["cloud"]}
    return {"hypothesis": "H1b: image economy (earliest-completion cover vs first-come-first-served)",
            "overall_status": _overall(a["status"] for a in assessments.values()), "assessments": assessments,
            "note": "Formulated after the 15 real-fleet runs of 2026-10-12: confirmatory only on fresh data."}


def combine_h1(h1a, h1b):
    """H1 as reframed in v1.4.0 = H1a AND H1b, per scenario: NOT SUPPORTED if
    either part is; SUPPORTED if both are; otherwise NOT EVALUABLE."""
    assessments = {}
    labels = list(h1a["assessments"]) + [lab for lab in h1b["assessments"] if lab not in h1a["assessments"]]
    for lab in labels:                                            # scenario order kept, as in the ledger
        a = h1a["assessments"].get(lab, {}).get("status", "NOT RUN")
        b = h1b["assessments"].get(lab, {}).get("status", "NOT RUN")
        st = ("NOT SUPPORTED" if "NOT SUPPORTED" in (a, b) else
              ("SUPPORTED" if a == b == "SUPPORTED" else "NOT EVALUABLE"))
        assessments[lab] = {"status": st, "H1a": a, "H1b": b}
    return {"hypothesis": "H1 (v1.4.0): latency non-inferiority (H1a) and image economy (H1b)",
            "overall_status": _overall(a["status"] for a in assessments.values()), "assessments": assessments}


def validate_h2_comparative(scenario_results):
    """H2 (v1.5.0), Fail-Signal adaptivity: per scenario, the status computed in
    Cell 7C (section 4) for the primary (time-aware) objective -- SUPPORTED if,
    with the same number of images, the adaptive selector covers more under
    cloud than an open-loop plan maximising expected coverage (one-sided 95%
    lower bound of the mean paired difference > 0 over the cloud
    realisations); NOT EVALUABLE when there was nothing to adapt or no room to
    gain. Coverage-first is reported alongside. Overall: SUPPORTED only if every
    evaluable scenario is supported."""
    assessments = {}
    for k, d in scenario_results.items():
        c = (d.get("cloud_robustness") or {})
        if not c.get("h2"):
            assessments[_scenario_label(k)] = {"status": "NOT RUN"}
            continue
        h, hc = c["h2"]["time_aware"], c["h2"].get("coverage_first", {})
        assessments[_scenario_label(k)] = {"status": h["status"], "status_coverage_first": hc.get("status"),
                                           "time_aware": h, "coverage_first": hc,
                                           "cloud_model": c.get("cloud_model"), "seeds": c.get("seeds")}
    return {"hypothesis": "H2 (v1.5.0): Fail-Signal adaptivity (adaptive vs open-loop plan with the same images, "
                          "over cloud realisations)",
            "overall_status": _overall(a["status"] for a in assessments.values()), "assessments": assessments,
            "note": "Replaces the original H2 (cloud coverage > 50% of clear-sky coverage), which a plan with no "
                    "Fail-Signal passes whenever p < 0.5. Formulated 2026-10-12 after the 15 runs of that day: "
                    "confirmatory only on fresh data."}


h1_revised_result = validate_h1_revised(scenario_results)
h1b_result = validate_h1b_image_economy(scenario_results)
h1_reframed_result = combine_h1(h1_revised_result, h1b_result)
h2c_result = validate_h2_comparative(scenario_results)

hypothesis_validation = {
    "H1_reframed": h1_reframed_result,
    "H1_revised": h1_revised_result,          # = H1a
    "H1b_image_economy": h1b_result,
    "H1": h1_result,
    "H2": h2c_result,                         # v1.5.0: comparative
    "H2_original_ratio": h2_result,           # cloud coverage > 50% of clear (kept for the record)
    "H3": h3_result,
    "H1_time_aware": h1_ta_result,
    "H1_onset_robustness": {_scenario_label(k): d.get("onset_robustness", {}) for k, d in scenario_results.items()},
    "approximation_guarantee": (
        "The k-step greedy attains at least 1 - 1/e of the optimum of a monotone submodular objective under a "
        "cardinality budget (Nemhauser, Wolsey & Fisher, 1978). This covers the coverage-first policy (the best "
        "utility per tile, summed) and, from v1.5.0, the time-aware policy, whose objective is the time-weighted "
        "coverage sum_t max w_e u_e,t -- also a weighted facility-location function. Cell 7C measures both against "
        "proven ILP optima of their own objectives. No guarantee is claimed for the Fail-Signal phases (re-planning "
        "on arrival and compensation beyond the budget), the earliest-completion cover's time (its greedy cover step "
        "has the classical logarithmic bound on the number of images), or the legacy v1.4.0 time-aware weighting."
        if globals().get("TIME_AWARE_OBJECTIVE", "facility") == "facility" else
        "(1-1/e) approximation guarantee (Nemhauser, Wolsey & Fisher, 1978) applies to the coverage-first policy. "
        "The legacy time-aware policy re-weights marginal gains per event; no guarantee is claimed for it."
    ),
}

# v1.5.3: this run's own statuses first. An "overall" status pools every
# scenario in the kernel's ledger (SUPPORTED only if every evaluable one is),
# so a run made after others in one kernel could read "H2 NOT SUPPORTED"
# overall while its own scenario was supported (15 October 2026).
_this_label = _scenario_label(_scenario_key()) if "_scenario_key" in globals() else None
_this_run = {"scenario": _this_label, "ledger_scenarios": [_scenario_label(k) for k in scenario_results],
             "statuses": {_hk: ((_hr.get("assessments") or {}).get(_this_label) or {}).get("status", "NOT RUN")
                          for _hk, _hr in (("H1a", h1_revised_result), ("H1b", h1b_result),
                                           ("H1 (H1a and H1b)", h1_reframed_result), ("H2", h2c_result),
                                           ("H3", h3_result))},
             "note": "statuses of this run's own scenario; the overall_status of each hypothesis pools "
                     "every scenario in ledger_scenarios"}
hypothesis_validation["this_run"] = _this_run
_ov = (f" (overall: pools the {len(scenario_results)} scenarios in this kernel)"
       if len(scenario_results) > 1 else "")
print("=" * 70)
print("HYPOTHESIS VALIDATION")
print("=" * 70)
print(f"\nThis run ({_this_label}): " + ", ".join(f"{k} {v}" for k, v in _this_run["statuses"].items()))
if len(scenario_results) > 1:
    print(f"NOTE: this kernel's ledger holds {len(scenario_results)} scenarios; each 'overall' status below "
          f"pools them all. Quote this run's line above or the per-scenario lines.")
print(f"\n{h1_reframed_result['hypothesis']}: {h1_reframed_result['overall_status']}{_ov}")
for n_label, a in h1_reframed_result['assessments'].items():
    print(f"  {n_label}: {a['status']} (H1a {a['H1a']}, H1b {a['H1b']})")
print(f"\n{h1_revised_result['hypothesis']}: {h1_revised_result['overall_status']}{_ov}")
for n_label, a in h1_revised_result['assessments'].items():
    _s = a.get(f"{a.get('primary_selector', 'time_aware')}_clear") or {}
    print(f"  {n_label}: {a['status']} -- {a.get('primary_selector')} within margin at "
          f"{_s.get('n_within_margin')}/{_s.get('n_onsets_evaluated')} onsets "
          f"(evaluable from {a.get('min_evaluable_onsets')}); time-aware: {a.get('status_time_aware')}")
print(f"\n{h2c_result['hypothesis']}: {h2c_result['overall_status']}{_ov}")
for n_label, a in h2c_result['assessments'].items():
    _h = a.get("time_aware") or {}
    _dx = _h.get("difference_vs_openloop_expected") or {}
    _f = lambda v: "n/a" if v is None else f"{v:.2f}"
    print(f"  {n_label}: {a['status']} -- adaptive minus open-loop (expected), same images: mean "
          f"{_f(_dx.get('mean'))} points, one-sided 95% lower bound {_f(_dx.get('lcb'))} over {_dx.get('n')} "
          f"realisations; coverage-first: {a.get('status_coverage_first')}"
          + (f"; not evaluable: {'; '.join(_h.get('not_evaluable_reasons') or [])}" if a['status'] == 'NOT EVALUABLE' else ""))
print(f"\n{h1b_result['hypothesis']}: {h1b_result['overall_status']}{_ov}")
for n_label, a in h1b_result['assessments'].items():
    _s = a.get("clear") or {}
    print(f"  {n_label}: {a['status']} -- fewer images than FCFS within the margin at "
          f"{_s.get('n_success')}/{_s.get('n_onsets_evaluated')} onsets; median images ECC "
          f"{_s.get('median_images_ecc')} vs FCFS {_s.get('median_images_fcfs')}")
for h_key, h_data in [("H1 (original, < 10 min)", h1_result), ("H1 time-aware", h1_ta_result),
                      ("H2 (original ratio)", h2_result), ("H3", h3_result)]:
    _sup = "SUPERSEDED (kept for the record): " if h_data.get("superseded") else ""
    print(f"\n{_sup}{h_data['hypothesis']}: {h_data['overall_status']}{_ov}")
    if h_data.get("superseded"):
        print(f"  ({h_data['superseded_note']})")
    for n_label, a in h_data['assessments'].items():
        print(f"  {n_label}: {a}")

print(f"\n{hypothesis_validation['approximation_guarantee']}")

try:
    with open(os.path.join(OUT_DIR if 'OUT_DIR' in globals() else os.getcwd(), 'arf_hypothesis_validation.json'), 'w') as f:
        json.dump(hypothesis_validation, f, indent=2, default=str)
    logger.info("Hypothesis validation saved to arf_hypothesis_validation.json")
except Exception as e:
    logger.warning(f"Could not save hypothesis validation: {e}")

# =============================================================================
# RESULTS SUMMARY (CSV) & FINAL REPORT (TXT)
# =============================================================================
# One row per scenario in the ledger, so a kernel that has run several
# (ROI, N) scenarios produces a cross-scenario comparison.
import textwrap
from datetime import datetime, timezone
_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()


def _r(v, nd=2):
    return None if v is None or (isinstance(v, float) and np.isnan(v)) else round(float(v), nd)


def _frontier_summary(rows, k_budget):
    """Per sky, for the reachable-tile target: the smallest image budget whose
    earliest-completion time equals the floor, and the time at the run's budget."""
    out = {}
    for sky in ("clear", "cloud"):
        f = [r for r in rows if r["sky"] == sky and r["target"] == "reachable"]
        if not f:
            continue
        fl = f[0]["floor_min"]
        at_floor = [r["k"] for r in f if r["t_complete_min"] is not None and fl is not None
                    and abs(r["t_complete_min"] - fl) < 1e-9]
        at_k = [r for r in f if r["k"] == k_budget]
        out[f"ECC Images to Match Floor - {sky}"] = min(at_floor) if at_floor else None
        out[f"ECC Time at Budget k (min) - {sky}"] = _r(at_k[0]["t_complete_min"], 1) if at_k else None
        out[f"Floor to Reachable Target (min) - {sky}"] = _r(fl, 1)
    return out


def _h1_v140_columns(key, ob):
    """Summary columns of the v1.4.0 H1 framing (H1a, H1b, quality at the level)."""
    lab = _scenario_label(key)
    out = {"H1 Status (v1.4.0: H1a and H1b)": h1_reframed_result["assessments"].get(lab, {}).get("status"),
           "H1a Status (latency non-inferiority)": h1_revised_result["assessments"].get(lab, {}).get("status"),
           "H1b Status (image economy)": h1b_result["assessments"].get(lab, {}).get("status")}
    a, b, q = ob.get("h1_revised"), ob.get("h1b_image_economy"), ob.get("quality_at_level")
    lvl = f"{(a or b or {}).get('reachable_coverage_pct', 95.0):g}%r"
    if a:
        out.update({"H1 Min Evaluable Onsets": a.get("min_evaluable_onsets"),
                    "H1a Primary Selector": a.get("primary_series", "time_aware"),
                    "H1a Within Margin 95% Lower Bound (%) - ECC clear":
                        _r(a["series"]["earliest_completion_clear"].get("pct_within_margin_lcb95"), 1),
                    "H1a Within Margin 95% Lower Bound (%) - time-aware clear":
                        _r(a["series"]["time_aware_clear"].get("pct_within_margin_lcb95"), 1)})
    if b:
        s = b["series"]["clear"]
        out.update({"H1b Onsets Evaluated - clear": s["n_onsets_evaluated"],
                    "H1b Onsets Needing One Image - clear": s["n_onsets_one_image"],
                    "H1b ECC Fewer Images Within Margin (%) - clear": _r(s["pct_success"], 1),
                    "H1b 95% Lower Bound (%) - clear": _r(s["pct_success_lcb95"], 1),
                    f"Median Images to {lvl} - ECC clear": _r(s["median_images_ecc"], 1),
                    f"Median Images to {lvl} - FCFS clear": _r(s["median_images_fcfs"], 1)})
    if q:
        s = q["series"]["clear"]
        out.update({f"Quality at {lvl} (median) - {lab_} clear": _r(s[k_]["median"], 3)
                    for k_, lab_ in (("time_aware", "time-aware"), ("fcfs_unbudgeted", "FCFS"),
                                     ("earliest_completion", "ECC"))})
        out[f"Quality at {lvl} time-aware vs FCFS (median relative difference %) - clear"] = _r(
            s["time_aware_vs_fcfs"]["median_relative_difference_pct"], 1)
    return out


_summary_rows = []
for _key, _d in scenario_results.items():
    _roi, _n = _key if isinstance(_key, tuple) and len(_key) == 2 else ("?", _key)
    _summary_rows.append({
        "ROI": _roi,
        "N": _n,
        "Candidate Events": _d.get("candidate_events"),
        "Selection Budget (k)": _d.get("selection_budget_k"),
        "Coverage - Clear Sky (%)": _r(_d.get("coverage_clear_pct")),
        "Coverage - Cloud/Fail-Signal (%)": _r(_d.get("coverage_cloud_pct")),
        "Cloud Knowledge": _d.get("cloud_knowledge", "oracle"),
        "Oracle Coverage - Cloud (%) (perfect cloud knowledge)": _r(_d.get("oracle_coverage_cloud_pct")),
        "Oracle H1 Latency - Cloud (min) (perfect cloud knowledge)": _r(_d.get("oracle_latency_cloud_min"), 1),
        "Cloud Re-plans": _d.get("cloud_replans"),
        "Cloud Compensatory Passes": _d.get("cloud_compensatory_passes"),
        "H3 Redundancy at 95% - Clear": _r(_d.get("h3_redundancy_clear"), 3),
        "H3 Redundancy at 95% - Cloud": _r(_d.get("h3_redundancy_cloud"), 3),
        "H3 Passes to 95% - Clear": _d.get("h3_passes_clear"),
        "Cloud Redundancy Floor (model)": _r(_d.get("cloud_redundancy_floor"), 3),
        "Redundancy all passes - Clear (budget-driven)": _r(_d.get("redundancy_clear"), 3),
        "Redundancy all passes - Cloud (budget-driven)": _r(_d.get("redundancy_cloud"), 3),
        "H1 Latency (completion) - Clear (min)": _r(_d.get("latency_clear_min"), 1),
        "H1 Latency (completion) - Cloud (min)": _r(_d.get("latency_cloud_min"), 1),
        "Last Selected Delivery - Clear (min)": _r(_d.get("last_delivery_clear_min"), 1),
        "Last Selected Delivery - Cloud (min)": _r(_d.get("last_delivery_cloud_min"), 1),
        "Reachable Tiles (%)": _r(_d.get("reachable_tiles_pct")),
        **(lambda _h: {} if not _h else {
            "H1r Margin (min)": _r(_h["floor_margin_min"], 1),
            **{f"H1r Onsets Within Margin (%) - {lab} {sky}": _r(_h["series"][f"{key}_{sky}"]["pct_within_margin"], 1)
               for key, lab in (("time_aware", "time-aware"), ("coverage_first", "coverage-first"), ("fcfs", "FCFS"),
                                ("earliest_completion", "ECC"))
               for sky in ("clear", "cloud")},
            "H1r Onsets Evaluated - clear": _h["series"]["time_aware_clear"]["n_onsets_evaluated"],
            "H1r Median Gap to Floor (min) - time-aware clear": _r(_h["series"]["time_aware_clear"]["median_gap_min"], 1),
        })((_d.get("onset_robustness") or {}).get("h1_revised")),
        **_h1_v140_columns(_key, _d.get("onset_robustness") or {}),
        **(lambda _c: {} if not _c.get("h2") else {
            "H2 Status (v1.5.0: adaptive vs open-loop, same images)": _c["h2"]["time_aware"]["status"],
            "H2 Status - coverage-first": _c["h2"]["coverage_first"]["status"],
            "H2 Cloud Realisations": _c["h2"]["time_aware"]["n_realisations"],
            "H2 Cloud Model": (_c.get("cloud_model") or {}).get("model"),
            "H2 Mean Images (adaptive, time-aware)": _r(_c["h2"]["time_aware"]["images_mean"], 1),
            **{f"H2 Mean Coverage (%) - {lab}": _r(_c["h2"]["time_aware"]["coverage_pct_mean"].get(t), 2)
               for t, lab in (("adaptive", "adaptive"), ("openloop_expected", "open-loop expected"),
                              ("openloop_clear", "open-loop clear-sky"), ("oracle", "oracle"))},
            "H2 Adaptive minus Open-loop (points, mean)": _r(_c["h2"]["time_aware"]["difference_vs_openloop_expected"]["mean"], 3),
            "H2 Adaptive minus Open-loop (points, one-sided 95% LB)": _r(_c["h2"]["time_aware"]["difference_vs_openloop_expected"]["lcb"], 3),
            "H2 Share of Oracle Gain Captured (median)": _r(_c["h2"]["time_aware"]["share_of_oracle_gain_captured_median"], 3),
        })(_d.get("cloud_robustness") or {}),
        "H2 Status (original ratio > 50%)": (lambda _st: _st)(
            "SUPPORTED" if (h2_result["assessments"].get(_scenario_label(_key), {}).get("coverage_maintenance_pct") or 0)
            > h2_result["maintenance_threshold_pct"] else "NOT SUPPORTED"),
        "Time-aware Objective": _d.get("time_aware_objective"),
        "Greedy/optimal time-weighted coverage - time-aware (median)": _r((_d.get("ilp_gap") or {}).get("time_aware_tw_ratio_median"), 4),
        "Greedy/optimal time-weighted coverage - time-aware (worst)": _r((_d.get("ilp_gap") or {}).get("time_aware_tw_ratio_min"), 4),
        "Decay Time Scale": _d.get("latency_decay_scale"),
        "Decay tau (min)": _r(_d.get("latency_decay_tau_min"), 1),
        "Reachable Usable Tiles - Clear (%)": _r(_d.get("reachable_usable_clear_pct")),
        "H3 Status": h3_result["assessments"].get(_scenario_label(_key), {}).get("status"),
        "Selected Events - Clear": _d.get("selected_events_clear"),
        "Selected Events - Cloud": _d.get("selected_events_cloud"),
        "Elbow k": _d.get("real_elbow_k"),
        "Saturation k": _d.get("saturation_k"),
        "Gap-Fill Enabled": _d.get("gap_fill_enabled"),
        "Decay Rate (time-aware)": _d.get("latency_decay_rate"),
        "H1 Latency time-aware - Clear (min)": _r(_d.get("ta_latency_clear_min"), 1),
        "H1 Latency time-aware - Cloud (min)": _r(_d.get("ta_latency_cloud_min"), 1),
        "Mean Quality - Clear (coverage-first)": _r(_d.get("mean_quality_clear"), 3),
        "Mean Quality - Clear (time-aware)": _r(_d.get("ta_mean_quality_clear"), 3),
        **{f"Onsets <H1 (%) - {lab}": _r(((_d.get("onset_robustness") or {}).get("series", {})
                                          .get(key, {}) or {}).get("pct_onsets_under_h1"), 1)
           for lab, key in (("best possible clear", "best_possible_clear"),
                            ("coverage-first clear", "coverage_first_clear"),
                            ("time-aware clear", "time_aware_clear"),
                            ("FCFS clear", "fcfs_clear"), ("random clear", "random_clear"))},
        **{f"Onset median (min) - {lab}": _r(((_d.get("onset_robustness") or {}).get("series", {})
                                            .get(key, {}) or {}).get("median_min"), 1)
           for lab, key in (("best possible clear", "best_possible_clear"),
                            ("coverage-first clear", "coverage_first_clear"),
                            ("time-aware clear", "time_aware_clear"),
                            ("FCFS clear", "fcfs_clear"), ("random clear", "random_clear"))},
        **{f"Onset median to {tg:g}% (min) - {lab} {sky}": _r(((_d.get("onset_robustness") or {})
                .get("equal_coverage_series", {}).get(f"{key}_{sky}_t{tg:g}", {}) or {}).get("median_min"), 1)
           for tg in COVERAGE_TARGETS_PCT for sky in ("clear", "cloud")
           for lab, key in (("best possible", "best_possible"), ("time-aware", "time_aware"), ("FCFS", "fcfs"),
                            ("ECC", "earliest_completion"))},
        **{f"Onsets reaching {tg:g}% (%) - {lab} {sky}": _r(((_d.get("onset_robustness") or {})
                .get("equal_coverage_series", {}).get(f"{key}_{sky}_t{tg:g}", {}) or {}).get("pct_onsets_reached"), 1)
           for tg in COVERAGE_TARGETS_PCT for sky in ("clear", "cloud")
           for lab, key in (("time-aware", "time_aware"), ("FCFS", "fcfs"), ("ECC", "earliest_completion"))},
        **{f"Greedy/optimal utility - {lab} (median)": _r((_d.get("ilp_gap") or {}).get(f"{key}_ratio_median"), 4)
           for lab, key in (("coverage-first", "coverage_first"), ("time-aware", "time_aware"), ("FCFS", "fcfs"))},
        **_frontier_summary(_d.get("cost_latency_frontier") or [], _d.get("selection_budget_k")),
        **(lambda _v: {"Validation Status": (_v.get("status") or "")[:60],
                       "Validation Precision": _r(_v.get("precision"), 3),
                       "Validation Recall": _r(_v.get("recall"), 3),
                       "Validation Predicted Tiles Confirmed (%)": _r(_v.get("predicted_tiles_confirmed_pct"), 1),
                       "Validation Real Cloud Cover Mean (%)": _r(_v.get("actual_cloud_cover_mean_pct"), 1)}
           )(_d.get("acquisition_validation") or {}),
    })
results_summary_df = pd.DataFrame(_summary_rows)
results_summary_df.to_csv(os.path.join(_out_dir, "arf_results_summary.csv"), index=False)

_labels = {_scenario_label(k): d for k, d in scenario_results.items()}
_bar, _sub = "=" * 70, "-" * 70
_lines = [_bar, "ALGORITHMIC RECONSTRUCTION FRAMEWORK (ARF)", "FINAL RESULTS REPORT", _bar, "",
          f"Date (UTC): {datetime.now(timezone.utc).strftime('%Y-%m-%d %H:%M:%S')}",
          f"Scenarios Evaluated: {list(_labels)}",
          f"Framework Version: {globals().get('__version__', 'unknown')}", "",
          _sub, "THIS RUN", _sub, "", f"Scenario: {_this_label}",
          *[f"  {k}: {v}" for k, v in _this_run["statuses"].items()],
          *([f"NOTE: the ledger holds {len(scenario_results)} scenarios (this kernel ran more than one); each "
             f"'overall' status under HYPOTHESIS VALIDATION pools them all. This folder's run is the one above."]
            if len(scenario_results) > 1 else []), "",
          _sub, "CROSS-SCENARIO COMPARISON", _sub, "",
          results_summary_df.to_string(index=False) if len(results_summary_df) else "(no scenarios in ledger)", "",
          _sub, "HYPOTHESIS VALIDATION", _sub, ""]
# v1.5.0: what the times mean in this run
_acq = {lab: d.get("acquisition_time_only") for lab, d in _labels.items()}
if _acq and all(v is True for v in _acq.values()):
    _lines += ["Times: delivery time = acquisition time + downlink wait. Every wait in this run is 0 (the modelled "
               "ground stations see each imaging pass), so all latencies below are ACQUISITION times; operator "
               "downlink, processing and distribution delays are not modelled.", ""]
elif _acq:
    _lines += [f"Times: delivery time = acquisition time + downlink wait to the modelled ground stations "
               f"(acquisition-only scenarios: {[k for k, v in _acq.items() if v]}). Operator processing and "
               f"distribution delays are not modelled.", ""]
# v1.5.1: times are measured from each scenario's own window start
_ref = {lab: d.get("reference_epoch") for lab, d in _labels.items()}
if len({v for v in _ref.values() if v}) > 1:
    _lines += ["WARNING: the scenarios in this ledger start at different times "
               + "; ".join(f"{k}: {v}" for k, v in _ref.items())
               + ". Their latencies are measured from different instants and are not comparable across "
                 "scenarios; set REFERENCE_EPOCH_UTC or ANALYSIS_EPOCH (Cell 0) to one value for runs to compare.", ""]
_lines += ["Onset statistics (H1a, H1b) describe one window: consecutive onsets share most of their passes, so "
           "their binomial bounds are optimistic; confirmatory tests use independent windows.", ""]

_pc = lambda v: "n/a" if v is None else f"{v:.1f}%"
_lines.append(f"{h1_reframed_result['hypothesis']}: {h1_reframed_result['overall_status']}{_ov}")
for _lab, _a in h1_reframed_result["assessments"].items():
    _lines.append(f"  {_lab}: {_a['status']} (H1a {_a['H1a']}, H1b {_a['H1b']})")
_lines.append("")
_lines.append(f"{h1_revised_result['hypothesis']}: {h1_revised_result['overall_status']}{_ov}")
for _lab, _a in h1_revised_result["assessments"].items():
    if "statement" not in _a:
        _lines.append(f"  {_lab}: {_a['status']}")
        continue
    _lines.append(f"  {_lab}: {_a['status']}")
    _lines.append(f"    {_a['statement']}  [margin {_a['floor_margin_source']}; evaluable from "
                  f"{_a.get('min_evaluable_onsets')} onsets]")
    _lines.append(f"    primary selector: {_a.get('primary_selector')}; time-aware selector: {_a.get('status_time_aware')}")
    for _n in ("earliest_completion_clear", "time_aware_clear", "coverage_first_clear", "fcfs_clear",
               "earliest_completion_cloud", "time_aware_cloud"):
        _s = _a[_n]
        _g = "n/a" if _s["median_gap_min"] is None else f"{_s['median_gap_min']:.0f} min"
        _lines.append(f"    {_n:22s} within margin at {_s['n_within_margin']}/{_s['n_onsets_evaluated']} onsets "
                      f"({_pc(_s['pct_within_margin'])}; 95% lower bound {_pc(_s.get('pct_within_margin_lcb95'))}); "
                      f"median gap to floor {_g}")
_lines.append(f"  Note: {h1_revised_result['note']}")
_lines.append("")
_lines.append(f"{h1b_result['hypothesis']}: {h1b_result['overall_status']}{_ov}")
for _lab, _a in h1b_result["assessments"].items():
    if "statement" not in _a:
        _lines.append(f"  {_lab}: {_a['status']}")
        continue
    _lines.append(f"  {_lab}: {_a['status']}")
    _lines.append(f"    {_a['statement']}  [evaluable from {_a.get('min_evaluable_onsets')} onsets]")
    for _sky in ("clear", "cloud"):
        _s = _a[_sky]
        _lines.append(f"    {_sky:5s} fewer images within the margin at {_s['n_success']}/{_s['n_onsets_evaluated']} "
                      f"onsets ({_pc(_s['pct_success'])}; 95% lower bound {_pc(_s['pct_success_lcb95'])}); median "
                      f"images ECC {_s['median_images_ecc']} vs FCFS {_s['median_images_fcfs']}; "
                      f"{_s['n_onsets_one_image']} onsets need one image (not evaluated)")
    _q = ((_labels.get(_lab, {}).get("onset_robustness") or {}).get("quality_at_level") or {}).get("series", {})
    for _sky in ("clear", "cloud"):
        _s = _q.get(_sky)
        if _s:
            _f3 = lambda v: "n/a" if v is None else f"{v:.3f}"
            _rd = _s["time_aware_vs_fcfs"]["median_relative_difference_pct"]
            _lines.append(f"    {_sky:5s} mosaic quality at the level (median; reported, not tested): time-aware "
                          f"{_f3(_s['time_aware']['median'])}, FCFS {_f3(_s['fcfs_unbudgeted']['median'])}, ECC "
                          f"{_f3(_s['earliest_completion']['median'])}; time-aware vs FCFS median "
                          f"{'n/a' if _rd is None else format(_rd, '+.1f') + '%'} "
                          f"(higher at {_pc(_s['time_aware_vs_fcfs']['pct_onsets_higher'])} of onsets)")
_lines.append(f"  Note: {h1b_result['note']}")
_lines.append("")
_lines.append("SUPERSEDED HYPOTHESES (kept for the record; not tested by the protocol, not results to quote):")
_lines.append(f"SUPERSEDED {h1_result['hypothesis']}: {h1_result['overall_status']}{_ov} "
              f"({h1_result['superseded_note']})")
_lines.append(f"  Latency = mosaic completion time (when delivered coverage reaches its final value);"
              f" target < {h1_result['threshold_min']:g} min.")
for _lab, _a in h1_result["assessments"].items():
    _lat = _a.get("latency_clear_min")
    _last = _labels.get(_lab, {}).get("last_delivery_clear_min")
    _lines.append(f"  {_lab}: " + ("N/A" if _lat is None else
                  f"{_lat:.1f} min ({'UNDER' if _a.get('under_threshold') else 'OVER'}-{h1_result['threshold_min']:g})")
                  + ("" if _last is None else f"  [last selected delivery: {_last:.1f} min]"))
_lines.append("")
_lines.append(f"SUPERSEDED {h1_ta_result['hypothesis']}: {h1_ta_result['overall_status']}{_ov} "
              f"({h1_ta_result['superseded_note']})")
for _lab, _a in h1_ta_result["assessments"].items():
    _lat = _a.get("latency_clear_min")
    _rate = _labels.get(_lab, {}).get("latency_decay_rate")
    _lines.append(f"  {_lab}: " + ("N/A" if _lat is None else
                  f"{_lat:.1f} min ({'UNDER' if _a.get('under_threshold') else 'OVER'}-{h1_ta_result['threshold_min']:g})")
                  + ("" if _rate is None else f"  [decay rate {_rate:g}; full sweep in arf_decay_sweep.csv]"))
_lines.append("")
_lines.append("H1 onset robustness (completion time across shifted onsets; see arf_onset_robustness.csv):")
for _lab, _d in _labels.items():
    _ob = _d.get("onset_robustness") or {}
    if not _ob:
        _lines.append(f"  {_lab}: not run")
        continue
    _lines.append(f"  {_lab}: {_ob['series']['best_possible_clear']['n_onsets']} onsets over "
                  f"{_ob['onset_shift_hours']} h, {_ob['horizon_hours']:g} h of passes each")
    _fm = lambda v: "n/a" if v is None else f"{v:.0f}"
    _lines.append("    Completion time (to each policy's OWN final coverage -- compare across policies only "
                  "where coverage is equal):")
    for _key, _s in _ob["series"].items():
        _lines.append(f"    {_key:22s} median {_fm(_s['median_min']):>5} | p90 {_fm(_s['p90_min']):>5} | "
                      f"max {_fm(_s['max_min']):>5} min | under the superseded 10-min H1 in {_s['pct_onsets_under_h1']:.1f}% of onsets")
    for _tg in _ob.get("coverage_targets_pct", []):
        _lines.append(f"    Equal coverage -- time to {_tg:g}% of ROI tiles (like-for-like across policies):")
        for _key, _s in _ob["equal_coverage_series"].items():
            if _key.endswith(f"_t{_tg:g}"):
                _lines.append(f"    {_key[:-len(f'_t{_tg:g}')]:22s} median {_fm(_s['median_min']):>5} | p90 {_fm(_s['p90_min']):>5} | "
                              f"reached in {_s['pct_onsets_reached']:5.1f}% | under the superseded 10-min H1 in {_s['pct_onsets_under_h1']:.1f}% of onsets")
for _lab, _d in _labels.items():
    _pcmp = _d.get("policy_comparison") or []    # v1.5.2: was _pc, which shadowed the % formatter above
    if _pcmp:
        _lines.append("")
        _lines.append(f"Policy comparison, {_lab} (full window, budget k, no gap-fill; see arf_policy_comparison.csv):")
        _fmt_t = lambda v: "never" if v is None else f"{v:.0f} min"
        for _p in _pcmp:
            _tt = " | ".join(f"to {tg:g}%: {_fmt_t(_p.get(f't{tg:g}_min')):>9}" for tg in COVERAGE_TARGETS_PCT)
            if _p["policy"] == "best_possible":
                _lines.append(f"    {_p['sky']:5s} {'best possible':16s} {'':>13} | {'':>16} | {'':>18} | {_tt}")
                continue
            _lat = "n/a" if _p["h1_latency_min"] is None else f"{_p['h1_latency_min']:.0f} min"
            _lines.append(f"    {_p['sky']:5s} {_p['policy']:16s} H1 {_lat:>9} | coverage {_p['coverage_pct']:6.2f}% "
                          f"| mean quality {_p['mean_quality']:.3f} | {_tt}")
    _fr = _d.get("cost_latency_frontier") or []
    if _fr:
        _lines.append(f"Cost-latency frontier, {_lab} (earliest-completion cover; see arf_cost_latency_frontier.csv):")
        for _sky in ("clear", "cloud"):
            _f = [r for r in _fr if r["sky"] == _sky and r["target"] == "reachable"]
            if not _f:
                continue
            _fl = _f[0]["floor_min"]
            _pts = ", ".join(f"k={r['k']}: {'never' if r['t_complete_min'] is None else format(r['t_complete_min'], '.0f') + ' min'}"
                             for r in _f)
            _lines.append(f"    {_sky:5s} to {_f[0]['target_pct']:g}% of {_f[0]['denominator_tiles']} reachable tiles "
                          f"(floor {'never' if _fl is None else format(_fl, '.0f') + ' min'}): {_pts}")
    _ig = _d.get("ilp_gap") or {}
    if _ig.get("solved"):
        _hz = _ig.get("horizons_used_min") or [int(_ig["horizon_hours"] * 60)]
        _lines.append(f"Optimality gap, {_lab}: {_ig['solved']}/{_ig['instances']} ILP instances solved to proven "
                      f"optimality ({_ig.get('time_limited', 0)} hit the time limit and are excluded); windows of "
                      f"{', '.join(f'{h} min' for h in _hz)} of passes, k={_ig['budget_k']}, clear sky")
        for _n in ("coverage_first", "time_aware", "fcfs"):
            _lines.append(f"    {_n:16s} utility / optimum: median {_ig[_n + '_ratio_median']:.4f}, "
                          f"worst {_ig[_n + '_ratio_min']:.4f}")
        if _ig.get("ecc_instances_exact"):
            _lines.append(f"    earliest-completion cover: greedy image count equalled the exact minimum in "
                          f"{_ig['ecc_matches_exact']}/{_ig['ecc_instances_exact']} windows "
                          f"(worst case {_ig['ecc_max_extra_images']} extra image(s))")
_lines.append("")
_lines.append(f"{h2c_result['hypothesis']}: {h2c_result['overall_status']}{_ov}")
for _lab, _a in h2c_result["assessments"].items():
    if _a.get("status") == "NOT RUN":
        _lines.append(f"  {_lab}: NOT RUN")
        continue
    _cm = _a.get("cloud_model") or {}
    _lines.append(f"  {_lab}: {_a['status']} (coverage-first: {_a.get('status_coverage_first')}); cloud model "
                  f"{_cm.get('model')}" + (f", L = {_cm.get('length_km'):g} km, T = {_cm.get('time_h'):g} h"
                                           if _cm.get('model') == 'correlated' else "")
                  + f"; seeds {_a.get('seeds')}")
    for _obj in ("time_aware", "coverage_first"):
        _h = _a.get(_obj) or {}
        if not _h:
            continue
        _cv = _h.get("coverage_pct_mean") or {}
        _dx, _dc = _h["difference_vs_openloop_expected"], _h["difference_vs_openloop_clear"]
        _f = lambda v, n=2: "n/a" if v is None else f"{v:.{n}f}"
        _lines.append(f"    {_obj}: mean images {_f(_h.get('images_mean'), 1)}; mean coverage adaptive "
                      f"{_f(_cv.get('adaptive'))}%, open-loop expected {_f(_cv.get('openloop_expected'))}%, open-loop "
                      f"clear-sky {_f(_cv.get('openloop_clear'))}%, oracle {_f(_cv.get('oracle'))}%")
        _lines.append(f"      adaptive minus open-loop (expected): mean {_f(_dx['mean'], 3)} points, one-sided 95% lower "
                      f"bound {_f(_dx['lcb'], 3)}, positive in {_f(_dx['pct_positive'], 0)}% of realisations; minus "
                      f"open-loop (clear-sky): mean {_f(_dc['mean'], 3)}; share of the oracle's gain captured "
                      f"(median) {_f(_h.get('share_of_oracle_gain_captured_median'), 3)}; {_h.get('replans_total')} "
                      f"re-plans, {_h.get('compensatory_total')} compensatory images in all")
        if _h.get("not_evaluable_reasons"):
            _lines.append(f"      not evaluable: {'; '.join(_h['not_evaluable_reasons'])}")
_lines.append(f"  Note: {h2c_result['note']}")
_lines.append("")
_lines.append(f"SUPERSEDED original {h2_result['hypothesis']} (kept for the record): "
              f"{h2_result['overall_status']}{_ov} (requires cloud coverage to maintain > "
              f"{h2_result['maintenance_threshold_pct']:g}% of clear-sky coverage; {h2_result['superseded_note']})")
for _lab, _a in h2_result["assessments"].items():
    _d = _labels.get(_lab, {})
    _lines.append(f"  {_lab}: Clear={_a['coverage_clear_pct']:.2f}%, Cloud={_a['coverage_cloud_pct']:.2f}% "
                  f"(maintained {_a['coverage_maintenance_pct']:.1f}%) [cloud knowledge: {_d.get('cloud_knowledge', 'oracle')}]")
    if _d.get("cloud_knowledge") == "realistic":
        _oc = _d.get("oracle_coverage_cloud_pct")
        _plan_txt = ("passes planned on expected coverage from each tile's cloud probability, each image's "
                     "cloud revealed on arrival, re-planned after every image"
                     if _d.get("cloud_planning") == "expected" else
                     "passes planned on clear-sky geometry, cloud revealed on arrival")
        _lines.append(f"    Realistic: {_plan_txt}; {_d.get('cloud_replans')} Fail-Signals, "
                      f"{_d.get('cloud_compensatory_passes')} compensatory passes. "
                      + ("" if _oc is None else f"Greedy given the cloud in advance: {_oc:.2f}%."))
_lines.append("")
_lines.append(f"{h3_result['hypothesis']}: {h3_result['overall_status']}{_ov} (requires coverage "
              f">{h3_result['coverage_threshold_pct']:g}% AND redundancy <{h3_result['redundancy_threshold']:g} "
              f"together, per the hypothesis as stated)")
for _lab, _a in h3_result["assessments"].items():
    _d = _labels.get(_lab, {})
    _ra = _a.get("redundancy_all_passes")
    if _a.get("status") == "NOT EVALUABLE":
        _lines.append(f"  {_lab}: NOT EVALUABLE -- the passes in the window can reach only "
                      f"{_a['reachable_clear_pct']:.2f}% of ROI tiles, below the {h3_result['coverage_threshold_pct']:g}% "
                      f"coverage at which redundancy is measured (selection covered {_a['coverage_clear_pct']:.2f}%)"
                      + ("" if _ra is None else f"  [redundancy over all selected passes: {_ra:.2f}, budget-driven]"))
        continue
    _red = _a['redundancy_clear']
    _lines.append(f"  {_lab}: {_a.get('status', '')} -- Coverage={_a['coverage_clear_pct']:.2f}%, Redundancy at "
                  f"{h3_result['coverage_threshold_pct']:g}% coverage="
                  + (f"{_red:.2f} " if np.isfinite(_red) else "never reached within the budget ")
                  + f"({_a.get('passes_to_coverage')} pass{'' if _a.get('passes_to_coverage') == 1 else 'es'})"
                  + ("" if _ra is None else f"  [over all selected passes: {_ra:.2f}, budget-driven]"))
    _rc, _fl = _d.get("h3_redundancy_cloud"), _d.get("cloud_redundancy_floor")
    if _rc is not None:
        _lines.append(f"    Cloud (reported, not part of the test): redundancy at {h3_result['coverage_threshold_pct']:g}% "
                      f"coverage={_rc:.2f} ({_d.get('h3_passes_cloud')} pass{'' if _d.get('h3_passes_cloud') == 1 else 'es'})"
                      + ("" if _fl is None else f"; model floor ~{_fl:.2f} at this cloud probability"))
_lines += ["", _sub, "VALIDATION AGAINST REAL ACQUISITIONS (Cell 7D)", _sub, ""]
for _lab, _d in _labels.items():
    _v = _d.get("acquisition_validation") or {}
    if not _v:
        continue
    _lines.append(f"  {_lab}: {_v.get('status')}")
    if _v.get("status") == "ok":
        _f = lambda x, nd=1: "n/a" if x is None else f"{x:.{nd}f}"
        _lines.append(f"    {_v['predicted_overpasses']} predicted and {_v['actual_overpasses']} real overpasses "
                      f"({', '.join(_v['platforms'])}) from {_v['window_start_utc']} to {_v['evaluated_until_utc']} UTC; "
                      f"{_v['matched']} matched within {_v['match_tolerance_min']:g} min.")
        _lines.append(f"    Precision {_f(_v['precision'], 3)}, recall {_f(_v['recall'], 3)}, median time offset "
                      f"{_f(_v['median_time_offset_min'], 2)} min; {_f(_v['predicted_tiles_confirmed_pct'])}% of predicted "
                      f"tiles lie inside the real scenes, which cover {_f(_v['actual_tiles_predicted_pct'])}% predicted.")
        _lines.append(f"    Real scene cloud cover: mean {_f(_v['actual_cloud_cover_mean_pct'])}%, median "
                      f"{_f(_v['actual_cloud_cover_median_pct'])}% (cloud model: {_v['model_cloud_probability_pct']:g}%). "
                      f"{_v['note']}")
        for _plat, _bp in (_v.get("by_platform") or {}).items():          # v1.5.2
            if not _bp.get("evaluated", True):
                _lines.append(f"    {_plat}: not evaluated yet (archive lag {_bp.get('archive_lag_h'):g} h)")
                continue
            _lines.append(f"    {_plat}: {_bp['matched']} of {_bp['predicted_overpasses']} predicted overpasses "
                          f"matched (precision {_f(_bp['precision'], 3)}), {_bp['actual_overpasses']} real "
                          f"(recall {_f(_bp['recall'], 3)}), evaluated to {_bp['evaluated_until_utc']} UTC")
_lines += ["", _sub, "THEORETICAL GUARANTEE", _sub, "",
           textwrap.fill(hypothesis_validation["approximation_guarantee"], width=70), ""]

with open(os.path.join(_out_dir, "arf_final_report.txt"), "w") as f:
    f.write("\n".join(_lines))
logger.info("Results summary and final report saved to arf_results_summary.csv / arf_final_report.txt")

print("\n=== CELL 8 COMPLETE ===")



In [ ]:
# =============================================================================
# CELL 9: REPRODUCIBILITY MANIFEST & RUN INSTRUCTIONS
# =============================================================================
#
# Records everything needed to reproduce this scenario's results:
#   - the execution environment (Python, platform, installed package versions)
#   - the scenario inputs (ROI, N, TLE source and epoch, and SHA-256 checksums
#     of the TLE cache and pinned satellite selection the run actually used)
#   - every output file the pipeline writes, with size and SHA-256 checksum
#   - the scenario_results ledger and the hypothesis validation (Cell 8)
#
# All outputs of Cells 0-9 are written to OUT_DIR (Cell 0): a per-scenario
# Google Drive folder on Colab (ARF_Outputs_<roi>_<fleet tag>[_from<date>]
# [_at<time>][_without-...], plus _v<release> when the folder already holds a
# complete run of another release), or the working directory when run locally.
# This cell validates files there.
#
# This cell does not execute other cells: notebook cells cannot re-run one
# another, so multi-scenario runs follow SCENARIO_RUN_INSTRUCTIONS below.
#
# Outputs (in OUT_DIR):
#   - arf_reproducibility_manifest.json (machine-readable)
#   - arf_pipeline_log.txt (human-readable)
# =============================================================================
#
# SPDX-License-Identifier: MIT
# SPDX-FileCopyrightText: 2026 ARF Contributors
# Author: Tatenda Allen Tobaiwa (BIUST)
# =============================================================================

# Version guard (v1.5.0; every cell from v1.5.1): stop if this cell and Cell 0 come
# from different ARF releases -- a mixed notebook would silently run old code under a new label.
_ARF_CELL_VERSION = "1.5.3"
if globals().get("__version__") != _ARF_CELL_VERSION:
    raise RuntimeError(f"Cell 9 is from ARF {_ARF_CELL_VERSION}, but Cell 0 reports ARF "
                       f"{globals().get('__version__', 'unknown')}. Run every cell from one release.")
globals().setdefault("_ARF_CELLS_RUN", {})["9"] = _ARF_CELL_VERSION

import os
import sys
import json
import logging
import platform
import hashlib
from datetime import datetime, timezone
from importlib import metadata as _importlib_metadata
from typing import Dict, List, Any

logger = logging.getLogger("arf.cell9")
if not logger.handlers:
    handler = logging.StreamHandler(sys.stdout)
    handler.setFormatter(logging.Formatter('%(asctime)s [%(name)s] %(levelname)s: %(message)s', datefmt='%H:%M:%S'))
    logger.addHandler(handler)
    logger.setLevel(logging.INFO)

_out_dir = OUT_DIR if 'OUT_DIR' in globals() else os.getcwd()
_framework_version = globals().get("__version__", "unknown")

# Every file Cells 0-8 write into OUT_DIR, grouped by the cell that writes it.
EXPECTED_OUTPUTS: Dict[str, List[str]] = {
    "Cell 0": ["arf_config.json"],
    "Cell 1": ["arf_cell1_validation.json"],
    "Cell 2": ["arf_cell2_summary.json"],
    "Cell 3": ["arf_cell3_metadata.json", "arf_coverage_map.png", "arf_coverage_map.pdf"],
    "Cell 4": ["arf_cell4_summary.json", "arf_access_events.csv"],
    "Cell 5": ["arf_cell5_summary.json", "arf_refined_events.csv", "arf_downlink_events.csv"],
    "Cell 6": ["arf_cell6_summary.json", "arf_selection_history_clear.csv",
               "arf_selection_history_cloud.csv", "arf_coverage_sweep.csv"],
    "Cell 7": ["arf_cell7_metadata.json",
               "arf_fig1_coverage_progression.png", "arf_fig1_coverage_progression.pdf",
               "arf_fig2_quality_progression.png", "arf_fig2_quality_progression.pdf",
               "arf_fig3_latency_analysis.png", "arf_fig3_latency_analysis.pdf",
               "arf_fig4_coverage_budget.png", "arf_fig4_coverage_budget.pdf",
               "arf_manuscript_figures.png", "arf_manuscript_figures.pdf",
               "arf_fig7_mosaic_map.png", "arf_fig7_mosaic_map.pdf", "arf_mosaic_tiles.csv"],
    "Cell 7B": ["arf_coverage_vs_time.csv"],                       # v1.5.2
    "Cell 7C": ["arf_decay_sweep.csv", "arf_policy_comparison.csv", "arf_cost_latency_frontier.csv",
                "arf_fig6_cost_latency.png", "arf_fig6_cost_latency.pdf"]
        + (["arf_ilp_gap.csv"] if globals().get("ILP_ENABLED", True) else []) + (
        ["arf_onset_robustness.csv", "arf_onset_robustness_summary.json",
         "arf_fig5_onset_robustness.png", "arf_fig5_onset_robustness.pdf",
         "arf_fig9_h1a_gap.png", "arf_fig9_h1a_gap.pdf"]                # fig9: v1.5.2
        if globals().get("ONSET_ROBUSTNESS_ENABLED", True) else [])
        # v1.5.0: cloud realisations (H2) and the cloud-scale sensitivity sweep
        + ["arf_cloud_robustness.csv", "arf_cloud_robustness_summary.json",
           "arf_fig8_cloud_robustness.png", "arf_fig8_cloud_robustness.pdf"]
        + (["arf_cloud_sensitivity.csv"] if (globals().get("CLOUD_SWEEP_LENGTHS_KM") or
                                             globals().get("CLOUD_SWEEP_TIMES_H")) else []),
    "Cell 7D": ["arf_acquisition_validation.csv", "arf_acquisition_validation_summary.json"],
    "Cell 8": ["arf_hypothesis_validation.json", "arf_results_summary.csv", "arf_final_report.txt"],
}

# Installed distributions to record. The orekit-data package is installed from
# git; its distribution name is tried under each common spelling.
_PACKAGE_DISTRIBUTIONS = {
    "numpy": ["numpy"], "pandas": ["pandas"], "matplotlib": ["matplotlib"],
    "h3": ["h3"], "shapely": ["shapely"], "orekit-jpype": ["orekit-jpype"],
    "jdk4py": ["jdk4py"], "orekit-data": ["orekitdata", "orekit-data", "orekit_data"],
    "scipy": ["scipy"], "rasterio": ["rasterio"],          # v1.5.0: ILP, cloud field; climatology reader (optional)
    "pyshp": ["pyshp"],                                    # v1.5.1: WDPA and Natural Earth shapefile reader (Cells 1, 3)
    "JPype1": ["JPype1", "jpype1"],                        # v1.5.2: the Python-Java bridge under orekit-jpype
}


def _java_version():
    """Version of the JVM running Orekit (v1.5.2), or None if no JVM is up."""
    try:
        import jpype
        if not jpype.isJVMStarted():
            return None
        from java.lang import System as _JSystem
        return {"java.version": str(_JSystem.getProperty("java.version")),
                "java.vendor": str(_JSystem.getProperty("java.vendor")),
                "java.vm.name": str(_JSystem.getProperty("java.vm.name"))}
    except Exception:
        return None


def _utc_now() -> str:
    return datetime.now(timezone.utc).isoformat()


def _sha256(path: str) -> str:
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for block in iter(lambda: f.read(1 << 20), b""):
            h.update(block)
    return h.hexdigest()


def _file_record(path: str) -> Dict[str, Any]:
    if os.path.exists(path):
        return {"status": "FOUND", "size_bytes": os.path.getsize(path), "sha256": _sha256(path)}
    return {"status": "MISSING"}


def capture_environment() -> Dict[str, Any]:
    """Python, platform and installed package versions (via importlib.metadata,
    which reports the version actually installed, unlike module __version__)."""
    env_info = {
        "timestamp_utc": _utc_now(),
        "python_version": sys.version,
        "python_executable": sys.executable,
        "platform": {
            "system": platform.system(),
            "release": platform.release(),
            "version": platform.version(),
            "machine": platform.machine(),
            "processor": platform.processor(),
        },
        "packages": {},
        "cwd": os.getcwd(),
        "out_dir": _out_dir,
    }
    for label, candidates in _PACKAGE_DISTRIBUTIONS.items():
        env_info["packages"][label] = "not installed"
        for dist in candidates:
            try:
                env_info["packages"][label] = _importlib_metadata.version(dist)
                break
            except _importlib_metadata.PackageNotFoundError:
                continue
    env_info["java"] = _java_version()                                       # v1.5.2
    env_info["validated_versions"] = globals().get("ARF_VALIDATED_VERSIONS")   # v1.5.2 (Cell 1)
    env_info["differs_from_validated"] = {
        k: {"validated": v, "installed": env_info["packages"].get(k)}
        for k, v in (globals().get("ARF_VALIDATED_VERSIONS") or {}).items()
        if env_info["packages"].get(k) not in (v, None) and k in env_info["packages"]}
    return env_info


def record_inputs() -> Dict[str, Any]:
    """The scenario definition and the exact input files this run used."""
    roi_def = ROI_DEFINITIONS.get(ACTIVE_ROI, {}) if 'ROI_DEFINITIONS' in globals() else {}
    inputs = {
        "active_roi": globals().get("ACTIVE_ROI"),
        "active_n": globals().get("ACTIVE_N"),
        "fleet": globals().get("FLEET", "starlink"),
        "fleet_tag": globals().get("FLEET_TAG"),
        "mission_table": globals().get("MISSION_TABLE") if globals().get("FLEET", "starlink") != "starlink" else None,
        "tle_source": roi_def.get("tle_source", "celestrak_live"),
        "historical_epoch": roi_def.get("historical_epoch"),
        "sim_epoch": str(sim_epoch) if 'sim_epoch' in globals() else None,
        # v1.5.1: where the window starts and why (runs are comparable only with
        # the same reference epoch)
        "reference_epoch": globals().get("_reference_epoch"),
        "reference_epoch_source": globals().get("REFERENCE_EPOCH_SOURCE"),
        "analysis_epoch": globals().get("ANALYSIS_EPOCH"),
        "n_satellites": len(constellation) if 'constellation' in globals() else None,
        "config": config.to_dict() if 'config' in globals() and hasattr(config, "to_dict") else None,
        "files": {},
    }
    for name in ("TLE_RAW_CACHE_PATH", "SELECTION_CACHE_PATH"):
        path = globals().get(name)
        if path:
            inputs["files"][name] = {"path": path, **_file_record(path)}
    # v1.5.1: every orbital-element file the run read (both CelesTrak groups of a
    # combined fleet, the reference-epoch groups, a Space-Track history file)
    for i, path in enumerate(globals().get("GP_INPUT_PATHS") or []):
        inputs["files"][f"GP_INPUT_{i}"] = {"path": path, **_file_record(path)}
    # WDPA boundary files (licence: not redistributed; recorded by checksum only)
    if roi_def.get("wdpa_sites") and "_find_wdpa_file" in globals():
        for sid in roi_def["wdpa_sites"]:
            p = _find_wdpa_file(sid, ARF_BOUNDARY_DIR)
            inputs["files"][f"WDPA_{sid}"] = ({"path": p, **_file_record(p)} if p
                                              else {"path": None, "status": "MISSING"})
    return inputs


def validate_all_outputs(out_dir: str = None) -> Dict[str, Any]:
    """Check every expected output (EXPECTED_OUTPUTS) exists in out_dir, with
    size and SHA-256 checksum."""
    out_dir = out_dir or _out_dir
    results = {"timestamp": _utc_now(), "out_dir": out_dir, "files": {},
               "n_expected": 0, "n_found": 0, "n_missing": 0}
    for cell, names in EXPECTED_OUTPUTS.items():
        for name in names:
            rec = {"cell": cell, **_file_record(os.path.join(out_dir, name))}
            results["files"][name] = rec
            results["n_expected"] += 1
            results["n_found" if rec["status"] == "FOUND" else "n_missing"] += 1
    results["all_present"] = results["n_missing"] == 0
    return results


# Every cell carries a version guard and reports in (v1.5.1; v1.5.0 guarded only
# the cells it changed). A cell that did not report was not run in this session,
# or is a copy from a release without the guard.
ARF_GUARDED_CELLS = ("0", "1", "2", "3", "4", "5", "6", "6B", "7", "7B", "7C", "7D", "8", "9")


def check_cell_versions() -> Dict[str, Any]:
    """Every guarded cell must have run in this session from this release.
    Raises unless ARF_ALLOW_MIXED_VERSIONS = True (the manifest then records
    the mismatch)."""
    run = dict(globals().get("_ARF_CELLS_RUN") or {})
    missing = [c for c in ARF_GUARDED_CELLS if c not in run]
    other = {c: v for c, v in run.items() if v != _framework_version}
    res = {"release": _framework_version, "cells_run": run, "missing": missing, "other_release": other,
           "consistent": not missing and not other}
    if not res["consistent"]:
        msg = (f"Mixed or incomplete notebook: cells not run from ARF {_framework_version}: {missing or 'none'}; "
               f"cells from another release: {other or 'none'}. Re-run Cells 0-9 from one release.")
        if not globals().get("ARF_ALLOW_MIXED_VERSIONS", False):
            raise RuntimeError(msg + " (Set ARF_ALLOW_MIXED_VERSIONS = True to write the manifest anyway; it "
                                     "will record the mismatch.)")
        logger.warning(msg)
    return res


def generate_reproducibility_manifest() -> Dict[str, Any]:
    """Build and save the manifest (JSON) and the pipeline log (TXT) in OUT_DIR."""
    logger.info("Generating reproducibility manifest...")
    cell_versions = check_cell_versions()

    def _json_scenario_key(k):
        return f"{k[0]}__N{k[1]}" if isinstance(k, tuple) and len(k) == 2 else str(k)

    manifest = {
        "manifest_version": "1.1.0",
        "framework": "Algorithmic Reconstruction Framework (ARF)",
        "framework_version": _framework_version,
        "cell_versions": cell_versions,                  # v1.5.0 version guard
        "config_overrides": dict(globals().get("ARF_CONFIG_OVERRIDES_APPLIED") or {}),   # v1.5.0 batch runs
        "manifest_timestamp": _utc_now(),
        "environment": capture_environment(),
        "inputs": record_inputs(),
        "output_validation": validate_all_outputs(),
        "scenario_results": ({_json_scenario_key(k): v for k, v in scenario_results.items()}
                             if 'scenario_results' in globals() else {}),
        "hypothesis_validation": hypothesis_validation if 'hypothesis_validation' in globals() else {},
    }

    manifest_path = os.path.join(_out_dir, "arf_reproducibility_manifest.json")
    try:
        with open(manifest_path, "w") as f:
            json.dump(manifest, f, indent=2, default=str)
        logger.info(f"Manifest saved to {manifest_path}")
    except Exception as e:
        logger.warning(f"Could not save manifest: {e}")

    log_path = os.path.join(_out_dir, "arf_pipeline_log.txt")
    rule = "-" * 70 + "\n"
    try:
        with open(log_path, "w") as f:
            f.write("=" * 70 + "\nALGORITHMIC RECONSTRUCTION FRAMEWORK (ARF)\n"
                    "REPRODUCIBILITY MANIFEST & PIPELINE LOG\n" + "=" * 70 + "\n\n")
            f.write(f"Generated: {manifest['manifest_timestamp']}\n"
                    f"Framework Version: {_framework_version}\n\n")

            f.write(rule + "EXECUTION ENVIRONMENT\n" + rule + "\n")
            env = manifest["environment"]
            f.write(f"Python: {env['python_version']}\n"
                    f"Platform: {env['platform']['system']} {env['platform']['release']}\n"
                    f"Machine: {env['platform']['machine']}\n\nPackage Versions:\n")
            for pkg, version in env["packages"].items():
                f.write(f"  {pkg}: {version}\n")

            f.write("\n" + rule + "SCENARIO INPUTS\n" + rule + "\n")
            inp = manifest["inputs"]
            f.write(f"ROI: {inp['active_roi']}   fleet: {inp['fleet']}   "
                    f"satellites propagated: {inp['n_satellites']}\n"
                    f"Window start (reference epoch): {inp.get('reference_epoch')} UTC "
                    f"[{inp.get('reference_epoch_source')}]\n"
                    f"TLE source: {inp['tle_source']}   historical epoch: {inp['historical_epoch']}\n"
                    f"Simulation epoch: {inp['sim_epoch']}\n")
            for name, rec in inp["files"].items():
                f.write(f"  [{rec['status']}] {name}: {rec['path']}"
                        + (f" ({rec['size_bytes']} bytes, sha256 {rec['sha256'][:16]}...)" if rec["status"] == "FOUND" else "")
                        + "\n")

            f.write("\n" + rule + "OUTPUT FILE VALIDATION\n" + rule + "\n")
            val = manifest["output_validation"]
            f.write(f"Directory: {val['out_dir']}\nExpected: {val['n_expected']}   Found: {val['n_found']}   "
                    f"Missing: {val['n_missing']}   All present: {val['all_present']}\n\n")
            for name, rec in val["files"].items():
                size = rec.get("size_bytes", "N/A")
                f.write(f"  [{rec['status']}] {rec['cell']}: {name} ({size} bytes)\n")

            if manifest["scenario_results"]:
                f.write("\n" + rule + "SCENARIO RESULTS\n" + rule + "\n")
                for label, data in manifest["scenario_results"].items():
                    f.write(f"{label}:\n")
                    for key, value in data.items():
                        if key not in ("sweep_results", "real_k_sweep"):   # detailed sweeps: see CSV/JSON
                            f.write(f"  {key}: {value}\n")
                    f.write("\n")

            if manifest["hypothesis_validation"]:
                f.write(rule + "HYPOTHESIS VALIDATION\n" + rule + "\n")
                for h_key, h_data in manifest["hypothesis_validation"].items():
                    if isinstance(h_data, dict) and "overall_status" in h_data:
                        _sup = "SUPERSEDED (kept for the record): " if h_data.get("superseded") else ""
                        f.write(f"{_sup}{h_data.get('hypothesis', h_key)}: {h_data['overall_status']}\n")
                        for label, a in h_data.get("assessments", {}).items():
                            f.write(f"  {label}: {a}\n")
                        f.write("\n")
        logger.info(f"Pipeline log saved to {log_path}")
    except Exception as e:
        logger.warning(f"Could not save pipeline log: {e}")

    return manifest


# =============================================================================
# RUN INSTRUCTIONS
# =============================================================================

SCENARIO_RUN_INSTRUCTIONS = """
=============================================================================
ARF -- HOW TO RUN SCENARIOS (v1.5.2)
=============================================================================
A scenario is a region (active_roi) and a fleet (fleet, planet_swath_case),
live or for a past analysis window (analysis_epoch). Default: CKGR + Okavango
Delta with the open-data + PlanetScope fleet (conservative Dove swath).

STEP 1: Configure (Cell 0)
   - Set active_roi (e.g. "ckgr_okavango"), fleet ("earth-resources",
     "planet", "earth-resources+planet") and, with PlanetScope,
     planet_swath_case ("conservative" / "optimistic").
   - For a past window set analysis_epoch (ISO UTC, e.g. "2026-08-17T00:00:00");
     its orbital elements come from Space-Track gp_history and need the Colab
     secrets SPACETRACK_USERNAME / SPACETRACK_PASSWORD.
   - Runs you will COMPARE (one region, several fleets) must share their
     window start: the same analysis_epoch, or the same reference_epoch_utc for
     live runs (by default live runs start at the newest element set over both
     CelesTrak groups, which runs fetched in one session share).
   - Fleet sensitivity: exclude_satellites=("SENTINEL-2A",) leaves a tracked
     satellite out (Sentinel-2A flies a reduced extension-campaign plan, so
     its predicted passes are an upper bound); run with and without it.
   - Run Cell 0. Outputs go to the Drive folder
     ARF_Outputs_<roi>_<fleet tag>[_from<YYYYMMDD>][_at<time>][_without-...],
     one per scenario. A folder holding a complete run of another release is
     kept; this release then writes to <folder>_v<release digits> (v1.5.2).

STEP 2: Run Cells 1 -> 2 -> 3 -> 4 -> 5 -> 6 -> 6B -> 7 -> 7B -> 7C -> 7D -> 8 -> 9
   in order, all from ONE release: every cell checks Cell 0's version and
   stops on a mismatch, and Cell 9 refuses a manifest from a mixed or
   incomplete notebook.

STEP 3: Check the results
   - Cell 6:  selections (coverage-first, time-aware on the time-weighted
              coverage objective), cloud model, H1 latency, H3 redundancy
   - Cell 6B: data-driven selection budget (elbow and saturation k)
   - Cell 7:  figures and the mosaic map
   - Cell 7B: time to 25/50/75/90% coverage
   - Cell 7C: decay sweep, baselines, onset robustness (H1a/H1b on the
              earliest-completion cover), exact ILP gaps (both objectives),
              cloud realisations (H2: adaptive vs open-loop with the same
              images) and the cloud-scale sensitivity sweep (Fig. 8)
   - Cell 7D: predicted Sentinel-2 / Landsat passes vs the real archive
   - Cell 8:  hypothesis validation, arf_results_summary.csv, final report
   - Cell 9:  this manifest -- every expected file FOUND, cell versions consistent

STEP 4: Further scenarios and batches
   - Return to Cell 0, change the settings, and repeat. Without restarting
     the kernel, the scenario_results ledger keeps every scenario so far.
   - Many scenarios or windows: arf_batch_runner.py runs each configuration
     through all cells in a fresh namespace, resumably; the confirmatory
     protocol's runs are listed there, and arf_confirmatory_analysis.py
     applies the protocol's decision rules to the finished runs.
   - arf_colab_parity.py re-runs Cells 6-8 of saved runs and compares them
     with the reference results computed outside Colab (environment check).
=============================================================================
"""


# =============================================================================
# EXECUTE
# =============================================================================

manifest = generate_reproducibility_manifest()
validation = manifest["output_validation"]

print("\n" + "=" * 70)
print("  CELL 9: Reproducibility Manifest")
print("=" * 70)
# v1.5.2: real fleets are identified by their fleet tag (N is the Starlink proxy's)
print(f"  Scenario:                  ROI={manifest['inputs']['active_roi']}, "
      + (f"N={manifest['inputs']['active_n']}" if manifest['inputs']['fleet'] == "starlink"
         else f"fleet {manifest['inputs']['fleet_tag']} ({manifest['inputs']['n_satellites']} satellites)"))
print(f"  Output directory:          {validation['out_dir']}")
print(f"  Output files:              {validation['n_found']} / {validation['n_expected']} found"
      f"{'' if validation['all_present'] else '  (see missing below)'}")
for name, rec in validation["files"].items():
    if rec["status"] == "MISSING":
        print(f"    - MISSING ({rec['cell']}): {name}")
for name, rec in manifest["inputs"]["files"].items():
    print(f"  Input {name:18s} {rec['status']}")
print(f"  Manifest:                  arf_reproducibility_manifest.json")
print(f"  Pipeline log:              arf_pipeline_log.txt")
print("=" * 70)
print(SCENARIO_RUN_INSTRUCTIONS)
print(f"\n=== CELL 9 COMPLETE ===")
